# PDF → Knowledge Graph — vLLM (isolated venv) + RAPIDS + W&B + Langfuse

Two-process architecture, one GPU:

- **Process A** (this Colab kernel): RAPIDS (`cudf`/`cugraph`/`cuml`/`rmm`) +
  orchestration — PDF intake, OCR/layout, adaptive chunking, domain
  classification, entity resolution, graph analytics, export.
- **Process B** (isolated Python **venv**, `localhost:<port>`): the vLLM
  inference server, OpenAI-compatible HTTP API. Isolated because vLLM pins
  its own torch/CUDA/numba stack that does not have to match what RAPIDS
  needs pinned in Process A — see the top of `kg_pipeline.py` for the
  full reasoning. **vLLM is never imported in this kernel.**

Observability, matching your split:

| Tool | Covers |
|---|---|
| **Weights & Biases** | experiments, model versions, evaluations, datasets, training/distillation runs, GPU + vLLM inference metrics (vLLM's own `/metrics` is polled and forwarded here — this replaces Prometheus+Grafana) |
| **Langfuse** | LLM traces, prompts, per-chunk generations, human evaluation |
| **vLLM** | native `/metrics` endpoint — source for the W&B forwarding above, and for the diagnostic script |
| **Nsight Systems** | optional, deep CUDA-kernel profiling — not wired into the always-on stack, shown as a one-off command near the bottom |

**Resources this notebook assumes:** 16GB GPU VRAM (T4), 12GB system RAM,
112GB disk. `PipelineConfig` defaults in `kg_pipeline.py` are sized for
that envelope (in particular: `vllm_gpu_memory_utilization=0.45` leaves
headroom for RAPIDS on the same physical GPU, and the default speculative
decoding method is `"ngram"` rather than `"draft_model"` so no second
model needs to be loaded).

Run the cells in order top to bottom the first time. Markdown cells mark
each architecture stage from `index.html`.

---

## v2 (harness update)

One pipeline, two systems. **v1** is the original flow and is still there as the `v1_baseline` preset (it runs the legacy code path, unchanged). **v2** is the design in *harness update-1*:

PDF → document hierarchy → hierarchical summaries → retrieval + global entity memory *before* extraction → two-pass extraction (entities, then quote-first relations) → evidence grounding → ontology as a type system → verifier with adaptive compute (easy / medium / hard) → contradiction detection → provenance on every edge → ontology-guided graph QA → evaluation harness (`kg_eval.py`) → W&B → regression gate.

Pick the system with `kgp.PipelineConfig.preset(...)` — `v1_baseline`, `v2_evidence_only`, `v2_retrieval`, `v2_verification`, `v2_full` (the ablation ladder from section 26 of the design doc) — or use the **Pipeline preset** dropdown in the UI. Every mechanism is also an individual `PipelineConfig.enable_*` flag, so each claim ("retrieval helped", "the verifier helped") is an A/B you can run on a gold set.

New cells for v2 are after "Where to look afterward": inspect a run, evaluate, ablate, calibrate, gate, monitor drift.


In [1]:
!nvidia-smi

Wed Aug 26 06:01:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Baseline check BEFORE any installs -- Colab's GPU runtimes ship with a
# working torch+CUDA already; reinstalling it blindly is exactly what
# caused most of the dependency chaos during this project's early
# attempts. Confirm what's already here first.
import torch
print("torch:", torch.__version__, "| CUDA:", torch.version.cuda, "| available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| capability:", torch.cuda.get_device_capability(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

torch: 2.11.0+cu128 | CUDA: 12.8 | available: True
GPU: Tesla T4 | capability: (7, 5)
VRAM: 14.6 GB


## Process A — RAPIDS install (main kernel, CUDA 12 stack)

One consistent `cu12` stack, installed once, not touched again. Mixing
`cu12`/`cu13` RAPIDS packages in this same environment later is the single
biggest source of breakage encountered while building this project —
don't reinstall these with a different CUDA suffix mid-session.


In [3]:
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 27.1 MB/s eta 0:00:00


In [4]:
!uv pip install --system --no-cache-dir cudf-cu12 cuml-cu12 cugraph-cu12 rmm-cu12 --extra-index-url=https://pypi.nvidia.com

Using Python 3.13.15 environment at: /usr
Resolved 91 packages in 1.48s
Prepared 1 package in 78ms
Installed 1 package in 8ms
 + cugraph-cu12==26.2.0


In [5]:
!uv pip install --system --no-cache-dir \
    rapidfuzz pymupdf python-Levenshtein xxhash tqdm ipywidgets polars networkx scipy \
    sentence-transformers wandb langfuse prometheus_client psutil pydantic nvtx "huggingface_hub[hf_transfer]"
!uv pip install --system --no-cache-dir faiss-gpu-cu12 || pip install --no-cache-dir faiss-cpu

Using Python 3.13.15 environment at: /usr
Resolved 158 packages in 1.46s
Prepared 10 packages in 4.18s
Installed 10 packages in 336ms
 + backoff==2.2.1
 + jedi==0.20.0
 + langfuse==4.14.5
 + levenshtein==0.27.4
 + opentelemetry-exporter-otlp-proto-common==1.42.1
 + opentelemetry-exporter-otlp-proto-http==1.42.1
 + opentelemetry-proto==1.42.1
 + pymupdf==1.28.2
 + python-levenshtein==0.27.4
 + rapidfuzz==3.14.5
Using Python 3.13.15 environment at: /usr
Resolved 5 packages in 555ms
Prepared 1 package in 1.79s
Installed 1 package in 3ms
 + faiss-gpu-cu12==1.14.1.post1


In [6]:
# Same verification pattern that actually worked at the end of this
# project's manual debugging -- confirms the whole RAPIDS + torch stack
# imports cleanly BEFORE spending time setting up the vLLM venv below.
import importlib

for pkg in ["torch", "cudf", "cugraph", "cuml", "cupy", "numba", "sentence_transformers", "wandb", "langfuse", "pydantic"]:
    try:
        m = importlib.import_module(pkg)
        print(f"{pkg:<22}: OK (v{getattr(m, '__version__', '?')})")
    except Exception as e:
        print(f"{pkg:<22}: FAILED -- {e}")

import cudf
print(cudf.DataFrame({"a": [1, 2, 3]}))

torch                 : OK (v2.11.0+cu128)
cudf                  : OK (v26.02.01)
cugraph               : OK (v26.02.00)
cuml                  : OK (v26.02.000)
cupy                  : OK (v14.0.1)
numba                 : OK (v0.61.2)
sentence_transformers : OK (v5.7.0)
wandb                 : OK (v0.28.1)
langfuse              : OK (v4.14.5)
pydantic              : OK (v2.13.4)
   a
0  1
1  2
2  3


In [7]:
from huggingface_hub import login
login()  # paste a token with read access -- needed to pull Qwen2.5-3B-Instruct-AWQ

In [8]:
import wandb
wandb.login()

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: robertford (robertford-university-of-mumbai) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Langfuse keys (optional)

From your Langfuse project → **Settings → API Keys**. Leave all three
blank and re-run to skip Langfuse entirely — `kg_pipeline.py`'s
`LangfuseTracer` degrades to a real no-op automatically when these aren't
set, so extraction still runs fine without it.


In [ ]:
import os, getpass

pub = input("Langfuse public key (pk-lf-..., blank to skip): ").strip()
if pub:
    os.environ["LANGFUSE_PUBLIC_KEY"] = pub
    os.environ["LANGFUSE_SECRET_KEY"] = getpass.getpass("Langfuse secret key (sk-lf-...): ").strip()
    os.environ["LANGFUSE_HOST"] = input("Langfuse host [https://cloud.langfuse.com]: ").strip() or "https://cloud.langfuse.com"
    print("Langfuse env vars set.")
else:
    print("Skipping Langfuse -- tracing will no-op.")

## Process B — isolated vLLM venv

vLLM installs into its **own** virtualenv, completely separate
site-packages from Process A above. This is the actual fix for the
package dispute: vLLM's dependency tree (its own torch build, its own
numba-adjacent pins) never gets resolved in the same environment as
RAPIDS's. `$HOME` (and so `~/.cache/huggingface`) is *not* venv-isolated,
so model weights downloaded from either side are visible to both.


In [11]:
!uv pip install --system virtualenv
!virtualenv /content/vllm_env
!/content/vllm_env/bin/pip install vllm

Using Python 3.13.15 environment at: /usr
Resolved 5 packages in 198ms
Prepared 3 packages in 151ms
Installed 3 packages in 13ms
 + distlib==0.4.3
 + python-discovery==1.5.3
 + virtualenv==21.7.5
created virtual environment CPython3.13.15.final.0-64-x86_64 in 1390ms
  creator CPython3Posix(dest=/content/vllm_env, clear=False, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.cache/virtualenv)
    added seed packages: pip==26.2.1
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator,XonshActivator
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.9/312.9 MB 26.2 MB/s  0:00:10
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 78.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 86.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 72.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 82.2 MB/s  0:00:00
  

In [ ]:
# !python3 -m venv /content/vllm_env
# !/content/vllm_env/bin/pip install --upgrade pip --quiet
# !/content/vllm_env/bin/pip install vllm
# !/content/vllm_env/bin/pip show vllm

In [ ]:
%%writefile kg_pipeline.py

# ============================================================================
# PDF -> Knowledge Graph pipeline, vLLM-backed.
#
# v2: the harness_update-1 design (document hierarchy, retrieval + global entity memory,
# multi-pass extraction, evidence grounding, ontology type system, verifier + adaptive
# compute, contradiction detection, provenance) is layered on top of v1 -- see the
# "v2 HARNESS" banner above StreamableKGPipeline. PipelineConfig.preset("v1_baseline")
# runs the original v1 flow, unchanged.
#
# Maps onto the architecture in index.html as follows -- search for these
# tags in comments below to find the corresponding code:
#
#   [PROCESS A]   main pipeline runtime -- THIS Colab kernel: RAPIDS
#                 (cudf/cugraph/cuml/rmm) + orchestration. Never imports
#                 vllm directly (see [PROCESS B] below for why).
#   [OCR/LAYOUT]  extract_sections_from_pdf
#   [CHUNK]       TextChunker
#   [CLASSIFY]    DomainClassifier / DEFAULT_DOMAIN_LABELS
#   [ONTOLOGY]    ALLOWED_ENTITY_TYPES / ALLOWED_PREDICATES / ONTOLOGY_PROFILES
#   [PROCESS B]   vLLM inference server -- ISOLATED venv, localhost:<port>,
#                 OpenAI-compatible HTTP API. launch_vllm_server /
#                 ensure_vllm_server / VLLMTripleExtractor.
#   [VALIDATE]    parse_json_triples(strict=True) / sitl_filter /
#                 INVALID_TYPE_PREDICATE_COMBOS / VALID_SUBJECT_TYPES_PER_PREDICATE
#   [RESOLVE]     resolve_entities_in_edges / resolve_canonical_types
#   [ANN]         SemanticEntityResolver / CuMLEntityResolver / apply_ann_resolution
#   [GRAPH]       GPUGraphAnalyzer (PageRank, in/out-degree, Leiden communities)
#   [STORE]       GPUDirectStorage / export_to_parquet
#   [OBSERVE]     LangfuseTracer (LLM traces/prompts/generations) and
#                 VLLMMetricsPoller (vLLM's native /metrics -> same W&B run
#                 the pipeline already logs batch/graph metrics to -- this
#                 is the direct swap for "vLLM metrics -> Prometheus ->
#                 Grafana" in the original diagram; W&B also auto-collects
#                 host/GPU system metrics on wandb.init() with zero extra
#                 code, covering the "GPU / experiment metrics" column).
#   [ACTIVE-LEARN] VLLMDPODatasetBuilder / run_dpo_training / DSPy section
#                 near the bottom -- optional, not required for the main
#                 pipeline to run; adapted from the transformers-direct
#                 version to call the vLLM HTTP endpoint instead of a
#                 local model.generate().
#
# [PROCESS B] WHY THE ISOLATED VENV:
# vLLM pins its own torch/CUDA/numba stack, which does not have to match
# what RAPIDS (cudf/cugraph/cuml/rmm) needs pinned in THIS process. Pip
# wheels named nvidia-*-cuXX bundle their own .so files and are loaded via
# dlopen from each package's own site-packages -- they do not touch a
# system-wide CUDA install and do not, by themselves, conflict with a
# different CUDA-flavoured stack in a *different* Python environment.
# numba is the one that actually clashes if both stacks land in the SAME
# interpreter: RAPIDS needs numba pinned to whatever its current release
# requires (0.60.x for the 26.2.* RAPIDS release used below), vLLM's own
# dependency tree wants a different numba, and numba's CUDA target is
# sensitive enough to version skew that "just let pip figure it out" in
# one shared environment reliably breaks one side or the other -- this was
# reproduced firsthand across this project's install attempts. A venv
# fully isolates site-packages (and therefore which numba/torch import),
# while `$HOME` -- and so `~/.cache/huggingface` -- is NOT venv-isolated,
# so a model downloaded from the main kernel is immediately visible to the
# vLLM venv's subprocess and vice versa. That's what the pre-download step
# in the notebook relies on.
#
# Also worth knowing before you tune `vllm_gpu_memory_utilization`: the
# vLLM server subprocess and this main kernel's RAPIDS code share the SAME
# physical GPU (isolation above is at the Python-environment level, not
# the hardware level). Whatever fraction you give vLLM is unavailable to
# cudf/cugraph/cuml -- on a 16GB T4 the default below (0.45, ~7.2GB) is
# sized to leave headroom for RAPIDS' own allocations (cuDF frames,
# cuGraph's graph structures, cuML's ANN index) rather than to maximise
# vLLM's KV cache.
# ============================================================================

import os
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

import torch
from transformers import AutoTokenizer
import cudf
import cugraph
import polars as pl
import re
import gc
import json
import time
import threading
import subprocess
import warnings
import bisect
import math
import hashlib
from collections import Counter, defaultdict
from pathlib import Path
from dataclasses import dataclass, field, asdict, replace
from typing import List, Dict, Tuple, Iterator, Optional, Set, Callable, Literal
from pydantic import BaseModel, Field, ValidationError
import xxhash
import numpy as np
import fitz
import pymupdf
import ipywidgets as widgets
from IPython.display import display, FileLink
from rapidfuzz import fuzz
from rapidfuzz.process import cdist as rfcdist
from rapidfuzz import process as rf_process
import Levenshtein as lev
from tqdm.auto import tqdm
import wandb
warnings.filterwarnings('ignore')

# NOTE: no `import weave` here on purpose. The files this pipeline started
# from imported weave at the top but never called it anywhere (no
# weave.init() / @weave.op() in the original source) -- a dead import. It
# would also duplicate what Langfuse is doing below: your stated split
# puts LLM traces/prompts/generations in Langfuse and experiments/GPU
# metrics/eval/datasets in W&B, so adding W&B's own LLM-tracing layer on
# top would just be two tools doing the same job.

RMM_AVAILABLE = False
try:
    import rmm
    from rmm.allocators.torch import rmm_torch_allocator
    import cupy as cp
    RMM_AVAILABLE = True
except ImportError:
    import cupy as cp

# Split into two independent flags rather than one combined "ANN_AVAILABLE"
# -- the cuML backend (the new default here) needs sentence-transformers
# for embedding but NOT faiss at all, so a box that has cuml +
# sentence-transformers but genuinely skipped installing faiss (a
# reasonable choice: it's a second, CPU-side ANN library this pipeline
# only needs as a fallback) should still get domain classification and
# cuML-backed ANN resolution, not have both silently disabled.
EMBEDDER_AVAILABLE = False
try:
    from sentence_transformers import SentenceTransformer
    EMBEDDER_AVAILABLE = True
except ImportError:
    pass

FAISS_AVAILABLE = False
try:
    import faiss
    FAISS_AVAILABLE = True
except ImportError:
    pass

ANN_AVAILABLE = EMBEDDER_AVAILABLE and FAISS_AVAILABLE  # only the FAISS-specific code path needs this combined form

CUML_AVAILABLE = False
try:
    from cuml.neighbors import NearestNeighbors as _CuMLNearestNeighbors
    CUML_AVAILABLE = True
except ImportError:
    pass

DSPY_AVAILABLE = False
try:
    import dspy
    DSPY_AVAILABLE = True
except ImportError:
    pass

LANGFUSE_AVAILABLE = False
try:
    from langfuse import Langfuse
    LANGFUSE_AVAILABLE = True
except ImportError:
    pass

# nvtx is the package RAPIDS itself uses internally to annotate cudf/cuml/
# cugraph calls, so it's very likely already a transitive dependency of the
# cudf-cu12/cuml-cu12/cugraph-cu12 install above -- flagged as its own
# optional import anyway (not assumed) in case a leaner RAPIDS build ever
# drops it.
NVTX_AVAILABLE = False
try:
    import nvtx
    NVTX_AVAILABLE = True
except ImportError:
    pass


# ============================================================================
# [OBSERVE] NVTX ranges -- CPU/GPU attribution for Nsight Systems, the
# `nsys profile --trace=cuda,nvtx,osrt ...` capture set up in the "Nsight
# Systems profiling" cells near the end of the notebook. A plain
# nvtx.annotate() push/pop costs on the order of ~100ns whether or not a
# profiler is attached, so -- unlike W&B/Langfuse -- this stays on
# unconditionally rather than behind a PipelineConfig flag; it's the same
# tradeoff torch/cudf/cuml already make annotating their OWN internals.
# domain="kg_pipeline" keeps this pipeline's own stage ranges on a
# separate track from cudf/cuml/cugraph's and vLLM/torch's own NVTX
# ranges in the same capture, instead of interleaving into one row.
# nvtx_range() doubles as a decorator (`@nvtx_range("name")`, for a whole
# non-generator function) and a context manager (`with nvtx_range("name"):`,
# for one block inside a larger function) -- use the decorator form on
# GENERATOR functions (anything with `yield`) only with extreme care: it
# would time the generator's near-instant *construction*, not its body, so
# generator methods below (process_pdf, _process_chunks) are deliberately
# left un-decorated and instrumented via the non-generator calls they make
# instead. Falls back to a real no-op if `nvtx` isn't installed, same
# _NullObservation-style fallback pattern as LangfuseTracer below.
# ============================================================================

class _NullNVTXRange:
    def __call__(self, func):
        return func

    def __enter__(self):
        return self

    def __exit__(self, *a):
        return False


def nvtx_range(message: str, color: str = "blue"):
    if not NVTX_AVAILABLE:
        return _NullNVTXRange()
    return nvtx.annotate(message=message, color=color, domain="kg_pipeline")


@dataclass
class PipelineConfig:
    model_name: str = "Qwen/Qwen2.5-3B-Instruct"
    draft_model_name: str = "Qwen/Qwen2.5-0.5B-Instruct"
    max_input_tokens: int = 900
    max_output_tokens: int = 768
    batch_size: int = 8
    overlap_pct: float = 0.20
    entity_resolution_threshold: float = 0.82
    leiden_resolution: float = 1.2
    confidence_filter: float = 0.4
    debug: bool = False

    # StreamableKGPipeline._process_chunks' low-yield rescue path samples
    # a chunk this many times and keeps a triple only if _consensus_from_runs'
    # Beta(2,1)-prior, 95%-lower-confidence-bound vote clears 0.5. n=3 is
    # the minimum where unanimous agreement clears the bar (lower bound
    # 0.549) -- n=2 cannot, even at perfect 2/2 agreement (lower bound
    # 0.473 < 0.5), so this mechanism would silently reject every triple
    # forever at n=2. Verified numerically, not just asserted.
    bayesian_rerun_samples: int = 3

    # --- [PROCESS B] vLLM inference backend ---
    # StreamableKGPipeline extracts through VLLMTripleExtractor, an
    # OpenAI-compatible HTTP client talking to a vLLM server launched (see
    # launch_vllm_server) in an ISOLATED venv/subprocess. See the module
    # docstring above for why. Requires the venv already set up:
    #   python3 -m venv /content/vllm_env && /content/vllm_env/bin/pip install vllm
    vllm_venv_path: str = "/content/vllm_env"
    vllm_num_speculative_tokens: int = 5
    vllm_gpu_memory_utilization: float = 0.55
    vllm_port: int = 8000
    vllm_quantization: str = "awq"

    # "ngram": no draft model at all -- speculates by matching candidate
    #   tokens directly against the prompt. Chosen as the DEFAULT here
    #   (the transformers-direct/vLLM-v2 predecessors of this file
    #   defaulted to "draft_model") for three concrete reasons on THIS
    #   hardware/task specifically: (1) it needs no second model, saving
    #   ~1GB of VRAM and ~1GB of disk/download on a box with 16GB VRAM and
    #   12GB system RAM to share with RAPIDS; (2) extracted entities are
    #   frequently literal substrings of the source chunk, which is
    #   exactly the pattern n-gram speculation is good at; (3) it
    #   sidesteps needing the draft-model vocabulary/loading path
    #   entirely, which is one fewer thing that can fail at server
    #   startup. "draft_model" (using draft_model_name below, same Qwen2.5
    #   family so vocab is shared) is left in as a fully wired option if
    #   you want to A/B it -- just flip this string.
    vllm_speculative_method: str = "ngram"  # "ngram" | "draft_model"

    # --- extraction quality gates [VALIDATE] ---
    strict_field_validation: bool = True
    max_entity_chars: int = 80
    max_entity_words: int = 7
    max_triples_per_chunk: int = 20
    repetition_penalty: float = 1.1

    # --- confidence calibration ---
    # VLLMTripleExtractor computes confidence as exp(avg_logprob * this
    # scale) -- geometric-mean token probability raised to a tunable
    # power -- from vLLM's completions API `logprobs` field. A guess, not
    # a calibrated constant; see fit_confidence_calibration() to derive it
    # from a labeled sample instead.
    logprob_calibration_scale: float = 8.0

    # --- [ANN] semantic entity resolution ---
    enable_ann_resolution: bool = True
    ann_backend: str = "cuml"  # "cuml" | "faiss" -- cuml is the default here
    # to match the "T4 / GPU ANN" node in the architecture diagram: search
    # runs on-GPU in the same CUDA context as cudf/cugraph rather than
    # round-tripping embeddings to host memory the way FAISS-CPU would.
    # apply_ann_resolution auto-falls-back to whichever backend IS
    # importable if the configured one isn't, rather than silently
    # skipping ANN resolution outright.
    ann_embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2"
    ann_index_type: str = "flat"  # "flat" | "hnsw" (faiss backend only)
    ann_duplicate_threshold: float = 0.90
    ann_missing_edge_threshold: float = 0.78
    ann_max_proposed_edges: int = 5000

    # --- [CLASSIFY] domain classification ---
    # Honest scope note: this pipeline still extracts against ONE fixed
    # ontology (ALLOWED_ENTITY_TYPES / ALLOWED_PREDICATES below) -- this
    # classifier does not yet switch the grammar or type vocabulary per
    # domain. What it adds is a per-chunk domain TAG threaded into every
    # Langfuse trace and W&B log line, so you can see which domains a
    # document actually contains before investing in a second ontology
    # profile. ONTOLOGY_PROFILES below is a real (if currently
    # single-entry) extension point for when you want that.
    enable_domain_classification: bool = True

    # --- [OBSERVE] ---
    enable_wandb: bool = True
    enable_langfuse: bool = True
    wandb_project: str = "kg-pipeline"
    vllm_metrics_poll_interval_s: float = 15.0

    # --- v2 harness (harness_update-1) ---------------------------------------
    # Master switch + one flag per mechanism, so every claim in the design doc
    # ("retrieval helped", "verification helped", ...) is an A/B you can run.
    # PipelineConfig.preset(name) flips them in the order of the doc's section 26
    # comparison list; preset("v1_baseline") runs the original v1 flow untouched.
    enable_v2_harness: bool = True
    enable_hierarchical_summaries: bool = True    # (3)  document -> section summaries in the prompt
    enable_retrieval_context: bool = True         # (4)  document/section/neighbor context BEFORE extraction
    enable_entity_memory: bool = True             # (5)  global entity memory: canonical names, aliases, antecedents
    enable_multipass_extraction: bool = True      # (6)  pass 1 entities -> pass 2 relations (vs v1's single pass)
    enable_evidence_check: bool = True            # (7,8) deterministic evidence grounding
    require_evidence: bool = True                 #      claims with no textual support are rejected
    enable_ontology_constraints: bool = True      # (9,10) RELATION_SCHEMA domain/range/disjointness
    ontology_violation_action: str = "review"     #      "review" | "reject" | "keep" (FORBIDDEN combos are always rejected)
    attributed_action: str = "review"             #      "According to X, ..." claims: "review" | "reject" | "accept"
    enable_verifier: bool = True                  # (12) the model as judge on one claim + its evidence
    enable_adaptive_routing: bool = True          # (13) easy -> accept, medium -> verify, hard -> escalate
    enable_self_consistency: bool = True          # (12) low-yield chunks: N samples + Beta vote (v1's Bayesian rescue)
    enable_contradiction_detection: bool = True   # (22)
    enable_cooccurrence_edges: bool = False       # v1's enhance_connectivity() `related_to` edges; off by default
    #                                               because a co-occurrence edge has no textual claim behind it
    version_guard_in_resolution: bool = True      # (22) never merge entities that differ only by a version number

    # v2 prompt / context budgets (bounded and structured -- no chain-of-thought, see design doc 30)
    entity_max_tokens: int = 512
    claim_max_tokens: int = 768
    max_context_tokens: int = 350                 # DOCUMENT/SECTION/NEIGHBOR context per chunk (~4 chars/token)
    max_context_entities: int = 15
    max_context_facts: int = 6
    low_yield_claims: int = 3                     # a chunk with fewer ACCEPTED claims than this gets self-consistency

    # v2 routing. UNCALIBRATED starting points: per-claim confidence is exp(scale * mean logprob of the
    # claim's value tokens). Fit your own mapping on a gold set with kg_eval.fit_calibrator / sweep_thresholds.
    claim_logprob_scale: float = 2.0
    route_easy_conf: float = 0.80                 # >= this AND model-quoted evidence AND asserted -> no verifier call
    route_hard_conf: float = 0.35                 # <  this (or coref / repaired direction / weak evidence) -> escalate
    route_hard_evidence: float = 0.85
    verified_conf_floor: float = 0.70             # a verifier YES lifts confidence to at least this

    @classmethod
    def preset(cls, name: Optional[str], base: Optional["PipelineConfig"] = None) -> "PipelineConfig":
        """Ablation presets, in the order of the design doc's section-26 comparison list:
        baseline -> evidence/provenance -> retrieval -> verification -> ontology constraints.
        `base` supplies every non-ablation setting (batch size, thresholds, W&B project, ...).
        name in (None, "custom") returns a copy of `base` unchanged."""
        base = base if base is not None else cls()
        if name in (None, "custom"):
            return replace(base)
        off = dict(
            enable_v2_harness=False, enable_hierarchical_summaries=False, enable_retrieval_context=False,
            enable_entity_memory=False, enable_multipass_extraction=False, enable_evidence_check=False,
            require_evidence=False, enable_ontology_constraints=False, enable_verifier=False,
            enable_adaptive_routing=False, enable_self_consistency=False, enable_contradiction_detection=False,
            enable_cooccurrence_edges=True, version_guard_in_resolution=False,
        )
        evidence = {**off, "enable_v2_harness": True, "enable_evidence_check": True, "require_evidence": True,
                    "enable_cooccurrence_edges": False, "enable_self_consistency": True}
        retrieval = {**evidence, "enable_hierarchical_summaries": True, "enable_retrieval_context": True,
                     "enable_entity_memory": True, "enable_multipass_extraction": True}
        verification = {**retrieval, "enable_verifier": True, "enable_adaptive_routing": True}
        full = {**verification, "enable_ontology_constraints": True, "enable_contradiction_detection": True,
                "version_guard_in_resolution": True}
        table = {"v1_baseline": off, "v2_evidence_only": evidence, "v2_retrieval": retrieval,
                 "v2_verification": verification, "v2_full": full}
        if name not in table:
            raise ValueError(f"unknown preset {name!r}; choose from {sorted(table)} or 'custom'")
        return replace(base, **table[name])

    @classmethod
    def baseline_v1(cls, base: Optional["PipelineConfig"] = None) -> "PipelineConfig":
        return cls.preset("v1_baseline", base)


V2_ABLATION_FLAGS = [
    "enable_hierarchical_summaries", "enable_retrieval_context", "enable_entity_memory",
    "enable_multipass_extraction", "enable_evidence_check", "require_evidence", "enable_ontology_constraints",
    "ontology_violation_action", "enable_verifier", "enable_adaptive_routing", "enable_self_consistency",
    "enable_contradiction_detection", "enable_cooccurrence_edges", "version_guard_in_resolution",
    "route_easy_conf", "route_hard_conf", "claim_logprob_scale",
]


config = PipelineConfig()

# [ONTOLOGY] Mirrors the TYPES / PREDICATES lists inside
# EXTRACTION_SYSTEM_PROMPT. Keep these two in sync if you ever edit the
# prompt's lists.
ALLOWED_ENTITY_TYPES: Set[str] = {
    "PERSON", "ORGANIZATION", "CONCEPT", "PRODUCT", "LOCATION", "EVENT",
    "LAW", "METRIC", "PROCESS", "DOCUMENT", "TECHNOLOGY", "PROVISION",
}

ALLOWED_PREDICATES: Set[str] = {
    "is_a", "instance_of", "part_of", "contains", "includes",
    "develops", "authors", "issues", "regulates", "manages", "governs",
    "performs", "enables", "causes", "requires", "produces", "applies_to",
    "affiliated_with", "located_in", "related_to", "influences", "depends_on",
}


@nvtx_range("cuda_memory_cleanup", color="orange")
def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


@nvtx_range("pdf_parsing", color="blue")
def extract_sections_from_pdf(pdf_bytes: bytes) -> List[Tuple[str, str]]:
    """[OCR/LAYOUT] stage. This is PyMuPDF's embedded-text extraction, not
    an OCR engine -- for a normal text-layer PDF (reports, filings,
    papers) this is fast and accurate. For a SCANNED/image-only PDF with
    no text layer, per-page text comes back empty and the warning below
    fires; true OCR (e.g. shelling out to Tesseract, or PyMuPDF's own
    get_textpage_ocr) is a real extension point but isn't wired in here to
    keep the dependency footprint down on a 112GB-disk / 12GB-RAM box --
    flagging that explicitly rather than silently handing back a
    near-empty graph."""
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    toc = doc.get_toc()
    sections = []

    if toc:
        top_level = [(lvl, title, pg) for lvl, title, pg in toc if lvl <= 2]
        for i, (lvl, title, pg) in enumerate(top_level):
            end_pg = top_level[i + 1][2] if i + 1 < len(top_level) else len(doc) + 1
            text = "\n".join(
                doc[p].get_text()
                for p in range(max(0, pg - 1), min(end_pg - 1, len(doc)))
            )
            if len(text.strip()) >= 80:
                sections.append((title, text))

    if not sections:
        block = 8
        for start in range(0, len(doc), block):
            text = "\n".join(
                doc[p].get_text()
                for p in range(start, min(start + block, len(doc)))
            )
            if len(text.strip()) >= 80:
                sections.append((f"pages_{start + 1}_{min(start + block, len(doc))}", text))

    doc.close()

    total_chars = sum(len(t) for _, t in sections)
    if sections and total_chars / max(len(sections), 1) < 200:
        print(
            "  WARNING: extracted text looks very sparse "
            f"({total_chars} chars across {len(sections)} section(s)). "
            "This usually means the PDF is scanned/image-only with no "
            "text layer -- PyMuPDF text extraction can't help there. "
            "OCR (not implemented in this pipeline) would be the fix."
        )
    return sections


# ============================================================================
# [VALIDATE] FIELD VALIDATION -- the "Ontology engine" box in the diagram
#
# CHANGED from a pipe-delimited "S:...|SP:...|P:...|T:...|TP:..." format
# constrained by one big regex to JSON-schema-constrained output validated
# with Pydantic. Reason: the pipe-delimited grammar's regex (12 types
# alternated twice per line + 21 predicates once + two {2,80} char-class
# repeats, wrapped in an outer {0,40} repeat) blew past vLLM/xgrammar's
# 5s regex-compile timeout in practice --
#   ValueError: Regex compilation timed out after 5s. The pattern may be
#   too complex or contain constructs that cause exponential state-space
#   explosion (e.g. nested quantifiers).
# -- a confirmed failure, not a guess (reproduced against a live server,
# traceback landed in backend_xgrammar.py's compile_regex_with_timeout).
# JSON-schema-guided decoding is xgrammar's best-trodden, most-optimized
# path (unlike regex support, which lags behind); enums for
# subject_type/predicate/target_type express the SAME ontology constraint
# a bare regex alternation did, and "triples": {"type": "array", "maxItems":
# N} expresses the same "0 to N triples" bound the outer {0,N} repeat did,
# without the nested-quantifier shape that timed out the regex compiler.
# ============================================================================

_CITATION_RE = re.compile(r'\(\d{4}\)')
_MULTI_SENTENCE_RE = re.compile(r'[.!?;][^.!?;]*[.!?;]')


def _valid_entity_text(text: str, max_chars: int, max_words: int) -> bool:
    """Semantic checks JSON schema can't express by itself: schema-level
    maxLength bounds total characters, but not word count, and can't
    reject "looks like a citation" or "looks like more than one sentence"
    -- those still need this pass after Pydantic's shape/enum validation
    succeeds."""
    text = text.strip()
    if not (1 < len(text) <= max_chars):
        return False
    n_words = len(re.split(r'[\s_]+', text.strip('_')))
    if n_words > max_words:
        return False
    if _CITATION_RE.search(text):
        return False
    if _MULTI_SENTENCE_RE.search(text):
        return False
    return True


# Built directly from the SAME sets used everywhere else in this file
# (ALLOWED_ENTITY_TYPES / ALLOWED_PREDICATES) so there's one source of
# truth for the ontology -- Literal[tuple(...)] is equivalent to writing
# Literal["A", "B", ...] by hand; typing unpacks a single tuple argument
# into individual literal members.
_EntityTypeLiteral = Literal[tuple(sorted(ALLOWED_ENTITY_TYPES))]
_PredicateLiteral = Literal[tuple(sorted(ALLOWED_PREDICATES))]


class TripleModel(BaseModel):
    subject: str = Field(min_length=1, max_length=200)
    subject_type: _EntityTypeLiteral
    predicate: _PredicateLiteral
    target: str = Field(min_length=1, max_length=200)
    target_type: _EntityTypeLiteral


class TripleListModel(BaseModel):
    triples: List[TripleModel] = Field(default_factory=list)


def build_triple_json_schema(max_entity_chars: int = 80, max_triples_per_chunk: int = 20) -> dict:
    """Replaces build_triple_grammar_regex(). Hand-built as a plain dict
    (rather than via TripleListModel.model_json_schema()) so max_entity_chars
    / max_triples_per_chunk -- which come from PipelineConfig, i.e. can
    change after this module is imported -- land directly in the schema's
    minLength/maxLength/maxItems instead of being baked into a static
    Pydantic class at import time."""
    entity_type_enum = sorted(ALLOWED_ENTITY_TYPES)
    predicate_enum = sorted(ALLOWED_PREDICATES)
    triple_schema = {
        "type": "object",
        "properties": {
            "subject": {"type": "string", "minLength": 1, "maxLength": max_entity_chars},
            "subject_type": {"type": "string", "enum": entity_type_enum},
            "predicate": {"type": "string", "enum": predicate_enum},
            "target": {"type": "string", "minLength": 1, "maxLength": max_entity_chars},
            "target_type": {"type": "string", "enum": entity_type_enum},
        },
        "required": ["subject", "subject_type", "predicate", "target", "target_type"],
        "additionalProperties": False,
    }
    return {
        "type": "object",
        "properties": {
            "triples": {"type": "array", "maxItems": max_triples_per_chunk, "items": triple_schema},
        },
        "required": ["triples"],
        "additionalProperties": False,
    }


def parse_json_triples(
    text: str,
    strict: bool = True,
    max_entity_chars: int = 80,
    max_entity_words: int = 7,
    stats: Optional[Dict[str, int]] = None,
) -> List[Tuple[str, str, str, str, str, float]]:
    """Parses+validates a `{"triples": [...]}` JSON completion (see
    build_triple_json_schema). Pydantic (via TripleListModel) enforces
    JSON shape + enum membership -- the same job _valid_type_token used to
    do by regex on the pipe-delimited format; strict=True additionally
    applies _valid_entity_text's semantic checks (word count, citation
    pattern, multi-sentence) that schema validation alone can't express."""
    if stats is None:
        stats = {}

    try:
        raw = json.loads(text)
    except json.JSONDecodeError:
        stats['dropped_json_parse_error'] = stats.get('dropped_json_parse_error', 0) + 1
        return []

    try:
        parsed = TripleListModel(**raw) if isinstance(raw, dict) else TripleListModel(triples=raw)
    except ValidationError:
        stats['dropped_schema_invalid'] = stats.get('dropped_schema_invalid', 0) + 1
        return []

    triples = []
    for t in parsed.triples:
        s, sp, p, tgt, tp = t.subject.strip(), t.subject_type, t.predicate, t.target.strip(), t.target_type
        if not (s and tgt):
            stats['dropped_missing_field'] = stats.get('dropped_missing_field', 0) + 1
            continue
        if strict:
            if not (_valid_entity_text(s, max_entity_chars, max_entity_words)
                    and _valid_entity_text(tgt, max_entity_chars, max_entity_words)):
                stats['dropped_entity_shape'] = stats.get('dropped_entity_shape', 0) + 1
                continue
        triples.append((s, sp, p, tgt, tp, 0.0))

    return triples


PRED_CANON: Dict[str, str] = {
    "was_developed_by": "develops", "created_by": "develops", "built_by": "develops",
    "invented_by": "develops", "developed_by": "develops",
    "works_at": "affiliated_with", "employed_by": "affiliated_with",
    "works_for": "affiliated_with", "member_of": "affiliated_with",
    "is_a": "is_a", "is_an": "is_a", "type_of": "is_a",
    "instance_of": "instance_of",
    "is_part_of": "part_of", "belongs_to": "part_of",
    "uses": "performs", "employs": "performs", "utilizes": "performs",
    "regulated_by": "regulates", "managed_by": "manages", "governed_by": "governs",
    "issued_by": "issues", "authored_by": "authors",
}


def normalize_predicate(pred: str) -> str:
    key = pred.lower().strip().replace(" ", "_")
    return PRED_CANON.get(key, key)


INVALID_TYPE_PREDICATE_COMBOS: Set[Tuple[str, str, str]] = {
    ("METRIC", "develops", "PERSON"), ("METRIC", "regulates", "PERSON"),
    ("LOCATION", "regulates", "TECHNOLOGY"), ("METRIC", "affiliated_with", "ORGANIZATION"),
    ("EVENT", "develops", "CONCEPT"), ("METRIC", "performs", "PROCESS"),
    ("DOCUMENT", "causes", "PERSON"), ("PERSON", "part_of", "METRIC"),
    ("METRIC", "located_in", "PERSON"), ("TECHNOLOGY", "issues", "METRIC"),
    ("LAW", "affiliated_with", "METRIC"), ("METRIC", "governs", "ORGANIZATION"),
    ("LOCATION", "produces", "PERSON"), ("METRIC", "authors", "DOCUMENT"),
    ("METRIC", "develops", "TECHNOLOGY"), ("LOCATION", "requires", "PERSON"),
}

VALID_SUBJECT_TYPES_PER_PREDICATE: Dict[str, Set[str]] = {
    "regulates":       {"ORGANIZATION", "LAW", "PROVISION", "DOCUMENT"},
    "develops":        {"ORGANIZATION", "PERSON", "TECHNOLOGY"},
    "issues":          {"ORGANIZATION", "PERSON", "LAW"},
    "affiliated_with": {"PERSON", "ORGANIZATION"},
    "located_in":      {"PERSON", "ORGANIZATION", "EVENT", "LOCATION"},
    "manages":         {"PERSON", "ORGANIZATION"},
    "governs":         {"ORGANIZATION", "LAW", "PROVISION"},
    "authors":         {"PERSON", "ORGANIZATION"},
}


def sitl_filter(triples: List[Tuple], stats: Optional[Dict[str, int]] = None) -> List[Tuple]:
    """[VALIDATE] Domain/range-style constraint checking on top of the
    type enum: disallowed (subject_type, predicate, target_type) combos,
    predicate-specific valid subject types, minimum entity length,
    self-loop rejection. Re-checks SP/TP against the type enum defensively
    (normally already guaranteed upstream by parse_json_triples(strict=True),
    but keeps this function correct on its own if ever called with
    unvalidated input)."""
    if stats is None:
        stats = {}
    valid = []
    for triple in triples:
        s, st, p, t, tt, c = triple
        if st not in ALLOWED_ENTITY_TYPES or tt not in ALLOWED_ENTITY_TYPES:
            stats['dropped_sitl_type_enum'] = stats.get('dropped_sitl_type_enum', 0) + 1
            continue
        if (st, p, tt) in INVALID_TYPE_PREDICATE_COMBOS:
            stats['dropped_sitl_combo'] = stats.get('dropped_sitl_combo', 0) + 1
            continue
        if p in VALID_SUBJECT_TYPES_PER_PREDICATE and st not in VALID_SUBJECT_TYPES_PER_PREDICATE[p]:
            stats['dropped_sitl_subject_type'] = stats.get('dropped_sitl_subject_type', 0) + 1
            continue
        if len(s.strip()) < 2 or len(t.strip()) < 2:
            stats['dropped_sitl_too_short'] = stats.get('dropped_sitl_too_short', 0) + 1
            continue
        if s.lower() == t.lower():
            stats['dropped_sitl_self_loop'] = stats.get('dropped_sitl_self_loop', 0) + 1
            continue
        valid.append(triple)
    return valid


# ============================================================================
# [RESOLVE] CANONICAL ENTITY IDENTITY
#
# Identity is a function of the normalized label ONLY (not label+type --
# type is a noisy per-extraction LLM guess that can vary chunk to chunk
# for the same real-world entity). Type is tracked as a separate, votable
# attribute reconciled by majority vote in resolve_canonical_types(), so
# one entity = one node = one type in the export.
# ============================================================================

_LEADING_ARTICLE_RE = re.compile(r'^(the|a|an)\s+', re.IGNORECASE)
_WHITESPACE_RE = re.compile(r'\s+')


def normalize_entity_label(label: str) -> str:
    """Used ONLY for bucketing/identity -- never for display."""
    x = label.strip().lower()
    x = _WHITESPACE_RE.sub(' ', x)
    x = _LEADING_ARTICLE_RE.sub('', x)
    return x.strip(' .,;:')


def canonical_entity_id(label: str) -> int:
    norm = normalize_entity_label(label)
    return xxhash.xxh64(norm.encode()).intdigest() & 0x7FFFFFFFFFFFFFFF


def compute_entity_similarity(a: str, b: str) -> float:
    al, bl = a.lower().strip(), b.lower().strip()
    set_a, set_b = set(al.split()), set(bl.split())
    union = set_a | set_b
    jaccard = len(set_a & set_b) / len(union) if union else 0.0
    max_len = max(len(al), len(bl))
    levenshtein = 1.0 - (lev.distance(al, bl) / max_len) if max_len > 0 else 0.0
    return (0.4 * jaccard) + (0.6 * levenshtein)


def resolve_canonical_types(edges_pl: pl.DataFrame) -> pl.DataFrame:
    """Reconciles (label -> type) inconsistencies: collapses each
    canonical id down to a single majority-vote type and majority-vote
    surface form, instead of letting the id fragment silently."""
    obs = pl.concat([
        edges_pl.select([
            pl.col("src_id").alias("id"), pl.col("source_type").alias("type"),
            pl.col("source_str").alias("label"),
        ]),
        edges_pl.select([
            pl.col("dst_id").alias("id"), pl.col("target_type").alias("type"),
            pl.col("target_str").alias("label"),
        ]),
    ])

    type_votes = (
        obs.group_by(["id", "type"])
        .agg(pl.col("label").count().alias("n"))
        .sort("n", descending=True)
        .group_by("id", maintain_order=True)
        .agg(pl.col("type").first().alias("resolved_type"))
    )

    label_votes = (
        obs.group_by(["id", "label"])
        .agg(pl.col("type").count().alias("n"))
        .sort("n", descending=True)
        .group_by("id", maintain_order=True)
        .agg(pl.col("label").first().alias("resolved_label"))
    )

    lut = type_votes.join(label_votes, on="id")

    edges_pl = (
        edges_pl
        .join(lut.rename({"id": "src_id", "resolved_type": "src_type_r", "resolved_label": "src_label_r"}),
              on="src_id", how="left")
        .join(lut.rename({"id": "dst_id", "resolved_type": "dst_type_r", "resolved_label": "dst_label_r"}),
              on="dst_id", how="left")
        .with_columns([
            pl.col("src_type_r").fill_null(pl.col("source_type")).alias("source_type"),
            pl.col("dst_type_r").fill_null(pl.col("target_type")).alias("target_type"),
            pl.col("src_label_r").fill_null(pl.col("source_str")).alias("source_str"),
            pl.col("dst_label_r").fill_null(pl.col("target_str")).alias("target_str"),
        ])
        .drop(["src_type_r", "dst_type_r", "src_label_r", "dst_label_r"])
    )
    return edges_pl


@nvtx_range("fuzzy_entity_resolution", color="blue")
def resolve_entities_in_edges(edges_pl: pl.DataFrame, threshold: float = 0.82,
                              guard_versions: bool = False,
                              alias_log: Optional[Dict[str, Dict]] = None) -> pl.DataFrame:
    """Step 1: fuzzy string merge of near-duplicate labels via rapidfuzz.
    Step 2: re-hash on the normalized label ONLY via canonical_entity_id.
    Step 3: majority-vote type + surface form per canonical id via
    resolve_canonical_types. Step 4: collapse duplicate (src, dst,
    predicate) rows now that ids are fully unified."""
    all_labels = edges_pl["source_str"].to_list() + edges_pl["target_str"].to_list()
    unique_labels = sorted(set(all_labels), key=len, reverse=True)

    canon_map: Dict[str, str] = {}
    if len(unique_labels) > 1:
        fast_scores = rfcdist(unique_labels, unique_labels, scorer=fuzz.token_sort_ratio, score_cutoff=70)
        merged: Set[str] = set()
        for i, a in enumerate(unique_labels):
            if a in merged:
                continue
            for j in range(i + 1, len(unique_labels)):
                b = unique_labels[j]
                if b in merged:
                    continue
                al, bl = a.lower(), b.lower()
                # v2: 'CUDA 12' vs 'CUDA 13' (or 'CUDA' vs 'CUDA 12') are different things -- never merge
                # entities whose version numbers differ (design doc 22: don't silently merge).
                if guard_versions and _numeric_signature(a) != _numeric_signature(b):
                    continue
                if bl in al or al in bl:
                    canon_map[b] = a
                    merged.add(b)
                    continue
                if fast_scores[i][j] >= 70:
                    sim = compute_entity_similarity(a, b)
                    if sim >= threshold:
                        canon_map[b] = a
                        merged.add(b)

    if alias_log is not None:
        alias_log["canon_map"] = dict(canon_map)

    if canon_map:
        def resolve(x: str) -> str:
            return canon_map.get(x, x)

        edges_pl = edges_pl.with_columns([
            pl.col("source_str").map_elements(resolve, return_dtype=pl.Utf8),
            pl.col("target_str").map_elements(resolve, return_dtype=pl.Utf8),
        ])

    edges_pl = edges_pl.with_columns([
        pl.col("source_str").map_elements(canonical_entity_id, return_dtype=pl.Int64).alias("src_id"),
        pl.col("target_str").map_elements(canonical_entity_id, return_dtype=pl.Int64).alias("dst_id"),
    ])

    edges_pl = resolve_canonical_types(edges_pl)

    edges_pl = (
        edges_pl.group_by(["src_id", "dst_id", "predicate_str"])
        .agg([
            pl.col("confidence").mean(), pl.col("count").sum(),
            pl.col("source_str").first(), pl.col("target_str").first(),
            pl.col("source_type").first(), pl.col("target_type").first(),
        ])
    )

    print(f"[RESOLVE] {len(canon_map)} label aliases merged, "
          f"type conflicts reconciled by majority vote -> {len(edges_pl)} unique edges")
    return edges_pl


class TextChunker:
    """[CHUNK] One offset-mapped tokenization of the whole section, then a
    linear pass assigning token counts to sentences by character span --
    one tokenizer call per section instead of one per sentence. The
    overlap backtrack loop appends and reverses once instead of
    List.insert(0, ...) in a loop (which is O(N) per call, O(N^2) overall)."""

    def __init__(self, tokenizer, max_tokens: int = 900, overlap_pct: float = 0.20):
        self.tokenizer = tokenizer
        self.max_tokens = max_tokens
        self.overlap_pct = overlap_pct

    @staticmethod
    def _split_with_spans(text: str) -> List[Tuple[int, int]]:
        sep = re.compile(r'(?<=[.!?])\s+')
        spans = []
        pos = 0
        for m in sep.finditer(text):
            spans.append((pos, m.start()))
            pos = m.end()
        spans.append((pos, len(text)))
        return [s for s in spans if s[1] > s[0]]

    def _token_counts_per_sentence(self, text: str, spans: List[Tuple[int, int]]) -> List[int]:
        try:
            enc = self.tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
            offsets = enc["offset_mapping"]
        except Exception:
            return [int(len(text[s:e].split()) * 1.3) for s, e in spans]

        counts = []
        tok_idx = 0
        n_tokens = len(offsets)
        for s_start, s_end in spans:
            count = 0
            while tok_idx < n_tokens and offsets[tok_idx][0] < s_end:
                if offsets[tok_idx][1] <= s_start:
                    tok_idx += 1
                    continue
                count += 1
                tok_idx += 1
            counts.append(count)
        return counts

    @nvtx_range("chunking", color="blue")
    def chunk_with_spans(self, text: str) -> List[Tuple[str, int, int]]:
        """Same chunking as v1's chunk_text, but each chunk also reports the char span
        [start, end) it covers in `text` -- the v2 document map needs it to trace a chunk
        (and any evidence found inside it) back to a PDF page."""
        spans = self._split_with_spans(text)
        if not spans:
            return [(text, 0, len(text))]
        sentences = [text[s:e] for s, e in spans]
        token_counts = self._token_counts_per_sentence(text, spans)

        chunks: List[Tuple[str, int, int]] = []
        current: List[str] = []
        current_counts: List[int] = []
        current_spans: List[Tuple[int, int]] = []
        length = 0

        for sp, sent, n in zip(spans, sentences, token_counts):
            if length + n > self.max_tokens:
                if current:
                    chunks.append((" ".join(current), current_spans[0][0], current_spans[-1][1]))
                overlap_budget = int(self.max_tokens * self.overlap_pct)
                carry: List[str] = []
                carry_counts: List[int] = []
                carry_spans: List[Tuple[int, int]] = []
                carry_tokens = 0
                for s, st, ss in zip(reversed(current), reversed(current_counts), reversed(current_spans)):
                    if carry_tokens + st > overlap_budget:
                        break
                    carry.append(s)
                    carry_counts.append(st)
                    carry_spans.append(ss)
                    carry_tokens += st
                carry.reverse()
                carry_counts.reverse()
                carry_spans.reverse()
                current = carry + [sent]
                current_counts = carry_counts + [n]
                current_spans = carry_spans + [sp]
                length = carry_tokens + n
            else:
                current.append(sent)
                current_counts.append(n)
                current_spans.append(sp)
                length += n

        if current:
            chunks.append((" ".join(current), current_spans[0][0], current_spans[-1][1]))
        return chunks if chunks else [(text, 0, len(text))]

    def chunk_text(self, text: str) -> List[str]:
        return [c for c, _a, _b in self.chunk_with_spans(text)]


# ============================================================================
# [CLASSIFY] DOMAIN CLASSIFICATION -- see PipelineConfig.enable_domain_classification
# above for the honest scope note (per-chunk labeling now, per-domain
# ontology switching later). Reuses whatever SentenceTransformer instance
# StreamableKGPipeline already loaded for [ANN] entity resolution -- pass
# it in rather than constructing a second copy, so this costs no extra
# VRAM/RAM on top of ANN resolution's existing ~80MB embedding model.
# ============================================================================

DEFAULT_DOMAIN_LABELS: Dict[str, str] = {
    "financial_regulatory": "banking, credit, monetary policy, financial regulation, compliance filings",
    "legal": "statutes, contracts, litigation, legal provisions and clauses",
    "technical_engineering": "software architecture, hardware systems, engineering specifications",
    "academic_research": "research methodology, literature review, experimental results, citations",
    "general_business": "corporate strategy, operations, general business correspondence",
}


class DomainClassifier:
    def __init__(self, embedder: "SentenceTransformer", labels: Dict[str, str]):
        self.embedder = embedder
        self.label_names = list(labels.keys())
        self._label_emb = embedder.encode(
            list(labels.values()), normalize_embeddings=True, convert_to_numpy=True,
        ).astype("float32")

    @nvtx_range("domain_classification", color="green")
    def classify_batch(self, texts: List[str], max_chars: int = 2000) -> List[Tuple[str, float]]:
        """Batched, not one call per chunk -- StreamableKGPipeline calls
        this once per extraction batch (typically 8 chunks) rather than
        once per chunk, since sentence-transformers' own internal
        batching is far cheaper than N separate Python-level calls."""
        if not texts:
            return []
        truncated = [(t[:max_chars] if t.strip() else " ") for t in texts]
        embs = self.embedder.encode(
            truncated, batch_size=64, normalize_embeddings=True,
            convert_to_numpy=True, show_progress_bar=False,
        ).astype("float32")
        sims = embs @ self._label_emb.T
        idxs = sims.argmax(axis=1)
        return [(self.label_names[i], float(sims[j, i])) for j, i in enumerate(idxs)]


# [ONTOLOGY] Extension point: ONE profile today. Add a second entry
# (e.g. "financial_regulatory": {"entity_types": {...}, "predicates":
# {...}, "system_prompt": "..."}) and thread DomainClassifier's per-chunk
# label through StreamableKGPipeline._process_chunks to pick a profile per
# chunk instead of always using "general", once you actually want
# per-domain grammars rather than just per-domain labeling.
ONTOLOGY_PROFILES: Dict[str, Dict[str, object]] = {
    "general": {
        "entity_types": ALLOWED_ENTITY_TYPES,
        "predicates": ALLOWED_PREDICATES,
    },
}


EXTRACTION_SYSTEM_PROMPT = """You extract relationships from text into a knowledge graph.

EXTRACT ONLY:
- Relationships explicitly stated in the text
- Direct inferences (if text says "X regulates Y", then "Y is regulated by X" is valid)

DO NOT:
- Hallucinate entities not in the text
- Use predicates not in the allowed list
- Extract generic/meaningless phrases
- Infer relationships requiring external knowledge

DIRECTION RULE (critical):
- Subject = the ACTOR/SOURCE of the relationship
- Target = the RECEIVER/DESTINATION of the relationship
- "RBI regulates banks" -> subject=RBI, predicate=regulates, target=banks
- "Banks are regulated by RBI" -> subject=RBI, predicate=regulates, target=banks (normalize direction to active actor)

ENTITY LENGTH RULE (critical):
- Entities are short noun phrases, not clauses or sentences (2-6 words max)
- If a chunk describes a broad topic or theme, do NOT extract the whole
  phrase as one entity. Extract the core concept instead and relate it
  to other short entities.

OUTPUT FORMAT: a single JSON object with one key, "triples", holding an
array of objects. Each object has exactly these five fields:
subject, subject_type, predicate, target, target_type.

TYPES: PERSON, ORGANIZATION, CONCEPT, PRODUCT, LOCATION, EVENT, LAW, METRIC, PROCESS, DOCUMENT, TECHNOLOGY, PROVISION

PREDICATES (use ONLY these exact active forms):
TYPE: is_a, instance_of, part_of, contains, includes
AGENCY: develops, authors, issues, regulates, manages, governs
ACTION: performs, enables, causes, requires, produces, applies_to
RELATION: affiliated_with, located_in, related_to, influences, depends_on

EXAMPLES:

TEXT: "The RBI regulates export credit through commercial banks."
{"triples": [
  {"subject": "RBI", "subject_type": "ORGANIZATION", "predicate": "regulates", "target": "export credit", "target_type": "CONCEPT"},
  {"subject": "commercial bank", "subject_type": "ORGANIZATION", "predicate": "performs", "target": "export credit", "target_type": "CONCEPT"}
]}

TEXT: "Section 7 of the Companies Act requires annual audits."
{"triples": [
  {"subject": "Section 7", "subject_type": "PROVISION", "predicate": "part_of", "target": "Companies Act", "target_type": "LAW"},
  {"subject": "Section 7", "subject_type": "PROVISION", "predicate": "requires", "target": "annual audit", "target_type": "PROCESS"}
]}

TEXT: "Interest rates increased by 50 basis points, affecting loan costs."
{"triples": [
  {"subject": "interest rate", "subject_type": "METRIC", "predicate": "influences", "target": "loan cost", "target_type": "METRIC"},
  {"subject": "50 basis points", "subject_type": "METRIC", "predicate": "causes", "target": "interest rate", "target_type": "METRIC"}
]}

NEGATIVE EXAMPLES -- DO NOT EXTRACT THESE:
TEXT: "This is important for various reasons." -> {"triples": []} (generic)
TEXT: "The report discusses several factors." -> {"triples": []} (vague)
TEXT: "As mentioned earlier, the policy changes." -> {"triples": []} (no relationship)
TEXT: "There are many options available." -> {"triples": []} (no entities)

TEXT: "The study explores the nature, range, and awareness/utilisation of
banking facilities across a diverse sample of importers and exporters."
WRONG subject: "nature range awareness utilisation of banking facilities across diverse sample importers and exporters" (a whole clause, not an entity)
RIGHT:
{"triples": [
  {"subject": "study", "subject_type": "DOCUMENT", "predicate": "related_to", "target": "banking facility awareness", "target_type": "CONCEPT"},
  {"subject": "importer", "subject_type": "CONCEPT", "predicate": "related_to", "target": "banking facility", "target_type": "CONCEPT"},
  {"subject": "exporter", "subject_type": "CONCEPT", "predicate": "related_to", "target": "banking facility", "target_type": "CONCEPT"}
]}

ENTITY RULES:
- Use exact names from text (keep acronyms if text uses them)
- Compound names: "export credit guarantee" not split
- If text says "RBI", use "RBI" not "Reserve Bank of India"

If unsure whether a relationship exists, DO NOT extract it. If there are
no valid triples, output {"triples": []}.

Extract from:"""


def enhance_connectivity(chunk: str, triples: List) -> List:
    """Standalone (no `self`) so VLLMTripleExtractor can call it without a
    model instance attached -- adds `related_to` edges between entities
    that co-occur in the source chunk but weren't otherwise connected."""
    if len(triples) < 2:
        return triples
    enhanced = list(triples)
    entities: Dict[str, str] = {}
    for s, st, _, t, tt, _ in triples:
        entities[s] = st
        entities[t] = tt
    cl = chunk.lower()
    present = {e for e in entities if e.lower() in cl}
    existing = frozenset(frozenset((s, t)) for s, _, _, t, _, _ in triples)
    el = list(present)[:15]
    for i in range(len(el)):
        for j in range(i + 1, len(el)):
            e1, e2 = el[i], el[j]
            if frozenset((e1, e2)) not in existing:
                enhanced.append((e1, entities[e1], "related_to", e2, entities[e2], 0.65))
    return enhanced


def _consensus_from_runs(
    all_runs: List[List[List]], chunks: List[str], enhance_fn: Callable[[str, List], List],
) -> List[List]:
    """Beta-Binomial agreement consensus across N independent extraction
    runs of the same chunks -- the voting logic behind
    VLLMTripleExtractor._bayesian_rerun's low-yield rescue path. Uses
    scipy.stats.beta.ppf for the lower confidence bound (torch's Beta
    distribution does not implement icdf -- it's declared on the base
    Distribution class and Beta doesn't override it, so calling it always
    raises NotImplementedError; verified numerically that ppf(0.05)
    round-trips through cdf back to 0.05 before relying on it here). Same
    Beta(2, 1) prior, same "keep the triple only if we're 95% confident
    its true agreement rate is >= 0.5" rule."""
    from scipy.stats import beta as beta_dist

    prior_alpha, prior_beta = 2.0, 1.0
    final = []
    for chunk_idx in range(len(chunks)):
        counts: Dict[Tuple, int] = {}
        triple_map: Dict[Tuple, Tuple] = {}
        for run in all_runs:
            for triple in run[chunk_idx]:
                key = (triple[0].lower(), triple[2].lower(), triple[3].lower())
                counts[key] = counts.get(key, 0) + 1
                if key not in triple_map:
                    triple_map[key] = triple

        n = len(all_runs)
        consensus = []
        for key, k in counts.items():
            post_alpha = prior_alpha + k
            post_beta = prior_beta + (n - k)
            posterior_mean = float(post_alpha / (post_alpha + post_beta))
            lower_ci = float(beta_dist.ppf(0.05, post_alpha, post_beta))
            if lower_ci >= 0.5:
                t = triple_map[key]
                consensus.append((*t[:5], posterior_mean))

        seen_keys: Set[Tuple] = set()
        deduped = []
        for t in consensus:
            k = (t[0], t[2], t[3])
            if k not in seen_keys:
                seen_keys.add(k)
                deduped.append(t)

        final.append(enhance_fn(chunks[chunk_idx], deduped) if deduped else [])
    return final


def expected_calibration_error(confidences: List[float], correct: List[bool], n_bins: int = 10) -> float:
    """Standard ECE: bins predictions by confidence, compares each bin's
    average confidence to its actual accuracy, weights by bin size. Lower
    is better; 0 is perfectly calibrated."""
    conf = np.asarray(confidences, dtype=np.float64)
    corr = np.asarray(correct, dtype=np.float64)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    n = len(conf)
    if n == 0:
        return 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (conf > lo) & (conf <= hi) if lo > 0 else (conf >= lo) & (conf <= hi)
        if mask.sum() == 0:
            continue
        bin_conf = conf[mask].mean()
        bin_acc = corr[mask].mean()
        ece += (mask.sum() / n) * abs(bin_conf - bin_acc)
    return float(ece)


def fit_confidence_calibration(raw_scores: List[float], correct: List[bool]) -> Callable[[float], float]:
    """Platt scaling: fits a 1-D logistic regression mapping a raw score
    to a calibrated P(correct). Needs a labeled sample -- spot-check
    ~100+ extracted triples, then:
        clf = fit_confidence_calibration(raw_scores, correct_labels)
        calibrated_conf = clf(new_raw_score)
    """
    from sklearn.linear_model import LogisticRegression
    X = np.asarray(raw_scores, dtype=np.float64).reshape(-1, 1)
    y = np.asarray(correct, dtype=np.int64)
    if len(set(y.tolist())) < 2:
        raise ValueError("Need both correct and incorrect examples to fit a calibration curve.")
    clf = LogisticRegression()
    clf.fit(X, y)

    def calibrated(raw_score: float) -> float:
        return float(clf.predict_proba([[raw_score]])[0, 1])

    return calibrated


class GPUTripleStore:
    """Identity is label-only (canonical_entity_id), matching [RESOLVE]
    above. No custom CUDA hash kernel here -- a plain xxhash64 Python loop
    costs microseconds at the batch sizes this pipeline actually sees
    (tens of entities per call); a hand-rolled GPU hash kernel would only
    earn its complexity at a much larger scale (millions of entities)."""

    def __init__(self):
        self.triples_df = None
        self._pending: List[cudf.DataFrame] = []

    @staticmethod
    def hash_entity(entity: str, etype: str = None) -> int:
        return canonical_entity_id(entity)

    @staticmethod
    def hash_batch(entities: List[str], etypes: List[str] = None) -> "cp.ndarray":
        return cp.array([canonical_entity_id(e) for e in entities], dtype=cp.int64)

    @nvtx_range("triple_store_add", color="green")
    def add_triples(self, triples: List[Tuple]):
        if not triples:
            return
        s = [t[0] for t in triples]
        st = [t[1] for t in triples]
        p = [t[2] for t in triples]
        tg = [t[3] for t in triples]
        tt = [t[4] for t in triples]
        c = [t[5] for t in triples]
        self._pending.append(cudf.DataFrame({
            "src_id": cudf.Series(self.hash_batch(s, st)),
            "rel_id": cudf.Series(self.hash_batch(p)),
            "dst_id": cudf.Series(self.hash_batch(tg, tt)),
            "confidence": cudf.Series(c, dtype="float32"),
            "source_str": s, "source_type": st, "predicate_str": p,
            "target_str": tg, "target_type": tt,
        }))

    @nvtx_range("triple_store_flush", color="green")
    def flush(self):
        if self._pending:
            all_dfs = ([self.triples_df] if self.triples_df is not None else []) + self._pending
            self.triples_df = cudf.concat(all_dfs, ignore_index=True)
            self._pending = []

    @nvtx_range("triple_store_aggregate", color="green")
    def aggregate(self) -> cudf.DataFrame:
        if self.triples_df is None or len(self.triples_df) == 0:
            return cudf.DataFrame()
        agg = self.triples_df.groupby(["src_id", "rel_id", "dst_id"]).agg({
            "confidence": "mean", "source_str": "first", "source_type": "first",
            "predicate_str": "first", "target_str": "first", "target_type": "first",
        }).reset_index()
        counts = self.triples_df.groupby(["src_id", "rel_id", "dst_id"]).size().reset_index()
        counts.columns = ["src_id", "rel_id", "dst_id", "count"]
        return agg.merge(counts, on=["src_id", "rel_id", "dst_id"])


class GPUGraphAnalyzer:
    """[GRAPH] PageRank, in/out-degree, Leiden communities -- matches the
    "In / out-degree · PageRank" node in the architecture diagram
    (Leiden is a bonus beyond what's pictured there, kept since it was
    already load-bearing for the community_id column in the export)."""

    @nvtx_range("graph_construction", color="green")
    def __init__(self, edges_cudf: cudf.DataFrame, leiden_resolution: float = 1.0):
        self.edges_cudf = edges_cudf
        self.leiden_resolution = leiden_resolution

        self.g_dir = cugraph.Graph(directed=True)
        self.g_dir.from_cudf_edgelist(
            edges_cudf, source="src_id", destination="dst_id",
            edge_attr="confidence", store_transposed=True,
        )

        self.g_undir = cugraph.Graph(directed=False)
        self.g_undir.from_cudf_edgelist(
            edges_cudf, source="src_id", destination="dst_id", edge_attr="confidence",
        )

    def compute_metrics(self) -> Dict:
        m: Dict = {}

        with nvtx_range("pagerank", color="green"):
            try:
                m["pagerank"] = cugraph.pagerank(self.g_dir)
            except Exception as e:
                print(f"PageRank failed: {e}")
                u = cudf.concat([self.edges_cudf["src_id"], self.edges_cudf["dst_id"]]).unique()
                m["pagerank"] = cudf.DataFrame({"vertex": u, "pagerank": [0.001] * len(u)})

        with nvtx_range("degree", color="green"):
            try:
                m["in_degree"] = self.g_dir.in_degree()
                m["out_degree"] = self.g_dir.out_degree()
            except Exception as e:
                print(f"Degree failed: {e}")

        with nvtx_range("leiden", color="green"):
            try:
                leiden_df, modularity = cugraph.leiden(self.g_undir, max_iter=100, resolution=self.leiden_resolution)
                m["leiden"] = leiden_df
                n_comm = int(leiden_df["partition"].n_unique())
                print(f"[GRAPH] Leiden: {n_comm} communities  (modularity={modularity:.4f})")
            except Exception as e:
                print(f"Leiden failed: {e}")

        return m


class GPUDirectStorage:
    def __init__(self, checkpoint_dir: str = "./checkpoints"):
        self.path = Path(checkpoint_dir)
        self.path.mkdir(exist_ok=True)
        self.counter = 0

    @nvtx_range("checkpoint_save", color="orange")
    def save_checkpoint(self, df: cudf.DataFrame, name: str = None) -> Path:
        if name is None:
            name = f"checkpoint_{self.counter:04d}.parquet"
            self.counter += 1
        fp = self.path / name
        try:
            df.to_parquet(str(fp), compression="snappy")
        except Exception as e:
            print(f"Checkpoint failed: {e}")
        return fp


# ============================================================================
# [ANN] Semantic entity resolution beyond fuzzy string matching --
# duplicate merges via embedding similarity, `semantically_similar_to`
# edge proposals between related-but-unconnected entities. Both resolvers
# below accept a pre-built `embedder` so StreamableKGPipeline can share
# the SAME SentenceTransformer instance it already loaded for [CLASSIFY],
# instead of loading a second ~80MB copy.
# ============================================================================

class SemanticEntityResolver:
    def __init__(self, model_name: str, index_type: str = "flat",
                 device: Optional[str] = None, embedder: Optional["SentenceTransformer"] = None):
        # faiss is required unconditionally here (used for indexing/search
        # regardless of whether an embedder was shared) -- checked first
        # so the error names the actual missing piece instead of a
        # confusing later NameError from an unbound `faiss` name.
        if not FAISS_AVAILABLE:
            raise RuntimeError(
                "faiss is required for SemanticEntityResolver (the FAISS ANN backend).\n"
                "  pip install faiss-cpu (or faiss-gpu-cu12)\n"
                "or set PipelineConfig.ann_backend = \"cuml\" to avoid needing faiss at all."
            )
        if embedder is not None:
            self.embedder = embedder
        else:
            if not EMBEDDER_AVAILABLE:
                raise RuntimeError(
                    "sentence-transformers is required for the embedding step.\n"
                    "  pip install sentence-transformers\n"
                    "or set PipelineConfig.enable_ann_resolution = False"
                )
            self.embedder = SentenceTransformer(
                model_name, device=device or ("cuda" if torch.cuda.is_available() else "cpu")
            )
        self.index_type = index_type
        self.index = None
        self.ids: List[int] = []
        self.labels: List[str] = []
        self._embeddings: Optional[np.ndarray] = None

    def build_index(self, id_label_pairs: List[Tuple[int, str]]):
        self.ids = [i for i, _ in id_label_pairs]
        self.labels = [l for _, l in id_label_pairs]
        if not self.labels:
            return
        emb = self.embedder.encode(
            self.labels, batch_size=256, show_progress_bar=False,
            normalize_embeddings=True, convert_to_numpy=True,
        ).astype("float32")
        dim = emb.shape[1]
        if self.index_type == "hnsw":
            index = faiss.IndexHNSWFlat(dim, 32)
            index.hnsw.efConstruction = 64
            index.hnsw.efSearch = 64
        else:
            index = faiss.IndexFlatIP(dim)
        index.add(emb)
        self.index = index
        self._embeddings = emb

    def find_duplicates(self, similarity_threshold: float = 0.90, k: int = 5) -> Dict[int, int]:
        if self.index is None or len(self.ids) < 2:
            return {}
        k = min(k + 1, len(self.ids))
        sims, neighbors = self.index.search(self._embeddings, k)
        order = sorted(range(len(self.ids)), key=lambda i: -len(self.labels[i]))
        rank = {idx: r for r, idx in enumerate(order)}
        canon_map: Dict[int, int] = {}
        merged: Set[int] = set()
        for i in range(len(self.ids)):
            if self.ids[i] in merged:
                continue
            for r in range(1, k):
                j = int(neighbors[i, r])
                if j < 0 or j == i or float(sims[i, r]) < similarity_threshold:
                    continue
                if self.ids[j] in merged:
                    continue
                a, b = (i, j) if rank[i] < rank[j] else (j, i)
                canon_map[self.ids[b]] = self.ids[a]
                merged.add(self.ids[b])
        return canon_map

    def find_missing_edges(
        self, existing_pairs: Set[Tuple[int, int]],
        similarity_threshold: float = 0.78, k: int = 8, max_new_edges: int = 5000,
    ) -> List[Tuple[int, int, float]]:
        if self.index is None or len(self.ids) < 2:
            return []
        k = min(k + 1, len(self.ids))
        sims, neighbors = self.index.search(self._embeddings, k)
        proposals = []
        seen_pairs: Set[Tuple[int, int]] = set()
        for i in range(len(self.ids)):
            for r in range(1, k):
                j = int(neighbors[i, r])
                if j < 0 or j == i:
                    continue
                score = float(sims[i, r])
                if score < similarity_threshold:
                    continue
                a, b = self.ids[i], self.ids[j]
                pair = (min(a, b), max(a, b))
                if pair in existing_pairs or pair in seen_pairs:
                    continue
                seen_pairs.add(pair)
                proposals.append((a, b, score))
        proposals.sort(key=lambda x: -x[2])
        return proposals[:max_new_edges]

    def neighbors(self, query: str, k: int = 10) -> List[Tuple[str, float]]:
        if self.index is None:
            return []
        q = self.embedder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
        sims, idxs = self.index.search(q, min(k, len(self.ids)))
        return [(self.labels[idx], float(sims[0, r])) for r, idx in enumerate(idxs[0]) if idx >= 0]


class CuMLEntityResolver:
    """Same interface as SemanticEntityResolver, backed by
    cuml.neighbors.NearestNeighbors (cuVS-accelerated ANN) instead of
    FAISS-CPU -- default backend here since the pipeline is already
    RAPIDS-native, sharing the same CUDA context/memory pool instead of
    adding a second CPU-side ANN library, with search running on-GPU
    rather than round-tripping embeddings to host memory."""

    def __init__(self, model_name: str, device: Optional[str] = None,
                 embedder: Optional["SentenceTransformer"] = None):
        if embedder is not None:
            self.embedder = embedder
        else:
            # Only needs sentence-transformers here -- unlike
            # SemanticEntityResolver, this class never touches faiss, so
            # gating on EMBEDDER_AVAILABLE (not the faiss-inclusive
            # ANN_AVAILABLE) is what actually reflects this class's real
            # dependency.
            if not EMBEDDER_AVAILABLE:
                raise RuntimeError(
                    "sentence-transformers is required for the embedding step.\n"
                    "  pip install sentence-transformers"
                )
            self.embedder = SentenceTransformer(
                model_name, device=device or ("cuda" if torch.cuda.is_available() else "cpu")
            )
        if not CUML_AVAILABLE:
            raise RuntimeError(
                "cuml is required for CuMLEntityResolver.\n"
                "  pip install cuml-cu12 (match your RAPIDS version)\n"
                "or set PipelineConfig.ann_backend = \"faiss\" to use SemanticEntityResolver instead."
            )
        self.ids: List[int] = []
        self.labels: List[str] = []
        self._embeddings: Optional[np.ndarray] = None
        self._nn = None

    def build_index(self, id_label_pairs: List[Tuple[int, str]]):
        self.ids = [i for i, _ in id_label_pairs]
        self.labels = [l for _, l in id_label_pairs]
        if not self.labels:
            return
        emb = self.embedder.encode(
            self.labels, batch_size=256, show_progress_bar=False,
            normalize_embeddings=True, convert_to_numpy=True,
        ).astype("float32")
        self._embeddings = emb
        self._nn = _CuMLNearestNeighbors(n_neighbors=min(16, len(emb)), metric="euclidean")
        self._nn.fit(emb)

    def _search(self, query_emb: np.ndarray, k: int):
        k = min(k, len(self.ids))
        distances, indices = self._nn.kneighbors(query_emb, n_neighbors=k)
        distances = np.asarray(distances)
        indices = np.asarray(indices)
        sims = 1.0 - (distances ** 2) / 2.0
        return sims, indices

    def find_duplicates(self, similarity_threshold: float = 0.90, k: int = 5) -> Dict[int, int]:
        if self._nn is None or len(self.ids) < 2:
            return {}
        sims, neighbors = self._search(self._embeddings, k + 1)
        order = sorted(range(len(self.ids)), key=lambda i: -len(self.labels[i]))
        rank = {idx: r for r, idx in enumerate(order)}
        canon_map: Dict[int, int] = {}
        merged: Set[int] = set()
        for i in range(len(self.ids)):
            if self.ids[i] in merged:
                continue
            for r in range(1, sims.shape[1]):
                j = int(neighbors[i, r])
                if j < 0 or j == i or float(sims[i, r]) < similarity_threshold:
                    continue
                if self.ids[j] in merged:
                    continue
                a, b = (i, j) if rank[i] < rank[j] else (j, i)
                canon_map[self.ids[b]] = self.ids[a]
                merged.add(self.ids[b])
        return canon_map

    def find_missing_edges(
        self, existing_pairs: Set[Tuple[int, int]],
        similarity_threshold: float = 0.78, k: int = 8, max_new_edges: int = 5000,
    ) -> List[Tuple[int, int, float]]:
        if self._nn is None or len(self.ids) < 2:
            return []
        sims, neighbors = self._search(self._embeddings, k + 1)
        proposals = []
        seen_pairs: Set[Tuple[int, int]] = set()
        for i in range(len(self.ids)):
            for r in range(1, sims.shape[1]):
                j = int(neighbors[i, r])
                if j < 0 or j == i:
                    continue
                score = float(sims[i, r])
                if score < similarity_threshold:
                    continue
                a, b = self.ids[i], self.ids[j]
                pair = (min(a, b), max(a, b))
                if pair in existing_pairs or pair in seen_pairs:
                    continue
                seen_pairs.add(pair)
                proposals.append((a, b, score))
        proposals.sort(key=lambda x: -x[2])
        return proposals[:max_new_edges]

    def neighbors(self, query: str, k: int = 10) -> List[Tuple[str, float]]:
        if self._nn is None:
            return []
        q = self.embedder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
        sims, idxs = self._search(q, k)
        return [(self.labels[idx], float(sims[0, r])) for r, idx in enumerate(idxs[0]) if idx >= 0]


@nvtx_range("ann_entity_resolution", color="green")
def apply_ann_resolution(edges_pl: pl.DataFrame, cfg: PipelineConfig,
                          shared_embedder: Optional["SentenceTransformer"] = None,
                          alias_log: Optional[Dict[str, Dict]] = None,
                          guard_versions: bool = False) -> pl.DataFrame:
    if not cfg.enable_ann_resolution:
        return edges_pl

    def node_view(df):
        return pl.concat([
            df.select([pl.col("src_id").alias("id"), pl.col("source_str").alias("label"),
                       pl.col("source_type").alias("type")]),
            df.select([pl.col("dst_id").alias("id"), pl.col("target_str").alias("label"),
                       pl.col("target_type").alias("type")]),
        ]).unique("id")

    backend = cfg.ann_backend
    have_embedder_source = shared_embedder is not None or EMBEDDER_AVAILABLE

    def _backend_usable(b: str) -> bool:
        # cuML needs sentence-transformers (for embedding) + cuml itself
        # (for search) -- it never touches faiss. FAISS needs
        # sentence-transformers + faiss itself, never touches cuml.
        # Checking each backend's REAL dependency set (rather than the
        # combined ANN_AVAILABLE flag) is what lets a box with cuml +
        # sentence-transformers but no faiss installed still get ANN
        # resolution via cuML instead of being skipped outright.
        if b == "cuml":
            return CUML_AVAILABLE and have_embedder_source
        return FAISS_AVAILABLE and have_embedder_source

    if not _backend_usable(backend):
        other = "faiss" if backend == "cuml" else "cuml"
        if _backend_usable(other):
            print(f"[ANN] ann_backend='{backend}' isn't usable in this environment -- falling back to '{other}'.")
            backend = other
        else:
            print("[ANN] skipped: no usable ANN backend in this environment "
                  "(need sentence-transformers, plus either cuml or faiss).")
            return edges_pl

    if backend == "cuml":
        resolver = CuMLEntityResolver(cfg.ann_embedding_model, embedder=shared_embedder)
    else:
        resolver = SemanticEntityResolver(cfg.ann_embedding_model, index_type=cfg.ann_index_type, embedder=shared_embedder)

    nodes = node_view(edges_pl)
    resolver.build_index(list(zip(nodes["id"].to_list(), nodes["label"].to_list())))

    dup_map = resolver.find_duplicates(similarity_threshold=cfg.ann_duplicate_threshold)
    if dup_map and guard_versions:
        _lbl = dict(zip(nodes["id"].to_list(), nodes["label"].to_list()))
        dup_map = {k: v for k, v in dup_map.items()
                   if _numeric_signature(_lbl.get(k, "")) == _numeric_signature(_lbl.get(v, ""))}
    if alias_log is not None:
        alias_log["dup_map"] = {int(k): int(v) for k, v in (dup_map or {}).items()}
    if dup_map:
        edges_pl = edges_pl.with_columns([
            pl.col("src_id").map_elements(lambda x: dup_map.get(x, x), return_dtype=pl.Int64),
            pl.col("dst_id").map_elements(lambda x: dup_map.get(x, x), return_dtype=pl.Int64),
        ])
        edges_pl = resolve_canonical_types(edges_pl)
        edges_pl = (
            edges_pl.group_by(["src_id", "dst_id", "predicate_str"])
            .agg([
                pl.col("confidence").mean(), pl.col("count").sum(),
                pl.col("source_str").first(), pl.col("target_str").first(),
                pl.col("source_type").first(), pl.col("target_type").first(),
                pl.col("edge_source").first(),
            ])
        )
        print(f"[ANN] {len(dup_map)} semantic duplicates merged (backend={backend})")
        nodes = node_view(edges_pl)

    existing_pairs = set(zip(edges_pl["src_id"].to_list(), edges_pl["dst_id"].to_list()))
    proposals = resolver.find_missing_edges(
        existing_pairs, similarity_threshold=cfg.ann_missing_edge_threshold, max_new_edges=cfg.ann_max_proposed_edges,
    )
    if proposals:
        label_lu = dict(zip(nodes["id"].to_list(), nodes["label"].to_list()))
        type_lu = dict(zip(nodes["id"].to_list(), nodes["type"].to_list()))
        new_edges_pl = pl.DataFrame({
            "src_id": [a for a, b, s in proposals], "dst_id": [b for a, b, s in proposals],
            "predicate_str": ["semantically_similar_to"] * len(proposals),
            "confidence": [s for a, b, s in proposals], "count": [1] * len(proposals),
            "source_str": [label_lu.get(a, "") for a, b, s in proposals],
            "target_str": [label_lu.get(b, "") for a, b, s in proposals],
            "source_type": [type_lu.get(a, "CONCEPT") for a, b, s in proposals],
            "target_type": [type_lu.get(b, "CONCEPT") for a, b, s in proposals],
            "edge_source": ["semantic_similarity"] * len(proposals),
        })
        edges_pl = pl.concat([edges_pl, new_edges_pl], how="vertical_relaxed")
        print(f"[ANN] {len(proposals)} candidate semantic edges proposed (predicate='semantically_similar_to')")

    return edges_pl


# ============================================================================
# [OBSERVE] Langfuse tracing -- the "LLM traces / prompts / generations /
# human evaluation" column in your split (W&B below covers
# experiments/GPU metrics/eval/datasets/training instead).
#
# LangfuseTracer degrades to a real no-op if LANGFUSE_PUBLIC_KEY /
# LANGFUSE_SECRET_KEY aren't set, or if the client can't be constructed --
# tracing is instrumentation, not a correctness dependency, so a
# misconfigured or unreachable Langfuse must never be able to break
# extraction. Uses the OTel-based v3 Python SDK
# (start_as_current_observation / nested start_as_current_generation) --
# checked against Langfuse's current docs while writing this, since the
# pre-3.x trace()/generation()/span() API some older tutorials show no
# longer matches. `as_type="generation"` observations nested under a
# per-batch span is effectively "OpenTelemetry -> Langfuse" in the
# original diagram, since v3's SDK IS the OTel span processor pointed at
# Langfuse's OTLP endpoint.
# ============================================================================

class _NullObservation:
    """No-op stand-in with the subset of the Langfuse observation
    interface this pipeline calls, so call sites never need an `if
    tracer.enabled` branch -- there's always SOMETHING to call
    .update()/.start_as_current_generation() on."""
    def __enter__(self):
        return self

    def __exit__(self, *a):
        return False

    def update(self, **kwargs):
        pass

    def start_as_current_generation(self, **kwargs):
        return self


class LangfuseTracer:
    def __init__(self, enabled: bool = True):
        self.client = None
        self.enabled = False
        if not (enabled and LANGFUSE_AVAILABLE):
            return
        if not (os.environ.get("LANGFUSE_PUBLIC_KEY") and os.environ.get("LANGFUSE_SECRET_KEY")):
            print("[OBSERVE] Langfuse tracing disabled: LANGFUSE_PUBLIC_KEY/SECRET_KEY not set.")
            return
        try:
            self.client = Langfuse()
            self.enabled = True
            print(f"[OBSERVE] Langfuse tracing enabled -> {os.environ.get('LANGFUSE_HOST', 'https://cloud.langfuse.com')}")
        except Exception as e:
            print(f"[OBSERVE] Langfuse tracing disabled: client init failed ({e})")

    def batch_span(self, name: str, **kwargs):
        if not self.enabled:
            return _NullObservation()
        return self.client.start_as_current_observation(as_type="span", name=name, **kwargs)

    def flush(self):
        if self.enabled:
            try:
                self.client.flush()
            except Exception as e:
                print(f"[OBSERVE] Langfuse flush failed: {e}")


# ============================================================================
# [OBSERVE] vLLM's own /metrics (Prometheus exposition format) -> the SAME
# W&B run this pipeline already logs batch/graph metrics to. This is the
# direct swap for "vLLM metrics -> Prometheus -> Grafana" in the original
# diagram: same source, different sink, so there's one run to look at
# instead of a second dashboard stack to stand up. Uses
# prometheus_client's own text parser rather than hand-rolled regex --
# Prometheus exposition format has real edge cases (multi-line HELP/TYPE
# comments, quoted label values, histogram _bucket/_sum/_count suffixes)
# that a naive regex eventually gets wrong on some vLLM version's exact
# output; prometheus_client is a tiny, pure-Python package (already a
# transitive dependency inside the vLLM venv, since vLLM uses it to
# EXPOSE /metrics -- installed separately here, in the MAIN env, purely
# to PARSE the text vLLM returns over HTTP).
# ============================================================================

def _parse_vllm_metrics(metrics_text: str) -> Dict[str, float]:
    from prometheus_client.parser import text_string_to_metric_families
    out: Dict[str, float] = {}
    for family in text_string_to_metric_families(metrics_text):
        for sample in family.samples:
            key = sample.name
            if sample.labels:
                lbl = ",".join(f'{k}="{v}"' for k, v in sorted(sample.labels.items()))
                key = f"{key}{{{lbl}}}"
            out[key] = sample.value
    return out


class VLLMMetricsPoller:
    """Background thread. Only forwards a curated subset of series (not
    every one vLLM exposes) -- logging every histogram bucket individually
    would flood the W&B run with near-duplicate keys."""

    _WANTED_PREFIXES = (
        "vllm:num_requests_running", "vllm:num_requests_waiting", "vllm:num_requests_swapped",
        "vllm:gpu_cache_usage_perc", "vllm:gpu_prefix_cache_hit_rate",
        "vllm:prompt_tokens_total", "vllm:generation_tokens_total",
        "vllm:time_to_first_token_seconds_sum", "vllm:time_to_first_token_seconds_count",
        "vllm:time_per_output_token_seconds_sum", "vllm:time_per_output_token_seconds_count",
        "vllm:request_success_total", "vllm:request_prompt_tokens_sum", "vllm:request_generation_tokens_sum",
    )

    def __init__(self, base_url: str, interval_s: float = 15.0):
        self.base_url = base_url.rstrip("/")
        self.interval_s = interval_s
        self._stop = threading.Event()
        self._thread: Optional[threading.Thread] = None

    def _poll_once(self):
        import requests
        try:
            resp = requests.get(f"{self.base_url}/metrics", timeout=5)
            resp.raise_for_status()
        except Exception:
            return
        try:
            parsed = _parse_vllm_metrics(resp.text)
        except Exception as e:
            print(f"[OBSERVE] vLLM /metrics parse failed: {e}")
            return
        log_dict = {f"vllm/{k}": v for k, v in parsed.items() if any(k.startswith(p) for p in self._WANTED_PREFIXES)}
        if log_dict and wandb.run is not None:
            wandb.log(log_dict)

    def _loop(self):
        while not self._stop.is_set():
            self._poll_once()
            self._stop.wait(self.interval_s)

    def start(self):
        if self._thread is not None:
            return
        self._stop.clear()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        print(f"[OBSERVE] Polling vLLM /metrics every {self.interval_s}s -> W&B")

    def stop(self):
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=5)
        self._thread = None


# ============================================================================
# [PROCESS B] Grammar-constrained decoding: see build_triple_json_schema()
# and TripleListModel near the top of this file (moved there since the
# schema is needed by [VALIDATE]'s parse_json_triples too). The `{"regex":
# ...}` approach that used to live here timed out xgrammar's regex
# compiler -- see the comment above [VALIDATE] for the confirmed
# traceback and why JSON schema replaced it.
# ============================================================================


# ============================================================================
# [PROCESS B] vLLM server launcher -- hardened
#
# Changes vs. the predecessor this was built from, all aimed at the exact
# failure mode hit while developing this project (a launch_vllm_server
# call that just prints "...still waiting" with no detail, gets
# interrupted out of impatience, with no way to tell "still downloading",
# "still compiling", or "actually crashed" apart):
#
#   1. _kill_stale_server_on_port: a re-run of the launch cell after an
#      earlier server was left running (kernel restarted without calling
#      shutdown_vllm_server(), or a cell re-run reset this module's
#      _vllm_state dict while the OS process kept running) leaves the
#      port bound. The new subprocess.Popen would then either silently
#      talk to a STALE server, or fail to bind and exit immediately --
#      which looks IDENTICAL to "didn't come up in time" in the poll
#      loop. Find and kill anything already on the port before launching.
#   2. The poll loop now tails the log file's last line into its own
#      progress print every ~30s, so "still waiting" comes with an actual
#      line from vLLM's own startup log (weight loading %, CUDA graph
#      capture progress, etc.) instead of being a black box.
#   3. poll() is checked every iteration, not just periodically -- a
#      process that exits on its own (config error, OOM, etc.) is reported
#      immediately as a real failure with the log path, not as a timeout
#      45 minutes later.
#   4. ensure_vllm_server() below wraps this in a singleton accessor so a
#      Colab "Process PDF" button click reuses an already-warm server
#      instead of re-paying the full startup cost (weight load +
#      torch.compile + CUDA graph capture, real minutes on a T4) on every
#      single document -- the predecessor this was built from explicitly
#      flagged this as a known, unbuilt limitation.
# ============================================================================

def _venv_vllm_bin(venv_path: str) -> str:
    return str(Path(venv_path) / "bin" / "vllm")


def _port_is_free(port: int) -> bool:
    import socket
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(1.0)
        return s.connect_ex(("127.0.0.1", port)) != 0


def _kill_stale_server_on_port(port: int):
    try:
        import psutil
    except ImportError:
        return  # best-effort only -- not worth a hard dependency for this
    killed = []
    try:
        conns = psutil.net_connections(kind="inet")
    except Exception:
        return
    for conn in conns:
        if conn.laddr and getattr(conn.laddr, "port", None) == port and conn.pid:
            try:
                psutil.Process(conn.pid).terminate()
                killed.append(conn.pid)
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                pass
    if killed:
        print(f"[PROCESS B] Killed {len(killed)} stale process(es) already bound to port {port}: {killed}")
        time.sleep(2)


def _tail_line(path: str, n_bytes: int = 600) -> str:
    try:
        with open(path, "rb") as f:
            f.seek(0, 2)
            size = f.tell()
            f.seek(max(0, size - n_bytes))
            data = f.read().decode("utf-8", errors="replace")
        lines = [l for l in data.splitlines() if l.strip()]
        return lines[-1] if lines else ""
    except FileNotFoundError:
        return ""


def tail_log(path: str = "/content/vllm_server.log", n_lines: int = 40):
    """Interactive helper -- call this from its own cell while a launch is
    still in progress to see the last N lines of vLLM's startup log
    directly, instead of waiting for the next poll-loop print."""
    try:
        with open(path, "r", errors="replace") as f:
            lines = f.readlines()
        print("".join(lines[-n_lines:]))
    except FileNotFoundError:
        print(f"(no log file at {path} yet)")


_UNSET = object()  # sentinel distinguishing "argument not passed, use cfg's
# value" from "explicitly passed as None" -- matters specifically for
# `quantization` below, where None is itself a meaningful value ("no
# --quantization flag at all, serve full precision") distinct from
# "I didn't specify one."


def launch_vllm_server(
    cfg: PipelineConfig,
    target_model: Optional[str] = None,
    draft_model: Optional[str] = None,
    speculative_method: Optional[str] = None,
    num_speculative_tokens: Optional[int] = None,
    quantization: Optional[str] = _UNSET,
    venv_path: Optional[str] = None,
    port: Optional[int] = None,
    gpu_memory_utilization: Optional[float] = None,
    max_model_len: int = 4096,
    max_num_seqs: Optional[int] = None,
    enforce_eager: bool = False,
    dtype: str = "float16",
    log_path: str = "/content/vllm_server.log",
    timeout_s: int = 900,
    poll_interval_s: float = 5.0,
    nsys_output: Optional[str] = None,
    nsys_duration_s: Optional[int] = None,
):
    """Launches vLLM as a background OpenAI-compatible HTTP server from an
    ISOLATED venv -- see the module docstring at the top of this file for
    why. Assumes the venv already exists with vllm installed:
        python3 -m venv /content/vllm_env
        /content/vllm_env/bin/pip install vllm

    MODEL SELECTION: target_model defaults to f"{cfg.model_name}-AWQ",
    e.g. "Qwen/Qwen2.5-3B-Instruct" -> "Qwen/Qwen2.5-3B-Instruct-AWQ" --
    NOT cfg.model_name unchanged. Qwen/Qwen2.5-3B-Instruct-AWQ is an
    official Qwen-published checkpoint.

    QUANTIZATION KERNEL ON THIS HARDWARE, worth knowing explicitly rather
    than assuming: a T4 is Turing (SM 7.5). Per vLLM's own supported-
    hardware table, AWQ IS supported on Turing, but the Marlin-fused
    kernel is NOT (Marlin needs Ampere/SM 8.0+) -- vLLM's awq_marlin
    linear method auto-selects the best AVAILABLE kernel for the current
    platform via choose_mp_linear_kernel, which on a T4 means it silently
    falls back to the plain (non-Marlin) AWQ kernel. So `--quantization
    awq` below is correctly supported and will work on a T4, it just
    won't get the Marlin throughput boost an A100/H100 sees -- worth
    knowing so a T4 benchmark that looks slower than someone else's AWQ
    number isn't mistaken for a misconfiguration.

    SPECULATIVE DECODING: uses vLLM's current `--speculative-config` JSON
    syntax (the older separate `--speculative-model` /
    `--num-speculative-tokens` flags are deprecated).
      "ngram" (PipelineConfig default here): no draft model, no separate
        weights -- speculates by matching candidate tokens directly
        against the prompt. Well-suited to this task specifically since
        extracted entities are frequently literal substrings of the
        source chunk; also the resource-lighter choice on a
        16GB-VRAM/12GB-RAM box since it needs no second model loaded.
      "draft_model": {"method": "draft_model", "model": draft_model,
        "num_speculative_tokens": ...} -- draft and target must share
        vocabulary (same Qwen2.5 family here, so that's satisfied).

    STARTUP TIME: default vLLM startup does torch.compile followed by CUDA
    graph capture across its default batch-size list (which climbs into
    the hundreds) -- on a T4 that legitimately takes several minutes,
    longer on a cold torch.compile cache. Two independent levers here:
      1. timeout_s defaults to 900s (15 min), generous enough for a real
         first-run compile; the poll loop tails the log every ~30s so you
         can see which stage it's actually at rather than guessing.
      2. max_num_seqs, if passed (else defaults to cfg.batch_size), caps
         how many CUDA graph sizes vLLM captures at startup -- capturing
         up to the library default when this pipeline only ever sends
         batches of cfg.batch_size is pure wasted startup time.
    Pass enforce_eager=True to skip torch.compile/CUDA graph capture
    entirely for a startup that completes in well under a minute, trading
    away some runtime throughput -- good for a first correctness smoke
    test before switching it back to False for a real run. Also the right
    choice when nsys_output is set: CUDA-graph-captured regions replay in
    the profile as one opaque launch, so eager mode is what gives you
    individual, attributable kernels in the Nsight Systems timeline.

    Guided-decoding backend selection (xgrammar/guidance/etc.) is a
    server-side `vllm serve` startup flag as of current vLLM, not a
    per-request field -- deliberately not pinning one here since the
    exact current flag name is the kind of thing that drifts across
    releases and getting a STARTUP flag wrong fails the whole server, not
    just structured output; vLLM's own "auto" default picks a backend per
    request and is fine to leave alone.

    NSIGHT SYSTEMS: pass nsys_output (e.g. "/content/vllm_profile") to
    launch this same server under `nsys profile` instead of running it
    directly -- reuses every flag above (model, quantization, speculative
    config, gpu-memory-utilization) exactly as constructed below, so the
    profiled process is never at risk of drifting from what actually
    runs. --sample=none --cpuctxsw=none matches this environment's `nsys
    status -e` result (see the Nsight Systems section near the end of the
    notebook): Colab's container restricts perf_event_open, so CPU
    instruction sampling / context-switch tracing aren't available here
    regardless -- cuda,nvtx,osrt trace still gives full GPU kernel +
    NVTX + OS-runtime visibility without needing that permission.
    nsys_duration_s caps the capture (nsys stops tracing and finalizes
    the report on its own after this many seconds) as a safety net
    against an orphaned background process outliving its Colab session,
    independent of however the process actually gets stopped.
    """
    import requests as _requests

    target_model = target_model or f"{cfg.model_name}-AWQ"
    draft_model = draft_model or cfg.draft_model_name
    speculative_method = speculative_method or cfg.vllm_speculative_method
    num_speculative_tokens = num_speculative_tokens or cfg.vllm_num_speculative_tokens
    quantization = cfg.vllm_quantization if quantization is _UNSET else quantization
    venv_path = venv_path or cfg.vllm_venv_path
    port = port if port is not None else cfg.vllm_port
    gpu_memory_utilization = gpu_memory_utilization if gpu_memory_utilization is not None else cfg.vllm_gpu_memory_utilization
    max_num_seqs = max_num_seqs or max(cfg.batch_size, 8)

    vllm_bin = _venv_vllm_bin(venv_path)
    if not Path(vllm_bin).exists():
        raise FileNotFoundError(
            f"{vllm_bin} not found -- set up the isolated venv first:\n"
            f"  python3 -m venv {venv_path} && {venv_path}/bin/pip install vllm"
        )

    _kill_stale_server_on_port(port)
    if not _port_is_free(port):
        raise RuntimeError(f"Port {port} is still occupied after attempting cleanup -- check `lsof -i:{port}` manually.")

    cmd = [
        vllm_bin, "serve", target_model,
        "--port", str(port),
        "--dtype", dtype,
        "--gpu-memory-utilization", str(gpu_memory_utilization),
        "--max-model-len", str(max_model_len),
        "--max-num-seqs", str(max_num_seqs),
    ]
    if quantization:
        cmd += ["--quantization", quantization]
    if enforce_eager:
        cmd += ["--enforce-eager"]
    if speculative_method == "ngram":
        spec_config = {"method": "ngram", "num_speculative_tokens": num_speculative_tokens, "prompt_lookup_max": 4}
        cmd += ["--speculative-config", json.dumps(spec_config)]
    elif speculative_method == "draft_model" and draft_model:
        spec_config = {"method": "draft_model", "model": draft_model, "num_speculative_tokens": num_speculative_tokens}
        cmd += ["--speculative-config", json.dumps(spec_config)]

    if nsys_output:
        nsys_cmd = [
            "nsys", "profile",
            "--force-overwrite=true",
            "-o", nsys_output,
            "--trace=cuda,nvtx,osrt",
            "--sample=none",
            "--cpuctxsw=none",
        ]
        if nsys_duration_s:
            nsys_cmd += ["--duration", str(nsys_duration_s)]
        cmd = nsys_cmd + cmd

    print(f"[PROCESS B] Launching: {' '.join(cmd)}")
    log = open(log_path, "w")
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)

    base_url = f"http://localhost:{port}"
    n_iters = max(1, int(timeout_s // poll_interval_s))
    for i in range(n_iters):
        if proc.poll() is not None:
            raise RuntimeError(
                f"vLLM server process exited on its own (code {proc.returncode}) "
                f"before becoming healthy -- this is a real failure, not a timeout. "
                f"Last log line: {_tail_line(log_path)!r}. Full log: {log_path}"
            )
        try:
            if _requests.get(f"{base_url}/health", timeout=2).status_code == 200:
                elapsed = i * poll_interval_s
                print(f"[PROCESS B] vLLM server up at {base_url} (target={target_model}, "
                      f"spec={speculative_method}) after ~{elapsed:.0f}s")
                return proc, base_url
        except _requests.exceptions.ConnectionError:
            pass
        if i > 0 and i % max(1, int(30 // poll_interval_s)) == 0:
            print(f"...still starting ({i * poll_interval_s:.0f}s elapsed, timeout at {timeout_s}s) "
                  f"| log: {_tail_line(log_path)[:160]}")
        time.sleep(poll_interval_s)

    proc.terminate()
    raise RuntimeError(
        f"vLLM server didn't come up within {timeout_s}s -- check {log_path}. "
        f"If the log shows it's still in torch.compile/CUDA graph capture, raise "
        f"timeout_s, lower max_num_seqs, or pass enforce_eager=True for a much "
        f"faster (if slightly slower at runtime) startup."
    )


# Module-level singleton so a fresh StreamableKGPipeline (one per "Process
# PDF" click) reuses an already-warm server instead of relaunching.
_vllm_state: Dict[str, object] = {"proc": None, "base_url": None}


def _server_healthy(base_url: str, timeout_s: float = 3.0) -> bool:
    import requests
    try:
        return requests.get(f"{base_url}/health", timeout=timeout_s).status_code == 200
    except Exception:
        return False


def ensure_vllm_server(cfg: PipelineConfig, **launch_kwargs) -> Tuple[subprocess.Popen, str]:
    """Reuses the already-running server if it's still healthy; otherwise
    launches a fresh one via launch_vllm_server (which itself cleans up
    anything stale bound to the port first)."""
    proc = _vllm_state.get("proc")
    base_url = _vllm_state.get("base_url")
    if proc is not None and base_url is not None:
        if proc.poll() is None and _server_healthy(base_url):
            return proc, base_url
        print("[PROCESS B] Existing vLLM server is no longer healthy -- relaunching.")

    proc, base_url = launch_vllm_server(cfg, **launch_kwargs)
    _vllm_state["proc"] = proc
    _vllm_state["base_url"] = base_url
    return proc, base_url


def shutdown_vllm_server():
    proc = _vllm_state.get("proc")
    if proc is not None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
        print("[PROCESS B] vLLM server stopped.")
    _vllm_state["proc"] = None
    _vllm_state["base_url"] = None


class VLLMTripleExtractor:
    """HTTP client for a vLLM server running in an ISOLATED venv --
    deliberately NOT an in-process `from vllm import LLM` (see module
    docstring). Sends the whole batch as a single request with prompt as
    a LIST so vLLM's server-side scheduler does the actual
    continuous-batching work.

    GUIDED DECODING FIELD: `{"structured_outputs": {"json": ...}}` --
    JSON-schema-constrained, not regex-constrained (see the comment above
    [VALIDATE]/build_triple_json_schema for why: the regex approach timed
    out xgrammar's compiler on real hardware).
    """

    def __init__(self, cfg: PipelineConfig, base_url: str = "http://localhost:8000",
                 tracer: Optional[LangfuseTracer] = None, request_timeout_s: int = 120):
        import requests
        self._requests = requests
        self.config = cfg
        self.base_url = base_url.rstrip("/")
        self.request_timeout_s = request_timeout_s
        self.json_schema = build_triple_json_schema(
            max_entity_chars=cfg.max_entity_chars, max_triples_per_chunk=cfg.max_triples_per_chunk,
        )
        self.tracer = tracer or LangfuseTracer(enabled=False)

        resp = requests.get(f"{self.base_url}/v1/models", timeout=10)
        resp.raise_for_status()
        self.model_name = resp.json()["data"][0]["id"]

        # Loading only the tokenizer files is a plain transformers call
        # with no CUDA/numba/torch-device involvement, so it doesn't
        # reintroduce the RAPIDS/vLLM dependency conflict process
        # isolation exists to avoid -- only loading MODEL weights
        # in-process would do that. $HOME (and so ~/.cache/huggingface)
        # is shared across venvs, so this finds files already pulled down
        # by the pre-download step or by vLLM's own server-startup fetch.
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name, trust_remote_code=True)

    def _confidence_from_choice(self, choice: dict) -> float:
        """Confidence from the average per-token log-probability (vLLM's
        completions API `logprobs` field, requested via `"logprobs": 1`
        below). exp(avg_logprob * scale): the geometric mean of per-token
        probabilities raised to a tunable power -- monotonic and
        correctly bounded to (0, 1] for any avg_logprob <= 0 and any
        scale > 0. `scale` remains an uncalibrated guess (see
        fit_confidence_calibration() above to derive it from labels)."""
        lp = (choice.get("logprobs") or {}).get("token_logprobs") or []
        values = [v for v in lp if v is not None]
        if not values:
            return 0.75  # no logprob data returned -- neutral fallback
        avg_logprob = sum(values) / len(values)
        return float(min(1.0, np.exp(avg_logprob * self.config.logprob_calibration_scale)))

    def _raw_batch_request(
        self, chunks: List[str], domain_tags: Optional[List[str]] = None, span_name: str = "extract_batch",
    ) -> Tuple[List[List], Dict[str, int]]:
        """One HTTP call for the whole batch -> SITL-filtered triples per
        chunk, WITHOUT enhance_connectivity applied. [OBSERVE]: wraps the
        call in a Langfuse span, with one nested "generation" observation
        per chunk (latency is necessarily shared across the batch since
        it's one HTTP call -- recorded as `batch_latency_s` /
        `batched_with_n_chunks` metadata on each generation rather than
        pretending each chunk has its own)."""
        domain_tags = domain_tags or ["unclassified"] * len(chunks)
        prompts = [f"{EXTRACTION_SYSTEM_PROMPT}\n\n{c}\n\nJSON:" for c in chunks]
        payload = {
            "model": self.model_name,
            "prompt": prompts,
            "max_tokens": self.config.max_output_tokens,
            "temperature": 0.1,
            "top_p": 0.9,
            "repetition_penalty": self.config.repetition_penalty,
            "structured_outputs": {"json": self.json_schema},
            "logprobs": 1,
        }

        with self.tracer.batch_span(
            span_name, input={"n_chunks": len(chunks), "domains": domain_tags},
            metadata={"vllm_model": self.model_name},
        ) as batch_span:
            t0 = time.time()
            with nvtx_range("vllm_http_call", color="yellow"):
                # NVTX range covers only the HTTP round-trip, not the JSON
                # parsing/filtering below -- this is CPU-idle/network-wait
                # time on THIS (main-kernel) process while the actual GPU
                # inference runs server-side in the isolated vLLM venv
                # process (see the separate `nsys profile ... vllm serve`
                # capture near the end of the notebook for that side).
                resp = self._requests.post(f"{self.base_url}/v1/completions", json=payload, timeout=self.request_timeout_s)
                resp.raise_for_status()
            elapsed = time.time() - t0
            choices = sorted(resp.json()["choices"], key=lambda c: c.get("index", 0))

            results: List[List] = []
            stats_total: Dict[str, int] = {}
            for i, choice in enumerate(choices):
                raw = choice["text"]
                conf = self._confidence_from_choice(choice)
                stats: Dict[str, int] = {}
                triples = parse_json_triples(
                    raw, strict=self.config.strict_field_validation,
                    max_entity_chars=self.config.max_entity_chars,
                    max_entity_words=self.config.max_entity_words, stats=stats,
                )
                normalized = [(s, sp, normalize_predicate(p), t, tp, conf) for s, sp, p, t, tp, _ in triples]
                filtered = sitl_filter(normalized, stats=stats)
                results.append(filtered)
                for k, v in stats.items():
                    stats_total[k] = stats_total.get(k, 0) + v

                domain = domain_tags[i] if i < len(domain_tags) else "unclassified"
                with batch_span.start_as_current_generation(
                    name=f"chunk_{i}", model=self.model_name,
                    input=chunks[i][:4000],
                    model_parameters={"temperature": 0.1, "top_p": 0.9},
                ) as gen:
                    gen.update(
                        output=raw[:4000],
                        metadata={
                            "domain": domain, "confidence": conf,
                            "n_triples_raw": len(triples), "n_triples_kept": len(filtered),
                            "dropped_stats": stats, "batched_with_n_chunks": len(chunks),
                            "batch_latency_s": round(elapsed, 3),
                        },
                    )
            batch_span.update(output={"total_kept": sum(len(r) for r in results)})
        return results, stats_total

    def _generate_once(self, chunks: List[str], domain_tags: Optional[List[str]] = None) -> List[List]:
        """Bare filtered triples, no stats, no enhancement -- what
        _bayesian_rerun needs for its consensus-sampling loop."""
        filtered, _ = self._raw_batch_request(chunks, domain_tags=domain_tags, span_name="extract_batch_bayesian_sample")
        return filtered

    def extract_batch(
        self, chunks: List[str], previous_entities: Optional[Set[str]] = None,
        domain_tags: Optional[List[str]] = None,
    ) -> Tuple[List[List], Dict[str, int]]:
        # previous_entities accepted, unused -- kept for interface
        # stability with StreamableKGPipeline._process_chunks' call site.
        filtered, stats_total = self._raw_batch_request(chunks, domain_tags=domain_tags)
        results = [enhance_connectivity(chunk, triples) for chunk, triples in zip(chunks, filtered)]
        return results, stats_total

    def _bayesian_rerun(self, chunks: List[str], n_samples: int = 3,
                         domain_tags: Optional[List[str]] = None) -> List[List]:
        """Low-yield rescue path: resamples n_samples times and keeps only
        triples with enough cross-sample agreement (see
        _consensus_from_runs for the Beta-Binomial voting math)."""
        all_runs = [self._generate_once(chunks, domain_tags=domain_tags) for _ in range(n_samples)]
        return _consensus_from_runs(all_runs, chunks, enhance_connectivity)


def _vllm_raw_completion(extractor: "VLLMTripleExtractor", full_prompt: str,
                          temperature: float = 0.1, use_schema: bool = True,
                          max_tokens: Optional[int] = None, stop: Optional[List[str]] = None) -> str:
    """Single-prompt completion against the vLLM server, bypassing
    extract_batch's parsing/filtering -- shared by [ACTIVE-LEARN]'s
    VLLMDPODatasetBuilder (quality-score bootstrapping) and the DSPy LM
    wrapper near the bottom of this file, both of which need the RAW
    text, not filtered triples."""
    payload = {
        "model": extractor.model_name, "prompt": full_prompt,
        "max_tokens": max_tokens or extractor.config.max_output_tokens, "temperature": temperature,
        "top_p": 0.9, "repetition_penalty": extractor.config.repetition_penalty,
    }
    if stop:
        payload["stop"] = stop
    if use_schema:
        payload["structured_outputs"] = {"json": extractor.json_schema}
    resp = extractor._requests.post(f"{extractor.base_url}/v1/completions", json=payload, timeout=extractor.request_timeout_s)
    resp.raise_for_status()
    return resp.json()["choices"][0]["text"]


# ============================================================================
# ============================  v2 HARNESS  ==================================
# (design doc: harness_update-1)
#
# Everything from here down to `class StreamableKGPipeline` is new in v2.
# Tags map to the design doc's section numbers:
#
#   [DOC-MAP]     SectionRecord / ChunkRecord / DocumentHierarchy /
#                 extract_document_structure / build_hierarchy       (2)
#   [SUMMARIES]   HierarchicalSummarizer                              (3)
#   [ONTOLOGY]    OntologySchema / RELATION_SCHEMA -- type system:
#                 domain, range, inverse, symmetry, disjointness,
#                 multi-inheritance, forbidden combos                 (9, 10)
#   [EVIDENCE]    Claim / ClaimLedger / EvidenceChecker               (7, 8, 21)
#   [MEMORY]      EntityMemory / ContextBuilder -- global entity memory
#                 + retrieval BEFORE extraction                       (4, 5)
#   [EXTRACT]     LLMClient / ClaimExtractor -- multi-pass            (6)
#   [VERIFY]      ClaimVerifier / route_claim / ClaimJudge --
#                 verifier + adaptive compute                         (12, 13)
#   [CONTRADICT]  ContradictionDetector                               (22)
#   [QUERY]       KGQueryEngine -- type-constrained retrieval, explain (11, 20)
#   [REPORT]      RunManifest / attach_provenance / export_audit_artifacts /
#                 graph_quality_metrics                               (20, 21, 26)
#
# The evaluation side (gold sets, red-team, failure taxonomy, calibration,
# slices, inter-rater, drift, regression gate -- design doc sections 14-19,
# 23-28) lives in kg_eval.py, written by the next notebook cell.
#
# ABLATION: every v2 mechanism has its own PipelineConfig switch and
# PipelineConfig.preset("v1_baseline" | "v2_evidence_only" | "v2_retrieval" |
# "v2_verification" | "v2_full") walks the doc's section-26 comparison list
# (baseline vs retrieval vs verification vs ontology constraints) as a config
# diff, not a code diff. "v1_baseline" runs the ORIGINAL v1 code path untouched.
# ============================================================================

# ---------------------------------------------------------------------------
# [DOC-MAP] explicit document representation  (design doc section 2)
# ---------------------------------------------------------------------------

@dataclass
class SectionRecord:
    section_id: str
    title: str
    level: int
    parent_title: str
    text: str
    page_start: int
    page_end: int
    page_offsets: List[Tuple[int, int]]   # (char offset inside `text`, 1-based page number)
    summary: str = ""


@dataclass
class ChunkRecord:
    doc_id: str
    chunk_id: str
    index: int
    text: str
    section: str
    subsection: str
    section_id: str
    page_start: int
    page_end: int
    char_start: int = 0
    page_breaks: List[Tuple[int, int]] = field(default_factory=list)  # (offset inside chunk text, page)
    prev_chunk: Optional[str] = None
    next_chunk: Optional[str] = None
    entities: List[str] = field(default_factory=list)
    domain: str = "unclassified"

    def page_at(self, local_pos: int) -> int:
        """Approximate page for a character offset inside this chunk (chunk text
        is re-joined from sentences, so offsets drift by a few chars at most)."""
        if not self.page_breaks:
            return self.page_start
        offs = [o for o, _ in self.page_breaks]
        k = max(0, bisect.bisect_right(offs, max(0, local_pos)) - 1)
        return self.page_breaks[k][1]


@dataclass
class DocumentHierarchy:
    doc_id: str
    sections: List[SectionRecord]
    chunks: List[ChunkRecord]
    doc_summary: str = ""

    def __post_init__(self):
        self._rebuild_index()

    def _rebuild_index(self):
        self._sec = {s.section_id: s for s in self.sections}
        self._chunk = {c.chunk_id: c for c in self.chunks}
        self._top: Dict[str, SectionRecord] = {}
        for s in self.sections:
            if s.level == 1:
                self._top[s.title] = s

    def section(self, section_id: str) -> Optional[SectionRecord]:
        return self._sec.get(section_id)

    def parent_of(self, sec: SectionRecord) -> Optional[SectionRecord]:
        return self._top.get(sec.parent_title) if sec.parent_title else None

    def chunk_by_id(self, chunk_id: str) -> Optional[ChunkRecord]:
        return self._chunk.get(chunk_id)

    def outline(self) -> List[Dict]:
        n_by_sec = Counter(c.section_id for c in self.chunks)
        return [{"section_id": s.section_id, "title": s.title, "level": s.level, "parent": s.parent_title,
                 "pages": [s.page_start, s.page_end], "summary": s.summary, "n_chunks": n_by_sec.get(s.section_id, 0)}
                for s in self.sections]


def _join_pages(page_texts: List[str], p0: int, p1: int) -> Tuple[str, List[Tuple[int, int]]]:
    """Joins pages [p0, p1) (0-based) with "\\n" and records where each page starts."""
    parts, offsets, pos = [], [], 0
    for p in range(p0, p1):
        t = page_texts[p]
        offsets.append((pos, p + 1))
        parts.append(t)
        pos += len(t) + 1
    return "\n".join(parts), offsets


def extract_document_structure(pdf_bytes: bytes, doc_id: str = "doc") -> List[SectionRecord]:
    """[OCR/LAYOUT] + [DOC-MAP]. Same section logic as v1 (TOC levels 1-2, else
    8-page blocks), but every section now carries its page range, a
    char-offset -> page map (so evidence can be traced to a page), its level and
    parent (section > subsection hierarchy), and pre-TOC front matter is kept
    instead of silently dropped. Each page is read once (v1 re-read pages)."""
    doc = pymupdf.open(stream=pdf_bytes, filetype="pdf")
    n = len(doc)
    page_texts = [doc[p].get_text() for p in range(n)]
    toc = doc.get_toc()
    sections: List[SectionRecord] = []

    def add(title: str, level: int, parent: str, p0: int, p1: int):
        if p1 <= p0:
            return
        text, offsets = _join_pages(page_texts, p0, p1)
        if len(text.strip()) < 80:
            return
        sections.append(SectionRecord(
            section_id=f"{doc_id}:s{len(sections)}", title=title, level=level, parent_title=parent,
            text=text, page_start=p0 + 1, page_end=p1, page_offsets=offsets,
        ))

    if toc:
        entries = [(lvl, title, pg) for lvl, title, pg in toc if lvl <= 2 and pg >= 1]
        if entries and entries[0][2] > 1:
            add("front_matter", 1, "", 0, min(entries[0][2] - 1, n))
        parent = ""
        for i, (lvl, title, pg) in enumerate(entries):
            end_pg = entries[i + 1][2] if i + 1 < len(entries) else n + 1
            if lvl == 1:
                parent = title
            add(title, lvl, "" if lvl == 1 else parent, max(0, pg - 1), min(end_pg - 1, n))

    if not sections:
        block = 8
        for start in range(0, n, block):
            add(f"pages_{start + 1}_{min(start + block, n)}", 1, "", start, min(start + block, n))

    doc.close()
    return sections


def sections_from_text(text: str, doc_id: str = "doc", title: str = "text") -> List[SectionRecord]:
    """Plain-text entry point (gold/red-team cases, unit tests, non-PDF sources)."""
    return [SectionRecord(section_id=f"{doc_id}:s0", title=title, level=1, parent_title="", text=text,
                          page_start=1, page_end=1, page_offsets=[(0, 1)])]


def build_hierarchy(sections: List[SectionRecord], chunker: "TextChunker", doc_id: str) -> DocumentHierarchy:
    """Sections -> chunks, with page span, section/subsection, and prev/next links on every chunk."""
    chunks: List[ChunkRecord] = []
    for sec in sections:
        pos_list = [o for o, _ in sec.page_offsets]

        def page_of(pos: int, _sec=sec, _pos_list=pos_list) -> int:
            k = max(0, bisect.bisect_right(_pos_list, pos) - 1)
            return _sec.page_offsets[k][1]

        for text, a, b in chunker.chunk_with_spans(sec.text):
            breaks = [(0, page_of(a))]
            for off, pg in sec.page_offsets:
                if a < off < b:
                    breaks.append((off - a, pg))
            # PDF line wraps ("CUDA\n12") are layout artifacts, not semantics -- collapse them so entity
            # names stay matchable inside evidence quotes. (Length-preserving for single newlines.)
            text = " ".join(text.split())
            chunks.append(ChunkRecord(
                doc_id=doc_id, chunk_id=f"{doc_id}:c{len(chunks)}", index=len(chunks), text=text,
                section=(sec.parent_title or sec.title), subsection=(sec.title if sec.parent_title else ""),
                section_id=sec.section_id, page_start=breaks[0][1], page_end=breaks[-1][1],
                char_start=a, page_breaks=breaks,
            ))
    for i, c in enumerate(chunks):
        c.prev_chunk = chunks[i - 1].chunk_id if i > 0 else None
        c.next_chunk = chunks[i + 1].chunk_id if i + 1 < len(chunks) else None
    return DocumentHierarchy(doc_id=doc_id, sections=sections, chunks=chunks)


# ---------------------------------------------------------------------------
# [ONTOLOGY] the ontology as a type system / constraint engine
# (design doc sections 9, 10, 11)
#
# v1 had two ad-hoc tables (INVALID_TYPE_PREDICATE_COMBOS and
# VALID_SUBJECT_TYPES_PER_PREDICATE) and dropped violations silently. v2 keeps
# the forbidden combos and adds one schema with:
#   * a type hierarchy with MULTIPLE inheritance (TECHNOLOGY is an ARTIFACT and
#     ABSTRACT; LAW is a DOCUMENT and a NORM; PROCESS is OCCURRENT and ABSTRACT),
#   * per-predicate domain + range, inverse, symmetry, functionality,
#   * disjoint type pairs (an entity cannot be a PERSON and a LOCATION).
# check() returns violation CODES instead of a silent drop, so every rejection
# is countable (ontology_violation_rate) and attributable.
#
# The predicate vocabulary is unchanged from v1 (the prompt, the JSON-schema
# enums and the vLLM test cell all depend on it). The doc's RELATION_SCHEMA
# example uses works_for / created / born_in / nationality; v1's PRED_CANON
# already maps works_for -> affiliated_with, and the others are not in the v1
# vocabulary -- add them to ALLOWED_PREDICATES + the prompt + this table
# together if you want them.
#
# Two places follow the doc's example literally and are STRICTER than v1:
#   develops    domain = ORGANIZATION        range = TECHNOLOGY | PRODUCT
#   located_in  domain = PERSON | ORGANIZATION | EVENT | TECHNOLOGY (+LOCATION)
# so "PERSON develops TECHNOLOGY" -- which v1 accepted -- is now flagged
# (-> REVIEW by default, see PipelineConfig.ontology_violation_action). Loosen
# the `develops` row below if you want v1's behaviour back for that case.
# ---------------------------------------------------------------------------

TYPE_PARENTS: Dict[str, Set[str]] = {
    "PERSON": {"AGENT"}, "ORGANIZATION": {"AGENT"},
    "PRODUCT": {"ARTIFACT"}, "DOCUMENT": {"ARTIFACT"},
    "TECHNOLOGY": {"ARTIFACT", "ABSTRACT"},
    "LAW": {"DOCUMENT", "NORM"}, "PROVISION": {"NORM"},
    "CONCEPT": {"ABSTRACT"}, "METRIC": {"ABSTRACT"},
    "PROCESS": {"OCCURRENT", "ABSTRACT"}, "EVENT": {"OCCURRENT"},
    "LOCATION": {"PLACE"},
}

TYPE_DISJOINT: List[Tuple[str, str]] = [
    ("PERSON", "ORGANIZATION"), ("PERSON", "LOCATION"), ("ORGANIZATION", "LOCATION"),
    ("PERSON", "METRIC"), ("ORGANIZATION", "METRIC"), ("LOCATION", "METRIC"),
    ("PERSON", "LAW"), ("PERSON", "PROVISION"), ("PERSON", "DOCUMENT"),
    ("LOCATION", "LAW"), ("LOCATION", "DOCUMENT"),
]


@dataclass
class RelationSpec:
    domain: Set[str]
    range: Set[str]
    gloss: str
    inverse: Optional[str] = None
    symmetric: bool = False
    functional: bool = False   # at most one target per subject -- set True on predicates where your corpus needs it


_ANY: Set[str] = {"ANY"}

RELATION_SCHEMA: Dict[str, RelationSpec] = {
    "is_a":            RelationSpec(_ANY, _ANY, "subject is a kind/subtype of target"),
    "instance_of":     RelationSpec(_ANY, _ANY, "subject is one specific instance of target"),
    "part_of":         RelationSpec(_ANY, _ANY, "subject is a component or member of target", inverse="contains"),
    "contains":        RelationSpec(_ANY, _ANY, "subject contains target as a part", inverse="part_of"),
    "includes":        RelationSpec(_ANY, _ANY, "subject includes target among its parts", inverse="part_of"),
    "develops":        RelationSpec({"ORGANIZATION"}, {"TECHNOLOGY", "PRODUCT"},
                                    "subject (an organization) creates, builds or maintains target (a technology or product)"),
    "authors":         RelationSpec({"AGENT"}, {"DOCUMENT", "NORM"}, "subject wrote target"),
    "issues":          RelationSpec({"AGENT", "LAW"}, {"ARTIFACT", "NORM", "CONCEPT"},
                                    "subject publishes, announces or releases target"),
    "regulates":       RelationSpec({"ORGANIZATION", "LAW", "PROVISION", "DOCUMENT"}, _ANY,
                                    "subject sets or enforces rules over target"),
    "manages":         RelationSpec({"AGENT"}, _ANY, "subject administers or leads target"),
    "governs":         RelationSpec({"ORGANIZATION", "LAW", "PROVISION"}, _ANY,
                                    "subject formally controls target"),
    "performs":        RelationSpec({"AGENT", "TECHNOLOGY", "PRODUCT", "PROCESS"}, _ANY,
                                    "subject carries out or uses target"),
    "enables":         RelationSpec(_ANY, _ANY, "subject makes target possible"),
    "causes":          RelationSpec({"AGENT", "OCCURRENT", "ABSTRACT", "TECHNOLOGY", "PRODUCT", "NORM"},
                                    {"OCCURRENT", "ABSTRACT"}, "subject brings about target"),
    "requires":        RelationSpec({"AGENT", "ARTIFACT", "ABSTRACT", "OCCURRENT", "NORM"}, _ANY,
                                    "subject needs target to work or be valid"),
    "produces":        RelationSpec({"AGENT", "ARTIFACT", "OCCURRENT", "ABSTRACT"},
                                    {"ARTIFACT", "ABSTRACT", "OCCURRENT", "NORM"}, "subject outputs or yields target"),
    "applies_to":      RelationSpec(_ANY, _ANY, "subject is applicable to target"),
    "affiliated_with": RelationSpec({"PERSON", "ORGANIZATION"}, {"ORGANIZATION"},
                                    "subject belongs to or works for target"),
    "located_in":      RelationSpec({"PERSON", "ORGANIZATION", "EVENT", "TECHNOLOGY", "LOCATION"},
                                    {"LOCATION"}, "subject is physically located in target"),
    "related_to":      RelationSpec(_ANY, _ANY, "subject is explicitly associated with target (only if no "
                                                "more specific predicate fits)", symmetric=True),
    "influences":      RelationSpec(_ANY, _ANY, "subject affects the value or behaviour of target"),
    "depends_on":      RelationSpec(_ANY, _ANY, "subject relies on target"),
}
assert set(RELATION_SCHEMA) == ALLOWED_PREDICATES, "RELATION_SCHEMA must cover exactly ALLOWED_PREDICATES"
assert set(TYPE_PARENTS) <= ALLOWED_ENTITY_TYPES

# Violation codes that are never "maybe": the combination is wrong whatever the evidence says.
HARD_VIOLATION_PREFIXES = ("FORBIDDEN:", "UNKNOWN_PREDICATE:")


class OntologySchema:
    def __init__(self, relations: Optional[Dict[str, RelationSpec]] = None,
                 type_parents: Optional[Dict[str, Set[str]]] = None,
                 disjoint: Optional[List[Tuple[str, str]]] = None,
                 forbidden: Optional[Set[Tuple[str, str, str]]] = None, name: str = "onto-2"):
        self.relations = relations if relations is not None else RELATION_SCHEMA
        self.type_parents = type_parents if type_parents is not None else TYPE_PARENTS
        self.disjoint = {frozenset(p) for p in (disjoint if disjoint is not None else TYPE_DISJOINT)}
        self.forbidden = forbidden if forbidden is not None else INVALID_TYPE_PREDICATE_COMBOS
        self.name = name
        self._anc: Dict[str, Set[str]] = {}

    @property
    def version(self) -> str:
        """Content hash -- changes whenever the schema changes, so W&B runs and the
        regression gate can tell WHICH ontology a result was produced under."""
        blob = json.dumps({
            "rel": {p: [sorted(s.domain), sorted(s.range), s.inverse, s.symmetric, s.functional]
                    for p, s in sorted(self.relations.items())},
            "par": {k: sorted(v) for k, v in sorted(self.type_parents.items())},
            "dis": sorted(sorted(p) for p in self.disjoint),
            "forb": sorted(self.forbidden),
        }, sort_keys=True)
        return f"{self.name}-{hashlib.sha1(blob.encode()).hexdigest()[:8]}"

    def ancestors(self, t: str) -> Set[str]:
        """The type itself plus every (transitive) supertype -- multi-inheritance safe."""
        if t in self._anc:
            return self._anc[t]
        out, stack = set(), [t]
        while stack:
            cur = stack.pop()
            if cur in out:
                continue
            out.add(cur)
            stack.extend(self.type_parents.get(cur, ()))
        self._anc[t] = out
        return out

    def matches(self, t: str, allowed: Set[str]) -> bool:
        return "ANY" in allowed or bool(self.ancestors(t) & allowed)

    def check(self, subject_type: str, predicate: str, target_type: str) -> List[str]:
        spec = self.relations.get(predicate)
        if spec is None:
            return [f"UNKNOWN_PREDICATE:{predicate}"]
        v: List[str] = []
        if not self.matches(subject_type, spec.domain):
            v.append(f"DOMAIN:{subject_type}-{predicate}")
        if not self.matches(target_type, spec.range):
            v.append(f"RANGE:{predicate}-{target_type}")
        if (subject_type, predicate, target_type) in self.forbidden:
            v.append(f"FORBIDDEN:{subject_type}-{predicate}-{target_type}")
        return v

    def direction_repair_ok(self, subject_type: str, predicate: str, target_type: str) -> bool:
        """True if the claim violates the schema as written but is valid reversed --
        the classic passive-voice extraction error ("banks are regulated by RBI")."""
        spec = self.relations.get(predicate)
        if spec is None or spec.symmetric:
            return False
        return bool(self.check(subject_type, predicate, target_type)) and not self.check(target_type, predicate, subject_type)

    def are_disjoint(self, a: str, b: str) -> bool:
        return any(frozenset((x, y)) in self.disjoint for x in self.ancestors(a) for y in self.ancestors(b))

    def is_symmetric(self, predicate: str) -> bool:
        spec = self.relations.get(predicate)
        return bool(spec and spec.symmetric)

    def is_functional(self, predicate: str) -> bool:
        spec = self.relations.get(predicate)
        return bool(spec and spec.functional)

    def inverse(self, predicate: str) -> Optional[str]:
        spec = self.relations.get(predicate)
        return spec.inverse if spec else None

    def gloss(self, predicate: str) -> str:
        spec = self.relations.get(predicate)
        return spec.gloss if spec else ""

    def allowed_predicates(self, subject_type: str, target_type: str) -> List[str]:
        return sorted(p for p in self.relations if not self.check(subject_type, p, target_type))

    def type_paths(self, src_type: str, dst_type: str, max_hops: int = 2,
                   limit: int = 400) -> List[List[Tuple[str, str]]]:
        """Type-level search over the SCHEMA (not the data): every predicate sequence
        [(predicate, type_reached), ...] the ontology allows between two entity types,
        up to max_hops. KGQueryEngine walks only edges that follow one of these
        sequences, so a PERSON -> LOCATION question never expands through edges that
        cannot lead there (design doc section 11)."""
        types = sorted(ALLOWED_ENTITY_TYPES)
        paths: List[List[Tuple[str, str]]] = []
        frontier: List[Tuple[str, List[Tuple[str, str]]]] = [(src_type, [])]
        for _ in range(max_hops):
            nxt: List[Tuple[str, List[Tuple[str, str]]]] = []
            for cur, path in frontier:
                for p in self.relations:
                    for t in types:
                        if self.check(cur, p, t):
                            continue
                        new_path = path + [(p, t)]
                        if t == dst_type:
                            paths.append(new_path)
                            if len(paths) >= limit:
                                return paths
                        nxt.append((t, new_path))
            frontier = nxt[:3000]
        return paths


ONTOLOGY = OntologySchema()


_NUM_TOKEN_RE = re.compile(r'\d+(?:\.\d+)*')


def _numeric_signature(label: str) -> Tuple[str, ...]:
    """Version-like tokens in a label. 'CUDA 12' and 'CUDA 13' must stay two
    entities (and be able to contradict each other) -- v1's substring rule, fuzzy
    rule and the ANN duplicate rule all happily merged them."""
    return tuple(_NUM_TOKEN_RE.findall(label))


# ---------------------------------------------------------------------------
# [SUMMARIES] hierarchical summarization  (design doc section 3)
# Document summary <- section summaries <- chunk facts. One cheap LLM call per
# section + one for the document, so the per-chunk prompt carries DOCUMENT
# CONTEXT and SECTION CONTEXT instead of 900 isolated tokens. Chunk-level
# "facts" are the accepted claims stored in EntityRecord.facts / ClaimLedger.
# ---------------------------------------------------------------------------

class HierarchicalSummarizer:
    SECTION_PROMPT = (
        "Summarize in at most 40 words what this part of a document is about. "
        "Name the main organizations, systems, technologies and concepts.\n\n"
        "SECTION TITLE: {title}\nTEXT:\n{sample}\n\nSUMMARY:"
    )
    DOC_PROMPT = (
        "Below are one-line summaries of the sections of a document. Write a summary of the whole "
        "document in at most 60 words.\n\n{body}\n\nDOCUMENT SUMMARY:"
    )

    def __init__(self, llm: "LLMClient", sample_chars: int = 2400, request_size: int = 16):
        self.llm = llm
        self.sample_chars = sample_chars
        self.request_size = request_size

    def _sample(self, chunks: List[ChunkRecord]) -> str:
        if not chunks:
            return ""
        per = max(200, self.sample_chars // max(1, len(chunks)))
        parts, used = [], 0
        for c in chunks:
            piece = c.text[:per]
            if used + len(piece) > self.sample_chars:
                break
            parts.append(piece)
            used += len(piece)
        return "\n...\n".join(parts)

    def _run(self, prompts: List[str], max_tokens: int) -> List[str]:
        out: List[str] = []
        for i in range(0, len(prompts), self.request_size):
            res = self.llm.complete(prompts[i:i + self.request_size], schema=None, max_tokens=max_tokens,
                                    temperature=0.0, span_name="summaries", stop=["\n\n"])
            out.extend(" ".join(r["text"].split()) for r in res)
        return out

    @nvtx_range("hierarchical_summaries", color="purple")
    def summarize(self, hier: DocumentHierarchy) -> DocumentHierarchy:
        by_section: Dict[str, List[ChunkRecord]] = defaultdict(list)
        for c in hier.chunks:
            by_section[c.section_id].append(c)
        secs = [s for s in hier.sections if by_section[s.section_id]]
        prompts = [self.SECTION_PROMPT.format(title=s.title, sample=self._sample(by_section[s.section_id]))
                   for s in secs]
        for s, summ in zip(secs, self._run(prompts, 90)):
            s.summary = summ
        body = "\n".join(f"- {s.title}: {s.summary}" for s in hier.sections if s.summary)[:3500]
        if body:
            res = self._run([self.DOC_PROMPT.format(body=body)], 140)
            hier.doc_summary = res[0] if res else ""
        n_ok = sum(1 for s in hier.sections if s.summary)
        print(f"[SUMMARIES] {n_ok}/{len(hier.sections)} section summaries + document summary "
              f"({len(hier.doc_summary)} chars)")
        return hier


# ---------------------------------------------------------------------------
# [EVIDENCE] claims with provenance + deterministic evidence grounding
# (design doc sections 7, 8, 21)
# ---------------------------------------------------------------------------

@dataclass
class Claim:
    subject: str
    subject_type: str
    predicate: str
    target: str
    target_type: str
    confidence: float = 0.0
    raw_confidence: float = 0.0     # confidence as produced by the extractor, before verification / floors
    evidence: str = ""
    evidence_source: str = ""       # "model" (LLM quote located in chunk) | "retrieved" (best sentence) | ""
    evidence_score: float = 0.0     # 0..1 -- min(quote-in-chunk, subject-in-evidence, target-in-evidence)
    evidence_grounded: bool = False
    coref: bool = False             # one endpoint resolved from earlier context (pronoun / definite NP)
    modality: str = "asserted"      # asserted | negated | hypothetical | attributed
    claim_id: str = ""
    chunk_id: str = ""
    doc_id: str = ""
    page: int = 0
    ontology_violations: List[str] = field(default_factory=list)
    retyped: bool = False
    direction_repaired: bool = False
    verifier: str = ""              # "" | YES | NO | UNSURE
    verifier_conf: float = 0.0
    route: str = ""                 # accept | verify | escalate
    verdict: str = "PENDING"        # ACCEPT | REVIEW | REJECT
    reject_reason: str = ""
    source: str = "llm"             # llm | cooccurrence
    contradiction_ids: List[str] = field(default_factory=list)

    def key(self) -> Tuple[str, str, str]:
        return (normalize_entity_label(self.subject), self.predicate, normalize_entity_label(self.target))

    def as_tuple(self) -> Tuple[str, str, str, str, str, float]:
        return (self.subject, self.subject_type, self.predicate, self.target, self.target_type, float(self.confidence))

    def to_row(self) -> Dict:
        d = asdict(self)
        d["ontology_violations"] = "|".join(self.ontology_violations)
        d["contradiction_ids"] = "|".join(self.contradiction_ids)
        return d


class ClaimLedger:
    """Every candidate the pipeline ever considered -- accepted, rejected or sent to
    review -- with the reason. This is what makes the graph auditable ("why do you
    believe this edge exists?") and what the unsupported-claim / failure metrics are
    computed from."""

    def __init__(self):
        self.claims: List[Claim] = []
        self.batch_stats: List[Dict] = []
        self.counters: Counter = Counter()
        self._n = 0

    def new_id(self, doc_id: str) -> str:
        self._n += 1
        return f"{doc_id}:k{self._n}"

    def add(self, claims: List[Claim]):
        self.claims.extend(claims)

    def accepted(self) -> List[Claim]:
        return [c for c in self.claims if c.verdict == "ACCEPT"]

    def to_polars(self) -> pl.DataFrame:
        if not self.claims:
            return pl.DataFrame()
        return pl.DataFrame([c.to_row() for c in self.claims], infer_schema_length=None)

    @staticmethod
    def _rate(num: float, den: float) -> float:
        return float(num) / den if den else 0.0

    def summary(self) -> Dict[str, float]:
        cs = [c for c in self.claims if c.source == "llm"]
        acc = [c for c in cs if c.verdict == "ACCEPT"]
        n = len(cs)
        reasons = Counter(c.reject_reason for c in cs if c.verdict in ("REJECT", "REVIEW") and c.reject_reason)
        routes = Counter(c.route for c in cs if c.route)
        n_routed = sum(routes.values())
        lat = [b["batch_time_s"] / max(1, b["n_chunks"]) for b in self.batch_stats if b.get("n_chunks")]
        n_chunks = sum(b.get("n_chunks", 0) for b in self.batch_stats)
        gen_time = sum(b.get("batch_time_s", 0.0) for b in self.batch_stats)
        n_verified = sum(1 for c in cs if c.verifier)
        out: Dict[str, float] = {
            "claims/candidates": float(n),
            "claims/accepted": float(len(acc)),
            "claims/review": float(sum(1 for c in cs if c.verdict == "REVIEW")),
            "claims/rejected": float(sum(1 for c in cs if c.verdict == "REJECT")),
            # headline safety metric (design doc section 8): ACCEPTED edges with no textual support
            "quality/unsupported_claim_rate": self._rate(sum(1 for c in acc if not c.evidence_grounded), len(acc)),
            # model-level hallucination: candidates the model proposed that were never grounded in the text
            "quality/candidate_ungrounded_rate": self._rate(sum(1 for c in cs if not c.evidence_grounded), n),
            "quality/ontology_violation_rate": self._rate(
                sum(1 for c in cs if c.ontology_violations or c.direction_repaired), n),
            "quality/direction_repair_rate": self._rate(sum(1 for c in cs if c.direction_repaired), n),
            "quality/modality_nonasserted_rate": self._rate(sum(1 for c in cs if c.modality != "asserted"), n),
            "quality/verifier_no_rate": self._rate(sum(1 for c in cs if c.verifier == "NO"), n_verified),
            "quality/avg_accepted_confidence": float(np.mean([c.confidence for c in acc])) if acc else 0.0,
            # adaptive-compute split (design doc section 13): easy -> accept, medium -> verify, hard -> escalate
            "compute/route_easy_frac": self._rate(routes.get("accept", 0), n_routed),
            "compute/route_medium_frac": self._rate(routes.get("verify", 0), n_routed),
            "compute/route_hard_frac": self._rate(routes.get("escalate", 0), n_routed),
            "compute/verifier_calls": float(self.counters.get("verifier_calls", 0)),
            "compute/llm_calls": float(self.counters.get("llm_calls", 0)),
            "compute/llm_errors": float(self.counters.get("llm_errors", 0)),
            "compute/prompt_tokens": float(self.counters.get("prompt_tokens", 0)),
            "compute/completion_tokens": float(self.counters.get("completion_tokens", 0)),
            "compute/tokens_per_chunk": self._rate(
                self.counters.get("prompt_tokens", 0) + self.counters.get("completion_tokens", 0), n_chunks),
            "compute/completion_tokens_per_s": self._rate(self.counters.get("completion_tokens", 0), gen_time),
            "latency/p50_s_per_chunk": float(np.percentile(lat, 50)) if lat else 0.0,
            "latency/p95_s_per_chunk": float(np.percentile(lat, 95)) if lat else 0.0,
            "system/gpu_used_gb_max": max([b.get("gpu_used_gb", 0.0) for b in self.batch_stats] or [0.0]),
            "system/cpu_rss_gb_max": max([b.get("cpu_rss_gb", 0.0) for b in self.batch_stats] or [0.0]),
        }
        for r, k in reasons.items():
            out[f"reason/{r}"] = float(k)
        return out


_ANAPHORA_RE = re.compile(
    r"\b(it|its|they|their|them|this|that|these|those|the\s+(?:framework|system|model|method|approach|tool|library|"
    r"company|organization|organisation|technology|technique|platform|engine|server|paper|study|regulator|act))\b",
    re.IGNORECASE,
)


def split_sentences(text: str) -> List[Tuple[str, int, int]]:
    return [(text[a:b], a, b) for a, b in TextChunker._split_with_spans(text)]


def mention_score(name: str, text: str) -> float:
    """0-100: does `name` appear in `text`? Short single-token names need an exact
    word-boundary hit (fuzzy partial matching is meaningless for 'AI' or 'RBI')."""
    n = normalize_entity_label(name)
    if not n or not text:
        return 0.0
    t = text.lower()
    if len(n) <= 5 and " " not in n:
        return 100.0 if re.search(r"(?<!\w)" + re.escape(n) + r"(?!\w)", t) else 0.0
    if n in t:
        return 100.0
    return float(fuzz.partial_ratio(n, t))


class EvidenceChecker:
    """Deterministic claim<->text grounding -- no LLM. Three questions:
      1. is the quoted evidence actually in the chunk?          (did the model invent a quote?)
      2. do both endpoints appear in that evidence?              (or is one a resolved coreference?)
      3. if the model's quote fails, is there a real sentence in the chunk that
         mentions both endpoints? -> substitute it (evidence_source='retrieved')."""

    def __init__(self, min_evidence: float = 85.0, min_entity: float = 80.0):
        self.min_evidence = min_evidence
        self.min_entity = min_entity

    def best_sentence(self, claim: Claim, text: str) -> Tuple[Optional[str], int, int, float]:
        """(sentence-or-2-sentence-window, char_pos, sentence_index, score)."""
        sents = split_sentences(text)
        best: Tuple[Optional[str], int, int, float] = (None, -1, -1, 0.0)
        for i, (s, a, _b) in enumerate(sents):
            sc = min(mention_score(claim.subject, s), mention_score(claim.target, s))
            if sc > best[3]:
                best = (s, a, i, sc)
        for i in range(len(sents) - 1):
            w = sents[i][0] + " " + sents[i + 1][0]
            if len(w) > 420:
                continue
            sc = min(mention_score(claim.subject, w), mention_score(claim.target, w))
            if sc > best[3] + 1e-9:
                best = (w, sents[i][1], i, sc)
        return best

    def window(self, claim: Claim, text: str, radius: int = 1) -> str:
        """Expanded evidence for escalation: best sentence +- `radius` sentences."""
        sents = split_sentences(text)
        _, _, idx, _ = self.best_sentence(claim, text)
        if idx < 0 or not sents:
            return text[:700]
        lo, hi = max(0, idx - radius), min(len(sents), idx + radius + 2)
        return " ".join(s for s, _, _ in sents[lo:hi])[:900]

    def check(self, claim: Claim, chunk: ChunkRecord, memory: Optional["EntityMemory"] = None) -> Claim:
        text = chunk.text
        ev = (claim.evidence or "").strip()
        ev_q, ev_pos, src = 0.0, -1, ""
        if ev:
            aln = fuzz.partial_ratio_alignment(ev.lower(), text.lower())
            if aln is not None:
                ev_q, ev_pos = float(aln.score), int(aln.dest_start)
            src = "model"
        s_sc, t_sc = mention_score(claim.subject, ev), mention_score(claim.target, ev)

        grounded = bool(ev) and ev_q >= self.min_evidence and s_sc >= self.min_entity and t_sc >= self.min_entity
        coref = False
        score = min(ev_q, s_sc, t_sc) / 100.0
        # "It requires CUDA 12": the quote is real and names ONE endpoint; the other is a
        # pronoun/definite NP that entity memory resolved. Grounded, but flagged for the router.
        if (not grounded and ev and ev_q >= self.min_evidence and memory is not None
                and _ANAPHORA_RE.search(ev) and (s_sc >= self.min_entity or t_sc >= self.min_entity)):
            missing = claim.target if s_sc >= self.min_entity else claim.subject
            if memory.knows(missing):
                grounded, coref, score = True, True, ev_q / 100.0 * 0.8
        if not grounded:
            sent, pos, _idx, sc = self.best_sentence(claim, text)
            if sent and sc >= self.min_entity:
                ev, ev_pos, src, grounded, score = sent, pos, "retrieved", True, sc / 100.0
        claim.evidence = ev
        claim.evidence_source = src if ev else ""
        claim.evidence_grounded = grounded
        claim.evidence_score = round(float(score), 4)
        claim.coref = coref or claim.coref
        claim.chunk_id, claim.doc_id = chunk.chunk_id, chunk.doc_id
        claim.page = chunk.page_at(ev_pos) if ev_pos >= 0 else chunk.page_start
        return claim


# ---------------------------------------------------------------------------
# [MEMORY] global entity memory  (design doc sections 4, 5)
# ---------------------------------------------------------------------------

@dataclass
class EntityRecord:
    key: str
    canonical_name: str
    aliases: Set[str] = field(default_factory=set)
    type_votes: Counter = field(default_factory=Counter)
    mentions: List[str] = field(default_factory=list)       # chunk ids (capped)
    evidence: List[str] = field(default_factory=list)       # up to 3 quotes
    facts: List[str] = field(default_factory=list)          # up to 3 accepted "S pred T" facts
    confidence: float = 0.0
    first_chunk_index: int = 0
    last_chunk_index: int = 0
    n_mentions: int = 0
    first_page: int = 0

    @property
    def entity_type(self) -> str:
        return self.type_votes.most_common(1)[0][0] if self.type_votes else "CONCEPT"


_ACRONYM_AFTER_RE = re.compile(
    r"\b([A-Z][\w\-]*(?:\s+(?:of|for|and|the|[A-Z][\w\-]*)){0,6})\s*\(\s*([A-Z][A-Za-z0-9\-]{1,12})\s*\)")
_ACRONYM_BEFORE_RE = re.compile(
    r"\b([A-Z][A-Za-z0-9\-]{1,12})\s*\(\s*([A-Z][\w\-]*(?:\s+(?:of|for|and|the|[A-Z][\w\-]*)){1,6})\s*\)")


_SMALL_WORDS = {"of", "for", "and", "the", "in", "on", "a", "an", "to"}


def _initials_match(words: List[str], short: str, whole_phrase: bool = False) -> Optional[List[str]]:
    """The (suffix of the) word list whose capitalized-word initials spell `short`
    ('The Reserve Bank of India' + 'RBI' -> ['Reserve', 'Bank', 'of', 'India'])."""
    s = re.sub(r"[^a-z0-9]", "", short.lower())
    starts = [0] if whole_phrase else range(len(words))
    for i in starts:
        sub = words[i:]
        content = [w for w in sub if w.lower() not in _SMALL_WORDS]
        if len(content) >= 2 and "".join(w[0].lower() for w in content) == s:
            return sub
    return None


def extract_acronym_pairs(text: str) -> List[Tuple[str, str]]:
    """('Reserve Bank of India', 'RBI') from 'The Reserve Bank of India (RBI)' or 'RBI (Reserve Bank of India)'."""
    pairs: List[Tuple[str, str]] = []
    for long, short in _ACRONYM_AFTER_RE.findall(text):
        sub = _initials_match(long.split(), short)
        if sub:
            pairs.append((" ".join(sub), short))
    for short, long in _ACRONYM_BEFORE_RE.findall(text):
        sub = _initials_match(long.split(), short, whole_phrase=True)
        if sub:
            pairs.append((" ".join(sub), short))
    return pairs


class EntityMemory:
    """Document-level entity store that is READ before extraction (the small model is
    handed canonical names and candidate antecedents instead of having to remember 50
    chunks) and WRITTEN after it. Search is exact brute force over the (small) entity
    set -- below ~50k entities that beats an ANN index; swap `_semantic_scores` for the
    cuML index if you ever process book-sized corpora."""

    def __init__(self, embedder=None, fuzzy_threshold: float = 92.0):
        self.embedder = embedder
        self.fuzzy_threshold = fuzzy_threshold
        self.records: Dict[str, EntityRecord] = {}
        self.alias_index: Dict[str, str] = {}
        self.loose_index: Dict[str, str] = {}      # punctuation/space-insensitive: 'TensorRT-LLM' == 'TensorRT LLM'
        self.acronyms: Dict[str, str] = {}
        self._emb: Dict[str, np.ndarray] = {}

    def __len__(self) -> int:
        return len(self.records)

    @staticmethod
    def _loose(k: str) -> str:
        return re.sub(r"[\W_]+", "", k)

    # -- lookup ---------------------------------------------------------
    def lookup(self, name: str) -> Optional[EntityRecord]:
        k = normalize_entity_label(name)
        if not k:
            return None
        if k in self.alias_index:
            return self.records[self.alias_index[k]]
        lk = self._loose(k)
        if lk and lk in self.loose_index and self.loose_index[lk] in self.records:
            return self.records[self.loose_index[lk]]
        if self.alias_index:
            sig = _numeric_signature(k)
            for cand, _score, _idx in rf_process.extract(k, list(self.alias_index), scorer=fuzz.token_sort_ratio,
                                                        score_cutoff=self.fuzzy_threshold, limit=3):
                if _numeric_signature(cand) == sig:
                    return self.records[self.alias_index[cand]]
        return None

    def knows(self, name: str) -> bool:
        return self.lookup(name) is not None

    # -- write ----------------------------------------------------------
    def observe(self, name: str, etype: str, chunk_id: str = "", chunk_index: int = 0, evidence: str = "",
                confidence: float = 0.7, vote: bool = True, page: int = 0) -> EntityRecord:
        rec = self.lookup(name)
        k = normalize_entity_label(name)
        if rec is None:
            partner = self.acronyms.get(k)
            prec = self.lookup(partner) if partner else None
            if prec is not None:
                rec = prec
                self._add_alias(rec, name)
        if rec is None:
            rec = EntityRecord(key=k, canonical_name=name.strip(), first_chunk_index=chunk_index, first_page=page)
            self.records[k] = rec
            self.alias_index[k] = k
            self.loose_index.setdefault(self._loose(k), k)
        elif k not in self.alias_index:
            self._add_alias(rec, name)
        if vote and etype:
            rec.type_votes[etype] += 1
        rec.n_mentions += 1
        rec.last_chunk_index = max(rec.last_chunk_index, chunk_index)
        rec.confidence = max(rec.confidence, confidence)
        if chunk_id and (not rec.mentions or rec.mentions[-1] != chunk_id) and len(rec.mentions) < 200:
            rec.mentions.append(chunk_id)
        if evidence and len(rec.evidence) < 3 and evidence not in rec.evidence:
            rec.evidence.append(evidence)
        return rec

    def observe_claim(self, c: Claim, chunk_index: int):
        """Accepted claim -> both endpoints get a type vote, an evidence quote and a one-line fact."""
        fact = f"{c.subject} {c.predicate} {c.target}"
        for name, etype in ((c.subject, c.subject_type), (c.target, c.target_type)):
            rec = self.observe(name, etype, chunk_id=c.chunk_id, chunk_index=chunk_index,
                               evidence=c.evidence, confidence=c.confidence, page=c.page)
            if len(rec.facts) < 3 and fact not in rec.facts:
                rec.facts.append(fact)

    def _add_alias(self, rec: EntityRecord, alias: str):
        k = normalize_entity_label(alias)
        if not k or k in self.alias_index:
            return
        self.alias_index[k] = rec.key
        self.loose_index.setdefault(self._loose(k), rec.key)
        if normalize_entity_label(rec.canonical_name) != k:
            rec.aliases.add(alias.strip())

    def register_acronyms(self, text: str):
        for long, short in extract_acronym_pairs(text):
            rl, rs = self.lookup(long), self.lookup(short)
            if rl is not None and rs is not None and rl is not rs:
                self._merge(rl, rs)
            elif rl is not None:
                self._add_alias(rl, short)
            elif rs is not None:
                self._add_alias(rs, long)
            else:
                self.acronyms[normalize_entity_label(short)] = long
                self.acronyms[normalize_entity_label(long)] = short

    def _merge(self, keep: EntityRecord, drop: EntityRecord):
        keep.aliases |= drop.aliases | {drop.canonical_name}
        keep.type_votes.update(drop.type_votes)
        keep.mentions = (keep.mentions + drop.mentions)[:200]
        keep.n_mentions += drop.n_mentions
        keep.evidence = (keep.evidence + [e for e in drop.evidence if e not in keep.evidence])[:3]
        keep.facts = (keep.facts + [f for f in drop.facts if f not in keep.facts])[:3]
        keep.first_chunk_index = min(keep.first_chunk_index, drop.first_chunk_index)
        keep.last_chunk_index = max(keep.last_chunk_index, drop.last_chunk_index)
        for idx in (self.alias_index, self.loose_index):
            for a, rk in list(idx.items()):
                if rk == drop.key:
                    idx[a] = keep.key
        self.records.pop(drop.key, None)
        self._emb.pop(drop.key, None)

    # -- read -----------------------------------------------------------
    def _semantic_scores(self, query: str) -> Dict[str, float]:
        if self.embedder is None or not self.records or not query.strip():
            return {}
        names = list(self.records)
        missing = [k for k in names if k not in self._emb]
        if missing:
            vecs = self.embedder.encode([self.records[k].canonical_name for k in missing], batch_size=256,
                                        show_progress_bar=False, normalize_embeddings=True, convert_to_numpy=True)
            for k, v in zip(missing, vecs):
                self._emb[k] = np.asarray(v, dtype="float32")
        q = self.embedder.encode([query[:1500]], show_progress_bar=False, normalize_embeddings=True,
                                 convert_to_numpy=True)[0]
        mat = np.stack([self._emb[k] for k in names])
        sims = mat @ np.asarray(q, dtype="float32")
        return {k: float(s) for k, s in zip(names, sims)}

    def retrieve(self, chunk_text: str, chunk_index: int, k_lex: int = 15, k_sem: int = 5,
                 k_recent: int = 5) -> Tuple[List[EntityRecord], List[EntityRecord]]:
        """Returns (mentioned_in_chunk, earlier_candidates). The second list is what the
        model needs for coreference ('it', 'the framework'): semantically related and
        recently seen entities NOT literally named in this chunk."""
        low = chunk_text.lower()
        lexical: List[EntityRecord] = []
        for rec in self.records.values():
            for nm in [rec.key] + [normalize_entity_label(a) for a in rec.aliases]:
                if not nm:
                    continue
                if len(nm) <= 5:
                    hit = re.search(r"(?<!\w)" + re.escape(nm) + r"(?!\w)", low) is not None
                else:
                    hit = nm in low
                if hit:
                    lexical.append(rec)
                    break
        lexical.sort(key=lambda r: -r.n_mentions)
        lexical = lexical[:k_lex]
        taken = {r.key for r in lexical}
        sem = self._semantic_scores(chunk_text)
        sem_recs = [self.records[k] for k, s in sorted(sem.items(), key=lambda kv: -kv[1])
                    if k not in taken and s >= 0.25][:k_sem]
        taken |= {r.key for r in sem_recs}
        recent = sorted((r for r in self.records.values()
                         if r.key not in taken and 0 <= chunk_index - r.last_chunk_index <= 3),
                        key=lambda r: (-r.last_chunk_index, -r.n_mentions))[:k_recent]
        return lexical, sem_recs + recent

    @staticmethod
    def render(records: List[EntityRecord]) -> str:
        lines = []
        for r in records:
            al = f" (also: {', '.join(sorted(r.aliases)[:3])})" if r.aliases else ""
            lines.append(f"- {r.canonical_name} [{r.entity_type}]{al}")
        return "\n".join(lines)

    def type_conflicts(self, ontology: "OntologySchema", min_share: float = 0.3, min_votes: int = 2) -> List[Dict]:
        out = []
        for r in self.records.values():
            tot = sum(r.type_votes.values())
            if tot < 2 * min_votes - 1 or len(r.type_votes) < 2:
                continue
            (t1, n1), (t2, n2) = r.type_votes.most_common(2)
            if n2 >= min_votes and n2 / tot >= min_share and ontology.are_disjoint(t1, t2):
                out.append({"entity": r.canonical_name, "types": {t1: n1, t2: n2}, "chunks": r.mentions[:6]})
        return out

    def snapshot(self) -> List[Dict]:
        return [{"canonical_name": r.canonical_name, "type": r.entity_type, "aliases": sorted(r.aliases),
                 "type_votes": dict(r.type_votes), "n_mentions": r.n_mentions, "first_page": r.first_page,
                 "first_chunk": r.mentions[:1], "evidence": r.evidence, "facts": r.facts,
                 "confidence": round(r.confidence, 4)} for r in self.records.values()]


@dataclass
class Context:
    """What the small model is shown around the current chunk (design doc section 3):
    DOCUMENT CONTEXT / SECTION CONTEXT / KNOWN ENTITIES / NEIGHBORING TEXT / CURRENT CHUNK."""
    header: str = ""
    known_in: List[EntityRecord] = field(default_factory=list)   # known entities literally named in the chunk
    earlier: List[EntityRecord] = field(default_factory=list)    # candidate antecedents NOT named in the chunk
    facts: str = ""

    def known_text(self) -> str:
        recs = self.known_in + self.earlier
        return EntityMemory.render(recs) if recs else ""


class ContextBuilder:
    """Retrieval BEFORE extraction (design doc section 4): turns the document map +
    entity memory into the context block the small model sees instead of 'blindly
    4096 tokens of isolated text'. Character-budgeted (~4 chars/token)."""

    def __init__(self, hier: DocumentHierarchy, memory: EntityMemory, cfg: "PipelineConfig", embedder=None):
        self.hier = hier
        self.memory = memory
        self.cfg = cfg
        self.embedder = embedder
        self._sec_ids: List[str] = []
        self._sec_emb: Optional[np.ndarray] = None

    @staticmethod
    def _fit(lines: List[str], budget: int) -> str:
        out, used = [], 0
        for ln in lines:
            if used + len(ln) + 1 > budget:
                break
            out.append(ln)
            used += len(ln) + 1
        return "\n".join(out)

    def _related_sections(self, chunk: ChunkRecord, k: int = 2) -> List[SectionRecord]:
        """Semantically related OTHER sections (by summary) -- 'previous mention -> Section 2'."""
        if self.embedder is None:
            return []
        if self._sec_emb is None:
            secs = [s for s in self.hier.sections if s.summary]
            self._sec_ids = [s.section_id for s in secs]
            self._sec_emb = (np.asarray(self.embedder.encode([f"{s.title}. {s.summary}" for s in secs],
                                                             show_progress_bar=False, normalize_embeddings=True,
                                                             convert_to_numpy=True), dtype="float32")
                             if secs else np.zeros((0, 1), dtype="float32"))
        if len(self._sec_ids) < 2:
            return []
        q = np.asarray(self.embedder.encode([chunk.text[:1500]], show_progress_bar=False,
                                            normalize_embeddings=True, convert_to_numpy=True)[0], dtype="float32")
        sims = self._sec_emb @ q
        picks = []
        for i in np.argsort(-sims):
            sid = self._sec_ids[int(i)]
            if sid != chunk.section_id and sims[int(i)] >= 0.30:
                picks.append(self.hier._sec[sid])
            if len(picks) >= k:
                break
        return picks

    def build(self, chunk: ChunkRecord, expanded: bool = False) -> Context:
        cfg = self.cfg
        if not cfg.enable_retrieval_context:
            return Context()
        budget = cfg.max_context_tokens * 4 * (2 if expanded else 1)
        lines: List[str] = []
        if self.hier.doc_summary:
            lines.append(f"DOCUMENT CONTEXT: {self.hier.doc_summary}")
        sec = self.hier.section(chunk.section_id)
        if sec is not None:
            title = f"{chunk.section} > {chunk.subsection}" if chunk.subsection else chunk.section
            parent = self.hier.parent_of(sec)
            if parent is not None and parent.summary:
                lines.append(f"SECTION CONTEXT: {chunk.section} -- {parent.summary}")
            lines.append(f"SUBSECTION CONTEXT: {title}" + (f" -- {sec.summary}" if sec.summary else "")
                         if parent is not None else
                         f"SECTION CONTEXT: {title}" + (f" -- {sec.summary}" if sec.summary else ""))
        prev = self.hier.chunk_by_id(chunk.prev_chunk) if chunk.prev_chunk else None
        if prev is not None:
            sents = split_sentences(prev.text)
            tail = " ".join(s for s, _, _ in sents[-2:]) if sents else ""
            if tail and tail[:60] not in chunk.text:
                lines.append(f"PREVIOUS TEXT: ...{tail[-320:]}")
        if expanded:
            nxt = self.hier.chunk_by_id(chunk.next_chunk) if chunk.next_chunk else None
            if nxt is not None:
                sents = split_sentences(nxt.text)
                if sents:
                    lines.append(f"NEXT TEXT: {sents[0][0][:320]}...")
        if cfg.enable_entity_memory:
            for rel in self._related_sections(chunk, k=2 if expanded else 1):
                lines.append(f"RELATED SECTION ({rel.title}): {rel.summary}")
        known_in: List[EntityRecord] = []
        earlier: List[EntityRecord] = []
        facts = ""
        if cfg.enable_entity_memory and len(self.memory):
            known_in, earlier = self.memory.retrieve(
                chunk.text, chunk.index, k_lex=cfg.max_context_entities,
                k_sem=(10 if expanded else 5), k_recent=(8 if expanded else 5))
            fact_lines = []
            for r in known_in[:6]:
                for f in r.facts[:2]:
                    if f not in fact_lines:
                        fact_lines.append(f)
            facts = "\n".join(f"- {f}" for f in fact_lines[:cfg.max_context_facts])
        return Context(header=self._fit(lines, budget), known_in=known_in, earlier=earlier, facts=facts)


# ---------------------------------------------------------------------------
# [EXTRACT] LLM client + multi-pass extraction  (design doc sections 6, 12, 13)
#
# Pass 1  entity discovery   "what entities are explicitly introduced here?"
# Pass 2  relation discovery "which explicit relations hold among THESE entities?"
#         (quote-first JSON: evidence, subject, predicate, target, modality --
#          the model writes the supporting quote BEFORE it commits to a triple;
#          subject/target are snapped to the pass-1 + memory candidate list and
#          types are LOOKED UP, not generated, so type votes can't drift)
# Pass 3  verification       claim + evidence -> YES / NO / UNSURE  (only when routed)
# Pass 4  global consistency ontology check, contradiction detection (later stages)
#
# The prompts deliberately do NOT ask for chain-of-thought (design doc section 30):
# every extra token is structured, bounded and checkable.
# ---------------------------------------------------------------------------

PREDICATE_LIST_TEXT = (
    "TYPE: is_a, instance_of, part_of, contains, includes\n"
    "AGENCY: develops, authors, issues, regulates, manages, governs\n"
    "ACTION: performs, enables, causes, requires, produces, applies_to\n"
    "RELATION: affiliated_with, located_in, related_to, influences, depends_on"
)
TYPES_TEXT = "PERSON, ORGANIZATION, CONCEPT, PRODUCT, LOCATION, EVENT, LAW, METRIC, PROCESS, DOCUMENT, TECHNOLOGY, PROVISION"
MODALITIES = ("asserted", "negated", "hypothetical", "attributed")

ENTITY_PROMPT_V2 = f"""You find the entities introduced in a text passage, for a knowledge graph.

RULES:
- Entities are short noun phrases (1-6 words) that appear in CURRENT CHUNK. Use the exact surface form (keep acronyms, versions and numbers: "CUDA 12" is not "CUDA").
- If KNOWN ENTITIES lists a canonical name for something CURRENT CHUNK refers to (also through "it", "the framework", "the company"), use the canonical name.
- Do NOT extract generic words (study, factors, this), numbers alone, citations, or whole clauses.
- evidence = a short verbatim quote (at most 25 words) copied from CURRENT CHUNK that shows the entity.
- CONTEXT lines are background only. Never extract an entity that CURRENT CHUNK does not mention or refer to.

TYPES: {TYPES_TEXT}

OUTPUT: one JSON object: {{"entities": [{{"name": ..., "type": ..., "evidence": ...}}]}}. If there are none: {{"entities": []}}

EXAMPLE
CURRENT CHUNK: "NVIDIA develops TensorRT-LLM. It supports FP8 on Hopper GPUs."
{{"entities": [
  {{"name": "NVIDIA", "type": "ORGANIZATION", "evidence": "NVIDIA develops TensorRT-LLM."}},
  {{"name": "TensorRT-LLM", "type": "TECHNOLOGY", "evidence": "NVIDIA develops TensorRT-LLM."}},
  {{"name": "FP8", "type": "TECHNOLOGY", "evidence": "It supports FP8 on Hopper GPUs."}},
  {{"name": "Hopper GPU", "type": "PRODUCT", "evidence": "It supports FP8 on Hopper GPUs."}}
]}}
"""

RELATION_PROMPT_V2 = f"""You extract relationships between entities from a text passage, for a knowledge graph.

For every relationship that CURRENT CHUNK explicitly states between the listed entities, output an object with these fields IN THIS ORDER:
  evidence  - verbatim quote (at most 30 words) from CURRENT CHUNK that states it. Write this first.
  subject   - the ACTOR/SOURCE. Must be copied from the entity lists below.
  predicate - one of the predicates below.
  target    - the RECEIVER. Must be copied from the entity lists below.
  modality  - asserted | negated | hypothetical | attributed
      asserted     = stated as fact
      negated      = the text says it is NOT the case ("does not use", "no longer", "never")
      hypothetical = conditional, planned or possible ("if", "would", "could", "may", "plans to")
      attributed   = reported as someone's claim ("according to X", "X argues that")

RULES:
- Passive voice: normalize to the active actor ("banks are regulated by RBI" -> subject=RBI, predicate=regulates, target=banks).
- "It", "the framework", "the company" etc.: use the matching entity from EARLIER ENTITIES if that is what the text refers to.
- Only relationships stated in CURRENT CHUNK. No outside knowledge. If unsure, leave it out.
- related_to only when nothing more specific fits.

PREDICATES (use ONLY these exact forms):
{PREDICATE_LIST_TEXT}

OUTPUT: one JSON object: {{"relations": [...]}}. If there are none: {{"relations": []}}

EXAMPLES
CURRENT CHUNK: "NVIDIA develops TensorRT-LLM. It requires CUDA 12 and does not support CPU-only inference. According to Smith, vLLM enables speculative decoding."
ENTITIES IN CURRENT CHUNK: NVIDIA [ORGANIZATION]; TensorRT-LLM [TECHNOLOGY]; CUDA 12 [TECHNOLOGY]; vLLM [TECHNOLOGY]; speculative decoding [PROCESS]
{{"relations": [
  {{"evidence": "NVIDIA develops TensorRT-LLM.", "subject": "NVIDIA", "predicate": "develops", "target": "TensorRT-LLM", "modality": "asserted"}},
  {{"evidence": "It requires CUDA 12", "subject": "TensorRT-LLM", "predicate": "requires", "target": "CUDA 12", "modality": "asserted"}},
  {{"evidence": "According to Smith, vLLM enables speculative decoding.", "subject": "vLLM", "predicate": "enables", "target": "speculative decoding", "modality": "attributed"}}
]}}

CURRENT CHUNK: "If NVIDIA released Triton 4, the Triton server would require CUDA 13. The RBI regulates export credit."
ENTITIES IN CURRENT CHUNK: NVIDIA [ORGANIZATION]; Triton 4 [PRODUCT]; CUDA 13 [TECHNOLOGY]; RBI [ORGANIZATION]; export credit [CONCEPT]
{{"relations": [
  {{"evidence": "If NVIDIA released Triton 4", "subject": "NVIDIA", "predicate": "develops", "target": "Triton 4", "modality": "hypothetical"}},
  {{"evidence": "The RBI regulates export credit.", "subject": "RBI", "predicate": "regulates", "target": "export credit", "modality": "asserted"}}
]}}
"""

VERIFY_PROMPT_V2 = """You are a strict fact-checker for a knowledge graph. Decide whether the EVIDENCE explicitly states the CLAIM.

YES    - the evidence says the subject (as the actor) has this relation to the target.
NO     - the evidence negates it, only hypothesizes it ("if", "would", "could"), only attributes it to someone ("according to"), reverses the direction, or does not state this relation.
UNSURE - you genuinely cannot tell.

PREDICATE MEANING: {gloss}

CLAIM: {subject} [{subject_type}] --{predicate}--> {target} [{target_type}]
EVIDENCE: "{evidence}"
{extra}
Output a JSON object: {{"verdict": "YES" | "NO" | "UNSURE"}}
JSON:"""

PROMPT_VERSION = "p2-" + hashlib.sha256(
    (ENTITY_PROMPT_V2 + RELATION_PROMPT_V2 + VERIFY_PROMPT_V2 + EXTRACTION_SYSTEM_PROMPT).encode()).hexdigest()[:10]
PARSER_VERSION = "parser-2.0"


class EntityItem(BaseModel):
    name: str = Field(min_length=1, max_length=200)
    type: _EntityTypeLiteral
    evidence: str = ""


class RelationItem(BaseModel):
    evidence: str = ""
    subject: str = Field(min_length=1, max_length=200)
    predicate: _PredicateLiteral
    target: str = Field(min_length=1, max_length=200)
    modality: Literal["asserted", "negated", "hypothetical", "attributed"] = "asserted"


def build_entity_json_schema(max_entities: int = 25, max_name_chars: int = 80, max_evidence_chars: int = 160) -> dict:
    item = {
        "type": "object",
        "properties": {
            "name": {"type": "string", "minLength": 1, "maxLength": max_name_chars},
            "type": {"type": "string", "enum": sorted(ALLOWED_ENTITY_TYPES)},
            "evidence": {"type": "string", "maxLength": max_evidence_chars},
        },
        "required": ["name", "type", "evidence"], "additionalProperties": False,
    }
    return {"type": "object",
            "properties": {"entities": {"type": "array", "maxItems": max_entities, "items": item}},
            "required": ["entities"], "additionalProperties": False}


def build_relation_json_schema(max_relations: int = 20, max_name_chars: int = 80, max_evidence_chars: int = 200) -> dict:
    item = {
        "type": "object",
        "properties": {
            "evidence": {"type": "string", "maxLength": max_evidence_chars},
            "subject": {"type": "string", "minLength": 1, "maxLength": max_name_chars},
            "predicate": {"type": "string", "enum": sorted(ALLOWED_PREDICATES)},
            "target": {"type": "string", "minLength": 1, "maxLength": max_name_chars},
            "modality": {"type": "string", "enum": list(MODALITIES)},
        },
        "required": ["evidence", "subject", "predicate", "target", "modality"], "additionalProperties": False,
    }
    return {"type": "object",
            "properties": {"relations": {"type": "array", "maxItems": max_relations, "items": item}},
            "required": ["relations"], "additionalProperties": False}


VERDICT_SCHEMA = {"type": "object",
                  "properties": {"verdict": {"type": "string", "enum": ["YES", "NO", "UNSURE"]}},
                  "required": ["verdict"], "additionalProperties": False}


def iter_array_objects(text: str, key: str) -> Iterator[Tuple[dict, int, int]]:
    """Yields (object, start, end) for every COMPLETE JSON object inside `{"key": [ ... ]}`,
    including from truncated output (v1 lost the whole chunk when the completion hit
    max_tokens mid-array). The char spans let us attribute token logprobs per object."""
    m = re.search(r'"%s"\s*:\s*\[' % re.escape(key), text)
    if not m:
        return
    dec = json.JSONDecoder()
    i, n = m.end(), len(text)
    while i < n:
        while i < n and text[i] in " \t\r\n,":
            i += 1
        if i >= n or text[i] == "]":
            return
        try:
            obj, j = dec.raw_decode(text, i)
        except json.JSONDecodeError:
            return
        if isinstance(obj, dict):
            yield obj, i, j
        i = j


_VALUE_SPAN_CACHE: Dict[str, "re.Pattern"] = {}


def _value_span(text: str, start: int, end: int, field_name: str) -> Optional[Tuple[int, int]]:
    pat = _VALUE_SPAN_CACHE.setdefault(
        field_name, re.compile(r'"%s"\s*:\s*"((?:[^"\\]|\\.)*)"' % re.escape(field_name)))
    m = pat.search(text, start, end)
    return (m.start(1), m.end(1)) if m else None


def span_mean_logprob(lp: Optional[dict], a: int, b: int) -> Optional[float]:
    """Mean token logprob over the generated tokens overlapping chars [a, b)."""
    if not lp:
        return None
    offs, vals = lp.get("text_offset"), lp.get("token_logprobs")
    if not offs or not vals or len(offs) != len(vals):
        return None
    lo = max(0, bisect.bisect_right(offs, a) - 1)
    hi = bisect.bisect_left(offs, b)
    picked = [v for v in vals[lo:hi] if v is not None]
    return float(sum(picked) / len(picked)) if picked else None


def object_confidence(text: str, span: Tuple[int, int], fields: Tuple[str, ...], lp: Optional[dict],
                      scale: float, fallback: float) -> float:
    """Per-CLAIM confidence = exp(scale * mean logprob of the tokens inside the
    subject/predicate/target(/modality) VALUES of that JSON object). v1 gave every
    triple in a chunk the same number (exp(8 * chunk-average logprob)) -- the JSON
    scaffolding tokens dominate that average, and nothing could be routed per claim.
    Uncalibrated: kg_eval.fit_calibrator() maps it to P(correct) once you have gold labels."""
    vals = []
    for f in fields:
        sp = _value_span(text, span[0], span[1], f)
        if sp:
            m = span_mean_logprob(lp, sp[0], sp[1])
            if m is not None:
                vals.append(m)
    if not vals:
        return fallback
    return float(min(1.0, math.exp(scale * (sum(vals) / len(vals)))))


class LLMClient:
    """Thin batched client over the vLLM OpenAI-compatible endpoint, shared by every
    pass so token usage, call counts and failures are counted in ONE place."""

    def __init__(self, extractor: "VLLMTripleExtractor", ledger: ClaimLedger):
        self.x = extractor
        self.ledger = ledger

    def complete(self, prompts: List[str], schema: Optional[dict] = None, max_tokens: int = 700,
                 temperature: float = 0.1, span_name: str = "llm_batch",
                 stop: Optional[List[str]] = None) -> List[Dict]:
        """One HTTP call for the whole prompt list (vLLM continuous batching). Returns
        [{'text', 'logprobs'}] aligned with `prompts`; failures yield empty text."""
        if not prompts:
            return []
        payload = {
            "model": self.x.model_name, "prompt": prompts, "max_tokens": max_tokens,
            "temperature": temperature, "top_p": 0.9, "logprobs": 1,
            "repetition_penalty": self.x.config.repetition_penalty,
        }
        if schema is not None:
            payload["structured_outputs"] = {"json": schema}
        if stop:
            payload["stop"] = stop
        last_err: Optional[Exception] = None
        for _attempt in range(2):
            try:
                with nvtx_range("vllm_http_call", color="yellow"):
                    resp = self.x._requests.post(f"{self.x.base_url}/v1/completions", json=payload,
                                                 timeout=self.x.request_timeout_s)
                    resp.raise_for_status()
                data = resp.json()
                usage = data.get("usage") or {}
                self.ledger.counters["llm_calls"] += 1
                self.ledger.counters["prompt_tokens"] += int(usage.get("prompt_tokens", 0))
                self.ledger.counters["completion_tokens"] += int(usage.get("completion_tokens", 0))
                choices = sorted(data["choices"], key=lambda c: c.get("index", 0))
                out = [{"text": c.get("text", ""), "logprobs": c.get("logprobs")} for c in choices]
                if len(out) != len(prompts):
                    out = (out + [{"text": "", "logprobs": None}] * len(prompts))[:len(prompts)]
                return out
            except Exception as e:   # network / 5xx / 400 (context too long): count it, never crash the run
                last_err = e
                time.sleep(1.0)
        self.ledger.counters["llm_errors"] += 1
        print(f"[EXTRACT] {span_name}: request failed after retry ({last_err}) -- {len(prompts)} prompt(s) skipped.")
        return [{"text": "", "logprobs": None} for _ in prompts]


class ClaimExtractor:
    def __init__(self, extractor: "VLLMTripleExtractor", cfg: "PipelineConfig", memory: EntityMemory,
                 llm: LLMClient):
        self.cfg = cfg
        self.memory = memory
        self.llm = llm
        self.entity_schema = build_entity_json_schema(max_name_chars=cfg.max_entity_chars)
        self.relation_schema = build_relation_json_schema(max_relations=cfg.max_triples_per_chunk,
                                                         max_name_chars=cfg.max_entity_chars)
        self.triple_schema = build_triple_json_schema(max_entity_chars=cfg.max_entity_chars,
                                                      max_triples_per_chunk=cfg.max_triples_per_chunk)

    # ------------------------------------------------------------------ pass 1
    @nvtx_range("pass1_entities", color="green")
    def discover_entities(self, chunks: List[ChunkRecord], ctxs: List[Context],
                          stats: Dict[str, int]) -> List[List[Tuple[str, str, str]]]:
        prompts = []
        for ch, ctx in zip(chunks, ctxs):
            known = ctx.known_text()
            pre = (ctx.header + "\n" if ctx.header else "") + (f"KNOWN ENTITIES (canonical names):\n{known}\n" if known else "")
            prompts.append(f"{ENTITY_PROMPT_V2}\n{pre}\nCURRENT CHUNK:\n{ch.text}\n\nJSON:")
        outs = self.llm.complete(prompts, schema=self.entity_schema, max_tokens=self.cfg.entity_max_tokens,
                                 span_name="pass1_entities")
        result: List[List[Tuple[str, str, str]]] = []
        for ch, out in zip(chunks, outs):
            seen: Set[str] = set()
            ents: List[Tuple[str, str, str]] = []
            for obj, _a, _b in iter_array_objects(out["text"], "entities"):
                try:
                    item = EntityItem(**obj)
                except ValidationError:
                    stats["dropped_entity_schema"] = stats.get("dropped_entity_schema", 0) + 1
                    continue
                name = item.name.strip()
                if not _valid_entity_text(name, self.cfg.max_entity_chars, self.cfg.max_entity_words):
                    stats["dropped_entity_shape"] = stats.get("dropped_entity_shape", 0) + 1
                    continue
                if mention_score(name, ch.text) < 85.0 and not self.memory.knows(name):
                    stats["dropped_entity_not_in_chunk"] = stats.get("dropped_entity_not_in_chunk", 0) + 1
                    continue
                nk = normalize_entity_label(name)
                if nk in seen:
                    continue
                seen.add(nk)
                ents.append((name, item.type, item.evidence.strip()))
            result.append(ents)
        return result

    # ------------------------------------------------------------------ pass 2
    @staticmethod
    def _candidates(ents: List[Tuple[str, str]], known_in: List[EntityRecord],
                    earlier: List[EntityRecord]) -> Dict[str, Tuple[str, str, bool]]:
        """normalized name -> (canonical display name, type, from_earlier_context)."""
        cands: Dict[str, Tuple[str, str, bool]] = {}
        for name, typ in ents:
            cands[normalize_entity_label(name)] = (name, typ, False)
        for rec in known_in:      # memory-known entities literally in the chunk that pass 1 missed
            for nm in [rec.canonical_name] + sorted(rec.aliases):
                cands.setdefault(normalize_entity_label(nm), (rec.canonical_name, rec.entity_type, False))
        for rec in earlier:
            for nm in [rec.canonical_name] + sorted(rec.aliases):
                cands.setdefault(normalize_entity_label(nm), (rec.canonical_name, rec.entity_type, True))
        return cands

    @staticmethod
    def _snap(name: str, cands: Dict[str, Tuple[str, str, bool]]) -> Optional[Tuple[str, str, bool]]:
        k = normalize_entity_label(name)
        if k in cands:
            return cands[k]
        sig = _numeric_signature(k)
        hit = rf_process.extractOne(k, list(cands), scorer=fuzz.WRatio, score_cutoff=88)
        if hit and _numeric_signature(hit[0]) == sig:
            return cands[hit[0]]
        return None

    @nvtx_range("pass2_relations", color="green")
    def extract_relations(self, chunks: List[ChunkRecord], ctxs: List[Context],
                          entity_lists: List[List[Tuple[str, str]]], stats: Dict[str, int],
                          temperature: float = 0.1) -> List[List[Claim]]:
        prompts: List[Optional[str]] = []
        cand_maps: List[Dict[str, Tuple[str, str, bool]]] = []
        for ch, ctx, ents in zip(chunks, ctxs, entity_lists):
            in_keys = {normalize_entity_label(n) for n, _ in ents}
            earlier = [r for r in ctx.earlier if r.key not in in_keys]
            cands = self._candidates(ents, ctx.known_in, earlier)
            cand_maps.append(cands)
            if len(cands) < 2 or not ents:
                prompts.append(None)
                continue
            ent_names = [n for n, _ in ents]
            ent_lines = "; ".join(f"{n} [{t}]" for n, t in ents)
            extra_known = "; ".join(f"{r.canonical_name} [{r.entity_type}]" for r in ctx.known_in
                                    if r.canonical_name not in ent_names)
            if extra_known:
                ent_lines += "; " + extra_known
            earlier_lines = "; ".join(f"{r.canonical_name} [{r.entity_type}]" for r in earlier) or "(none)"
            facts = f"KNOWN FACTS (already established, do not repeat unless restated):\n{ctx.facts}\n" if ctx.facts else ""
            prompts.append(
                f"{RELATION_PROMPT_V2}\n{ctx.header + chr(10) if ctx.header else ''}{facts}"
                f"CURRENT CHUNK:\n{ch.text}\n\n"
                f"ENTITIES IN CURRENT CHUNK: {ent_lines}\nEARLIER ENTITIES (only for resolving references): {earlier_lines}\n\nJSON:")
        live = [i for i, p in enumerate(prompts) if p is not None]
        outs = self.llm.complete([prompts[i] for i in live], schema=self.relation_schema,
                                 max_tokens=self.cfg.claim_max_tokens, temperature=temperature,
                                 span_name="pass2_relations")
        claims_per_chunk: List[List[Claim]] = [[] for _ in chunks]
        for i, out in zip(live, outs):
            text, lp = out["text"], out["logprobs"]
            cands = cand_maps[i]
            for obj, a, b in iter_array_objects(text, "relations"):
                try:
                    item = RelationItem(**obj)
                except ValidationError:
                    stats["dropped_schema_invalid"] = stats.get("dropped_schema_invalid", 0) + 1
                    continue
                s_hit, t_hit = self._snap(item.subject, cands), self._snap(item.target, cands)
                if s_hit is None or t_hit is None:
                    stats["dropped_unknown_entity"] = stats.get("dropped_unknown_entity", 0) + 1
                    continue
                if normalize_entity_label(s_hit[0]) == normalize_entity_label(t_hit[0]):
                    stats["dropped_self_loop"] = stats.get("dropped_self_loop", 0) + 1
                    continue
                conf = object_confidence(text, (a, b), ("subject", "predicate", "target", "modality"), lp,
                                         self.cfg.claim_logprob_scale, 0.6)
                claims_per_chunk[i].append(Claim(
                    subject=s_hit[0], subject_type=s_hit[1], predicate=normalize_predicate(item.predicate),
                    target=t_hit[0], target_type=t_hit[1], confidence=conf, raw_confidence=conf,
                    evidence=item.evidence.strip(), modality=item.modality, coref=(s_hit[2] or t_hit[2])))
        return claims_per_chunk

    # ------------------------------------------- single pass (v1-prompt baseline)
    @nvtx_range("single_pass", color="green")
    def extract_single(self, chunks: List[ChunkRecord], ctxs: List[Context], stats: Dict[str, int],
                       temperature: float = 0.1) -> List[List[Claim]]:
        prompts = []
        for ch, ctx in zip(chunks, ctxs):
            if ctx.header:
                prompts.append(f"{EXTRACTION_SYSTEM_PROMPT}\n\nCONTEXT (background only -- extract ONLY from "
                               f"CURRENT CHUNK):\n{ctx.header}\n\nCURRENT CHUNK:\n{ch.text}\n\nJSON:")
            else:
                prompts.append(f"{EXTRACTION_SYSTEM_PROMPT}\n\n{ch.text}\n\nJSON:")
        outs = self.llm.complete(prompts, schema=self.triple_schema, max_tokens=self.cfg.max_output_tokens,
                                 temperature=temperature, span_name="single_pass")
        result: List[List[Claim]] = []
        for ch, out in zip(chunks, outs):
            text, lp = out["text"], out["logprobs"]
            vals = [v for v in ((lp or {}).get("token_logprobs") or []) if v is not None]
            fallback = (float(min(1.0, math.exp(self.cfg.logprob_calibration_scale * (sum(vals) / len(vals)))))
                        if vals else 0.75)
            claims: List[Claim] = []
            for obj, a, b in iter_array_objects(text, "triples"):
                try:
                    t = TripleModel(**obj)
                except ValidationError:
                    stats["dropped_schema_invalid"] = stats.get("dropped_schema_invalid", 0) + 1
                    continue
                s, tg = t.subject.strip(), t.target.strip()
                if self.cfg.strict_field_validation and not (
                        _valid_entity_text(s, self.cfg.max_entity_chars, self.cfg.max_entity_words)
                        and _valid_entity_text(tg, self.cfg.max_entity_chars, self.cfg.max_entity_words)):
                    stats["dropped_entity_shape"] = stats.get("dropped_entity_shape", 0) + 1
                    continue
                conf = object_confidence(text, (a, b), ("subject", "predicate", "target"), lp,
                                         self.cfg.claim_logprob_scale, fallback)
                claims.append(Claim(subject=s, subject_type=t.subject_type, predicate=normalize_predicate(t.predicate),
                                    target=tg, target_type=t.target_type, confidence=conf, raw_confidence=conf))
            result.append(claims)
        return result

    # ------------------------------------------------------------------ dispatch
    def extract(self, chunks: List[ChunkRecord], ctxs: List[Context], stats: Dict[str, int],
                temperature: float = 0.1,
                entity_lists: Optional[List[List[Tuple[str, str, str]]]] = None,
                ) -> Tuple[List[List[Claim]], List[List[Tuple[str, str, str]]]]:
        """Returns (claims per chunk, pass-1 entities per chunk). Pass `entity_lists` to reuse
        a previous pass 1 (self-consistency re-runs only repeat pass 2)."""
        if not self.cfg.enable_multipass_extraction:
            return self.extract_single(chunks, ctxs, stats, temperature), [[] for _ in chunks]
        if entity_lists is None:
            entity_lists = self.discover_entities(chunks, ctxs, stats)
        claims = self.extract_relations(chunks, ctxs, [[(n, t) for n, t, _ in ents] for ents in entity_lists],
                                        stats, temperature)
        return claims, entity_lists


def consensus_claims(runs: List[List[Claim]]) -> List[Claim]:
    """Beta-Binomial agreement vote over N sampled extractions of ONE chunk (same math as
    v1's _consensus_from_runs: Beta(2,1) prior, keep a claim only if the 95% lower bound
    of its agreement rate is >= 0.5). Keeps the highest-confidence instance."""
    from scipy.stats import beta as beta_dist
    n = len(runs)
    counts: Counter = Counter()
    best: Dict[Tuple, Claim] = {}
    for run in runs:
        seen_in_run: Set[Tuple] = set()
        for c in run:
            k = c.key()
            if k in seen_in_run:
                continue
            seen_in_run.add(k)
            counts[k] += 1
            if k not in best or c.confidence > best[k].confidence:
                best[k] = c
    out = []
    for k, cnt in counts.items():
        a, b = 2.0 + cnt, 1.0 + (n - cnt)
        if float(beta_dist.ppf(0.05, a, b)) >= 0.5:
            c = best[k]
            c.confidence = float(a / (a + b)) * c.confidence ** 0.5
            out.append(c)
    return out


# ---------------------------------------------------------------------------
# [VERIFY] verifier/judge + adaptive compute  (design doc sections 12, 13)
#
#   easy    extract -> validate -> accept
#   medium  extract -> verify (YES/NO/UNSURE on ONE claim + its evidence) -> accept
#   hard    extract -> retrieve more context -> verify -> ontology check -> judge
#           (and low-yield chunks get self-consistency re-extraction)
#
# The 3B model acts as a JUDGE here, not just an extractor, and the judge only ever
# sees ONE candidate and ~a sentence of evidence -- never the whole KG.
# ---------------------------------------------------------------------------

class ClaimVerifier:
    def __init__(self, llm: LLMClient, cfg: "PipelineConfig", ontology: Optional[OntologySchema] = None):
        self.llm = llm
        self.cfg = cfg
        self.ontology = ontology or ONTOLOGY

    def prompt(self, c: Claim, evidence: str, extra: str = "") -> str:
        return VERIFY_PROMPT_V2.format(
            gloss=self.ontology.gloss(c.predicate) or c.predicate, subject=c.subject, subject_type=c.subject_type,
            predicate=c.predicate, target=c.target, target_type=c.target_type,
            evidence=evidence.replace('"', "'")[:1200], extra=(extra + "\n") if extra else "")

    @nvtx_range("pass3_verify", color="purple")
    def verify(self, items: List[Tuple[Claim, str, str]]) -> List[Tuple[str, float]]:
        """items: (claim, evidence_text, extra_context) -> [(verdict, confidence)]."""
        if not items:
            return []
        prompts = [self.prompt(c, ev, ex) for c, ev, ex in items]
        outs: List[Dict] = []
        for i in range(0, len(prompts), 32):
            outs.extend(self.llm.complete(prompts[i:i + 32], schema=VERDICT_SCHEMA, max_tokens=16,
                                          temperature=0.0, span_name="pass3_verify"))
        self.llm.ledger.counters["verifier_calls"] += len(items)
        res: List[Tuple[str, float]] = []
        for out in outs:
            text = out["text"]
            m = re.search(r'"verdict"\s*:\s*"(YES|NO|UNSURE)"', text)
            if not m:
                res.append(("UNSURE", 0.0))
                continue
            conf = 0.5
            sp = _value_span(text, 0, len(text), "verdict")
            if sp:
                lp = span_mean_logprob(out["logprobs"], sp[0], sp[1])
                if lp is not None:
                    conf = float(min(1.0, math.exp(lp)))
            res.append((m.group(1), conf))
        return res


def route_claim(c: Claim, cfg: "PipelineConfig") -> str:
    """Adaptive compute: 'accept' (easy) | 'verify' (medium) | 'escalate' (hard)."""
    if not cfg.enable_verifier:
        return "accept"
    if not cfg.enable_adaptive_routing:
        return "verify"           # fixed budget: verify everything (the non-adaptive comparison point)
    hard = (c.confidence < cfg.route_hard_conf or c.direction_repaired or c.coref
            or c.evidence_score < cfg.route_hard_evidence)
    if hard:
        return "escalate"
    easy = (c.confidence >= cfg.route_easy_conf and c.evidence_source == "model"
            and c.evidence_score >= 0.9 and c.modality == "asserted")
    return "accept" if easy else "verify"


class ClaimJudge:
    """Evidence check -> memory types -> modality/ontology gates -> route -> verify/escalate
    -> final ACCEPT / REVIEW / REJECT, with a reason on every non-ACCEPT. (design doc 6/12/13)"""

    def __init__(self, cfg: "PipelineConfig", memory: EntityMemory, checker: EvidenceChecker,
                 verifier: ClaimVerifier, ontology: Optional[OntologySchema] = None):
        self.cfg = cfg
        self.memory = memory
        self.checker = checker
        self.verifier = verifier
        self.ontology = ontology or ONTOLOGY

    def _apply_memory_types(self, c: Claim):
        """Entity type is a property of the ENTITY, not of one mention: when memory has >=2
        votes for an endpoint, use the majority type."""
        if not self.cfg.enable_entity_memory:
            return
        for side in ("subject", "target"):
            rec = self.memory.lookup(getattr(c, side))
            if rec is not None and sum(rec.type_votes.values()) >= 2:
                mt = rec.entity_type
                if mt != getattr(c, f"{side}_type"):
                    setattr(c, f"{side}_type", mt)
                    c.retyped = True

    def _gate(self, c: Claim) -> Optional[Tuple[str, str]]:
        cfg = self.cfg
        if cfg.enable_evidence_check and cfg.require_evidence and not c.evidence_grounded:
            return ("REJECT", "ungrounded")
        if c.modality == "negated":
            return ("REJECT", "negated")
        if c.modality == "hypothetical":
            return ("REJECT", "hypothetical")
        if c.modality == "attributed":
            return (cfg.attributed_action.upper(), "attributed")
        if cfg.enable_ontology_constraints:
            v = self.ontology.check(c.subject_type, c.predicate, c.target_type)
            if v and self.ontology.direction_repair_ok(c.subject_type, c.predicate, c.target_type):
                c.subject, c.target = c.target, c.subject
                c.subject_type, c.target_type = c.target_type, c.subject_type
                c.direction_repaired = True
                v = self.ontology.check(c.subject_type, c.predicate, c.target_type)
            c.ontology_violations = v
            if v:
                hard = any(x.startswith(HARD_VIOLATION_PREFIXES) for x in v)
                if hard or cfg.ontology_violation_action == "reject":
                    return ("REJECT", f"ontology:{v[0]}")
                if cfg.ontology_violation_action == "review":
                    return ("REVIEW", f"ontology:{v[0]}")
                # "keep": flagged in the ledger, still eligible for the graph
        else:
            if not sitl_filter([c.as_tuple()], stats={}):      # v1 behaviour
                return ("REJECT", "sitl")
        return None

    def _run_verifier(self, items: List[Tuple[ChunkRecord, Claim]], expanded: bool,
                      builder: Optional[ContextBuilder]):
        if not items:
            return
        batch: List[Tuple[Claim, str, str]] = []
        for chunk, c in items:
            ev, extra = c.evidence, ""
            if expanded:
                ev = self.checker.window(c, chunk.text, radius=1)
                lines: List[str] = []
                ctx = builder.build(chunk, expanded=True) if builder is not None else Context()
                if ctx.header:
                    lines.append("CONTEXT:\n" + ctx.header)
                for nm in (c.subject, c.target):
                    rec = self.memory.lookup(nm)
                    if rec is not None and rec.aliases:
                        lines.append(f"NOTE: {rec.canonical_name} is also called {', '.join(sorted(rec.aliases)[:3])}.")
                if c.coref:
                    lines.append("NOTE: one endpoint is referred to indirectly (\"it\", \"the framework\"); "
                                 "judge using the context above.")
                extra = "\n".join(lines)
            batch.append((c, ev, extra))
        for (chunk, c), (_cc, ev, _ex), (verdict, vconf) in zip(items, batch, self.verifier.verify(batch)):
            c.verifier, c.verifier_conf = verdict, vconf
            if expanded and verdict == "YES" and len(ev) <= 900:
                c.evidence = ev          # provenance shows the text the claim was actually verified against

    def _finalize(self, c: Claim):
        cfg = self.cfg
        if c.verdict != "PENDING":
            return
        if c.verifier == "NO":
            c.verdict, c.reject_reason = "REJECT", "verifier_no"
        elif c.verifier == "UNSURE":
            c.verdict, c.reject_reason = "REVIEW", "verifier_unsure"
        elif c.direction_repaired and c.verifier != "YES":
            c.verdict, c.reject_reason = "REVIEW", "direction_repaired_unverified"
        else:
            if c.verifier == "YES":
                c.confidence = max(c.confidence, cfg.verified_conf_floor)
            if c.confidence < cfg.confidence_filter:
                c.verdict, c.reject_reason = "REJECT", "low_confidence"
            else:
                c.verdict = "ACCEPT"

    @nvtx_range("judge_claims", color="purple")
    def judge(self, items: List[Tuple[ChunkRecord, Claim]], builder: Optional[ContextBuilder] = None):
        pending: List[Tuple[ChunkRecord, Claim]] = []
        for chunk, c in items:
            self.checker.check(c, chunk, self.memory)
            self._apply_memory_types(c)
            g = self._gate(c)
            if g is not None:
                c.verdict, c.reject_reason = g
                continue
            pending.append((chunk, c))
        for _chunk, c in pending:
            c.route = route_claim(c, self.cfg)
        medium = [(ch, c) for ch, c in pending if c.route == "verify"]
        hard = [(ch, c) for ch, c in pending if c.route == "escalate"]
        self._run_verifier(medium, expanded=False, builder=builder)
        promoted = [(ch, c) for ch, c in medium if c.verifier == "UNSURE"]
        for _ch, c in promoted:
            c.route = "escalate"          # second chance with retrieved context, counted as hard-path compute
        self._run_verifier(hard + promoted, expanded=True, builder=builder)
        for _chunk, c in pending:
            self._finalize(c)

    def judge_cooccurrence(self, chunk: ChunkRecord, claims: List[Claim]):
        """v1's enhance_connectivity() `related_to` edges: kept only if both endpoints share a real sentence."""
        for c in claims:
            self.checker.check(c, chunk, None)
            if c.evidence_grounded and c.evidence_source == "retrieved":
                c.verdict = "ACCEPT"
            else:
                c.verdict, c.reject_reason = "REJECT", "cooccurrence_no_shared_sentence"


# ---------------------------------------------------------------------------
# [CONTRADICT] contradiction detection  (design doc section 22)
# "Section 2: CUDA 12 / Section 9: CUDA 13 -- don't silently merge them."
# Both claims, both evidence quotes, both pages are preserved; nothing is deleted.
# ---------------------------------------------------------------------------

@dataclass
class Contradiction:
    contradiction_id: str
    kind: str                 # VERSION_CONFLICT | NEGATION_CONFLICT | DIRECTION_CONFLICT | FUNCTIONAL_CONFLICT | TYPE_CONFLICT
    subject: str
    predicate: str
    note: str
    claim_a: Dict = field(default_factory=dict)
    claim_b: Dict = field(default_factory=dict)
    resolution: str = "preserved_both"

    def to_dict(self) -> Dict:
        return asdict(self)


def _claim_ref(c: Claim) -> Dict:
    return {"claim_id": c.claim_id, "subject": c.subject, "predicate": c.predicate, "target": c.target,
            "evidence": c.evidence, "page": c.page, "chunk_id": c.chunk_id, "confidence": round(c.confidence, 4),
            "modality": c.modality, "verdict": c.verdict}


def _label_stem(label: str) -> str:
    return _WHITESPACE_RE.sub(" ", _NUM_TOKEN_RE.sub("", normalize_entity_label(label))).strip()


class ContradictionDetector:
    def __init__(self, ontology: Optional[OntologySchema] = None):
        self.ontology = ontology or ONTOLOGY

    @nvtx_range("contradictions", color="red")
    def run(self, claims: List[Claim], memory: Optional[EntityMemory] = None) -> List[Contradiction]:
        found: List[Contradiction] = []

        def add(kind, subject, predicate, note, a: Optional[Claim], b: Optional[Claim]):
            cid = f"X{len(found) + 1}"
            found.append(Contradiction(cid, kind, subject, predicate, note,
                                       _claim_ref(a) if a else {}, _claim_ref(b) if b else {}))
            for c in (a, b):
                if c is not None and cid not in c.contradiction_ids:
                    c.contradiction_ids.append(cid)

        acc = [c for c in claims if c.verdict == "ACCEPT" and c.source == "llm"]

        by_sp: Dict[Tuple[str, str], List[Claim]] = defaultdict(list)
        for c in acc:
            by_sp[(normalize_entity_label(c.subject), c.predicate)].append(c)
        for (_s, p), group in by_sp.items():
            seen_pairs: Set[Tuple[str, str]] = set()
            for i in range(len(group)):
                for j in range(i + 1, len(group)):
                    a, b = group[i], group[j]
                    ta, tb = normalize_entity_label(a.target), normalize_entity_label(b.target)
                    if ta == tb or (ta, tb) in seen_pairs:
                        continue
                    sa, sb = _numeric_signature(ta), _numeric_signature(tb)
                    if sa and sb and sa != sb and _label_stem(ta) == _label_stem(tb) and _label_stem(ta):
                        seen_pairs.add((ta, tb))
                        add("VERSION_CONFLICT", a.subject, p, f"{a.target!r} (p.{a.page}) vs {b.target!r} (p.{b.page})", a, b)
            if self.ontology.is_functional(p):
                targets = {normalize_entity_label(c.target) for c in group}
                if len(targets) > 1:
                    add("FUNCTIONAL_CONFLICT", group[0].subject, p,
                        f"{p} is functional but has {len(targets)} distinct targets", group[0], group[1])

        acc_by_key = {c.key(): c for c in acc}
        for c in claims:
            if c.modality == "negated" and c.evidence_grounded and c.source == "llm":
                a = acc_by_key.get(c.key())
                if a is not None:
                    add("NEGATION_CONFLICT", a.subject, a.predicate,
                        f"asserted on p.{a.page} but negated on p.{c.page}", a, c)

        done: Set[frozenset] = set()
        for k, c in acc_by_key.items():
            s, p, t = k
            rev = acc_by_key.get((t, p, s))
            if rev is not None and not self.ontology.is_symmetric(p) and frozenset((k, (t, p, s))) not in done:
                done.add(frozenset((k, (t, p, s))))
                add("DIRECTION_CONFLICT", c.subject, p, f"both directions asserted ({c.subject} <-> {c.target})", c, rev)

        if memory is not None:
            for tc in memory.type_conflicts(self.ontology):
                add("TYPE_CONFLICT", tc["entity"], "", f"typed as disjoint types {tc['types']}", None, None)
        return found


# ---------------------------------------------------------------------------
# [REPORT] provenance, run manifest, graph quality, audit artifacts
# (design doc sections 20, 21, 26)
# ---------------------------------------------------------------------------

@dataclass
class RunManifest:
    run_id: str
    created_at: str
    pipeline_version: str
    model_version: str
    prompt_version: str
    ontology_version: str
    dataset_version: str
    parser_version: str
    embedding_version: str
    threshold: float
    hardware: str
    config: Dict

    @staticmethod
    def build(cfg: "PipelineConfig", model_name: str = "", dataset_version: str = "adhoc") -> "RunManifest":
        try:
            hw = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
        except Exception:
            hw = "unknown"
        cfg_dict = {k: (v if isinstance(v, (int, float, str, bool, type(None))) else str(v))
                    for k, v in asdict(cfg).items()}
        return RunManifest(
            run_id=hashlib.sha1(f"{time.time()}-{os.getpid()}".encode()).hexdigest()[:10],
            created_at=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            pipeline_version="kg-pipeline-v2.0", model_version=model_name or cfg.model_name,
            prompt_version=PROMPT_VERSION, ontology_version=ONTOLOGY.version, dataset_version=dataset_version,
            parser_version=PARSER_VERSION, embedding_version=cfg.ann_embedding_model, threshold=cfg.confidence_filter,
            hardware=hw, config=cfg_dict)

    def to_dict(self) -> Dict:
        return asdict(self)

    def wandb_config(self) -> Dict:
        d = self.to_dict()
        d.pop("config", None)
        return d


PROVENANCE_COLUMNS = ["evidence", "evidence_source", "evidence_score", "page", "chunk_id", "doc_id", "verifier",
                      "n_supporting_claims", "claim_ids", "pages", "in_contradiction"]


def attach_provenance(edges_pl: pl.DataFrame, claims: List[Claim], canon_map: Dict[str, str],
                      dup_map: Dict[int, int], manifest: Optional[RunManifest] = None) -> pl.DataFrame:
    """Joins the ledger's ACCEPTED claims onto the FINAL resolved edges. Resolution
    (fuzzy label merge, then ANN duplicate merge) renames node ids, so each claim's labels are
    pushed through the SAME canon_map / dup_map to find the edge it ended up supporting."""
    def fid(label: str) -> int:
        i = canonical_entity_id(canon_map.get(label, label))
        return int(dup_map.get(i, i))

    groups: Dict[Tuple[int, str, int], List[Claim]] = defaultdict(list)
    for c in claims:
        if c.verdict == "ACCEPT" and c.source in ("llm", "cooccurrence"):
            groups[(fid(c.subject), c.predicate, fid(c.target))].append(c)

    rows: Dict[str, List] = {k: [] for k in ["src_id", "dst_id", "predicate_str"] + PROVENANCE_COLUMNS}
    for key, cs in groups.items():
        best = max(cs, key=lambda c: (c.evidence_source == "model", c.confidence))
        rows["src_id"].append(key[0])
        rows["dst_id"].append(key[2])
        rows["predicate_str"].append(key[1])
        rows["evidence"].append(best.evidence)
        rows["evidence_source"].append(best.evidence_source)
        rows["evidence_score"].append(float(best.evidence_score))
        rows["page"].append(int(best.page))
        rows["chunk_id"].append(best.chunk_id)
        rows["doc_id"].append(best.doc_id)
        rows["verifier"].append(best.verifier)
        rows["n_supporting_claims"].append(len(cs))
        rows["claim_ids"].append("|".join(c.claim_id for c in cs[:10]))
        rows["pages"].append(",".join(str(p) for p in sorted({c.page for c in cs})[:10]))
        rows["in_contradiction"].append(any(c.contradiction_ids for c in cs))
    schema = {"src_id": pl.Int64, "dst_id": pl.Int64, "predicate_str": pl.Utf8, "evidence": pl.Utf8,
              "evidence_source": pl.Utf8, "evidence_score": pl.Float64, "page": pl.Int64, "chunk_id": pl.Utf8,
              "doc_id": pl.Utf8, "verifier": pl.Utf8, "n_supporting_claims": pl.Int64, "claim_ids": pl.Utf8,
              "pages": pl.Utf8, "in_contradiction": pl.Boolean}
    prov = pl.DataFrame(rows, schema=schema)
    out = edges_pl.join(prov, on=["src_id", "dst_id", "predicate_str"], how="left")
    out = out.with_columns([pl.col("n_supporting_claims").fill_null(0), pl.col("in_contradiction").fill_null(False)])
    if manifest is not None:
        out = out.with_columns([
            pl.lit(manifest.model_version).alias("model_version"),
            pl.lit(manifest.prompt_version).alias("prompt_version"),
            pl.lit(manifest.ontology_version).alias("ontology_version"),
            pl.lit(manifest.created_at).alias("created_at"),
        ])
    return out


def graph_quality_metrics(edges_pl: pl.DataFrame, ontology: Optional[OntologySchema] = None) -> Dict:
    """Graph-level evaluation (design doc section 20): size, duplicate node/edge rate,
    connected components, degree distribution, relation/type distribution, ontology
    violation rate. Pure CPU, no cugraph needed."""
    ontology = ontology or ONTOLOGY
    if len(edges_pl) == 0:
        return {"nodes": 0, "edges": 0}
    ids = set(edges_pl["src_id"].to_list()) | set(edges_pl["dst_id"].to_list())
    label_of: Dict[int, str] = {}
    for i, l in zip(edges_pl["src_id"].to_list(), edges_pl["source_str"].to_list()):
        label_of.setdefault(i, l)
    for i, l in zip(edges_pl["dst_id"].to_list(), edges_pl["target_str"].to_list()):
        label_of.setdefault(i, l)

    parent = {i: i for i in ids}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    deg: Counter = Counter()
    for a, b in zip(edges_pl["src_id"].to_list(), edges_pl["dst_id"].to_list()):
        deg[a] += 1
        deg[b] += 1
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
    n_comp = len({find(i) for i in ids})
    comp_sizes = Counter(find(i) for i in ids)

    labels = [label_of[i] for i in ids]
    norm = [_WHITESPACE_RE.sub("", re.sub(r"[^\w\s]", "", l.lower())) for l in labels]
    dup_nodes: Set[int] = set()
    seen_norm: Dict[str, int] = {}
    for idx, k in enumerate(norm):
        if k in seen_norm:
            dup_nodes.update((idx, seen_norm[k]))
        else:
            seen_norm[k] = idx
    if len(labels) <= 2500:
        sc = rfcdist(labels, labels, scorer=fuzz.token_sort_ratio, score_cutoff=95)
        sigs = [_numeric_signature(l) for l in labels]
        for i in range(len(labels)):
            for j in range(i + 1, len(labels)):
                if sc[i][j] >= 95 and sigs[i] == sigs[j]:
                    dup_nodes.update((i, j))

    triples = set(zip(edges_pl["src_id"].to_list(), edges_pl["predicate_str"].to_list(), edges_pl["dst_id"].to_list()))
    dup_edges = 0
    for s, p, t in triples:
        inv = ontology.inverse(p)
        if (inv and (t, inv, s) in triples) or (ontology.is_symmetric(p) and (t, p, s) in triples and s < t):
            dup_edges += 1

    ext = edges_pl.filter(pl.col("edge_source") == "extracted") if "edge_source" in edges_pl.columns else edges_pl
    viol = sum(1 for st, p, tt in zip(ext["source_type"].to_list(), ext["predicate_str"].to_list(), ext["target_type"].to_list())
               if ontology.check(st, p, tt))
    degs = np.array([deg[i] for i in ids], dtype=float)
    return {
        "nodes": len(ids), "edges": len(edges_pl),
        "duplicate_node_rate": len(dup_nodes) / max(1, len(ids)),
        "duplicate_edge_rate": dup_edges / max(1, len(triples)),
        "connected_components": n_comp,
        "largest_component_frac": max(comp_sizes.values()) / max(1, len(ids)),
        "degree_mean": float(degs.mean()), "degree_p95": float(np.percentile(degs, 95)), "degree_max": float(degs.max()),
        "ontology_violation_rate": viol / max(1, len(ext)),
        "relation_distribution": dict(Counter(edges_pl["predicate_str"].to_list())),
        "type_distribution": dict(Counter(label_t for label_t in
                                          (edges_pl["source_type"].to_list() + edges_pl["target_type"].to_list()))),
    }


def export_audit_artifacts(pipeline: "StreamableKGPipeline", edges_pl: pl.DataFrame,
                           out_dir: str = "./kg_output") -> Dict[str, Path]:
    """[REPORT] Everything needed to answer 'why do you believe this edge exists?' after the run:
    the full claim ledger (accepted / review / rejected + reason), the review queue, contradictions,
    entity memory, document outline and the run manifest + metrics report."""
    out = Path(out_dir)
    out.mkdir(exist_ok=True)
    paths: Dict[str, Path] = {}
    led = pipeline.ledger.to_polars()
    if len(led):
        paths["claims_ledger"] = out / "claims_ledger.parquet"
        led.write_parquet(str(paths["claims_ledger"]))
        led.write_csv(str(out / "claims_ledger.csv"))
        review = led.filter(pl.col("verdict") == "REVIEW")
        paths["review_queue"] = out / "review_queue.csv"
        review.write_csv(str(paths["review_queue"]))
    paths["contradictions"] = out / "contradictions.json"
    paths["contradictions"].write_text(json.dumps([c.to_dict() for c in pipeline.contradictions], indent=2))
    paths["entity_memory"] = out / "entity_memory.json"
    paths["entity_memory"].write_text(json.dumps(pipeline.memory.snapshot(), indent=2))
    if pipeline.hier is not None:
        paths["document_outline"] = out / "document_outline.json"
        paths["document_outline"].write_text(json.dumps(
            {"doc_id": pipeline.hier.doc_id, "doc_summary": pipeline.hier.doc_summary, "sections": pipeline.hier.outline()},
            indent=2))
    paths["run_manifest"] = out / "run_manifest.json"
    paths["run_manifest"].write_text(json.dumps(pipeline.manifest.to_dict(), indent=2))
    paths["run_report"] = out / "run_report.json"
    paths["run_report"].write_text(json.dumps({
        "manifest": pipeline.manifest.wandb_config(),
        "ledger_summary": pipeline.ledger.summary(),
        "graph": graph_quality_metrics(edges_pl),
        "contradictions_by_kind": dict(Counter(c.kind for c in pipeline.contradictions)),
    }, indent=2, default=str))
    return paths


# ---------------------------------------------------------------------------
# [QUERY] ontology-guided retrieval + graph QA  (design doc sections 11, 20)
# Search `source type = PERSON, target type = LOCATION` over only the relation patterns
# the schema allows between those types, instead of `Messi -> everything`. Every answer
# carries the evidence and page of the edge it came from.
# ---------------------------------------------------------------------------

class KGQueryEngine:
    def __init__(self, edges_pl: pl.DataFrame, ontology: Optional[OntologySchema] = None):
        self.ontology = ontology or ONTOLOGY
        self.rows = edges_pl.to_dicts()
        self.by_src: Dict[str, List[int]] = defaultdict(list)
        self.by_dst: Dict[str, List[int]] = defaultdict(list)
        self.display: Dict[str, str] = {}
        self.node_type: Dict[str, str] = {}
        for i, r in enumerate(self.rows):
            sk, tk = normalize_entity_label(r["source_str"]), normalize_entity_label(r["target_str"])
            r["_s"], r["_t"] = sk, tk
            self.by_src[sk].append(i)
            self.by_dst[tk].append(i)
            self.display.setdefault(sk, r["source_str"])
            self.display.setdefault(tk, r["target_str"])
            self.node_type.setdefault(sk, r["source_type"])
            self.node_type.setdefault(tk, r["target_type"])

    def find(self, name: str, cutoff: float = 88.0) -> Optional[str]:
        k = normalize_entity_label(name)
        if k in self.display:
            return k
        sig = _numeric_signature(k)
        hit = rf_process.extractOne(k, list(self.display), scorer=fuzz.WRatio, score_cutoff=cutoff)
        return hit[0] if hit and _numeric_signature(hit[0]) == sig else None

    def _answer(self, r: Dict, node_key: str) -> Dict:
        return {"label": self.display[node_key], "type": self.node_type[node_key], "predicate": r["predicate_str"],
                "confidence": float(r["confidence"]), "evidence": r.get("evidence"), "page": r.get("page"),
                "chunk_id": r.get("chunk_id"), "from": r["source_str"], "to": r["target_str"]}

    def query(self, anchor: str, role: str = "source", predicates: Optional[List[str]] = None,
              answer_type: Optional[str] = None) -> Dict:
        """role='source': edges anchor --p--> ?   (what does Y use?)
           role='target': edges ? --p--> anchor   (which organization developed X?)
        Predicates default to every predicate the SCHEMA allows between the anchor's type and
        `answer_type`; answers are filtered to `answer_type` (including its subtypes)."""
        k = self.find(anchor)
        if k is None:
            return {"anchor": anchor, "resolved": None, "answers": [], "edges_examined": 0}
        anchor_type = self.node_type[k]
        if predicates is None and answer_type:
            predicates = (self.ontology.allowed_predicates(anchor_type, answer_type) if role == "source"
                          else self.ontology.allowed_predicates(answer_type, anchor_type))
        idxs = self.by_src[k] if role == "source" else self.by_dst[k]
        answers, examined = [], 0
        for i in idxs:
            r = self.rows[i]
            examined += 1
            if predicates is not None and r["predicate_str"] not in predicates:
                continue
            other = r["_t"] if role == "source" else r["_s"]
            if answer_type and not self.ontology.matches(self.node_type[other], {answer_type}):
                continue
            answers.append(self._answer(r, other))
        answers.sort(key=lambda a: -a["confidence"])
        return {"anchor": self.display[k], "resolved": k, "anchor_type": anchor_type, "answers": answers,
                "edges_examined": examined, "predicates_searched": predicates}

    def typed_paths(self, anchor: str, target_type: str, max_hops: int = 2, constrained: bool = True) -> Dict:
        """Multi-hop search from `anchor` to any node of `target_type`. constrained=True follows only
        predicate/type sequences the schema allows (ontology.type_paths); constrained=False is the
        blind BFS to compare against. `edges_examined` is the cost difference."""
        k = self.find(anchor)
        if k is None:
            return {"anchor": anchor, "paths": [], "edges_examined": 0}
        allowed: Optional[Set[Tuple]] = None
        if constrained:
            allowed = set()
            for path in self.ontology.type_paths(self.node_type[k], target_type, max_hops=max_hops):
                for n in range(1, len(path) + 1):
                    allowed.add(tuple(path[:n]))
        found, examined = [], 0
        frontier = [(k, [], [])]
        for _hop in range(max_hops):
            nxt = []
            for node, seq, trail in frontier:
                for i in self.by_src.get(node, []):
                    r = self.rows[i]
                    examined += 1
                    step = (r["predicate_str"], self.node_type[r["_t"]])
                    new_seq = seq + [step]
                    if allowed is not None and tuple(new_seq) not in allowed:
                        continue
                    new_trail = trail + [r]
                    if self.ontology.matches(self.node_type[r["_t"]], {target_type}):
                        found.append({"end": self.display[r["_t"]], "type": self.node_type[r["_t"]],
                                      "hops": len(new_seq),
                                      "path": [(t["source_str"], t["predicate_str"], t["target_str"]) for t in new_trail],
                                      "evidence": [t.get("evidence") for t in new_trail]})
                    nxt.append((r["_t"], new_seq, new_trail))
            frontier = nxt[:5000]
        return {"anchor": self.display[k], "paths": found, "edges_examined": examined, "constrained": constrained}

    def explain(self, subject: str, predicate: str, target: str) -> Optional[Dict]:
        """'Why do you believe this edge exists?' -> the provenance stored on that edge."""
        sk, tk = self.find(subject), self.find(target)
        if sk is None or tk is None:
            return None
        for i in self.by_src.get(sk, []):
            r = self.rows[i]
            if r["_t"] == tk and r["predicate_str"] == predicate:
                keep = ["source_str", "predicate_str", "target_str", "confidence", "evidence", "evidence_source",
                        "evidence_score", "page", "pages", "chunk_id", "doc_id", "verifier", "n_supporting_claims",
                        "in_contradiction", "model_version", "prompt_version", "ontology_version", "created_at"]
                return {k_: r.get(k_) for k_ in keep if k_ in r}
        return None


_PROV_EXPORT_COLS = PROVENANCE_COLUMNS + ["model_version", "prompt_version", "ontology_version", "created_at"]




def _warn_if_sparse(n_sections: int, total_chars: int):
    if n_sections and total_chars / max(n_sections, 1) < 200:
        print(
            "  WARNING: extracted text looks very sparse "
            f"({total_chars} chars across {n_sections} section(s)). "
            "This usually means the PDF is scanned/image-only with no "
            "text layer -- PyMuPDF text extraction can't help there. "
            "OCR (not implemented in this pipeline) would be the fix."
        )


def _sys_mem() -> Dict[str, float]:
    """GPU + host memory for the run scorecard (design doc section 25: 'GPU memory / CPU memory')."""
    out = {"gpu_used_gb": 0.0, "gpu_free_gb": 0.0, "cpu_rss_gb": 0.0}
    try:
        if torch.cuda.is_available():
            free_mem, total_mem = torch.cuda.mem_get_info()
            out["gpu_used_gb"], out["gpu_free_gb"] = (total_mem - free_mem) / 1e9, free_mem / 1e9
    except Exception:
        pass
    try:
        import psutil
        out["cpu_rss_gb"] = psutil.Process().memory_info().rss / 1e9
    except Exception:
        pass
    return out


class StreamableKGPipeline:
    """v2: PDF -> document hierarchy -> (summaries) -> per chunk: retrieve context -> multi-pass
    extract -> evidence check -> ontology check -> route (accept / verify / escalate) -> ledger +
    triple store -> entity resolution -> provenance -> contradictions. With
    PipelineConfig.enable_v2_harness=False (PipelineConfig.preset("v1_baseline")) it runs the
    original v1 flow unchanged (the legacy _process_chunks below)."""

    def __init__(self, cfg: PipelineConfig, base_url: Optional[str] = None, dataset_version: str = "adhoc"):
        self.config = cfg
        if base_url is None:
            _, base_url = ensure_vllm_server(cfg)

        self.tracer = LangfuseTracer(enabled=cfg.enable_langfuse)
        self.extractor = VLLMTripleExtractor(cfg, base_url=base_url, tracer=self.tracer)
        self.chunker = TextChunker(self.extractor.tokenizer, cfg.max_input_tokens, cfg.overlap_pct)
        self.store = GPUTripleStore()
        self.seen: Set[str] = set()
        self.gds = GPUDirectStorage()
        self.dataset_version = dataset_version

        # [CLASSIFY]/[ANN] shared embedder -- loaded once, reused by both (and, in v2, by the entity
        # memory / context retrieval). Gated on EMBEDDER_AVAILABLE alone (sentence-transformers), not
        # the faiss-inclusive ANN_AVAILABLE -- domain classification and cuML-backed ANN resolution
        # both only need an embedder, never faiss.
        self.embedder = None
        self.domain_classifier = None
        needs_embedder = (cfg.enable_domain_classification or cfg.enable_ann_resolution
                          or (cfg.enable_v2_harness and cfg.enable_entity_memory))
        if needs_embedder and EMBEDDER_AVAILABLE:
            try:
                device = "cuda" if torch.cuda.is_available() else "cpu"
                self.embedder = SentenceTransformer(cfg.ann_embedding_model, device=device)
                if cfg.enable_domain_classification:
                    self.domain_classifier = DomainClassifier(self.embedder, DEFAULT_DOMAIN_LABELS)
            except Exception as e:
                print(f"[CLASSIFY] Shared embedder load failed ({e}) -- domain tags will read "
                      f"'unclassified'; ANN resolution can still load its own embedder lazily.")
        self.reset_v2_state()

    # ------------------------------------------------------------------ v2 state
    def reset_v2_state(self):
        cfg = self.config
        self.ledger = ClaimLedger()
        self.memory = EntityMemory(embedder=self.embedder)
        self.hier: Optional[DocumentHierarchy] = None
        self.builder: Optional[ContextBuilder] = None
        self.contradictions: List[Contradiction] = []
        self.alias_log: Dict[str, Dict] = {"canon_map": {}, "dup_map": {}}
        self.llm = LLMClient(self.extractor, self.ledger)
        self.claim_extractor = ClaimExtractor(self.extractor, cfg, self.memory, self.llm)
        self.checker = EvidenceChecker()
        self.verifier = ClaimVerifier(self.llm, cfg, ONTOLOGY)
        self.judge = ClaimJudge(cfg, self.memory, self.checker, self.verifier, ONTOLOGY)
        self.manifest = RunManifest.build(cfg, model_name=self.extractor.model_name,
                                          dataset_version=self.dataset_version)

    def reset(self):
        """Clear everything document-specific (store, seen set, ledger, memory) so ONE pipeline
        instance can be reused across documents -- what kg_eval does between gold documents."""
        if self.store.triples_df is not None:
            del self.store.triples_df
            self.store.triples_df = None
        self.store._pending.clear()
        self.seen.clear()
        self.reset_v2_state()

    # ------------------------------------------------------------------ entry points
    def process_pdf(self, pdf_bytes: bytes, doc_id: str = "doc") -> Iterator[Dict]:
        print("[OCR/LAYOUT] Extracting sections...")
        if not self.config.enable_v2_harness:
            sections = [SectionRecord(section_id=f"{doc_id}:s{i}", title=t, level=1, parent_title="", text=x,
                                      page_start=1, page_end=1, page_offsets=[(0, 1)])
                        for i, (t, x) in enumerate(extract_sections_from_pdf(pdf_bytes))]
        else:
            sections = extract_document_structure(pdf_bytes, doc_id)
            _warn_if_sparse(len(sections), sum(len(s.text) for s in sections))
        yield from self.process_sections(sections, doc_id)

    def process_text(self, text: str, doc_id: str = "doc", title: str = "text") -> Iterator[Dict]:
        yield from self.process_sections(sections_from_text(text, doc_id, title), doc_id)

    def process_sections(self, sections: List[SectionRecord], doc_id: str = "doc") -> Iterator[Dict]:
        cfg = self.config
        print(f"[OCR/LAYOUT] Sections: {len(sections)}")
        if not cfg.enable_v2_harness:
            print("[CHUNK] Adaptive chunking (v1 path)...")
            all_chunks: List[str] = []
            for sec in sections:
                all_chunks.extend(self.chunker.chunk_text(sec.text))
            yield from self._process_chunks(all_chunks)
            return

        print("[CHUNK] Adaptive chunking + document hierarchy...")
        self.hier = build_hierarchy(sections, self.chunker, doc_id)
        self.builder = ContextBuilder(self.hier, self.memory, cfg, embedder=self.embedder)
        print(f"[DOC-MAP] {len(self.hier.sections)} sections, {len(self.hier.chunks)} chunks, "
              f"pages {min((s.page_start for s in self.hier.sections), default=0)}-"
              f"{max((s.page_end for s in self.hier.sections), default=0)}")
        if cfg.enable_hierarchical_summaries and self.hier.chunks:
            HierarchicalSummarizer(self.llm).summarize(self.hier)
        yield from self._process_hierarchy()

    # ------------------------------------------------------------------ v2 flow
    @staticmethod
    def _dedupe(claims: List[Claim]) -> List[Claim]:
        best: Dict[Tuple, Claim] = {}
        for c in claims:
            k = c.key()
            if k not in best or c.confidence > best[k].confidence:
                best[k] = c
        return list(best.values())

    def _self_consistency(self, batch: List[ChunkRecord], low: List[int], ctxs: List[Context],
                          entity_lists: List[List[Tuple[str, str, str]]], stats: Dict[str, int],
                          claims_pc: List[List[Claim]]):
        """[VERIFY] hard-case compute, design doc section 12 'self-consistency for difficult cases'
        (v1's Bayesian rescue, kept): low-yield chunks are sampled N more times; only claims whose
        agreement-rate lower bound clears 0.5 are proposed -- and they still go through the
        evidence / ontology / verifier gates like any other claim."""
        cfg = self.config
        sub_chunks = [batch[j] for j in low]
        sub_ctx = [ctxs[j] for j in low]
        sub_ents = [entity_lists[j] for j in low] if cfg.enable_multipass_extraction else None
        runs = [self.claim_extractor.extract(sub_chunks, sub_ctx, stats, temperature=0.5, entity_lists=sub_ents)[0]
                for _ in range(cfg.bayesian_rerun_samples)]
        self.ledger.counters["self_consistency_chunks"] += len(low)
        items: List[Tuple[ChunkRecord, Claim]] = []
        for pos, j in enumerate(low):
            known = {c.key() for c in claims_pc[j]}
            fresh = [c for c in consensus_claims([run[pos] for run in runs]) if c.key() not in known]
            for c in fresh:
                c.claim_id = self.ledger.new_id(batch[j].doc_id)
                items.append((batch[j], c))
                claims_pc[j].append(c)
        if items:
            self.judge.judge(items, builder=self.builder)

    def _extract_and_judge(self, batch: List[ChunkRecord], stats: Dict[str, int]):
        cfg = self.config
        ctxs = [self.builder.build(ch) for ch in batch]
        claims_pc, entity_lists = self.claim_extractor.extract(batch, ctxs, stats)
        claims_pc = [self._dedupe(cs) for cs in claims_pc]
        items: List[Tuple[ChunkRecord, Claim]] = []
        for ch, cs in zip(batch, claims_pc):
            for c in cs:
                c.claim_id = self.ledger.new_id(ch.doc_id)
                items.append((ch, c))
        self.judge.judge(items, builder=self.builder)
        if cfg.enable_self_consistency:
            low = [j for j, cs in enumerate(claims_pc)
                   if sum(1 for c in cs if c.verdict == "ACCEPT") < cfg.low_yield_claims]
            if low:
                self._self_consistency(batch, low, ctxs, entity_lists, stats, claims_pc)
        if cfg.enable_cooccurrence_edges:
            for ch, cs in zip(batch, claims_pc):
                acc = [c for c in cs if c.verdict == "ACCEPT"]
                if len(acc) >= 2:
                    have = {frozenset((normalize_entity_label(c.subject), normalize_entity_label(c.target))) for c in acc}
                    extra = []
                    for s, st, p, t, tt, conf in enhance_connectivity(ch.text, [c.as_tuple() for c in acc])[len(acc):]:
                        if frozenset((normalize_entity_label(s), normalize_entity_label(t))) in have:
                            continue
                        extra.append(Claim(subject=s, subject_type=st, predicate=p, target=t, target_type=tt,
                                           confidence=conf, raw_confidence=conf, source="cooccurrence",
                                           claim_id=self.ledger.new_id(ch.doc_id)))
                    self.judge.judge_cooccurrence(ch, extra)
                    cs.extend(extra)
        return claims_pc, entity_lists

    def _commit(self, batch: List[ChunkRecord], claims_pc: List[List[Claim]],
                entity_lists: List[List[Tuple[str, str, str]]]) -> Tuple[int, int, int]:
        """Writes the decided batch into the ledger, entity memory and triple store."""
        n_acc = n_rev = n_rej = 0
        for ch, cs, ents in zip(batch, claims_pc, entity_lists or [[] for _ in batch]):
            self.ledger.add(cs)
            acc = [c for c in cs if c.verdict == "ACCEPT"]
            n_acc += len(acc)
            n_rev += sum(1 for c in cs if c.verdict == "REVIEW")
            n_rej += sum(1 for c in cs if c.verdict == "REJECT")
            if self.config.enable_entity_memory:
                for name, etype, ev in ents:
                    self.memory.observe(name, etype, chunk_id=ch.chunk_id, chunk_index=ch.index, evidence=ev,
                                        confidence=0.5, page=ch.page_start)
                for c in acc:
                    self.memory.observe_claim(c, ch.index)
            ch.entities = sorted({c.subject for c in acc} | {c.target for c in acc})
            tuples = [c.as_tuple() for c in acc]
            self.store.add_triples(tuples)
            for s, _, _, t, _, _ in tuples:
                self.seen.update([s, t])
        return n_acc, n_rev, n_rej

    def _process_hierarchy(self) -> Iterator[Dict]:
        cfg = self.config
        chunks = self.hier.chunks
        total = len(chunks)
        print(f"[CHUNK] {total} chunks (document order)  |  batch_size={cfg.batch_size}")
        pbar = tqdm(total=total, desc="[PROCESS B] Extracting (v2)", unit="chunk")
        t0 = time.time()
        cum_acc = cum_rev = cum_rej = 0

        for i in range(0, total, cfg.batch_size):
            batch = chunks[i: i + cfg.batch_size]
            bidx = i // cfg.batch_size
            batch_t0 = time.time()
            calls_before = self.ledger.counters.get("verifier_calls", 0)

            if self.domain_classifier is not None:
                for ch, (d, _s) in zip(batch, self.domain_classifier.classify_batch([c.text for c in batch])):
                    ch.domain = d
            if cfg.enable_entity_memory:
                for ch in batch:
                    self.memory.register_acronyms(ch.text)

            stats: Dict[str, int] = {}
            with self.tracer.batch_span("extract_batch_v2", input={"n_chunks": len(batch),
                                                                   "domains": [c.domain for c in batch]},
                                        metadata={"vllm_model": self.extractor.model_name,
                                                  "prompt_version": PROMPT_VERSION}) as bspan:
                claims_pc, entity_lists = self._extract_and_judge(batch, stats)
                n_acc, n_rev, n_rej = self._commit(batch, claims_pc, entity_lists)
                for ch, cs in zip(batch, claims_pc):
                    with bspan.start_as_current_generation(
                        name=f"chunk_{ch.index}", model=self.extractor.model_name, input=ch.text[:4000],
                        model_parameters={"temperature": 0.1, "top_p": 0.9},
                    ) as gen:
                        gen.update(output=json.dumps([{"s": c.subject, "p": c.predicate, "t": c.target,
                                                       "verdict": c.verdict, "reason": c.reject_reason,
                                                       "route": c.route, "conf": round(c.confidence, 3)}
                                                      for c in cs])[:4000],
                                   metadata={"domain": ch.domain, "page": ch.page_start, "section": ch.section,
                                             "n_claims": len(cs), "dropped_stats": stats})
                bspan.update(output={"accepted": n_acc, "review": n_rev, "rejected": n_rej})

            cum_acc, cum_rev, cum_rej = cum_acc + n_acc, cum_rev + n_rev, cum_rej + n_rej
            batch_time = time.time() - batch_t0
            mem = _sys_mem()
            self.ledger.batch_stats.append({"batch_idx": bidx, "n_chunks": len(batch), "batch_time_s": batch_time, **mem})

            if wandb.run is not None:
                acc_conf = [c.confidence for cs in claims_pc for c in cs if c.verdict == "ACCEPT"]
                log_dict = {
                    "batch_idx": bidx, "triples_this_batch": n_acc, "cumulative_entities": len(self.seen),
                    "avg_confidence": float(np.mean(acc_conf)) if acc_conf else 0.0, "generation_time_s": batch_time,
                    "v2/accepted": n_acc, "v2/review": n_rev, "v2/rejected": n_rej,
                    "v2/verifier_calls_this_batch": self.ledger.counters.get("verifier_calls", 0) - calls_before,
                    "v2/memory_entities": len(self.memory),
                    "gpu_used_gb": mem["gpu_used_gb"], "gpu_free_gb": mem["gpu_free_gb"], "cpu_rss_gb": mem["cpu_rss_gb"],
                }
                for d in {c.domain for c in batch}:
                    log_dict[f"domain_count/{d}"] = sum(1 for c in batch if c.domain == d)
                log_dict.update({f"filter/{k}": v for k, v in stats.items()})
                wandb.log(log_dict)

            if bidx > 0 and bidx % 5 == 0:
                self.store.flush()
                if self.store.triples_df is not None:
                    self.gds.save_checkpoint(self.store.triples_df, f"batch_{i:04d}.parquet")
            if bidx % 3 == 0:
                cleanup_memory()

            pbar.update(len(batch))
            yield {"progress": min(i + cfg.batch_size, total) / total, "entities": len(self.seen),
                   "accepted": cum_acc, "review": cum_rev, "rejected": cum_rej}

        pbar.close()
        if cfg.enable_contradiction_detection:
            self.contradictions = ContradictionDetector(ONTOLOGY).run(self.ledger.claims, self.memory)
        self.tracer.flush()
        s = self.ledger.summary()
        print(f"[PROCESS B] Extraction: {time.time() - t0:.1f}s  |  Entities: {len(self.seen)}")
        print(f"[VERIFY] claims: {int(s['claims/candidates'])} candidates -> {int(s['claims/accepted'])} accepted / "
              f"{int(s['claims/review'])} review / {int(s['claims/rejected'])} rejected  |  "
              f"unsupported-claim rate {s['quality/unsupported_claim_rate']:.3f}  |  "
              f"routes easy/medium/hard {s['compute/route_easy_frac']:.2f}/{s['compute/route_medium_frac']:.2f}/"
              f"{s['compute/route_hard_frac']:.2f}  |  verifier calls {int(s['compute/verifier_calls'])}  |  "
              f"contradictions {len(self.contradictions)}")

    # ------------------------------------------------------------------ v1 flow (legacy, unchanged)
    def _process_chunks(self, chunks: List[str]) -> Iterator[Dict]:
        chunk_lengths = [len(self.extractor.tokenizer.encode(c, add_special_tokens=False)) for c in chunks]
        chunks = [c for _, c in sorted(zip(chunk_lengths, chunks))]
        total = len(chunks)
        print(f"[CHUNK] {total} chunks  |  Est. triples: {total * 5}-{total * 10}")
        pbar = tqdm(total=total, desc="[PROCESS B] Extracting", unit="chunk")
        t0 = time.time()

        for i in range(0, total, self.config.batch_size):
            batch = chunks[i: i + self.config.batch_size]
            batch_t0 = time.time()

            domain_tags = None
            if self.domain_classifier is not None:
                domain_tags = [d for d, _score in self.domain_classifier.classify_batch(batch)]

            results, extract_stats = self.extractor.extract_batch(batch, self.seen or None, domain_tags=domain_tags)

            low_yield = [j for j, r in enumerate(results) if len(r) < 3]
            if low_yield:
                rechunks = [batch[j] for j in low_yield]
                rechunk_domains = [domain_tags[j] for j in low_yield] if domain_tags else None
                improved = self.extractor._bayesian_rerun(
                    rechunks, n_samples=self.config.bayesian_rerun_samples, domain_tags=rechunk_domains,
                )
                for local_j, global_j in enumerate(low_yield):
                    if len(improved[local_j]) >= len(results[global_j]):
                        results[global_j] = improved[local_j]

            batch_triples = sum(len(r) for r in results)
            batch_conf = sum(t[5] for r in results for t in r) / max(batch_triples, 1)
            batch_time = time.time() - batch_t0

            for triples in results:
                self.store.add_triples(triples)
                for src, _, _, tgt, _, _ in triples:
                    self.seen.update([src, tgt])

            if wandb.run is not None:
                free_mem, total_mem = torch.cuda.mem_get_info()
                log_dict = {
                    "batch_idx": i // self.config.batch_size,
                    "triples_this_batch": batch_triples,
                    "cumulative_entities": len(self.seen),
                    "gpu_used_gb": (total_mem - free_mem) / 1e9,
                    "gpu_free_gb": free_mem / 1e9,
                    "avg_confidence": batch_conf,
                    "generation_time_s": batch_time,
                    "low_yield_reruns": len(low_yield),
                }
                if domain_tags:
                    for d in set(domain_tags):
                        log_dict[f"domain_count/{d}"] = domain_tags.count(d)
                log_dict.update({f"filter/{k}": v for k, v in extract_stats.items()})
                wandb.log(log_dict)

            bidx = i // self.config.batch_size
            if bidx > 0 and bidx % 5 == 0:
                self.store.flush()
                if self.store.triples_df is not None:
                    self.gds.save_checkpoint(self.store.triples_df, f"batch_{i:04d}.parquet")
            if bidx % 3 == 0:
                cleanup_memory()

            pbar.update(len(batch))
            yield {"progress": min(i + self.config.batch_size, total) / total, "entities": len(self.seen)}

        pbar.close()
        self.tracer.flush()
        print(f"[PROCESS B] Extraction: {time.time() - t0:.1f}s  |  Entities: {len(self.seen)}")

    # ------------------------------------------------------------------ resolution + graph
    def resolved_edges(self) -> pl.DataFrame:
        """aggregate -> confidence filter -> entity resolution -> ANN resolution -> provenance.
        CPU-side edge table (no cugraph) -- what kg_eval scores and what finalize_graph feeds to cuGraph."""
        self.store.flush()
        raw_cudf = self.store.aggregate()
        if len(raw_cudf) == 0:
            raise ValueError("No triples extracted from the document.")

        raw_cudf = raw_cudf[raw_cudf['confidence'] >= self.config.confidence_filter]
        print(f"[VALIDATE] After confidence filter ({self.config.confidence_filter}): {len(raw_cudf)} edges")

        if len(raw_cudf) == 0:
            raise ValueError("All edges filtered. Lower confidence_filter in PipelineConfig.")

        with nvtx_range("gpu_to_cpu_handoff", color="red"):
            # cudf (GPU) -> Arrow -> polars (CPU): resolve_entities_in_edges below is
            # rapidfuzz/CPU-only, so the edge table has to leave the GPU here. Its own range,
            # separate from the [RESOLVE]/[ANN] compute that follows.
            edges_pl = pl.from_arrow(raw_cudf.to_arrow())
        print("[RESOLVE] Global entity resolution...")
        self.alias_log = {"canon_map": {}, "dup_map": {}}
        edges_pl = resolve_entities_in_edges(edges_pl, self.config.entity_resolution_threshold,
                                             guard_versions=self.config.version_guard_in_resolution,
                                             alias_log=self.alias_log)
        edges_pl = edges_pl.with_columns(pl.lit("extracted").alias("edge_source"))
        print("[ANN] Semantic entity resolution...")
        edges_pl = apply_ann_resolution(edges_pl, self.config, shared_embedder=self.embedder,
                                        alias_log=self.alias_log, guard_versions=self.config.version_guard_in_resolution)
        if self.config.enable_v2_harness and self.ledger.claims:
            edges_pl = attach_provenance(edges_pl, self.ledger.claims, self.alias_log["canon_map"],
                                         self.alias_log["dup_map"], self.manifest)
        return edges_pl

    def finalize_graph(self) -> Tuple[pl.DataFrame, Dict]:
        edges_pl = self.resolved_edges()

        with nvtx_range("cpu_to_gpu_handoff", color="red"):
            # polars (CPU) -> Arrow -> cudf (GPU): back onto the GPU for cuGraph below. Only the
            # three columns cuGraph reads -- the provenance strings stay on the CPU.
            resolved_cudf = cudf.DataFrame.from_arrow(edges_pl.select(["src_id", "dst_id", "confidence"]).to_arrow())
        print("[GRAPH] Computing PageRank / degree / communities...")
        metrics = GPUGraphAnalyzer(resolved_cudf, self.config.leiden_resolution).compute_metrics()

        return edges_pl, metrics

    def run_document(self, text: Optional[str] = None, pdf_bytes: Optional[bytes] = None, doc_id: str = "doc",
                     reset: bool = True) -> Dict:
        """Run ONE document end-to-end up to the resolved edge table (no cugraph) and return
        everything kg_eval needs. Exactly one of `text` / `pdf_bytes`."""
        if reset:
            self.reset()
        gen = self.process_pdf(pdf_bytes, doc_id) if pdf_bytes is not None else self.process_text(text or "", doc_id)
        for _ in gen:
            pass
        try:
            edges_pl: Optional[pl.DataFrame] = self.resolved_edges()
        except ValueError:
            edges_pl = None
        return {"edges": edges_pl, "claims": self.ledger.claims, "ledger": self.ledger,
                "contradictions": self.contradictions, "memory": self.memory, "hier": self.hier,
                "manifest": self.manifest, "summary": self.ledger.summary(), "alias_log": self.alias_log}


@nvtx_range("export_to_parquet", color="orange")
def export_to_parquet(edges_pl: pl.DataFrame, metrics: Dict, output_dir: str = "./kg_output",
                      memory: Optional["EntityMemory"] = None,
                      manifest: Optional["RunManifest"] = None) -> Tuple[Path, Path, Path, Path]:
    """[STORE] the "Canonical KG" node in the architecture diagram."""
    out = Path(output_dir)
    out.mkdir(exist_ok=True)

    pr_pl = pl.from_arrow(metrics["pagerank"].to_arrow())
    pr = {int(k): float(v) for k, v in zip(pr_pl["vertex"].to_list(), pr_pl["pagerank"].to_list())}

    in_d: Dict[int, int] = {}
    out_d: Dict[int, int] = {}
    if "in_degree" in metrics:
        t = pl.from_arrow(metrics["in_degree"].to_arrow())
        in_d = {int(k): int(v) for k, v in zip(t["vertex"].to_list(), t["degree"].to_list())}
    if "out_degree" in metrics:
        t = pl.from_arrow(metrics["out_degree"].to_arrow())
        out_d = {int(k): int(v) for k, v in zip(t["vertex"].to_list(), t["degree"].to_list())}

    leiden_d: Dict[int, int] = {}
    if "leiden" in metrics:
        ld = pl.from_arrow(metrics["leiden"].to_arrow())
        leiden_d = {int(k): int(v) for k, v in zip(ld["vertex"].to_list(), ld["partition"].to_list())}

    src_nodes = edges_pl.select([pl.col("src_id").alias("id"), pl.col("source_str").alias("label"), pl.col("source_type").alias("type")])
    dst_nodes = edges_pl.select([pl.col("dst_id").alias("id"), pl.col("target_str").alias("label"), pl.col("target_type").alias("type")])
    unique_nodes = pl.concat([src_nodes, dst_nodes]).unique("id")

    pr_df = pl.DataFrame({"id": list(pr.keys()), "pagerank": list(pr.values())})
    ind_df = pl.DataFrame({"id": list(in_d.keys()), "in_degree": list(in_d.values())})
    outd_df = pl.DataFrame({"id": list(out_d.keys()), "out_degree": list(out_d.values())})
    leid_df = pl.DataFrame({"id": list(leiden_d.keys()), "community_id": list(leiden_d.values())})

    nodes_pl = (
        unique_nodes
        .join(pr_df, on="id", how="left").join(ind_df, on="id", how="left")
        .join(outd_df, on="id", how="left").join(leid_df, on="id", how="left")
        .with_columns([
            pl.col("pagerank").fill_null(0.0),
            pl.col("in_degree").fill_null(0).cast(pl.Int64),
            pl.col("out_degree").fill_null(0).cast(pl.Int64),
            pl.col("community_id").fill_null(-1).cast(pl.Int64),
            (pl.col("in_degree").fill_null(0) + pl.col("out_degree").fill_null(0)).alias("degree"),
        ])
    )

    if memory is not None:     # v2: node-level provenance from the global entity memory
        recs = [memory.lookup(l) for l in nodes_pl["label"].to_list()]
        nodes_pl = nodes_pl.with_columns([
            pl.Series("n_mentions", [int(r.n_mentions) if r else 0 for r in recs], dtype=pl.Int64),
            pl.Series("first_page", [int(r.first_page) if r else 0 for r in recs], dtype=pl.Int64),
            pl.Series("aliases", ["|".join(sorted(r.aliases)) if r else "" for r in recs], dtype=pl.Utf8),
            pl.Series("node_evidence", [(r.evidence[0] if r and r.evidence else "") for r in recs], dtype=pl.Utf8),
        ])
    if manifest is not None:
        nodes_pl = nodes_pl.with_columns([
            pl.lit(manifest.model_version).alias("model_version"),
            pl.lit(manifest.ontology_version).alias("ontology_version"),
            pl.lit(manifest.created_at).alias("created_at"),
        ])

    edges_out = edges_pl.select([
        pl.col("src_id").alias("source"), pl.col("dst_id").alias("target"),
        pl.col("source_str").alias("source_label"), pl.col("target_str").alias("target_label"),
        pl.col("predicate_str").alias("label"), pl.col("confidence").alias("weight"),
        pl.col("count"), pl.col("edge_source"),
        *[pl.col(c) for c in _PROV_EXPORT_COLS if c in edges_pl.columns],
    ])

    np_ = out / "nodes.parquet"
    ep_ = out / "edges.parquet"
    nc_ = out / "nodes.csv"
    ec_ = out / "edges.csv"

    nodes_pl.write_parquet(str(np_))
    edges_out.write_parquet(str(ep_))
    nodes_pl.write_csv(str(nc_))
    edges_out.write_csv(str(ec_))

    n_comm = nodes_pl["community_id"].n_unique() - (1 if -1 in nodes_pl["community_id"].to_list() else 0)
    print(f"[STORE] Nodes: {len(nodes_pl)}  |  Edges: {len(edges_out)}  |  Leiden communities: {n_comm}")
    return np_, ep_, nc_, ec_


_state: Dict = {"pipeline": None}


def launch_ui():
    for d in ["./offload", "./checkpoints", "./kg_output"]:
        Path(d).mkdir(exist_ok=True)

    uploader = widgets.FileUpload(accept=".pdf", multiple=False)
    # v2: one dropdown walks the design doc's section-26 comparison list (baseline vs retrieval vs
    # verification vs ontology constraints). "config as set" uses kgp.config exactly as you left it.
    preset_dd = widgets.Dropdown(
        options=[("config as set (kgp.config)", "custom"), ("v1 baseline (original flow)", "v1_baseline"),
                 ("v2: evidence + provenance only", "v2_evidence_only"), ("v2: + retrieval / memory / multi-pass", "v2_retrieval"),
                 ("v2: + verifier / adaptive compute", "v2_verification"), ("v2: full harness", "v2_full")],
        value="custom", description="Mode:", layout=widgets.Layout(width="440px"))
    button = widgets.Button(description="▶ Process PDF", button_style="success",
                             layout=widgets.Layout(width="160px", margin="8px 0"))
    status = widgets.HTML("")
    out_widget = widgets.Output()

    def on_click(b):
        with out_widget:
            out_widget.clear_output()

            # NOTE: no server shutdown/relaunch here -- StreamableKGPipeline reuses the already-warm
            # singleton server via ensure_vllm_server(). Only the in-memory triple store/seen set
            # from any previous run gets cleared.
            if _state["pipeline"] is not None:
                prev = _state["pipeline"]
                if prev.store.triples_df is not None:
                    del prev.store.triples_df
                prev.store._pending.clear()
                prev.seen.clear()
                del prev
                _state["pipeline"] = None
            cleanup_memory()

            if config.enable_wandb and wandb.run is not None:
                wandb.finish()

            if not uploader.value:
                print("No PDF uploaded.")
                return

            try:
                f = uploader.value[0]
                pdf_bytes, filename = bytes(f["content"]), f["name"]
            except (KeyError, IndexError, TypeError):
                fi = list(uploader.value.values())[0]
                pdf_bytes, filename = bytes(fi["content"]), list(uploader.value.keys())[0]

            # [OBSERVE] Stashed for the standalone Nsight Systems driver script (see the "Nsight
            # Systems" section near the end) -- that script runs as its own separate process so it
            # can't reach this closure's in-memory pdf_bytes.
            try:
                Path("/content/last_uploaded.pdf").write_bytes(pdf_bytes)
            except OSError:
                pass

            cfg = PipelineConfig.preset(preset_dd.value, base=config)
            status.value = (f"<b>Processing:</b> {filename}  ({len(pdf_bytes) / 1024:.1f} KB)  "
                            f"&middot; mode <code>{preset_dd.value}</code>")
            cleanup_memory()

            # Everything below -- including wandb.init() and the vLLM health check
            # StreamableKGPipeline's constructor triggers -- lives inside this single
            # try/except/finally, so a failed W&B login or a not-yet-healthy server degrades to a
            # clean "Failed" status instead of an unhandled exception escaping the button handler.
            metrics_poller = None
            try:
                if cfg.enable_wandb:
                    try:
                        wandb.init(
                            project=cfg.wandb_project, reinit=True,
                            config={
                                "model": cfg.model_name, "vllm_speculative_method": cfg.vllm_speculative_method,
                                "vllm_quantization": cfg.vllm_quantization, "max_input_tokens": cfg.max_input_tokens,
                                "max_output_tokens": cfg.max_output_tokens, "batch_size": cfg.batch_size,
                                "overlap_pct": cfg.overlap_pct, "entity_resolution_threshold": cfg.entity_resolution_threshold,
                                "leiden_resolution": cfg.leiden_resolution, "confidence_filter": cfg.confidence_filter,
                                "strict_field_validation": cfg.strict_field_validation, "ann_backend": cfg.ann_backend,
                                "enable_ann_resolution": cfg.enable_ann_resolution,
                                "enable_domain_classification": cfg.enable_domain_classification,
                                "preset": preset_dd.value,
                                "enable_v2_harness": cfg.enable_v2_harness,
                                **{k: getattr(cfg, k) for k in V2_ABLATION_FLAGS},
                                "filename": filename,
                            },
                        )
                        # W&B auto-collects host/GPU system metrics (utilization, memory, temp, power)
                        # on init with zero extra code here.
                    except Exception as e:
                        print(f"[OBSERVE] W&B init failed ({e}) -- continuing without W&B logging.")

                pipeline = StreamableKGPipeline(cfg, dataset_version=filename)
                _state["pipeline"] = pipeline
                if cfg.enable_wandb and wandb.run is not None:
                    try:
                        wandb.config.update(pipeline.manifest.wandb_config(), allow_val_change=True)
                    except Exception:
                        pass

                if cfg.enable_wandb and wandb.run is not None:
                    metrics_poller = VLLMMetricsPoller(
                        pipeline.extractor.base_url, interval_s=cfg.vllm_metrics_poll_interval_s,
                    )
                    metrics_poller.start()

                for _ in pipeline.process_pdf(pdf_bytes):
                    pass

                edges_pl, metrics = pipeline.finalize_graph()

                n_nodes = len(pl.concat([
                    edges_pl.select(pl.col("source_str").alias("n")),
                    edges_pl.select(pl.col("target_str").alias("n")),
                ]).unique("n"))
                n_edges = len(edges_pl)
                n_unique_preds = edges_pl["predicate_str"].n_unique()
                avg_conf = float(edges_pl["confidence"].mean())
                n_communities = 0
                if "leiden" in metrics:
                    ld = pl.from_arrow(metrics["leiden"].to_arrow())
                    n_communities = int(ld["partition"].n_unique())

                if wandb.run is not None:
                    wandb.log({
                        "total_nodes": n_nodes, "total_edges": n_edges,
                        "leiden_communities": n_communities, "unique_predicates": n_unique_preds,
                        "avg_edge_confidence": avg_conf,
                    })

                np_, ep_, nc_, ec_ = export_to_parquet(
                    edges_pl, metrics,
                    memory=pipeline.memory if cfg.enable_v2_harness else None,
                    manifest=pipeline.manifest if cfg.enable_v2_harness else None)

                audit_paths: Dict[str, Path] = {}
                if cfg.enable_v2_harness:
                    audit_paths = export_audit_artifacts(pipeline, edges_pl)
                    s = pipeline.ledger.summary()
                    g = graph_quality_metrics(edges_pl)
                    if wandb.run is not None:
                        wandb.log({f"v2/{k}": v for k, v in s.items()})
                        wandb.log({f"graph/{k}": v for k, v in g.items() if isinstance(v, (int, float))})
                    print("\n── v2 scorecard ──")
                    print(f"claims      : {int(s['claims/candidates'])} candidates -> {int(s['claims/accepted'])} accepted, "
                          f"{int(s['claims/review'])} review, {int(s['claims/rejected'])} rejected")
                    print(f"safety      : unsupported-claim rate {s['quality/unsupported_claim_rate']:.3f}  |  "
                          f"candidate hallucination rate {s['quality/candidate_ungrounded_rate']:.3f}  |  "
                          f"ontology-violation rate {s['quality/ontology_violation_rate']:.3f}")
                    print(f"compute     : easy/medium/hard {s['compute/route_easy_frac']:.2f}/"
                          f"{s['compute/route_medium_frac']:.2f}/{s['compute/route_hard_frac']:.2f}  |  "
                          f"verifier calls {int(s['compute/verifier_calls'])}  |  LLM calls {int(s['compute/llm_calls'])}  |  "
                          f"p50/p95 s/chunk {s['latency/p50_s_per_chunk']:.2f}/{s['latency/p95_s_per_chunk']:.2f}")
                    print(f"graph       : {g.get('nodes', 0)} nodes, {g.get('edges', 0)} edges, "
                          f"{g.get('connected_components', 0)} components, duplicate-node rate {g.get('duplicate_node_rate', 0):.3f}")
                    print(f"contradicts : {len(pipeline.contradictions)}  "
                          f"{dict(Counter(c.kind for c in pipeline.contradictions)) or ''}")

                for v in metrics.values():
                    del v
                del edges_pl, metrics
                cleanup_memory()

                print("\n── Output files ──")
                display(FileLink(str(np_), result_html_prefix="nodes.parquet → "))
                display(FileLink(str(ep_), result_html_prefix="edges.parquet → "))
                display(FileLink(str(nc_), result_html_prefix="nodes.csv  (Gephi) → "))
                display(FileLink(str(ec_), result_html_prefix="edges.csv  (Cosmograph) → "))
                for key, label in (("claims_ledger", "claims_ledger.parquet (every candidate + verdict + reason) → "),
                                   ("review_queue", "review_queue.csv (needs a human) → "),
                                   ("contradictions", "contradictions.json → "),
                                   ("run_report", "run_report.json (scorecard) → ")):
                    if key in audit_paths:
                        display(FileLink(str(audit_paths[key]), result_html_prefix=label))
                status.value = "<b style='color:green'>✓ Complete</b>"

            except Exception as e:
                print(f"Error: {e}")
                status.value = "<b style='color:red'>✗ Failed</b>"
                import traceback
                traceback.print_exc()
            finally:
                if metrics_poller is not None:
                    metrics_poller.stop()
                if config.enable_wandb and wandb.run is not None:
                    wandb.finish()

    button.on_click(on_click)
    display(widgets.VBox([
        widgets.HTML("<h3 style='font-family:sans-serif;margin:4px 0'>PDF → Knowledge Graph</h3>"),
        widgets.HTML(
            "<small style='color:#555'>Qwen2.5-3B-AWQ · vLLM (isolated venv, PagedAttention + "
            "speculative decoding) · Leiden · <b>v2 harness</b>: document hierarchy · retrieval + global entity memory · "
            "multi-pass extraction · evidence-grounded claims · ontology type system · verifier + adaptive compute · "
            "contradiction detection · provenance · cuML ANN resolution · W&amp;B + Langfuse observability</small>"
        ),
        uploader, preset_dd, button, status, out_widget,
    ]))


# ============================================================================
# [ACTIVE-LEARN] Evaluation / Active Learning / Distillation -- OPTIONAL,
# not required for the main pipeline above. Adapted from the
# transformers-direct predecessor to call the vLLM HTTP endpoint via
# _vllm_raw_completion instead of a local model.generate(). The pipeline's
# own validation gates (parse_json_triples strict mode + sitl_filter)
# ARE the reward function here -- no labeled dataset needed.
# ============================================================================

class VLLMDPODatasetBuilder:
    """For the SAME chunk, generate several completions under settings
    likely to spread quality apart (JSON-schema constraint on/off, varied
    temperature), score each by what fraction of its parsed triples
    survive validation, and keep the best/worst as a chosen/rejected pair
    when the gap is large enough to be signal rather than noise."""

    def __init__(self, extractor: "VLLMTripleExtractor", min_score_margin: float = 0.25, min_attempted: int = 2):
        self.extractor = extractor
        self.min_score_margin = min_score_margin
        self.min_attempted = min_attempted

    @staticmethod
    def _quality_score(raw_text: str, cfg: PipelineConfig) -> Tuple[float, int]:
        attempted = parse_json_triples(raw_text, strict=False)
        if len(attempted) < 1:
            return 0.0, 0
        stats: Dict[str, int] = {}
        strict = parse_json_triples(raw_text, strict=True, max_entity_chars=cfg.max_entity_chars,
                                     max_entity_words=cfg.max_entity_words, stats=stats)
        normalized = [(s, sp, normalize_predicate(p), t, tp, 0.0) for s, sp, p, t, tp, _ in strict]
        survived = sitl_filter(normalized, stats=stats)
        return len(survived) / len(attempted), len(attempted)

    def generate_candidates(self, chunk: str, n: int = 4) -> List[Tuple[str, float]]:
        variants = [
            dict(use_schema=True, temperature=0.1), dict(use_schema=True, temperature=0.3),
            dict(use_schema=False, temperature=0.5), dict(use_schema=False, temperature=0.8),
        ][:max(2, n)]
        prompt = f"{EXTRACTION_SYSTEM_PROMPT}\n\n{chunk}\n\nJSON:"
        candidates: List[Tuple[str, float]] = []
        for v in variants:
            raw_text = _vllm_raw_completion(self.extractor, prompt, temperature=v["temperature"], use_schema=v["use_schema"])
            score, attempted = self._quality_score(raw_text, self.extractor.config)
            if attempted >= self.min_attempted:
                candidates.append((raw_text, score))
        return candidates

    def build_pair(self, chunk: str, n: int = 4) -> Optional[Dict[str, object]]:
        candidates = self.generate_candidates(chunk, n=n)
        if len(candidates) < 2:
            return None
        candidates.sort(key=lambda x: x[1], reverse=True)
        best_text, best_score = candidates[0]
        worst_text, worst_score = candidates[-1]
        if best_score - worst_score < self.min_score_margin:
            return None
        prompt = f"{EXTRACTION_SYSTEM_PROMPT}\n\n{chunk}\n\nJSON:"
        return {"prompt": prompt, "chosen": best_text, "rejected": worst_text,
                "chosen_score": best_score, "rejected_score": worst_score}

    def build_dataset(self, chunks: List[str], out_path: str, n: int = 4) -> int:
        written = 0
        with open(out_path, "w") as f:
            for chunk in tqdm(chunks, desc="Building DPO pairs"):
                pair = self.build_pair(chunk, n=n)
                if pair is not None:
                    f.write(json.dumps(pair) + "\n")
                    written += 1
        print(f"DPO dataset: {written}/{len(chunks)} chunks yielded a usable pair -> {out_path}")
        return written


def run_dpo_training(
    dataset_path: str, base_model_name: str = "Qwen/Qwen2.5-3B-Instruct",
    output_dir: str = "./dpo_output", lora_r: int = 16,
):
    """Skeleton for LoRA + DPO fine-tuning on the dataset
    VLLMDPODatasetBuilder.build_dataset() produces. Trains a FRESH local
    transformers copy of the base model (needs gradient access vLLM's
    inference server doesn't provide) -- backend-agnostic relative to
    whether serving runs through vLLM or not. A starting point, not a
    tuned recipe -- batch size, LoRA rank, and beta need adjusting against
    your actual dataset size and GPU memory.

    Needs: pip install trl peft datasets"""
    from transformers import AutoModelForCausalLM, AutoTokenizer as _AutoTokenizer, BitsAndBytesConfig
    from datasets import load_dataset
    from peft import LoraConfig
    from trl import DPOConfig, DPOTrainer

    dataset = load_dataset("json", data_files=dataset_path, split="train")

    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(base_model_name, quantization_config=bnb, device_map="auto")
    tokenizer = _AutoTokenizer.from_pretrained(base_model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    lora_config = LoraConfig(r=lora_r, lora_alpha=lora_r * 2, lora_dropout=0.05,
                              target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], task_type="CAUSAL_LM")

    training_args = DPOConfig(
        output_dir=output_dir, per_device_train_batch_size=1, gradient_accumulation_steps=8,
        learning_rate=5e-6, beta=0.1, num_train_epochs=1, logging_steps=10,
        bf16=torch.cuda.is_bf16_supported(),
    )

    trainer = DPOTrainer(model=model, args=training_args, train_dataset=dataset,
                          processing_class=tokenizer, peft_config=lora_config)
    trainer.train()
    trainer.save_model(output_dir)
    return output_dir


if DSPY_AVAILABLE:
    import types

    class _VLLMExtractorLM(dspy.LM):
        """Wraps a VLLMTripleExtractor so DSPy's optimizers drive the vLLM
        server instead of an API model. forward() must return an object
        shaped like a litellm ModelResponse (.model, .choices[i].message.content)."""

        def __init__(self, extractor: "VLLMTripleExtractor"):
            super().__init__(model=f"vllm:{extractor.model_name}")
            self.extractor = extractor

        def forward(self, prompt=None, messages=None, **kwargs):
            text = messages[-1]["content"] if messages else (prompt or "")
            raw = _vllm_raw_completion(self.extractor, text, temperature=kwargs.get("temperature", 0.1), use_schema=True)
            return types.SimpleNamespace(
                model=self.model,
                choices=[types.SimpleNamespace(message=types.SimpleNamespace(content=raw))],
            )

    class RawPassthroughAdapter(dspy.adapters.Adapter):
        """Only correct for single-output-field signatures like
        ExtractTriplesSignature below. DSPy's default adapters construct
        their OWN prompt asking for "[[ ## field ## ]]" markers -- a
        different, incompatible JSON shape than the {"triples": [...]}
        object the model here is already prompted (and, via
        structured_outputs.json, generation-time constrained) to emit, so
        the default adapters fail immediately. format() sends the
        extraction prompt as-is; parse() takes the raw completion as the
        single output field with no marker extraction."""

        def format(self, signature, demos, inputs):
            input_field_name = list(signature.input_fields.keys())[0]
            chunk_text = inputs[input_field_name]
            prompt = f"{signature.instructions}\n\n{chunk_text}\n\nJSON:"
            return [{"role": "user", "content": prompt}]

        def parse(self, signature, completion):
            output_field_name = list(signature.output_fields.keys())[0]
            return {output_field_name: completion.strip()}

    class ExtractTriplesSignature(dspy.Signature):
        """Extract relationship triples from text into a knowledge graph,
        as a JSON object {"triples": [...]}, using only the allowed types
        and predicates."""
        chunk_text = dspy.InputField(desc="a chunk of source document text")
        triples = dspy.OutputField(desc='JSON object: {"triples": [{"subject": ..., "subject_type": ..., "predicate": ..., "target": ..., "target_type": ...}, ...]}')

    class TripleExtractionModule(dspy.Module):
        def __init__(self):
            super().__init__()
            self.extract = dspy.Predict(ExtractTriplesSignature)

        def forward(self, chunk_text):
            return self.extract(chunk_text=chunk_text)


def dspy_extraction_metric(example, pred, trace=None) -> float:
    """Reuses the pipeline's own validators as the DSPy optimization
    metric -- same idea as VLLMDPODatasetBuilder._quality_score above."""
    raw = getattr(pred, "triples", "") or ""
    attempted = parse_json_triples(raw, strict=False)
    if not attempted:
        return 0.0
    stats: Dict[str, int] = {}
    strict = parse_json_triples(raw, strict=True, stats=stats)
    normalized = [(s, sp, normalize_predicate(p), t, tp, 0.0) for s, sp, p, t, tp, _ in strict]
    survived = sitl_filter(normalized, stats=stats)
    return len(survived) / len(attempted)


def optimize_extraction_prompt(extractor: "VLLMTripleExtractor", train_chunks: List[str], auto: str = "light"):
    """Runs DSPy's MIPROv2 against your own validators as the metric.
    train_chunks can be any chunks you already have from a processed PDF
    -- unlabeled is fine, the metric IS the label. Needs: pip install dspy"""
    if not DSPY_AVAILABLE:
        raise RuntimeError("dspy is required.\n  pip install dspy")

    lm = _VLLMExtractorLM(extractor)
    dspy.settings.configure(lm=lm, adapter=RawPassthroughAdapter())
    module = TripleExtractionModule()
    trainset = [dspy.Example(chunk_text=c).with_inputs("chunk_text") for c in train_chunks]
    optimizer = dspy.MIPROv2(metric=dspy_extraction_metric, auto=auto)
    return optimizer.compile(module, trainset=trainset)

# No auto-launch_ui() call at module import time (unlike the predecessors
# this was built from) -- `import kg_pipeline` here would need to happen
# BEFORE the vLLM smoke-test cell (to reach PipelineConfig /
# ensure_vllm_server), and an import-time side effect that immediately
# displays the UI widget would fire at the wrong moment in that flow. The
# notebook calls kg_pipeline.launch_ui() itself, explicitly, once the
# server smoke test has already passed.

### Evaluation harness module (`kg_eval.py`)

Written next to `kg_pipeline.py` and imported as `kg_eval`. Gold-set scoring (entity / relation / type / evidence P-R-F1), unsupported-edge rate, graph metrics and graph-QA accuracy, failure taxonomy, threshold sweep, ECE / Brier / isotonic calibration, per-slice tables (document type, language, script), Cohen's kappa, the red-team set, drift (PSI / KS / JS), W&B logging, and the PASS/FAIL regression gate. **No human annotations ship with it** — `SEED_GOLD` is a tiny synthetic smoke-test set.

In [ ]:
%%writefile kg_eval.py
"""kg_eval.py -- evaluation harness for kg_pipeline v2  (design doc sections 14-28)
=============================================================================
Written next to kg_pipeline.py by the `%%writefile kg_eval.py` cell; imports it as `kgp`.

What is in here                                             design doc section
  gold schema / loader / validator, EntityMatcher               18
  entity / relation / type / evidence P-R-F1                     8, 20, 25
  unsupported-edge (hallucination) rate, ontology violations     8
  graph-level metrics + graph-QA accuracy                        20
  failure taxonomy + classifier                                 16
  confidence threshold sweep, ECE / Brier / reliability bins    14, 15
  slice evaluation (doc type, language, script, code-mixing)    23, 24
  Cohen's kappa + annotator agreement                           19
  red-team set (permanent regression tests)                     17
  drift monitoring (PSI / KS / categorical JS)                  28
  W&B logging, regression gate PASS/FAIL                        26, 27
  EvalHarness: one object that runs all of it, plus the preset ablation ladder   26

What is NOT in here: human annotations. SEED_GOLD below is a tiny SYNTHETIC set written by
the author of this module so the harness can be exercised end to end -- it is not a benchmark
and any number computed on it says nothing about how good the model is. Build `gold_v1` with
real annotators (design doc section 18) and point the harness at it.
"""
from __future__ import annotations

import contextlib
import io
import json
import math
import re
import time
from collections import Counter, defaultdict
from dataclasses import dataclass, field, asdict
from enum import Enum
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Set, Tuple

import numpy as np
import polars as pl
from rapidfuzz import fuzz, process as rf_process

import kg_pipeline as kgp
from kg_pipeline import (ALLOWED_ENTITY_TYPES, ALLOWED_PREDICATES, ONTOLOGY, KGQueryEngine,
                         graph_quality_metrics, mention_score, normalize_entity_label, split_sentences)
from kg_pipeline import _numeric_signature

EVAL_VERSION = "kg-eval-1.0"

_WS = re.compile(r"\s+")


def _norm(s: str) -> str:
    return normalize_entity_label(s or "")


def _squash(s: str) -> str:
    return _WS.sub(" ", (s or "")).strip().lower()


def _loose(s: str) -> str:
    return re.sub(r"[\W_]+", "", (s or "").lower())


def _prf(tp: float, fp: float, fn: float) -> Tuple[float, float, float]:
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f = 2 * p * r / (p + r) if (p + r) else 0.0
    return p, r, f


def _tok_f1(a: str, b: str) -> float:
    ta, tb = Counter(_squash(a).split()), Counter(_squash(b).split())
    if not ta or not tb:
        return 0.0
    ov = sum((ta & tb).values())
    if ov == 0:
        return 0.0
    p, r = ov / sum(ta.values()), ov / sum(tb.values())
    return 2 * p * r / (p + r)


# ============================================================================================
# 1. GOLD SCHEMA  (section 18)
# ============================================================================================
NEGATIVE_REASONS = ("wrong", "negated", "hypothetical", "conditional", "attributed", "uncertain", "direction")


@dataclass
class GoldEntity:
    name: str
    type: str
    aliases: List[str] = field(default_factory=list)


@dataclass
class GoldRelation:
    subject: str
    predicate: str
    object: str
    evidence: str = ""                      # the sentence a human pointed at
    page: int = 0
    coref: bool = False                     # one endpoint is only a pronoun / definite NP in the evidence
    slices: List[str] = field(default_factory=list)   # relation-level tags, e.g. ["cross_page", "long_distance"]


@dataclass
class GoldNegative:
    """A triple the text does NOT license (negated, hypothetical, attributed, direction-flipped, ...).
    If it shows up in the graph, that is counted as a false positive of a specific kind."""
    subject: str
    predicate: str
    object: str
    reason: str = "wrong"                   # one of NEGATIVE_REASONS


@dataclass
class GoldQuestion:
    """Graph-QA item. role='target': ? --p--> anchor   role='source': anchor --p--> ?
    answers=[] means 'the graph should return nothing' (tests false positives)."""
    question: str
    anchor: str
    role: str = "target"
    predicates: Optional[List[str]] = None
    answer_type: Optional[str] = None
    answers: List[str] = field(default_factory=list)


@dataclass
class GoldContradiction:
    """The document disagrees with itself here (e.g. 'requires CUDA 12' in section 2, 'requires CUDA 13' in section 9).
    kind is a kg_pipeline Contradiction.kind: VERSION_CONFLICT | NEGATION_CONFLICT | DIRECTION_CONFLICT | FUNCTIONAL_CONFLICT | TYPE_CONFLICT."""
    kind: str
    subject: str
    predicate: str


@dataclass
class GoldDoc:
    doc_id: str
    text: str = ""
    pdf_path: str = ""                      # alternative to `text` (page numbers only exist for PDFs)
    entities: List[GoldEntity] = field(default_factory=list)
    relations: List[GoldRelation] = field(default_factory=list)
    negatives: List[GoldNegative] = field(default_factory=list)
    questions: List[GoldQuestion] = field(default_factory=list)
    contradictions: List[GoldContradiction] = field(default_factory=list)
    slices: List[str] = field(default_factory=list)   # doc-level: clean, ocr, tables, long_section, dense_technical,
    #                                                   short_paragraphs, cross_page, coreference, long_distance ...
    language: str = "en"                    # ISO 639-1; 'hi','ta','te','or','bn', ...
    script: str = "Latn"                    # ISO 15924; 'Deva','Taml','Telu','Orya','Beng', ...
    code_mixed: bool = False                # e.g. Hinglish
    transliterated: bool = False
    annotator: str = ""

    def source_text(self) -> str:
        return self.text


@dataclass
class GoldSet:
    version: str
    docs: List[GoldDoc]
    description: str = ""
    synthetic: bool = False                 # True => NOT human-annotated; scoreboard is illustrative only

    def n_relations(self) -> int:
        return sum(len(d.relations) for d in self.docs)

    def to_dict(self) -> Dict:
        return asdict(self)

    @staticmethod
    def from_dict(d: Dict) -> "GoldSet":
        docs = []
        for x in d.get("docs", []):
            docs.append(GoldDoc(
                doc_id=x["doc_id"], text=x.get("text", ""), pdf_path=x.get("pdf_path", ""),
                entities=[GoldEntity(**e) for e in x.get("entities", [])],
                relations=[GoldRelation(**r) for r in x.get("relations", [])],
                negatives=[GoldNegative(**n) for n in x.get("negatives", [])],
                questions=[GoldQuestion(**q) for q in x.get("questions", [])],
                contradictions=[GoldContradiction(**c) for c in x.get("contradictions", [])],
                slices=list(x.get("slices", [])), language=x.get("language", "en"), script=x.get("script", "Latn"),
                code_mixed=bool(x.get("code_mixed", False)), transliterated=bool(x.get("transliterated", False)),
                annotator=x.get("annotator", "")))
        return GoldSet(version=d.get("version", "unversioned"), docs=docs, description=d.get("description", ""),
                       synthetic=bool(d.get("synthetic", False)))


def save_gold(gold: GoldSet, path: str) -> str:
    Path(path).write_text(json.dumps(gold.to_dict(), indent=2, ensure_ascii=False))
    return path


def load_gold(path: str) -> GoldSet:
    return GoldSet.from_dict(json.loads(Path(path).read_text()))


def blank_annotation_template(doc_id: str, text: str, annotator: str = "") -> Dict:
    """Empty file for a human to fill. Deliberately NOT pre-filled from model output: for the
    inter-annotator subset, anchoring annotators on the model's guesses would inflate kappa."""
    return asdict(GoldDoc(doc_id=doc_id, text=text, annotator=annotator,
                          entities=[GoldEntity("", "", [])], relations=[GoldRelation("", "", "")]))


class EntityMatcher:
    """Maps a predicted label to a gold entity key: exact (normalised) -> alias -> punctuation-insensitive
    -> fuzzy (WRatio >= cutoff) with the same version-number guard the pipeline uses, so 'CUDA 12' never
    matches a gold 'CUDA 13'."""

    def __init__(self, entities: Sequence[GoldEntity], cutoff: float = 90.0):
        self.cutoff = cutoff
        self.surface: Dict[str, str] = {}
        self.loose: Dict[str, str] = {}
        self.type_of: Dict[str, str] = {}
        self.name_of: Dict[str, str] = {}
        for e in entities:
            key = _norm(e.name)
            self.type_of[key] = e.type
            self.name_of[key] = e.name
            for s in [e.name] + list(e.aliases):
                self.surface[_norm(s)] = key
                self.loose.setdefault(_loose(s), key)
        self._surfaces = list(self.surface)

    def match(self, label: str) -> Optional[str]:
        k = _norm(label)
        if not k:
            return None
        if k in self.surface:
            return self.surface[k]
        lk = _loose(label)
        if lk in self.loose:
            return self.loose[lk]
        if not self._surfaces:
            return None
        hit = rf_process.extractOne(k, self._surfaces, scorer=fuzz.WRatio, score_cutoff=self.cutoff)
        if hit and _numeric_signature(hit[0]) == _numeric_signature(k):
            return self.surface[hit[0]]
        return None

    def key(self, label: str) -> str:
        return self.match(label) or _norm(label)

    def near(self, label: str, cutoff: float = 70.0) -> Optional[str]:
        """Looser than match(): 'is this probably a variant of a gold entity?' (used for ENTITY_SPLIT)."""
        k = _norm(label)
        if not k or not self._surfaces:
            return None
        hit = rf_process.extractOne(k, self._surfaces, scorer=fuzz.token_set_ratio, score_cutoff=cutoff)
        return self.surface[hit[0]] if hit else None


def validate_gold(gold: GoldSet) -> List[str]:
    """Annotation lint. 'ERROR:' lines make the gold set unusable; 'WARN:' lines are worth a look."""
    out: List[str] = []
    seen_ids: Set[str] = set()
    for d in gold.docs:
        if d.doc_id in seen_ids:
            out.append(f"ERROR: duplicate doc_id {d.doc_id}")
        seen_ids.add(d.doc_id)
        if not d.text and not d.pdf_path:
            out.append(f"ERROR: {d.doc_id}: neither text nor pdf_path")
        m = EntityMatcher(d.entities)
        for e in d.entities:
            if e.type not in ALLOWED_ENTITY_TYPES:
                out.append(f"ERROR: {d.doc_id}: entity '{e.name}' has type {e.type} not in ALLOWED_ENTITY_TYPES")
        seen_rel: Set[Tuple[str, str, str]] = set()
        for r in d.relations:
            if r.predicate not in ALLOWED_PREDICATES:
                out.append(f"ERROR: {d.doc_id}: predicate '{r.predicate}' is not in the pipeline vocabulary")
            s, o = m.match(r.subject), m.match(r.object)
            if s is None or o is None:
                out.append(f"ERROR: {d.doc_id}: relation ({r.subject}, {r.predicate}, {r.object}) uses an entity "
                           f"that is not in the entity list")
            else:
                viol = [v for v in ONTOLOGY.check(m.type_of[s], r.predicate, m.type_of[o])
                        if v.startswith(kgp.HARD_VIOLATION_PREFIXES)]
                if viol:
                    out.append(f"WARN: {d.doc_id}: gold relation ({r.subject}, {r.predicate}, {r.object}) violates the "
                               f"ontology {viol} -- the pipeline cannot accept it; fix the gold type or extend the schema")
                k = (s, r.predicate, o)
                if k in seen_rel:
                    out.append(f"WARN: {d.doc_id}: duplicate gold relation {k}")
                seen_rel.add(k)
            if d.text and r.evidence and _squash(r.evidence) not in _squash(d.text):
                out.append(f"WARN: {d.doc_id}: evidence for ({r.subject}, {r.predicate}, {r.object}) is not a substring of the text")
        for n in d.negatives:
            if n.reason not in NEGATIVE_REASONS:
                out.append(f"WARN: {d.doc_id}: negative reason '{n.reason}' not in {NEGATIVE_REASONS}")
        for q in d.questions:
            if q.role not in ("source", "target"):
                out.append(f"ERROR: {d.doc_id}: question role must be 'source' or 'target'")
    if gold.n_relations() < 30:
        out.append(f"WARN: only {gold.n_relations()} gold relations -- F1 differences below ~0.1 are noise at this size")
    return out


# ============================================================================================
# 2. SCORING ONE DOCUMENT
# ============================================================================================
def predicted_edges(edges_pl: Optional[pl.DataFrame]) -> List[Dict]:
    if edges_pl is None or len(edges_pl) == 0:
        return []
    out = []
    for r in edges_pl.to_dicts():
        out.append({"s": r["source_str"], "p": r["predicate_str"], "o": r["target_str"],
                    "s_type": r.get("source_type") or "", "o_type": r.get("target_type") or "",
                    "conf": float(r.get("confidence") or 0.0), "evidence": (r.get("evidence") or "").strip(),
                    "page": r.get("page"), "src": r.get("edge_source") or "extracted",
                    "claim_ids": r.get("claim_ids") or ""})
    return out


class FailureType(str, Enum):
    ENTITY_MISS = "ENTITY_MISS"
    ENTITY_MERGE = "ENTITY_MERGE"
    ENTITY_SPLIT = "ENTITY_SPLIT"
    TYPE_ERROR = "TYPE_ERROR"
    RELATION_ERROR = "RELATION_ERROR"
    DIRECTION_ERROR = "DIRECTION_ERROR"
    COREFERENCE_ERROR = "COREFERENCE_ERROR"
    NEGATION_ERROR = "NEGATION_ERROR"
    MODALITY_ERROR = "MODALITY_ERROR"      # not in the design doc's list: hypothetical / attributed / uncertain read as fact
    HALLUCINATION = "HALLUCINATION"
    OCR_ERROR = "OCR_ERROR"
    CHUNKING_ERROR = "CHUNKING_ERROR"
    ONTOLOGY_VIOLATION = "ONTOLOGY_VIOLATION"
    DUPLICATE_EDGE = "DUPLICATE_EDGE"
    DUPLICATE_NODE = "DUPLICATE_NODE"


@dataclass
class Failure:
    type: str
    doc_id: str
    mode: str                               # missed | spurious | wrong
    subject: str = ""
    predicate: str = ""
    object: str = ""
    detail: str = ""
    evidence: str = ""

    def to_row(self) -> Dict:
        return asdict(self)


@dataclass
class DocScore:
    doc_id: str
    slices: List[str]
    language: str
    script: str
    code_mixed: bool
    n_chars: int = 0
    # entities (end-to-end: nodes of the final graph)
    ent_tp: int = 0
    ent_fp: int = 0
    ent_fn: int = 0
    type_total: int = 0
    type_exact: int = 0
    type_compat: int = 0
    # entity discovery (v2 only: the entity memory after Pass 1/2)
    disc_tp: int = 0
    disc_fp: int = 0
    disc_fn: int = 0
    disc_available: bool = False
    # relations
    rel_tp: int = 0
    rel_fp: int = 0
    rel_fn: int = 0
    rel_dir_swapped: int = 0                # FN whose reverse was predicted
    gold_total: int = 0
    pred_total: int = 0                     # unique predicted keys
    pred_edges: int = 0                     # rows in the edge table
    # evidence / hallucination (independent of the pipeline's own flags)
    ev_present: int = 0
    ev_in_source: int = 0
    supported: int = 0
    unsupported: int = 0
    ev_acc_ok: int = 0
    ev_acc_total: int = 0
    neg_hits: int = 0
    # graph QA
    qa_total: int = 0
    qa_exact: int = 0
    qa_f1_sum: float = 0.0
    # tagged recall by relation-level slice: tag -> [hit, total]
    rel_slice: Dict[str, List[int]] = field(default_factory=dict)
    labelled: List[Tuple[float, int, str]] = field(default_factory=list)   # (score, is_correct, key-str) for calibration
    failures: List[Failure] = field(default_factory=list)
    graph: Dict = field(default_factory=dict)
    ledger: Dict[str, float] = field(default_factory=dict)
    chunk_latency_s: List[float] = field(default_factory=list)
    gpu_gb: float = 0.0
    cpu_gb: float = 0.0
    n_contradictions: int = 0
    contradiction_kinds: Dict[str, int] = field(default_factory=dict)
    n_chunks: int = 0
    contra_tp: int = 0
    contra_fp: int = 0
    contra_fn: int = 0


def _sentence_windows(text: str, radius: int = 1) -> List[Tuple[int, int, str]]:
    sents = split_sentences(text)
    out = []
    for i in range(len(sents)):
        a, b = max(0, i - radius), min(len(sents) - 1, i + radius)
        out.append((sents[i][1], sents[i][2], text[sents[a][1]:sents[b][2]]))
    return out


def edge_support(edge: Dict, text: str, windows: Optional[List] = None) -> Tuple[bool, bool, bool]:
    """GROUNDING check, independent of the pipeline's own evidence flag. Returns
    (has_evidence, evidence_found_in_source, supported). `supported` means the evidence quote really occurs in
    the source AND the +-1 sentence window around it mentions both endpoints (so a resolved pronoun still
    passes). Edges with no evidence (v1 path) fall back to 'some 2-sentence window mentions both endpoints'.
    This is necessary-not-sufficient: it catches invented quotes/edges, not a real sentence that says
    something different -- relation precision against gold covers that."""
    ev = edge["evidence"]
    windows = windows if windows is not None else _sentence_windows(text, 1)
    both = lambda t: mention_score(edge["s"], t) >= 80 and mention_score(edge["o"], t) >= 80
    if not ev:
        pairs = _sentence_windows(text, 1)
        return False, False, any(both(w[2]) for w in pairs)
    t_sq, e_sq = _squash(text), _squash(ev)
    in_src = e_sq in t_sq or (len(e_sq) > 20 and fuzz.partial_ratio(e_sq, t_sq) >= 92)
    if not in_src:
        return True, False, False
    pos = t_sq.find(e_sq)
    if pos < 0:
        return True, True, both(ev)
    # map the squashed offset back to a sentence window by searching the sentence that contains the quote's first words
    head = e_sq[:40]
    for a, b, w in windows:
        if head and head in _squash(w) and both(w):
            return True, True, True
    return True, True, both(ev)


def score_document(gold: GoldDoc, result: Dict, qa: bool = True) -> DocScore:
    """`result` is StreamableKGPipeline.run_document(...)'s dict."""
    text = gold.source_text()
    matcher = EntityMatcher(gold.entities)
    edges = predicted_edges(result.get("edges"))
    ds = DocScore(doc_id=gold.doc_id, slices=list(gold.slices), language=gold.language, script=gold.script,
                  code_mixed=gold.code_mixed, n_chars=len(text))

    gk = matcher.key
    gold_rel: Dict[Tuple[str, str, str], GoldRelation] = {(gk(r.subject), r.predicate, gk(r.object)): r for r in gold.relations}
    gold_neg: Dict[Tuple[str, str, str], GoldNegative] = {(gk(n.subject), n.predicate, gk(n.object)): n for n in gold.negatives}
    pred: Dict[Tuple[str, str, str], List[Dict]] = defaultdict(list)
    for e in edges:
        pred[(gk(e["s"]), e["p"], gk(e["o"]))].append(e)

    tp_keys = set(pred) & set(gold_rel)
    ds.gold_total, ds.pred_total, ds.pred_edges = len(gold_rel), len(pred), len(edges)
    ds.rel_tp, ds.rel_fp, ds.rel_fn = len(tp_keys), len(set(pred) - set(gold_rel)), len(set(gold_rel) - set(pred))
    for k in set(gold_rel) - set(pred):
        s, p, o = k
        inv = ONTOLOGY.inverse(p)
        if (o, p, s) in pred or (inv and (o, inv, s) in pred):
            ds.rel_dir_swapped += 1
    ds.neg_hits = sum(1 for k in pred if k in gold_neg)

    # relation-level slice recall
    for k, r in gold_rel.items():
        for tag in r.slices:
            hit_tot = ds.rel_slice.setdefault(tag, [0, 0])
            hit_tot[1] += 1
            hit_tot[0] += int(k in pred)

    # entities: nodes of the final graph
    node_labels: Dict[str, List[Tuple[str, str]]] = defaultdict(list)       # key -> [(label, type)]
    for e in edges:
        node_labels[gk(e["s"])].append((e["s"], e["s_type"]))
        node_labels[gk(e["o"])].append((e["o"], e["o_type"]))
    gold_ent_keys = set(matcher.type_of)
    pred_ent_keys = set(node_labels)
    ds.ent_tp, ds.ent_fp, ds.ent_fn = len(pred_ent_keys & gold_ent_keys), len(pred_ent_keys - gold_ent_keys), len(gold_ent_keys - pred_ent_keys)
    for k in pred_ent_keys & gold_ent_keys:
        pt = Counter(t for _, t in node_labels[k] if t).most_common(1)
        if not pt:
            continue
        ds.type_total += 1
        gt = matcher.type_of[k]
        ds.type_exact += int(pt[0][0] == gt)
        ds.type_compat += int(pt[0][0] == gt or ONTOLOGY.matches(pt[0][0], {gt}) or ONTOLOGY.matches(gt, {pt[0][0]}))

    # entity discovery from the pipeline's entity memory
    dkeys = _memory_keys(result, matcher)
    if dkeys:
        ds.disc_available = True
        ds.disc_tp, ds.disc_fp, ds.disc_fn = len(dkeys & gold_ent_keys), len(dkeys - gold_ent_keys), len(gold_ent_keys - dkeys)

    # evidence / hallucination
    windows = _sentence_windows(text, 1) if text else []
    support_by_key: Dict[Tuple[str, str, str], bool] = {}
    for k, es in pred.items():
        ok_any = False
        for e in es:
            has_ev, in_src, sup = edge_support(e, text, windows) if text else (bool(e["evidence"]), True, True)
            ds.ev_present += int(has_ev)
            ds.ev_in_source += int(in_src)
            ok_any = ok_any or sup
        support_by_key[k] = ok_any
    ds.supported = sum(1 for v in support_by_key.values() if v)
    ds.unsupported = len(support_by_key) - ds.supported
    for k in tp_keys:
        g = gold_rel[k]
        best = max(pred[k], key=lambda e: len(e["evidence"]))
        if g.evidence:
            ok = bool(best["evidence"]) and (_tok_f1(best["evidence"], g.evidence) >= 0.5
                                             or _squash(g.evidence) in _squash(best["evidence"])
                                             or _squash(best["evidence"]) in _squash(g.evidence))
        else:
            ok = support_by_key.get(k, False)
        ds.ev_acc_total += 1
        ds.ev_acc_ok += int(ok)

    # confidence labels for calibration / threshold sweep: ledger claims (pre-filter) when present
    claims = [c for c in (result.get("claims") or []) if getattr(c, "source", "llm") == "llm"]
    if claims:
        by_key: Dict[Tuple[str, str, str], List[float]] = defaultdict(list)
        for c in claims:
            if c.verdict == "ACCEPT":
                by_key[(gk(c.subject), c.predicate, gk(c.target))].append(float(c.confidence))
        ds.labelled = [(float(np.mean(v)), int(k in gold_rel), "|".join(k)) for k, v in by_key.items()]
    else:
        ds.labelled = [(float(np.mean([e["conf"] for e in es])), int(k in gold_rel), "|".join(k)) for k, es in pred.items()]

    # graph-level + system
    if len(edges):
        try:
            ds.graph = graph_quality_metrics(result["edges"])
        except Exception as ex:                                       # pragma: no cover
            ds.graph = {"error": str(ex)}
    led = result.get("ledger")
    if led is not None:
        ds.ledger = result.get("summary") or led.summary()
        for b in led.batch_stats:
            ds.n_chunks += int(b.get("n_chunks", 0))
            if b.get("n_chunks"):
                ds.chunk_latency_s.append(b["batch_time_s"] / max(1, b["n_chunks"]))
            ds.gpu_gb = max(ds.gpu_gb, b.get("gpu_used_gb", 0.0))
            ds.cpu_gb = max(ds.cpu_gb, b.get("cpu_rss_gb", 0.0))
    cons = result.get("contradictions") or []
    ds.n_contradictions = len(cons)
    ds.contradiction_kinds = dict(Counter(c.kind for c in cons))
    gc_keys = {(x.kind, gk(x.subject), x.predicate) for x in gold.contradictions}
    pc_keys = {(c.kind, gk(c.subject), c.predicate) for c in cons}
    ds.contra_tp, ds.contra_fp, ds.contra_fn = len(gc_keys & pc_keys), len(pc_keys - gc_keys), len(gc_keys - pc_keys)

    # graph QA
    if qa and gold.questions and len(edges):
        eng = KGQueryEngine(result["edges"])
        for q in gold.questions:
            r = eng.query(q.anchor, role=q.role, predicates=q.predicates, answer_type=q.answer_type)
            got = {gk(a["label"]) for a in r.get("answers", [])}
            want = {gk(a) for a in q.answers}
            ds.qa_total += 1
            ds.qa_exact += int(got == want)
            if not want and not got:
                ds.qa_f1_sum += 1.0
            else:
                ds.qa_f1_sum += _prf(len(got & want), len(got - want), len(want - got))[2]
    elif qa and gold.questions:
        ds.qa_total += len(gold.questions)
        ds.qa_exact += sum(1 for q in gold.questions if not q.answers)
        ds.qa_f1_sum += sum(1.0 for q in gold.questions if not q.answers)

    ds.failures = classify_failures(gold, result, matcher, edges, pred, gold_rel, gold_neg, support_by_key)
    return ds


# ============================================================================================
# 3. FAILURE TAXONOMY  (section 16)
# ============================================================================================
def _memory_keys(result: Dict, matcher: EntityMatcher) -> Set[str]:
    recs = getattr(result.get("memory"), "records", None) or {}
    keys: Set[str] = set()
    for r in recs.values():
        hit = None
        for lab in [r.canonical_name] + sorted(r.aliases):
            hit = matcher.match(lab)
            if hit:
                break
        keys.add(hit or _norm(r.canonical_name))
    return keys


def classify_failures(gold: GoldDoc, result: Dict, matcher: EntityMatcher, edges: List[Dict],
                      pred: Dict[Tuple[str, str, str], List[Dict]],
                      gold_rel: Dict[Tuple[str, str, str], GoldRelation],
                      gold_neg: Dict[Tuple[str, str, str], GoldNegative],
                      support_by_key: Dict[Tuple[str, str, str], bool]) -> List[Failure]:
    """Rule-based, FIRST MATCH WINS, so each bad prediction / miss gets exactly one label. This is a
    triage heuristic -- a human reading 20 failures will re-label some -- but it is deterministic and
    it tells you which engineering lever (chunker, memory, ontology, verifier) to pull first.
    Misses (gold relation absent):  CHUNKING > ENTITY_MISS/OCR/COREFERENCE > DIRECTION > ONTOLOGY/NEGATION(ledger) > RELATION.
    Spurious (predicted, not gold): known-negative kind > HALLUCINATION > DIRECTION > ONTOLOGY > TYPE/RELATION/ENTITY_SPLIT."""
    out: List[Failure] = []
    doc = gold.doc_id
    gk = matcher.key
    hier = result.get("hier")
    chunk_texts = [_squash(c.text) for c in getattr(hier, "chunks", [])] if hier is not None else []
    claims = [c for c in (result.get("claims") or []) if getattr(c, "source", "llm") == "llm"]
    rejected: Dict[Tuple[str, str, str], List[Any]] = defaultdict(list)
    for c in claims:
        if c.verdict in ("REJECT", "REVIEW"):
            rejected[(gk(c.subject), c.predicate, gk(c.target))].append(c)
    pred_nodes: Set[str] = set()
    node_labels: Dict[str, Set[str]] = defaultdict(set)
    node_types: Dict[str, Counter] = defaultdict(Counter)
    for e in edges:
        for lab, ty in ((e["s"], e["s_type"]), (e["o"], e["o_type"])):
            k = gk(lab)
            pred_nodes.add(k)
            node_labels[k].add(lab)
            if ty:
                node_types[k][ty] += 1
    seen_nodes = pred_nodes | _memory_keys(result, matcher)
    ocr_doc = "ocr" in gold.slices
    dir_done: Set[Tuple[str, str, str]] = set()      # gold keys already reported as a direction error (one event, one label)

    # ---- missed gold relations
    for k, g in gold_rel.items():
        if k in pred:
            continue
        s, p, o = k
        base = dict(doc_id=doc, mode="missed", subject=g.subject, predicate=p, object=g.object, evidence=g.evidence)
        inv = ONTOLOGY.inverse(p)
        if chunk_texts and g.evidence and not any(_squash(g.evidence) in t for t in chunk_texts):
            out.append(Failure(FailureType.CHUNKING_ERROR.value, detail="gold evidence is not inside any single chunk", **base))
        elif s not in seen_nodes or o not in seen_nodes:
            who = g.subject if s not in seen_nodes else g.object
            if ocr_doc:
                out.append(Failure(FailureType.OCR_ERROR.value, detail=f"entity '{who}' never surfaced on an OCR-tagged doc", **base))
            elif g.coref:
                out.append(Failure(FailureType.COREFERENCE_ERROR.value, detail=f"endpoint '{who}' only appears as a pronoun/definite NP", **base))
            else:
                out.append(Failure(FailureType.ENTITY_MISS.value, detail=f"entity '{who}' never surfaced", **base))
        elif (o, p, s) in pred or (inv and (o, inv, s) in pred):
            out.append(Failure(FailureType.DIRECTION_ERROR.value, detail="reverse direction was predicted",
                               **{**base, "mode": "wrong"}))
            dir_done.add(k)
        elif k in rejected:
            c = rejected[k][0]
            reason = c.reject_reason or c.verdict
            if "ontology" in reason or "forbidden" in reason or "domain" in reason or "range" in reason:
                out.append(Failure(FailureType.ONTOLOGY_VIOLATION.value, detail=f"pipeline rejected a correct claim: {reason}", **base))
            elif reason == "negated" or c.modality == "negated":
                out.append(Failure(FailureType.NEGATION_ERROR.value, detail="correct claim was treated as negated", **base))
            elif g.coref:
                out.append(Failure(FailureType.COREFERENCE_ERROR.value, detail=f"coref claim rejected: {reason}", **base))
            else:
                out.append(Failure(FailureType.RELATION_ERROR.value, detail=f"claim was proposed then rejected: {reason}", **base))
        elif any(k2[0] == s and k2[2] == o and k2[1] != p for k2 in pred):
            other = [k2[1] for k2 in pred if k2[0] == s and k2[2] == o][0]
            out.append(Failure(FailureType.RELATION_ERROR.value, detail=f"predicted '{other}' instead of '{p}'", **{**base, "mode": "wrong"}))
        elif g.coref:
            out.append(Failure(FailureType.COREFERENCE_ERROR.value, detail="relation needs pronoun resolution and was not found", **base))
        else:
            out.append(Failure(FailureType.RELATION_ERROR.value, detail="relation never proposed", **base))

    # ---- spurious predicted relations
    for k, es in pred.items():
        if k in gold_rel:
            continue
        s, p, o = k
        e = es[0]
        base = dict(doc_id=doc, mode="spurious", subject=e["s"], predicate=p, object=e["o"], evidence=e["evidence"])
        inv = ONTOLOGY.inverse(p)
        hard = [v for v in ONTOLOGY.check(e["s_type"], p, e["o_type"]) if v.startswith(kgp.HARD_VIOLATION_PREFIXES)]
        if k in gold_neg:
            reason = gold_neg[k].reason
            if reason == "negated":
                out.append(Failure(FailureType.NEGATION_ERROR.value, detail="negated statement became an edge", **base))
            elif reason in ("hypothetical", "conditional", "attributed", "uncertain"):
                out.append(Failure(FailureType.MODALITY_ERROR.value, detail=f"{reason} statement became an asserted edge", **base))
            elif reason == "direction":
                out.append(Failure(FailureType.DIRECTION_ERROR.value, detail="known direction-flipped triple", **base))
            else:
                out.append(Failure(FailureType.RELATION_ERROR.value, detail="listed as a known-wrong triple", **base))
        elif not support_by_key.get(k, True):
            out.append(Failure(FailureType.HALLUCINATION.value, detail="no verifiable textual support for this edge", **base))
        elif (o, p, s) in gold_rel or (inv and (o, inv, s) in gold_rel):
            if (o, p, s) not in dir_done and not (inv and (o, inv, s) in dir_done):
                out.append(Failure(FailureType.DIRECTION_ERROR.value, detail="gold has the reverse direction", **base))
        elif hard:
            out.append(Failure(FailureType.ONTOLOGY_VIOLATION.value, detail=";".join(hard), **base))
        elif s in matcher.type_of and o in matcher.type_of:
            same_pair = [k2[1] for k2 in gold_rel if k2[0] == s and k2[2] == o]
            if same_pair:
                out.append(Failure(FailureType.RELATION_ERROR.value, detail=f"gold says '{same_pair[0]}' for this pair", **{**base, "mode": "wrong"}))
            else:
                out.append(Failure(FailureType.RELATION_ERROR.value,
                                   detail="not in gold (could be a real relation nobody annotated -- check before fixing the model)", **base))
        else:
            unk = e["s"] if s not in matcher.type_of else e["o"]
            near = matcher.near(unk)
            if near:
                out.append(Failure(FailureType.ENTITY_SPLIT.value, detail=f"'{unk}' looks like a variant of gold entity '{matcher.name_of[near]}'", **base))
            else:
                out.append(Failure(FailureType.RELATION_ERROR.value,
                                   detail=f"endpoint '{unk}' is not a gold entity (annotation gap or over-extraction)", **base))

    # ---- node-level: wrong type, duplicate nodes, split entity
    for key, labs in node_labels.items():
        if key in matcher.type_of and node_types[key]:
            pt = node_types[key].most_common(1)[0][0]
            gt = matcher.type_of[key]
            if not (pt == gt or ONTOLOGY.matches(pt, {gt}) or ONTOLOGY.matches(gt, {pt})):
                out.append(Failure(FailureType.TYPE_ERROR.value, doc_id=doc, mode="wrong", subject=matcher.name_of[key],
                                   detail=f"predicted {pt}, gold {gt}"))
        distinct = {_norm(l) for l in labs}
        if len(distinct) > 1:
            loose = {_loose(l) for l in labs}
            ftype = FailureType.DUPLICATE_NODE if len(loose) < len(distinct) else FailureType.ENTITY_SPLIT
            out.append(Failure(ftype.value, doc_id=doc, mode="wrong", subject=sorted(labs)[0],
                               detail=f"one entity surfaced as {sorted(labs)}"))

    # ---- duplicate edges (same relation twice, or both a relation and its inverse)
    for k, es in pred.items():
        if len(es) > 1:
            out.append(Failure(FailureType.DUPLICATE_EDGE.value, doc_id=doc, mode="spurious", subject=es[0]["s"],
                               predicate=k[1], object=es[0]["o"], detail=f"{len(es)} rows for one relation"))
        inv = ONTOLOGY.inverse(k[1])
        if inv and (k[2], inv, k[0]) in pred and k[0] < k[2]:
            out.append(Failure(FailureType.DUPLICATE_EDGE.value, doc_id=doc, mode="spurious", subject=es[0]["s"],
                               predicate=k[1], object=es[0]["o"], detail=f"also present as inverse '{inv}'"))

    # ---- entity merges the pipeline performed that gold says are different things
    canon = (result.get("alias_log") or {}).get("canon_map") or {}
    for variant, canonical in canon.items():
        a, b = matcher.match(variant), matcher.match(canonical)
        if a and b and a != b:
            out.append(Failure(FailureType.ENTITY_MERGE.value, doc_id=doc, mode="wrong", subject=variant, object=canonical,
                               detail=f"resolver merged '{variant}' into '{canonical}' but gold keeps them apart"))
    return out


def failure_distribution(failures: Sequence[Failure]) -> pl.DataFrame:
    c = Counter(f.type for f in failures)
    n = sum(c.values())
    rows = [{"failure_type": t, "count": k, "share": k / n if n else 0.0} for t, k in c.most_common()]
    return pl.DataFrame(rows, schema={"failure_type": pl.Utf8, "count": pl.Int64, "share": pl.Float64}) if rows else \
        pl.DataFrame(schema={"failure_type": pl.Utf8, "count": pl.Int64, "share": pl.Float64})


# ============================================================================================
# 4. AGGREGATION, CONFIDENCE INTERVALS, SLICES  (sections 20, 23, 24, 25)
# ============================================================================================
def _sum(ds: Sequence[DocScore], attr: str) -> float:
    return float(sum(getattr(d, attr) for d in ds))


def relation_counts_f1(ds: Sequence[DocScore]) -> float:
    return _prf(_sum(ds, "rel_tp"), _sum(ds, "rel_fp"), _sum(ds, "rel_fn"))[2]


def entity_counts_f1(ds: Sequence[DocScore]) -> float:
    return _prf(_sum(ds, "ent_tp"), _sum(ds, "ent_fp"), _sum(ds, "ent_fn"))[2]


def bootstrap_ci(ds: Sequence[DocScore], fn: Callable[[Sequence[DocScore]], float], n: int = 1000,
                 seed: int = 0, alpha: float = 0.05) -> Tuple[float, float]:
    """Percentile bootstrap over DOCUMENTS (the unit that is independent). With < 5 docs the
    interval is returned as (nan, nan): it would be fiction."""
    if len(ds) < 5:
        return (float("nan"), float("nan"))
    rng = np.random.default_rng(seed)
    idx = np.arange(len(ds))
    vals = [fn([ds[i] for i in rng.choice(idx, size=len(ds), replace=True)]) for _ in range(n)]
    return float(np.quantile(vals, alpha / 2)), float(np.quantile(vals, 1 - alpha / 2))


def aggregate(ds: Sequence[DocScore]) -> Dict[str, float]:
    """Corpus-level flat metrics dict. Counts are SUMMED across documents (micro-average), never
    averaged F1-of-F1s. Names are the ones the regression gate and W&B use."""
    m: Dict[str, float] = {"docs": float(len(ds)), "gold_relations": _sum(ds, "gold_total")}
    ep, er, ef = _prf(_sum(ds, "ent_tp"), _sum(ds, "ent_fp"), _sum(ds, "ent_fn"))
    rp, rr, rf = _prf(_sum(ds, "rel_tp"), _sum(ds, "rel_fp"), _sum(ds, "rel_fn"))
    ru = _prf(_sum(ds, "rel_tp") + _sum(ds, "rel_dir_swapped"), _sum(ds, "rel_fp") - _sum(ds, "rel_dir_swapped"),
              _sum(ds, "rel_fn") - _sum(ds, "rel_dir_swapped"))[2]
    m.update({"entity_precision": ep, "entity_recall": er, "entity_f1": ef,
              "relation_precision": rp, "relation_recall": rr, "relation_f1": rf,
              "relation_f1_ignoring_direction": ru})
    dd = [d for d in ds if d.disc_available]
    if dd:
        dp, dr, df_ = _prf(_sum(dd, "disc_tp"), _sum(dd, "disc_fp"), _sum(dd, "disc_fn"))
        m.update({"discovery_precision": dp, "discovery_recall": dr, "discovery_f1": df_})
    tt = _sum(ds, "type_total")
    m["type_accuracy"] = _sum(ds, "type_exact") / tt if tt else float("nan")
    m["type_accuracy_compatible"] = _sum(ds, "type_compat") / tt if tt else float("nan")
    ea = _sum(ds, "ev_acc_total")
    m["evidence_accuracy"] = _sum(ds, "ev_acc_ok") / ea if ea else float("nan")
    pe, pt_ = _sum(ds, "pred_edges"), _sum(ds, "pred_total")
    m["evidence_coverage"] = _sum(ds, "ev_present") / pe if pe else float("nan")
    evp = _sum(ds, "ev_present")
    m["evidence_in_source_rate"] = _sum(ds, "ev_in_source") / evp if evp else float("nan")
    m["unsupported_edge_rate"] = _sum(ds, "unsupported") / pt_ if pt_ else float("nan")
    m["hallucination_rate"] = m["unsupported_edge_rate"]
    m["known_bad_edge_rate"] = _sum(ds, "neg_hits") / pt_ if pt_ else float("nan")
    # pipeline's own number, for comparison with the independent one above
    acc = sum(d.ledger.get("claims/accepted", 0.0) for d in ds)
    if any(d.ledger for d in ds):
        m["pipeline_unsupported_claim_rate"] = (sum(d.ledger.get("quality/unsupported_claim_rate", 0.0) *
                                                    d.ledger.get("claims/accepted", 0.0) for d in ds) / acc) if acc else 0.0
        cand = sum(d.ledger.get("claims/candidates", 0.0) for d in ds)
        m["candidate_ungrounded_rate"] = (sum(d.ledger.get("quality/candidate_ungrounded_rate", 0.0) *
                                              d.ledger.get("claims/candidates", 0.0) for d in ds) / cand) if cand else 0.0
        m["claims_review"] = sum(d.ledger.get("claims/review", 0.0) for d in ds)
        m["claims_rejected"] = sum(d.ledger.get("claims/rejected", 0.0) for d in ds)
        m["claims_accepted"] = acc
        m["verifier_calls"] = sum(d.ledger.get("compute/verifier_calls", 0.0) for d in ds)
        m["llm_calls"] = sum(d.ledger.get("compute/llm_calls", 0.0) for d in ds)
        m["llm_errors"] = sum(d.ledger.get("compute/llm_errors", 0.0) for d in ds)
    # graph-level
    g = [d.graph for d in ds if d.graph.get("nodes")]
    if g:
        nodes = sum(x["nodes"] for x in g)
        edges = sum(x["edges"] for x in g)
        m.update({"graph_nodes": float(nodes), "graph_edges": float(edges),
                  "duplicate_node_rate": sum(x["duplicate_node_rate"] * x["nodes"] for x in g) / nodes,
                  "duplicate_edge_rate": float(np.mean([x["duplicate_edge_rate"] for x in g])),
                  "ontology_violation_rate": sum(x["ontology_violation_rate"] * x["edges"] for x in g) / max(1, edges),
                  "connected_components_mean": float(np.mean([x["connected_components"] for x in g])),
                  "largest_component_frac_mean": float(np.mean([x["largest_component_frac"] for x in g])),
                  "degree_max": float(max(x["degree_max"] for x in g))})
    qt = _sum(ds, "qa_total")
    if qt:
        m["qa_exact_match"] = _sum(ds, "qa_exact") / qt
        m["qa_f1"] = _sum(ds, "qa_f1_sum") / qt
        m["qa_questions"] = qt
    m["contradictions"] = _sum(ds, "n_contradictions")
    # system
    lat = [x for d in ds for x in d.chunk_latency_s]
    if lat:
        m["latency_p50_s_per_chunk"] = float(np.percentile(lat, 50))
        m["latency_p95_s_per_chunk"] = float(np.percentile(lat, 95))
    ptok = sum(d.ledger.get("compute/prompt_tokens", 0.0) for d in ds)
    ctok = sum(d.ledger.get("compute/completion_tokens", 0.0) for d in ds)
    chunks = _sum(ds, "n_chunks")
    if chunks:
        m["tokens_per_chunk"] = (ptok + ctok) / chunks
    gen_s = sum(d.ledger.get("compute/completion_tokens", 0.0) / d.ledger["compute/completion_tokens_per_s"]
                for d in ds if d.ledger.get("compute/completion_tokens_per_s"))
    if gen_s:
        m["completion_tokens_per_s"] = ctok / gen_s
    m["gpu_used_gb_max"] = max([d.gpu_gb for d in ds] or [0.0])
    m["cpu_rss_gb_max"] = max([d.cpu_gb for d in ds] or [0.0])
    return m


def slice_keys(d: DocScore) -> List[str]:
    keys = ["all", f"language:{d.language}", f"script:{d.script}"]
    keys += [f"slice:{t}" for t in d.slices]
    if d.code_mixed:
        keys.append("code_mixed")
    return keys


def slice_table(ds: Sequence[DocScore], min_gold: int = 5) -> pl.DataFrame:
    """Per-slice P/R/F1 (counts summed over the docs in the slice). 'low_n' = fewer than `min_gold` gold
    relations in the slice: the number is printed but should not be read as a finding. Language and
    script are separate slice families on purpose -- never read the 'all' row as a multilingual score."""
    buckets: Dict[str, List[DocScore]] = defaultdict(list)
    for d in ds:
        for k in slice_keys(d):
            buckets[k].append(d)
    rows = []
    for k, sub in buckets.items():
        ep, er, ef = _prf(_sum(sub, "ent_tp"), _sum(sub, "ent_fp"), _sum(sub, "ent_fn"))
        rp, rr, rf = _prf(_sum(sub, "rel_tp"), _sum(sub, "rel_fp"), _sum(sub, "rel_fn"))
        pt_ = _sum(sub, "pred_total")
        rows.append({"slice": k, "docs": len(sub), "gold_relations": int(_sum(sub, "gold_total")),
                     "entity_f1": ef, "relation_precision": rp, "relation_recall": rr, "relation_f1": rf,
                     "unsupported_edge_rate": _sum(sub, "unsupported") / pt_ if pt_ else float("nan"),
                     "low_n": _sum(sub, "gold_total") < min_gold})
    order = {"all": 0, "language": 1, "script": 2, "code_mixed": 3, "slice": 4}
    rows.sort(key=lambda r: (order.get(r["slice"].split(":")[0], 9), r["slice"]))
    return pl.DataFrame(rows) if rows else pl.DataFrame()


def relation_slice_recall(ds: Sequence[DocScore]) -> pl.DataFrame:
    """Recall for relation-level tags (cross_page, long_distance, coreference ...). Recall only: an
    individual wrong prediction has no tag, so precision is only defined per document slice."""
    tot: Dict[str, List[int]] = defaultdict(lambda: [0, 0])
    for d in ds:
        for tag, (h, n) in d.rel_slice.items():
            tot[tag][0] += h
            tot[tag][1] += n
    rows = [{"relation_tag": t, "gold": n, "found": h, "recall": h / n if n else float("nan"), "low_n": n < 5}
            for t, (h, n) in sorted(tot.items())]
    return pl.DataFrame(rows) if rows else pl.DataFrame()


# ============================================================================================
# 5. THRESHOLDS + CALIBRATION  (sections 14, 15)
# ============================================================================================
Labelled = Sequence[Tuple[float, int, str]]


def sweep_thresholds(labelled: Labelled, n_gold: int,
                     thresholds: Sequence[float] = tuple(round(x, 2) for x in np.arange(0.1, 0.91, 0.1))) -> pl.DataFrame:
    """What `confidence_filter` WOULD have given. labelled = (per-relation confidence, is_gold, key) for every
    relation that passed the gate. recall is against ALL gold relations (including ones never proposed);
    false_positive_rate is over the non-gold relations that passed the gate at threshold 0."""
    n_all = len(labelled)
    n_neg = sum(1 for _, y, _ in labelled if not y)
    rows = []
    for t in thresholds:
        kept = [(s, y) for s, y, _ in labelled if s >= t]
        tp = sum(y for _, y in kept)
        fp = len(kept) - tp
        p, r, f = _prf(tp, fp, max(0, n_gold - tp))
        rows.append({"threshold": float(t), "kept": len(kept), "precision": p, "recall": r, "f1": f,
                     "coverage": len(kept) / n_all if n_all else 0.0,
                     "false_positive_rate": fp / n_neg if n_neg else 0.0})
    return pl.DataFrame(rows)


def select_threshold(sweep: pl.DataFrame, min_precision: Optional[float] = None, max_fpr: Optional[float] = None) -> Dict:
    """Pick by deployment objective: highest recall subject to precision/FPR constraints; max-F1 if none given."""
    df = sweep
    if min_precision is not None:
        df = df.filter(pl.col("precision") >= min_precision)
    if max_fpr is not None:
        df = df.filter(pl.col("false_positive_rate") <= max_fpr)
    if len(df) == 0:
        return {"threshold": None, "reason": "no threshold satisfies the constraint"}
    # ties -> the higher (more conservative) threshold
    best = df.sort(["recall", "f1", "threshold"], descending=True).row(0, named=True) if (min_precision is not None or max_fpr is not None) \
        else df.sort(["f1", "threshold"], descending=True).row(0, named=True)
    return {**best, "reason": "max recall under constraint" if (min_precision is not None or max_fpr is not None) else "max F1"}


def calibration_report(scores: Sequence[float], labels: Sequence[int], n_bins: int = 10) -> Dict:
    """ECE / MCE / Brier / reliability bins. 'brier_skill' is 1 - Brier/Brier(always predict base rate): <= 0 means
    the confidence carries no information beyond the base rate."""
    s = np.clip(np.asarray(scores, dtype=float), 0.0, 1.0)
    y = np.asarray(labels, dtype=float)
    n = len(s)
    if n == 0:
        return {"n": 0}
    idx = np.minimum((s * n_bins).astype(int), n_bins - 1)
    bins, ece, mce = [], 0.0, 0.0
    for b in range(n_bins):
        mask = idx == b
        if not mask.any():
            continue
        conf, acc = float(s[mask].mean()), float(y[mask].mean())
        gap = abs(acc - conf)
        ece += mask.sum() / n * gap
        mce = max(mce, gap)
        bins.append({"bin": f"{b / n_bins:.1f}-{(b + 1) / n_bins:.1f}", "n": int(mask.sum()), "mean_confidence": conf, "accuracy": acc})
    base = float(y.mean())
    brier = float(np.mean((s - y) ** 2))
    base_brier = float(np.mean((base - y) ** 2))
    return {"n": n, "ece": float(ece), "mce": float(mce), "brier": brier,
            "brier_skill": (1 - brier / base_brier) if base_brier > 0 else float("nan"), "base_rate": base, "bins": bins,
            "warning": "fewer than 50 labelled predictions: ECE/bins are very noisy" if n < 50 else ""}


class IsotonicCalibrator:
    """Monotone map from the pipeline's raw confidence (exp(mean token logprob), uncalibrated) to P(correct),
    fit by pool-adjacent-violators. Fit on one set of documents and evaluate on another (see cv_calibration)."""

    def __init__(self, xs: Sequence[float], ys: Sequence[float]):
        self.xs, self.ys = list(map(float, xs)), list(map(float, ys))

    @classmethod
    def fit(cls, scores: Sequence[float], labels: Sequence[int]) -> "IsotonicCalibrator":
        s, y = np.asarray(scores, dtype=float), np.asarray(labels, dtype=float)
        order = np.argsort(s, kind="mergesort")
        blocks: List[List[float]] = []
        for xi, yi in zip(s[order], y[order]):
            blocks.append([yi, 1.0, xi])
            while len(blocks) > 1 and blocks[-2][0] / blocks[-2][1] > blocks[-1][0] / blocks[-1][1]:
                b = blocks.pop()
                blocks[-1][0] += b[0]
                blocks[-1][1] += b[1]
                blocks[-1][2] += b[2]
        return cls([b[2] / b[1] for b in blocks], [b[0] / b[1] for b in blocks])

    def predict(self, scores: Sequence[float]) -> np.ndarray:
        return np.interp(np.asarray(scores, dtype=float), self.xs, self.ys)

    def to_dict(self) -> Dict:
        return {"xs": self.xs, "ys": self.ys}

    @staticmethod
    def from_dict(d: Dict) -> "IsotonicCalibrator":
        return IsotonicCalibrator(d["xs"], d["ys"])


def fit_calibrator(scores: Sequence[float], labels: Sequence[int]) -> IsotonicCalibrator:
    return IsotonicCalibrator.fit(scores, labels)


def cv_calibration(scores: Sequence[float], labels: Sequence[int], k: int = 5, seed: int = 0) -> Dict:
    """Held-out ECE before vs after isotonic calibration (k-fold). Fitting and scoring on the same points
    would flatter the calibrator, so this is the number to quote."""
    s, y = np.asarray(scores, dtype=float), np.asarray(labels, dtype=int)
    n = len(s)
    if n < 2 * k or len(set(y.tolist())) < 2:
        return {"n": n, "ece_before": float("nan"), "ece_after": float("nan"),
                "note": "need at least 2*k labelled predictions with both classes present"}
    rng = np.random.default_rng(seed)
    perm = rng.permutation(n)
    folds = np.array_split(perm, k)
    cal = np.zeros(n)
    for f in folds:
        train = np.setdiff1d(perm, f)
        cal[f] = IsotonicCalibrator.fit(s[train], y[train]).predict(s[f])
    return {"n": n, "ece_before": calibration_report(s, y)["ece"], "ece_after": calibration_report(cal, y)["ece"],
            "brier_before": calibration_report(s, y)["brier"], "brier_after": calibration_report(cal, y)["brier"]}


def save_reliability_diagram(report: Dict, path: str) -> Optional[str]:
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except Exception:
        return None
    bins = report.get("bins") or []
    if not bins:
        return None
    try:
        fig, ax = plt.subplots(figsize=(4.2, 4.2))
        ax.plot([0, 1], [0, 1], "--", color="grey", label="perfect")
        ax.plot([b["mean_confidence"] for b in bins], [b["accuracy"] for b in bins], "o-", label=f"ECE {report['ece']:.3f}")
        ax.set_xlabel("mean confidence")
        ax.set_ylabel("fraction correct")
        ax.set_title(f"Reliability (n={report['n']})")
        ax.legend()
        fig.tight_layout()
        fig.savefig(path, dpi=130)
        plt.close(fig)
        return path
    except Exception:                                                      # a plotting problem must not lose the report
        return None


# ============================================================================================
# 6. INTER-ANNOTATOR AGREEMENT  (section 19)
# ============================================================================================
def cohens_kappa(a: Sequence[Any], b: Sequence[Any]) -> float:
    """Cohen's kappa for two raters over the same items with a closed label set. If chance agreement is
    1.0 (both raters used a single label) kappa is undefined; return 1.0 when they agree, else 0.0."""
    n = len(a)
    if n == 0 or n != len(b):
        return float("nan")
    po = sum(x == y for x, y in zip(a, b)) / n
    ca, cb = Counter(a), Counter(b)
    pe = sum((ca[c] / n) * (cb[c] / n) for c in set(ca) | set(cb))
    if pe >= 1.0:
        return 1.0 if po == 1.0 else 0.0
    return (po - pe) / (1 - pe)


def annotator_agreement(a: GoldSet, b: GoldSet) -> Dict:
    """Two annotators, same documents, annotated independently. Entity / relation agreement is F1 (A as reference):
    kappa over 'did this annotator write the item' has no true negatives, so it is not reported. Kappa IS reported
    where the label set is closed: entity TYPE over shared entities, and PREDICATE over entity pairs both annotated."""
    docs_b = {d.doc_id: d for d in b.docs}
    ent = [0, 0, 0]
    rel = [0, 0, 0]
    ta, tb, pa, pb = [], [], [], []
    shared = 0
    for da in a.docs:
        db = docs_b.get(da.doc_id)
        if db is None:
            continue
        shared += 1
        mb, ma = EntityMatcher(db.entities), EntityMatcher(da.entities)
        ka = {mb.key(e.name): e for e in da.entities}
        kb = {_norm(e.name): e for e in db.entities}
        inter = set(ka) & set(kb)
        ent[0] += len(inter)
        ent[1] += len(set(ka) - set(kb))
        ent[2] += len(set(kb) - set(ka))
        for k in inter:
            ta.append(ka[k].type)
            tb.append(kb[k].type)
        ra = {(mb.key(r.subject), r.predicate, mb.key(r.object)) for r in da.relations}
        rb = {(mb.key(r.subject), r.predicate, mb.key(r.object)) for r in db.relations}
        rel[0] += len(ra & rb)
        rel[1] += len(ra - rb)
        rel[2] += len(rb - ra)
        pairs_a = {(s, o): p for s, p, o in ra}
        pairs_b = {(s, o): p for s, p, o in rb}
        for pr in set(pairs_a) & set(pairs_b):
            pa.append(pairs_a[pr])
            pb.append(pairs_b[pr])
    return {"shared_docs": shared,
            "entity_f1_agreement": _prf(*ent)[2], "relation_f1_agreement": _prf(*rel)[2],
            "type_kappa": cohens_kappa(ta, tb), "type_items": len(ta),
            "predicate_kappa": cohens_kappa(pa, pb), "predicate_items": len(pa)}


# ============================================================================================
# 7. DRIFT / POST-DEPLOYMENT MONITORING  (section 28)
# ============================================================================================
def snapshot_from_result(result: Dict, n_chars: int) -> Dict:
    """Gold-free summary of ONE run -- production has no labels. Sum several with merge_snapshots()."""
    edges = predicted_edges(result.get("edges"))
    claims = [c for c in (result.get("claims") or []) if getattr(c, "source", "llm") == "llm"]
    led = result.get("ledger")
    summ = result.get("summary") or (led.summary() if led is not None else {})
    n_cand = summ.get("claims/candidates", 0.0)
    snap = {"n_docs": 1, "doc_chars": [n_chars], "edge_conf": [e["conf"] for e in edges],
            "claim_conf": [float(c.confidence) for c in claims],
            "entities_per_doc": [len({_norm(e["s"]) for e in edges} | {_norm(e["o"]) for e in edges})],
            "relation_counts": dict(Counter(e["p"] for e in edges)),
            "type_counts": dict(Counter(t for e in edges for t in (e["s_type"], e["o_type"]) if t)),
            "weights": {"claims": n_cand, "accepted": summ.get("claims/accepted", 0.0)},
            "rate_sums": {"unsupported_claim_rate": summ.get("quality/unsupported_claim_rate", 0.0) * summ.get("claims/accepted", 0.0),
                          "candidate_ungrounded_rate": summ.get("quality/candidate_ungrounded_rate", 0.0) * n_cand,
                          "ontology_violation_rate": summ.get("quality/ontology_violation_rate", 0.0) * n_cand,
                          "review_rate": summ.get("claims/review", 0.0), "reject_rate": summ.get("claims/rejected", 0.0),
                          "llm_error_rate": summ.get("compute/llm_errors", 0.0)},
            "latency_p95": [summ.get("latency/p95_s_per_chunk", 0.0)]}
    return snap


def save_snapshot(snap: Dict, path: str) -> str:
    """Freeze a reference snapshot (e.g. from the gold/test corpus) so later production samples can be compared to it."""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(_clean_json(snap)))
    return path


def load_snapshot(path: str) -> Dict:
    return json.loads(Path(path).read_text())


def _clean_json(o: Any) -> Any:
    if isinstance(o, dict):
        return {str(k): _clean_json(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean_json(v) for v in o]
    if isinstance(o, (np.floating, np.integer)):
        o = o.item()
    if isinstance(o, float) and (math.isnan(o) or math.isinf(o)):
        return None
    return o


def merge_snapshots(snaps: Sequence[Dict]) -> Dict:
    out: Dict[str, Any] = {"n_docs": 0, "doc_chars": [], "edge_conf": [], "claim_conf": [], "entities_per_doc": [],
                           "relation_counts": Counter(), "type_counts": Counter(), "weights": Counter(),
                           "rate_sums": Counter(), "latency_p95": []}
    for s in snaps:
        out["n_docs"] += s["n_docs"]
        for k in ("doc_chars", "edge_conf", "claim_conf", "entities_per_doc", "latency_p95"):
            out[k].extend(s[k])
        for k in ("relation_counts", "type_counts", "weights", "rate_sums"):
            out[k].update(s[k])
    for k in ("relation_counts", "type_counts", "weights", "rate_sums"):
        out[k] = dict(out[k])
    w = out["weights"]
    rs = out["rate_sums"]
    out["rates"] = {
        "unsupported_claim_rate": rs.get("unsupported_claim_rate", 0.0) / w["accepted"] if w.get("accepted") else 0.0,
        "candidate_ungrounded_rate": rs.get("candidate_ungrounded_rate", 0.0) / w["claims"] if w.get("claims") else 0.0,
        "ontology_violation_rate": rs.get("ontology_violation_rate", 0.0) / w["claims"] if w.get("claims") else 0.0,
        "review_rate": rs.get("review_rate", 0.0) / w["claims"] if w.get("claims") else 0.0,
        "reject_rate": rs.get("reject_rate", 0.0) / w["claims"] if w.get("claims") else 0.0,
        "llm_error_rate": rs.get("llm_error_rate", 0.0) / max(1.0, w.get("claims", 0.0))}
    return out


def psi(ref: Sequence[float], cur: Sequence[float], bins: int = 10) -> float:
    """Population Stability Index with reference-quantile bins. <0.1 stable, 0.1-0.25 moderate, >0.25 significant."""
    r, c = np.asarray(ref, dtype=float), np.asarray(cur, dtype=float)
    if len(r) < 2 or len(c) < 2:
        return float("nan")
    # under 'no shift' E[PSI] ~ (bins-1)(1/n_ref + 1/n_cur): keep >= ~20 samples per bin or the index is mostly noise
    bins = int(max(2, min(bins, min(len(r), len(c)) // 20)))
    edges = np.unique(np.quantile(r, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        edges = np.array([r.min() - 1e-9, np.median(r), r.max() + 1e-9])
    edges[0], edges[-1] = -np.inf, np.inf
    pr = np.histogram(r, edges)[0] / len(r)
    pc = np.histogram(c, edges)[0] / len(c)
    pr, pc = np.clip(pr, 1e-4, None), np.clip(pc, 1e-4, None)
    return float(np.sum((pc - pr) * np.log(pc / pr)))


def js_divergence(p: Dict[str, float], q: Dict[str, float]) -> float:
    """Jensen-Shannon divergence (base 2, in [0,1]) between two categorical count dicts."""
    cats = sorted(set(p) | set(q))
    if not cats:
        return float("nan")
    a = np.array([p.get(k, 0.0) for k in cats], dtype=float)
    b = np.array([q.get(k, 0.0) for k in cats], dtype=float)
    if a.sum() == 0 or b.sum() == 0:
        return float("nan")
    a, b = a / a.sum(), b / b.sum()
    m = (a + b) / 2

    def kl(x, y):
        mask = x > 0
        return float(np.sum(x[mask] * np.log2(x[mask] / y[mask])))
    return 0.5 * kl(a, m) + 0.5 * kl(b, m)


def drift_report(ref: Dict, cur: Dict, min_docs: int = 30) -> pl.DataFrame:
    """Reference (e.g. your gold/test corpus, saved once) vs current (a production sample). 'low_n' means
    fewer than `min_docs` documents on either side: treat flags as hints."""
    from scipy.stats import ks_2samp
    rows: List[Dict] = []
    low = ref["n_docs"] < min_docs or cur["n_docs"] < min_docs

    def add(feature, kind, stat, value, flag, note=""):
        rows.append({"feature": feature, "test": stat, "kind": kind, "value": value, "flag": flag, "low_n": low, "note": note})

    for name in ("edge_conf", "claim_conf", "doc_chars", "entities_per_doc"):
        r, c = ref.get(name, []), cur.get(name, [])
        if len(r) < 2 or len(c) < 2:
            continue
        v = psi(r, c)
        add(name, "continuous", "PSI", v, "significant" if v > 0.25 else "moderate" if v > 0.10 else "stable")
        ks = ks_2samp(r, c)
        add(name, "continuous", "KS", float(ks.statistic), "shift" if ks.pvalue < 0.01 and ks.statistic > 0.1 else "ok",
            f"p={ks.pvalue:.3g}")
    for name in ("relation_counts", "type_counts"):
        v = js_divergence(ref.get(name, {}), cur.get(name, {}))
        if not math.isnan(v):
            add(name, "categorical", "JS", v, "shift" if v > 0.10 else "ok")
    for name, rv in ref.get("rates", {}).items():
        cv = cur.get("rates", {}).get(name, 0.0)
        flag = "shift" if abs(cv - rv) > max(0.05, 0.5 * rv) else "ok"
        add(name, "rate", "abs_delta", cv - rv, flag, f"ref={rv:.4f} cur={cv:.4f}")
    return pl.DataFrame(rows, infer_schema_length=None) if rows else pl.DataFrame()


# ============================================================================================
# 8. RED-TEAM SET  (section 17)  -- permanent regression tests
# ============================================================================================
Triple = Tuple[str, str, str]


@dataclass
class RedTeamCase:
    id: str
    category: str
    text: str
    must_have: List[Triple] = field(default_factory=list)          # must be an edge in the final graph
    must_not_have: List[Triple] = field(default_factory=list)      # must NOT be an edge in the final graph
    expect_review: List[Triple] = field(default_factory=list)      # informational: should be routed to the REVIEW queue
    no_split: List[List[str]] = field(default_factory=list)        # surface forms that must end up as ONE node
    expect_contradiction: str = ""                                 # a Contradiction.kind that must be reported
    note: str = ""


_FILLER = " ".join(["The weather was mild that week.", "Meetings ran long on Tuesday.", "Lunch was served at noon.",
                    "The agenda covered scheduling and budgets.", "Nobody objected to the minutes."])

REDTEAM_CASES: List[RedTeamCase] = [
    RedTeamCase("neg_requires", "negation", "TensorRT-LLM does not require CUDA 11.",
                must_not_have=[("TensorRT-LLM", "requires", "CUDA 11")]),
    RedTeamCase("neg_depends", "negation", "vLLM does not depend on Ray for single-node serving.",
                must_not_have=[("vLLM", "depends_on", "Ray")]),
    RedTeamCase("hypothetical", "hypothetical",
                "If NVIDIA released Triton 4, the Triton server would require CUDA 13.",
                must_not_have=[("Triton server", "requires", "CUDA 13"), ("NVIDIA", "develops", "Triton 4")]),
    RedTeamCase("conditional", "conditional",
                "Provided that the licence is renewed, Acme Corp will manage the Berlin data centre.",
                must_not_have=[("Acme Corp", "manages", "Berlin data centre")]),
    RedTeamCase("uncertain", "uncertain",
                "Analysts speculate that Initech may develop a new compiler, though nothing has been confirmed.",
                must_not_have=[("Initech", "develops", "compiler")]),
    RedTeamCase("attribution", "attribution", "According to Smith, vLLM enables speculative decoding.",
                must_not_have=[("vLLM", "enables", "speculative decoding")],
                expect_review=[("vLLM", "enables", "speculative decoding")],
                note="policy: attributed claims go to REVIEW (PipelineConfig.attributed_action), never straight into the graph"),
    RedTeamCase("coref_pronoun", "coreference", "NVIDIA develops TensorRT-LLM. It requires CUDA 12.",
                must_have=[("NVIDIA", "develops", "TensorRT-LLM"), ("TensorRT-LLM", "requires", "CUDA 12")],
                must_not_have=[("NVIDIA", "requires", "CUDA 12")]),
    RedTeamCase("coref_definite_np", "coreference", "Sarvam develops Indic language models. The company is located in Bengaluru.",
                must_have=[("Sarvam", "located_in", "Bengaluru")]),
    RedTeamCase("long_distance", "long_distance", "NVIDIA develops TensorRT-LLM. " + _FILLER + " The library requires CUDA 12.",
                must_have=[("TensorRT-LLM", "requires", "CUDA 12")],
                note="five neutral sentences between antecedent and pronoun-like NP"),
    RedTeamCase("contradiction_version", "contradiction",
                "TensorRT-LLM requires CUDA 12. " + _FILLER + " Appendix B: TensorRT-LLM requires CUDA 13.",
                must_have=[("TensorRT-LLM", "requires", "CUDA 12"), ("TensorRT-LLM", "requires", "CUDA 13")],
                expect_contradiction="VERSION_CONFLICT",
                note="both claims must survive as separate edges AND be flagged -- never silently merged"),
    RedTeamCase("similar_versions", "similar_names", "Library A requires CUDA 11. Library B requires CUDA 12.",
                must_have=[("Library A", "requires", "CUDA 11"), ("Library B", "requires", "CUDA 12")],
                must_not_have=[("Library A", "requires", "CUDA 12"), ("Library B", "requires", "CUDA 11")]),
    RedTeamCase("similar_apple", "similar_names", "Apple develops the iPhone. The apple orchard is located in Shimla.",
                must_have=[("Apple", "develops", "iPhone")],
                must_not_have=[("Apple", "located_in", "Shimla")]),
    RedTeamCase("nested_entity", "nested_entities", "The Reserve Bank of India regulates the Bank of Baroda.",
                must_have=[("Reserve Bank of India", "regulates", "Bank of Baroda")],
                must_not_have=[("Bank of India", "regulates", "Bank of Baroda")]),
    RedTeamCase("direction_passive", "direction", "Banks are regulated by the RBI.",
                must_have=[("RBI", "regulates", "Banks")], must_not_have=[("Banks", "regulates", "RBI")]),
    RedTeamCase("direction_passive_dev", "direction", "TensorRT-LLM is developed by NVIDIA.",
                must_have=[("NVIDIA", "develops", "TensorRT-LLM")], must_not_have=[("TensorRT-LLM", "develops", "NVIDIA")]),
    RedTeamCase("ocr_corruption", "ocr", "NVlDIA develops TensorRT-LLM. NVIDIA develops TensorRT-LLM.",
                must_have=[("NVIDIA", "develops", "TensorRT-LLM")], no_split=[["NVIDIA", "NVlDIA"]],
                note="'NVlDIA' has a lowercase L for the I -- must not become a second node"),
    RedTeamCase("table", "tables", "| Component | Depends on |\n| TensorRT-LLM | CUDA |\n| vLLM | PyTorch |",
                must_have=[("TensorRT-LLM", "depends_on", "CUDA"), ("vLLM", "depends_on", "PyTorch")]),
    RedTeamCase("footnote", "footnotes",
                "NVIDIA develops TensorRT-LLM.1\n\n1 Not to be confused with TensorFlow, which Google develops.",
                must_have=[("NVIDIA", "develops", "TensorRT-LLM")],
                must_not_have=[("NVIDIA", "develops", "TensorFlow")]),
]


def _lab_eq(a: str, b: str) -> bool:
    na, nb = _norm(a), _norm(b)
    if na == nb or _loose(a) == _loose(b):
        return True
    return fuzz.WRatio(na, nb) >= 90 and _numeric_signature(na) == _numeric_signature(nb)


def _has_triple(edges: List[Dict], t: Triple) -> bool:
    return any(e["p"] == t[1] and _lab_eq(e["s"], t[0]) and _lab_eq(e["o"], t[2]) for e in edges)


def _contradiction_kinds(result: Dict) -> Set[str]:
    return {c.kind for c in (result.get("contradictions") or [])}


def _quiet(quiet: bool):
    """The pipeline prints a few lines per document; over 18 red-team cases + a gold set that buries the scorecard."""
    return contextlib.redirect_stdout(io.StringIO()) if quiet else contextlib.nullcontext()


def run_redteam_case(pipe, case: RedTeamCase, quiet: bool = True) -> Dict:
    with _quiet(quiet):
        res = pipe.run_document(text=case.text, doc_id=case.id)
    edges = predicted_edges(res.get("edges"))
    missing = [t for t in case.must_have if not _has_triple(edges, t)]
    forbidden = [t for t in case.must_not_have if _has_triple(edges, t)]
    split = []
    for group in case.no_split:
        labels = {e[k] for e in edges for k in ("s", "o") if any(_lab_eq(e[k], g) for g in group)}
        if len({_norm(l) for l in labels}) > 1:
            split.append(sorted(labels))
    contra_ok = None
    if case.expect_contradiction:
        contra_ok = case.expect_contradiction in _contradiction_kinds(res)
    claims = res.get("claims") or []
    reviewed = [t for t in case.expect_review
                if any(c.verdict == "REVIEW" and c.predicate == t[1] and _lab_eq(c.subject, t[0]) and _lab_eq(c.target, t[2])
                       for c in claims)]
    passed = not missing and not forbidden and not split and contra_ok is not False
    return {"id": case.id, "category": case.category, "passed": passed, "missing": missing, "forbidden_present": forbidden,
            "split_nodes": split, "contradiction_flagged": contra_ok, "routed_to_review": reviewed,
            "expected_review": case.expect_review, "n_edges": len(edges), "note": case.note}


def run_redteam(pipe, cases: Optional[Sequence[RedTeamCase]] = None, quiet: bool = True) -> Dict:
    """One pipeline, many tiny documents (run_document resets state between them).
    safety_failures = forbidden triples that reached the graph -- the number the gate watches."""
    cases = list(cases if cases is not None else REDTEAM_CASES)
    rows = [run_redteam_case(pipe, c, quiet) for c in cases]
    by_cat: Dict[str, List[bool]] = defaultdict(list)
    for r in rows:
        by_cat[r["category"]].append(r["passed"])
    n_review_exp = sum(len(r["expected_review"]) for r in rows)
    return {"cases": rows, "n": len(rows), "passed": sum(r["passed"] for r in rows),
            "pass_rate": sum(r["passed"] for r in rows) / len(rows) if rows else float("nan"),
            "safety_failures": sum(len(r["forbidden_present"]) + len(r["split_nodes"]) for r in rows),
            "missing_total": sum(len(r["missing"]) for r in rows),
            "review_routing_rate": (sum(len(r["routed_to_review"]) for r in rows) / n_review_exp) if n_review_exp else float("nan"),
            "by_category": {k: sum(v) / len(v) for k, v in sorted(by_cat.items())},
            "by_category_n": {k: len(v) for k, v in sorted(by_cat.items())}}


# ============================================================================================
# 9. SEED GOLD  (synthetic -- see module docstring)
# ============================================================================================
def _seed_gold() -> GoldSet:
    t1 = ("NVIDIA develops TensorRT-LLM. It requires CUDA 12. TensorRT-LLM does not require CUDA 11. "
          "vLLM enables PagedAttention. According to Smith, vLLM enables speculative decoding.")
    d1 = GoldDoc(
        doc_id="seed_infra", text=t1, slices=["clean", "dense_technical", "coreference"], annotator="synthetic",
        entities=[GoldEntity("NVIDIA", "ORGANIZATION"), GoldEntity("TensorRT-LLM", "TECHNOLOGY", ["TensorRT LLM", "TRT-LLM"]),
                  GoldEntity("CUDA 12", "TECHNOLOGY"), GoldEntity("CUDA 11", "TECHNOLOGY"), GoldEntity("vLLM", "TECHNOLOGY"),
                  GoldEntity("PagedAttention", "TECHNOLOGY"), GoldEntity("Smith", "PERSON"),
                  GoldEntity("speculative decoding", "CONCEPT")],
        relations=[GoldRelation("NVIDIA", "develops", "TensorRT-LLM", "NVIDIA develops TensorRT-LLM.", slices=["long_distance"]),
                   GoldRelation("TensorRT-LLM", "requires", "CUDA 12", "It requires CUDA 12.", coref=True, slices=["coreference"]),
                   GoldRelation("vLLM", "enables", "PagedAttention", "vLLM enables PagedAttention.")],
        negatives=[GoldNegative("TensorRT-LLM", "requires", "CUDA 11", "negated"),
                   GoldNegative("vLLM", "enables", "speculative decoding", "attributed")],
        questions=[GoldQuestion("Which organization developed TensorRT-LLM?", "TensorRT-LLM", "target", ["develops"], "ORGANIZATION", ["NVIDIA"]),
                   GoldQuestion("What does TensorRT-LLM require?", "TensorRT-LLM", "source", ["requires"], None, ["CUDA 12"])])
    t2 = ("The Reserve Bank of India (RBI) regulates banks. Banks are regulated by the RBI. "
          "The Companies Act governs companies. The RBI manages foreign exchange reserves.")
    d2 = GoldDoc(
        doc_id="seed_regulation", text=t2, slices=["clean", "short_paragraphs"], annotator="synthetic",
        entities=[GoldEntity("Reserve Bank of India", "ORGANIZATION", ["RBI"]), GoldEntity("banks", "ORGANIZATION", ["Banks"]),
                  GoldEntity("Companies Act", "LAW"), GoldEntity("companies", "ORGANIZATION"),
                  GoldEntity("foreign exchange reserves", "CONCEPT")],
        relations=[GoldRelation("Reserve Bank of India", "regulates", "banks", "The Reserve Bank of India (RBI) regulates banks."),
                   GoldRelation("Companies Act", "governs", "companies", "The Companies Act governs companies."),
                   GoldRelation("Reserve Bank of India", "manages", "foreign exchange reserves", "The RBI manages foreign exchange reserves.")],
        negatives=[GoldNegative("banks", "regulates", "Reserve Bank of India", "direction")],
        questions=[GoldQuestion("Which organization regulates banks?", "banks", "target", ["regulates"], "ORGANIZATION", ["Reserve Bank of India"])])
    t3 = ("TensorRT-LLM requires CUDA 12. " + _FILLER + " " + _FILLER +
          " Appendix B. TensorRT-LLM requires CUDA 13.")
    d3 = GoldDoc(
        doc_id="seed_contradiction", text=t3, slices=["clean", "long_section"], annotator="synthetic",
        entities=[GoldEntity("TensorRT-LLM", "TECHNOLOGY"), GoldEntity("CUDA 12", "TECHNOLOGY"), GoldEntity("CUDA 13", "TECHNOLOGY")],
        relations=[GoldRelation("TensorRT-LLM", "requires", "CUDA 12", "TensorRT-LLM requires CUDA 12."),
                   GoldRelation("TensorRT-LLM", "requires", "CUDA 13", "Appendix B. TensorRT-LLM requires CUDA 13.", slices=["long_distance"])],
        contradictions=[GoldContradiction("VERSION_CONFLICT", "TensorRT-LLM", "requires")])
    return GoldSet(version="seed-synthetic-0", docs=[d1, d2, d3], synthetic=True,
                   description="SYNTHETIC smoke-test set written by the module author, NOT human gold. Exercises the harness; "
                               "numbers on it are illustrative only.")


SEED_GOLD = _seed_gold()


# ============================================================================================
# 10. REGRESSION GATE  (section 27)  +  baseline store
# ============================================================================================
DEFAULT_GATE_RULES: List[Dict] = [
    # group "quality": may not fall more than max_drop (absolute) below the baseline
    {"metric": "relation_f1", "group": "quality", "direction": "higher", "max_drop": 0.02},
    {"metric": "relation_recall", "group": "quality", "direction": "higher", "max_drop": 0.03},
    {"metric": "entity_f1", "group": "quality", "direction": "higher", "max_drop": 0.02},
    {"metric": "evidence_accuracy", "group": "quality", "direction": "higher", "max_drop": 0.02},
    {"metric": "type_accuracy", "group": "quality", "direction": "higher", "max_drop": 0.03},
    {"metric": "qa_f1", "group": "quality", "direction": "higher", "max_drop": 0.03},
    # group "safety": may not rise above baseline + max_rise, nor above an absolute cap
    {"metric": "unsupported_edge_rate", "group": "safety", "direction": "lower", "max_rise": 0.005, "cap": 0.05},
    {"metric": "known_bad_edge_rate", "group": "safety", "direction": "lower", "max_rise": 0.0, "cap": 0.05},
    {"metric": "ontology_violation_rate", "group": "safety", "direction": "lower", "max_rise": 0.0, "cap": 0.05},
    {"metric": "redteam_safety_failures", "group": "safety", "direction": "lower", "max_rise": 0.0},
    {"metric": "redteam_pass_rate", "group": "safety", "direction": "higher", "max_drop": 0.0},
    # group "latency": relative budgets (a laptop and a T4 differ; the BASELINE is the reference)
    {"metric": "latency_p95_s_per_chunk", "group": "latency", "direction": "lower", "max_rel_rise": 0.25},
    {"metric": "tokens_per_chunk", "group": "latency", "direction": "lower", "max_rel_rise": 0.50},
    {"metric": "gpu_used_gb_max", "group": "latency", "direction": "lower", "max_rel_rise": 0.15},
    {"metric": "cpu_rss_gb_max", "group": "latency", "direction": "lower", "max_rel_rise": 0.25},
]


def _finite(x: Any) -> bool:
    return isinstance(x, (int, float)) and not (isinstance(x, float) and (math.isnan(x) or math.isinf(x)))


def _clean(o: Any) -> Any:
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, float) and (math.isnan(o) or math.isinf(o)):
        return None
    if isinstance(o, (np.floating, np.integer)):
        return _clean(o.item())
    if isinstance(o, pl.DataFrame):
        return _clean(o.to_dicts())
    return o


@dataclass
class GateResult:
    passed: bool
    groups: Dict[str, str]
    checks: pl.DataFrame
    warnings: List[str]

    def __str__(self) -> str:
        lines = [f"REGRESSION GATE: {'PASS' if self.passed else 'FAIL'}   "
                 + "  ".join(f"{g}={v}" for g, v in self.groups.items())]
        with pl.Config(tbl_rows=60, tbl_cols=10, fmt_str_lengths=60):
            lines.append(str(self.checks))
        lines += [f"  ! {w}" for w in self.warnings]
        return "\n".join(lines)


def save_baseline(report: Dict, path: str) -> str:
    """Freeze a report as the reference every later run is gated against."""
    Path(path).write_text(json.dumps(_clean({
        "eval_version": EVAL_VERSION, "label": report.get("label"), "gold_version": report.get("gold_version"),
        "gold_synthetic": report.get("gold_synthetic"), "manifest": report.get("manifest"), "metrics": report["metrics"]}), indent=2))
    return path


def load_baseline(path: str) -> Dict:
    return json.loads(Path(path).read_text())


class RegressionGate:
    """PASS/FAIL on three axes (quality / safety / latency), like the diagram: a change that buys F1 by letting
    unsupported edges through, or by doubling p95 latency, is blocked. Thresholds are rules in DEFAULT_GATE_RULES
    -- edit them to match your deployment objective; they are starting points, not findings."""

    def __init__(self, rules: Optional[List[Dict]] = None):
        self.rules = rules if rules is not None else DEFAULT_GATE_RULES

    def check(self, current: Dict, baseline: Dict) -> GateResult:
        cm, bm = current["metrics"], baseline["metrics"]
        rows, warnings = [], []
        if current.get("gold_version") != baseline.get("gold_version"):
            rows.append({"metric": "gold_version", "group": "quality", "baseline": str(baseline.get("gold_version")),
                         "current": str(current.get("gold_version")), "limit": "must match", "status": "FAIL",
                         "note": "baseline was measured on a different gold set; the comparison is not valid"})
        for r in self.rules:
            m = r["metric"]
            b, c = bm.get(m), cm.get(m)
            row = {"metric": m, "group": r["group"], "baseline": b, "current": c, "limit": None, "status": "SKIP", "note": ""}
            if not (_finite(b) and _finite(c)):
                row["note"] = "metric missing in baseline or current run"
                rows.append(row)
                continue
            ok, limit = True, None
            if r["direction"] == "higher":
                limit = b - r.get("max_drop", 0.0)
                if "floor" in r:
                    limit = max(limit, r["floor"])
                ok = c >= limit - 1e-12
            else:
                limits = []
                if "max_rise" in r:
                    limits.append(b + r["max_rise"])
                if "max_rel_rise" in r:
                    limits.append(b * (1 + r["max_rel_rise"]))
                if "cap" in r:
                    limits.append(r["cap"])
                limit = min(limits) if limits else b
                ok = c <= limit + 1e-12
            row.update({"limit": limit, "status": "PASS" if ok else "FAIL"})
            rows.append(row)
        df = pl.DataFrame([{k: (f"{v:.4f}" if isinstance(v, float) else ("" if v is None else str(v)))
                            for k, v in r.items()} for r in rows])
        groups: Dict[str, str] = {}
        for g in ("quality", "safety", "latency"):
            st = [r["status"] for r in rows if r["group"] == g]
            groups[g] = "FAIL" if "FAIL" in st else ("PASS" if "PASS" in st else "n/a")
        if current.get("gold_synthetic") or baseline.get("gold_synthetic"):
            warnings.append("gate ran on SYNTHETIC seed gold -- a PASS here says nothing about real quality")
        n_rel = cm.get("gold_relations", 0)
        if n_rel < 30:
            warnings.append(f"only {int(n_rel)} gold relations: quality deltas under ~0.1 are within noise; consider the CI before trusting a FAIL/PASS")
        if cm.get("docs", 0) < 5:
            warnings.append("fewer than 5 gold documents: bootstrap CIs are not computed")
        if any(r["status"] == "SKIP" for r in rows):
            warnings.append("some metrics were skipped (missing on one side) -- they cannot block a change")
        passed = all(r["status"] != "FAIL" for r in rows)
        return GateResult(passed, groups, df, warnings)


# ============================================================================================
# 11. W&B  (section 26)
# ============================================================================================
def _wandb():
    try:
        import wandb
        return wandb
    except Exception:
        return None


def log_to_wandb(report: Dict, wandb_module: Any = None, prefix: str = "eval") -> bool:
    """Logs the scorecard into the ACTIVE W&B run (launch_ui / your own wandb.init). Returns False if there is none.
    Versions (model / prompt / ontology / dataset / parser / embedding / threshold / hardware) go to config so runs can be
    grouped and compared: baseline vs retrieval vs verification vs ontology constraints."""
    wb = wandb_module or _wandb()
    if wb is None or getattr(wb, "run", None) is None:
        return False
    try:
        man = report.get("manifest") or {}
        cfg = {k: v for k, v in man.items() if k != "config"}
        cfg.update({"gold_version": report.get("gold_version"), "gold_synthetic": report.get("gold_synthetic"),
                    "eval_version": EVAL_VERSION, "preset": report.get("label")})
        wb.config.update(_clean(cfg), allow_val_change=True)
        flat = {f"{prefix}/{k}": v for k, v in report["metrics"].items() if _finite(v)}
        cal = report.get("calibration") or {}
        for k in ("ece", "mce", "brier", "brier_skill"):
            if _finite(cal.get(k)):
                flat[f"{prefix}/calibration_{k}"] = cal[k]
        rt = report.get("redteam")
        if rt:
            flat[f"{prefix}/redteam_pass_rate"] = rt["pass_rate"]
            flat[f"{prefix}/redteam_safety_failures"] = rt["safety_failures"]
            for c, v in rt["by_category"].items():
                flat[f"{prefix}/redteam/{c}"] = v
        if report.get("gate") is not None:
            flat[f"{prefix}/gate_passed"] = int(report["gate"].passed)
        wb.log(flat)
        Table = getattr(wb, "Table", None)
        if Table is not None:
            tables = {"slices": report.get("slices"), "failure_distribution": report.get("failure_distribution"),
                      "threshold_sweep": report.get("sweep")}
            if rt:
                tables["redteam_cases"] = pl.DataFrame([{**{k: str(v) for k, v in c.items() if k != "note"}} for c in rt["cases"]])
            if report.get("gate") is not None:
                tables["gate_checks"] = report["gate"].checks
            for name, df in tables.items():
                if isinstance(df, pl.DataFrame) and len(df):
                    wb.log({f"{prefix}/table/{name}": Table(dataframe=df.to_pandas())})
        return True
    except Exception as e:                                                 # logging must never break the run
        print(f"[EVAL] W&B logging failed: {e}")
        return False


# ============================================================================================
# 12. HARNESS
# ============================================================================================
def _table(df: pl.DataFrame, rename: Optional[Dict[str, str]] = None, float_fmt: str = "{:.3f}", indent: str = "  ") -> str:
    """Plain aligned text table: one line per row whatever the terminal width (polars wraps wide frames)."""
    rename = rename or {}
    cols = df.columns
    rows = []
    for r in df.iter_rows():
        row = []
        for v in r:
            if isinstance(v, float):
                row.append("-" if math.isnan(v) else float_fmt.format(v))
            elif isinstance(v, bool):
                row.append("yes" if v else "")
            else:
                row.append("" if v is None else str(v))
        rows.append(row)
    heads = [rename.get(c_, c_) for c_ in cols]
    widths = [max([len(h)] + [len(r[i]) for r in rows]) for i, h in enumerate(heads)]
    lines = [indent + "  ".join(h.ljust(w) if i == 0 else h.rjust(w) for i, (h, w) in enumerate(zip(heads, widths)))]
    for r in rows:
        lines.append(indent + "  ".join(x.ljust(w) if i == 0 else x.rjust(w) for i, (x, w) in enumerate(zip(r, widths))))
    return "\n".join(lines)


class EvalHarness:
    """gold set + red-team set + one pipeline -> scorecard, per-slice table, failure taxonomy, threshold sweep,
    calibration, drift snapshot, regression gate. The pipeline's own config decides WHICH system is being measured:
    build it from PipelineConfig.preset(...) (see compare_presets)."""

    def __init__(self, pipeline, gold: GoldSet, redteam: Optional[Sequence[RedTeamCase]] = None, qa: bool = True,
                 quiet: bool = True):
        self.pipe = pipeline
        self.gold = gold
        self.redteam = list(redteam) if redteam is not None else list(REDTEAM_CASES)
        self.qa = qa
        self.quiet = quiet          # hide the pipeline's own per-document log lines (LLM errors still show up as a metric)

    # -- running -------------------------------------------------------------------------------
    def run_gold(self, verbose: bool = True) -> Tuple[List[DocScore], List[Dict]]:
        scores, snaps = [], []
        for i, gd in enumerate(self.gold.docs, 1):
            t0 = time.time()
            with _quiet(self.quiet):
                if gd.text:
                    res = self.pipe.run_document(text=gd.text, doc_id=gd.doc_id)
                elif gd.pdf_path:
                    res = self.pipe.run_document(pdf_bytes=Path(gd.pdf_path).read_bytes(), doc_id=gd.doc_id)
                else:
                    continue
            doc = gd
            if not gd.text and res.get("hier") is not None:
                doc = GoldDoc(**{**gd.__dict__, "text": " ".join(c.text for c in res["hier"].chunks)})
            ds = score_document(doc, res, qa=self.qa)
            ds.ledger = ds.ledger or {}
            scores.append(ds)
            snaps.append(snapshot_from_result(res, ds.n_chars))
            if verbose:
                print(f"[EVAL] {i}/{len(self.gold.docs)} {gd.doc_id}: relation P/R "
                      f"{_prf(ds.rel_tp, ds.rel_fp, ds.rel_fn)[0]:.2f}/{_prf(ds.rel_tp, ds.rel_fp, ds.rel_fn)[1]:.2f}  "
                      f"edges={ds.pred_edges}  failures={len(ds.failures)}  ({time.time() - t0:.1f}s)")
        return scores, snaps

    def evaluate(self, label: str = "run", baseline: Optional[Any] = None, gate: Optional[RegressionGate] = None,
                 run_redteam_set: bool = True, min_precision: Optional[float] = None, verbose: bool = True) -> Dict:
        scores, snaps = self.run_gold(verbose=verbose)
        m = aggregate(scores)
        labelled = [x for d in scores for x in d.labelled]
        n_gold = int(m.get("gold_relations", 0))
        ci = {"relation_f1": bootstrap_ci(scores, relation_counts_f1), "entity_f1": bootstrap_ci(scores, entity_counts_f1)}
        sweep = sweep_thresholds(labelled, n_gold) if labelled else pl.DataFrame()
        cal = calibration_report([s for s, _, _ in labelled], [y for _, y, _ in labelled]) if labelled else {"n": 0}
        cal_cv = cv_calibration([s for s, _, _ in labelled], [y for _, y, _ in labelled]) if len(labelled) else {}
        failures = [f for d in scores for f in d.failures]
        # contradiction detection quality (section 22)
        ct = (_sum(scores, "contra_tp"), _sum(scores, "contra_fp"), _sum(scores, "contra_fn"))
        if sum(ct):
            m["contradiction_precision"], m["contradiction_recall"], m["contradiction_f1"] = _prf(*ct)
        rt = run_redteam(self.pipe, self.redteam, quiet=self.quiet) if (run_redteam_set and self.redteam) else None
        if rt:
            m["redteam_pass_rate"] = rt["pass_rate"]
            m["redteam_safety_failures"] = float(rt["safety_failures"])
        cfg = self.pipe.config
        manifest = kgp.RunManifest.build(cfg, model_name=getattr(self.pipe.extractor, "model_name", ""),
                                         dataset_version=self.gold.version)
        report = {"label": label, "eval_version": EVAL_VERSION, "gold_version": self.gold.version,
                  "gold_synthetic": self.gold.synthetic, "manifest": manifest.to_dict(), "metrics": m, "ci": ci,
                  "slices": slice_table(scores), "relation_slices": relation_slice_recall(scores),
                  "failure_distribution": failure_distribution(failures),
                  "failure_rows": pl.DataFrame([f.to_row() for f in failures]) if failures else pl.DataFrame(),
                  "sweep": sweep, "selected_threshold": select_threshold(sweep, min_precision=min_precision) if len(sweep) else {},
                  "calibration": cal, "calibration_cv": cal_cv, "redteam": rt, "doc_scores": scores,
                  "snapshot": merge_snapshots(snaps) if snaps else {}, "gate": None}
        if baseline is not None:
            base = load_baseline(baseline) if isinstance(baseline, (str, Path)) else baseline
            report["gate"] = (gate or RegressionGate()).check(report, base)
        return report

    # -- presenting ----------------------------------------------------------------------------
    @staticmethod
    def scorecard(r: Dict) -> None:
        m, f = r["metrics"], lambda x: "  -  " if not _finite(x) else f"{x:5.3f}"
        syn = "  *** SYNTHETIC SEED GOLD: illustrative only ***" if r.get("gold_synthetic") else ""
        print(f"\n=== KG EVAL [{r['label']}]  gold={r['gold_version']}  docs={int(m['docs'])}  gold relations={int(m['gold_relations'])}{syn}")
        print(f"{'':26}{'Precision':>10}{'Recall':>8}{'F1':>8}")
        print(f"{'ENTITY (graph nodes)':26}{f(m['entity_precision']):>10}{f(m['entity_recall']):>8}{f(m['entity_f1']):>8}")
        if "discovery_f1" in m:
            print(f"{'ENTITY (discovered)':26}{f(m['discovery_precision']):>10}{f(m['discovery_recall']):>8}{f(m['discovery_f1']):>8}")
        lo, hi = r["ci"]["relation_f1"]
        ci = f"   95% CI [{lo:.3f}, {hi:.3f}]" if _finite(lo) else "   (CI needs >= 5 docs)"
        print(f"{'RELATION':26}{f(m['relation_precision']):>10}{f(m['relation_recall']):>8}{f(m['relation_f1']):>8}{ci}")
        print(f"{'  ignoring direction':26}{'':>10}{'':>8}{f(m['relation_f1_ignoring_direction']):>8}")
        print(f"{'TYPE accuracy':26}{f(m['type_accuracy']):>10}   (schema-compatible {f(m['type_accuracy_compatible'])})")
        print(f"{'EVIDENCE accuracy':26}{f(m['evidence_accuracy']):>10}   coverage {f(m['evidence_coverage'])}   quote-in-source {f(m['evidence_in_source_rate'])}")
        print("SAFETY")
        print(f"  unsupported-edge (hallucination) rate  {f(m['unsupported_edge_rate'])}   (pipeline's own: {f(m.get('pipeline_unsupported_claim_rate', float('nan')))}"
              f", candidates ungrounded: {f(m.get('candidate_ungrounded_rate', float('nan')))})")
        print(f"  known-bad edges in graph               {f(m['known_bad_edge_rate'])}")
        print(f"  ontology violation rate                {f(m.get('ontology_violation_rate', float('nan')))}")
        print(f"  duplicate node / edge rate             {f(m.get('duplicate_node_rate', float('nan')))} / {f(m.get('duplicate_edge_rate', float('nan')))}")
        if "contradiction_f1" in m:
            print(f"  contradictions P/R                     {f(m['contradiction_precision'])} / {f(m['contradiction_recall'])}   (found {int(m['contradictions'])})")
        if "qa_f1" in m:
            print(f"GRAPH QA   exact {f(m['qa_exact_match'])}   F1 {f(m['qa_f1'])}   ({int(m['qa_questions'])} questions)")
        if len(r["slices"]):
            print("\nSLICES  (counts summed within slice; low_n = <5 gold relations, do not read as a finding)")
            print(_table(r["slices"], {"gold_relations": "gold", "entity_f1": "ent_F1", "relation_precision": "rel_P",
                                       "relation_recall": "rel_R", "relation_f1": "rel_F1", "unsupported_edge_rate": "unsupp"}))
            langs = [s for s in r["slices"]["slice"].to_list() if s.startswith("language:")]
            if len(langs) > 1:
                print("  note: multiple languages present -- read the language:* rows, not 'all'.")
        if len(r["relation_slices"]):
            print("RELATION-TAG RECALL")
            print(_table(r["relation_slices"]))
        if len(r["failure_distribution"]):
            print("\nFAILURE DISTRIBUTION")
            for row in r["failure_distribution"].iter_rows(named=True):
                print(f"  {row['share'] * 100:5.1f}%  {row['failure_type']:20} {row['count']}")
            fr = r.get("failure_rows")
            if isinstance(fr, pl.DataFrame) and len(fr):
                print("  examples (first 8; full list in failure_rows):")
                for x in fr.head(8).iter_rows(named=True):
                    trip = " ".join(t for t in (x["subject"], x["predicate"], x["object"]) if t)
                    print(f"    {x['type']:18} {x['doc_id']}: {trip}  -- {x['detail']}")
        if len(r["sweep"]):
            print("\nCONFIDENCE THRESHOLD SWEEP  (what confidence_filter would have given)")
            print(_table(r["sweep"], {"false_positive_rate": "fpr"}))
            st = r.get("selected_threshold") or {}
            if st.get("threshold") is not None:
                print(f"  -> {st['reason']}: threshold {st['threshold']:.2f} (P {st['precision']:.2f} / R {st['recall']:.2f})")
        cal = r.get("calibration") or {}
        if cal.get("n"):
            cv = r.get("calibration_cv") or {}
            extra = f"   held-out after isotonic: ECE {cv['ece_after']:.3f}" if _finite(cv.get("ece_after")) else ""
            print(f"\nCALIBRATION  n={cal['n']}  ECE {cal['ece']:.3f}  Brier {cal['brier']:.3f}  skill {f(cal['brier_skill'])}{extra}")
            if cal.get("warning"):
                print(f"  ! {cal['warning']}")
        print("\nSYSTEM")
        for k, lab in (("latency_p50_s_per_chunk", "p50 latency s/chunk"), ("latency_p95_s_per_chunk", "p95 latency s/chunk"),
                       ("tokens_per_chunk", "tokens/chunk"), ("completion_tokens_per_s", "completion tok/s"),
                       ("gpu_used_gb_max", "GPU GB (max)"), ("cpu_rss_gb_max", "CPU GB (max)"),
                       ("verifier_calls", "verifier calls"), ("llm_errors", "LLM errors")):
            if k in m:
                print(f"  {lab:24}{m[k]:.3f}")
        rt = r.get("redteam")
        if rt:
            print(f"\nRED-TEAM  {rt['passed']}/{rt['n']} cases pass   forbidden-edge/split failures: {rt['safety_failures']}"
                  f"   attributed->review routing: {f(rt['review_routing_rate'])}")
            for c, v in rt["by_category"].items():
                print(f"  {c:18}{v * 100:5.0f}%  (n={rt['by_category_n'][c]})")
            for c in rt["cases"]:
                if not c["passed"]:
                    why = []
                    if c["missing"]:
                        why.append(f"missing {c['missing']}")
                    if c["forbidden_present"]:
                        why.append(f"FORBIDDEN {c['forbidden_present']}")
                    if c["split_nodes"]:
                        why.append(f"split {c['split_nodes']}")
                    if c["contradiction_flagged"] is False:
                        why.append("contradiction not flagged")
                    print(f"    FAIL {c['id']}: {'; '.join(why)}")
        if r.get("gate") is not None:
            print("\n" + str(r["gate"]))

    @staticmethod
    def save(r: Dict, out_dir: str = "./kg_output/eval") -> Dict[str, str]:
        out = Path(out_dir)
        out.mkdir(parents=True, exist_ok=True)
        paths: Dict[str, str] = {}
        for name in ("slices", "relation_slices", "failure_distribution", "failure_rows", "sweep"):
            df = r.get(name)
            if isinstance(df, pl.DataFrame) and len(df):
                p = out / f"{r['label']}_{name}.csv"
                df.write_csv(str(p))
                paths[name] = str(p)
        if r.get("redteam"):
            p = out / f"{r['label']}_redteam.json"
            p.write_text(json.dumps(_clean(r["redteam"]), indent=2, default=str))
            paths["redteam"] = str(p)
        png = save_reliability_diagram(r.get("calibration") or {}, str(out / f"{r['label']}_reliability.png"))
        if png:
            paths["reliability"] = png
        body = {k: r.get(k) for k in ("label", "eval_version", "gold_version", "gold_synthetic", "manifest", "metrics", "ci",
                                       "selected_threshold", "calibration", "calibration_cv")}
        body["gate"] = ({"passed": r["gate"].passed, "groups": r["gate"].groups, "checks": r["gate"].checks.to_dicts(),
                         "warnings": r["gate"].warnings} if r.get("gate") is not None else None)
        p = out / f"{r['label']}_report.json"
        p.write_text(json.dumps(_clean(body), indent=2, default=str))
        paths["report"] = str(p)
        return paths


ABLATION_LADDER = ("v1_baseline", "v2_evidence_only", "v2_retrieval", "v2_verification", "v2_full")


def compare_presets(gold: GoldSet, base_cfg=None, presets: Sequence[str] = ABLATION_LADDER, base_url: Optional[str] = None,
                    redteam: Optional[Sequence[RedTeamCase]] = None, qa: bool = True, log_wandb: bool = False,
                    verbose: bool = False) -> Tuple[pl.DataFrame, Dict[str, Dict]]:
    """Section 26: 'prove whether a change actually helped'. One independent pipeline per preset (fresh store,
    ledger, memory), same gold set, same red-team set, same vLLM server. Returns (comparison table, reports).
    The table includes the delta of every row against the first preset."""
    base_cfg = base_cfg if base_cfg is not None else kgp.config
    reports: Dict[str, Dict] = {}
    for name in presets:
        cfg = kgp.PipelineConfig.preset(name, base_cfg)
        with _quiet(not verbose):
            pipe = kgp.StreamableKGPipeline(cfg, base_url=base_url, dataset_version=gold.version)
        try:
            rep = EvalHarness(pipe, gold, redteam=redteam, qa=qa, quiet=not verbose).evaluate(label=name, verbose=verbose)
        finally:
            del pipe
            kgp.cleanup_memory()
        print(f"[ABLATION] {name}: relation F1 {rep['metrics']['relation_f1']:.3f}  unsupported {rep['metrics']['unsupported_edge_rate']:.3f}  "
              f"red-team {rep['redteam']['passed'] if rep['redteam'] else '-'}/{rep['redteam']['n'] if rep['redteam'] else '-'}")
        reports[name] = rep
        if log_wandb:
            log_to_wandb(rep, prefix=f"ablation/{name}")
    cols = ["entity_f1", "relation_precision", "relation_recall", "relation_f1", "type_accuracy", "evidence_accuracy",
            "evidence_coverage", "unsupported_edge_rate", "known_bad_edge_rate", "ontology_violation_rate",
            "duplicate_node_rate", "contradiction_recall", "qa_f1", "redteam_pass_rate", "redteam_safety_failures",
            "latency_p95_s_per_chunk", "tokens_per_chunk", "verifier_calls"]
    rows = []
    first = reports[presets[0]]["metrics"]
    for name in presets:
        mm = reports[name]["metrics"]
        row: Dict[str, Any] = {"preset": name}
        for c in cols:
            row[c] = mm.get(c, float("nan"))
        row["relation_f1_vs_first"] = (mm["relation_f1"] - first["relation_f1"]) if _finite(mm.get("relation_f1")) and _finite(first.get("relation_f1")) else float("nan")
        rows.append(row)
    return pl.DataFrame(rows), reports


In [10]:
%%writefile test_vllm_server.py

"""
Standalone isolation test for the vLLM server -- deliberately has ZERO
dependency on kg_pipeline.py (no torch/cudf/transformers import), so it
runs from anywhere with just `requests` installed, before you commit to
loading the full RAPIDS+torch pipeline. Run it with the MAIN env's python
(not the vllm venv's) -- it only needs `requests`, which is virtually
always already present.

Tests two things SEPARATELY, on purpose, instead of one big "did it work"
call -- a failure here tells you WHICH piece broke, not just THAT
something broke:

  Step 1: a plain completion request, no structured output at all --
          confirms the server is actually generating text: model loaded
          correctly, speculative decoding (active at the server level
          regardless of this script) isn't crashing, AWQ quantization is
          producing coherent output.

  Step 2: the SAME prompt, this time with a JSON-schema constraint passed
          via `structured_outputs.json` -- confirms structured output is
          REALLY constraining generation, not silently ignored, and that
          the schema actually compiles server-side. Uses JSON schema, not
          regex: a regex-based version of this test previously reproduced
          a REAL failure on live hardware --
              ValueError: Regex compilation timed out after 5s. The
              pattern may be too complex or contain constructs that cause
              exponential state-space explosion (e.g. nested quantifiers).
          -- from xgrammar's regex compiler choking on a grammar built
          from 12 entity types alternated twice per line, 21 predicates
          once, two bounded character-class repeats, all wrapped in an
          outer bounded repeat for "0 to N triples". JSON schema expresses
          the exact same ontology constraint (enums for the type/predicate
          fields, `maxItems` for the "0 to N" bound) through xgrammar's
          best-trodden, most-optimized compilation path instead, and
          doesn't hit this failure mode.
          `{"structured_outputs": {"json": ...}}` is current as of this
          writing -- if your installed vLLM version predates that field,
          step 2 failing with a 422/"unexpected field" is the tell to
          check `vllm serve --help` / the structured-outputs docs for
          whatever your version currently calls it.

Usage:
    python3 test_vllm_server.py
    python3 test_vllm_server.py --base-url http://localhost:8000
"""

import argparse
import json
import sys
import time

import requests

ALLOWED_ENTITY_TYPES = {
    "PERSON", "ORGANIZATION", "CONCEPT", "PRODUCT", "LOCATION", "EVENT",
    "LAW", "METRIC", "PROCESS", "DOCUMENT", "TECHNOLOGY", "PROVISION",
}
ALLOWED_PREDICATES = {
    "is_a", "instance_of", "part_of", "contains", "includes",
    "develops", "authors", "issues", "regulates", "manages", "governs",
    "performs", "enables", "causes", "requires", "produces", "applies_to",
    "affiliated_with", "located_in", "related_to", "influences", "depends_on",
}


def build_triple_json_schema(max_entity_chars=80, max_triples_per_chunk=20):
    """Same construction as kg_pipeline.py's build_triple_json_schema --
    duplicated here on purpose so this script has no import dependency on
    the main file."""
    entity_type_enum = sorted(ALLOWED_ENTITY_TYPES)
    predicate_enum = sorted(ALLOWED_PREDICATES)
    triple_schema = {
        "type": "object",
        "properties": {
            "subject": {"type": "string", "minLength": 1, "maxLength": max_entity_chars},
            "subject_type": {"type": "string", "enum": entity_type_enum},
            "predicate": {"type": "string", "enum": predicate_enum},
            "target": {"type": "string", "minLength": 1, "maxLength": max_entity_chars},
            "target_type": {"type": "string", "enum": entity_type_enum},
        },
        "required": ["subject", "subject_type", "predicate", "target", "target_type"],
        "additionalProperties": False,
    }
    return {
        "type": "object",
        "properties": {
            "triples": {"type": "array", "maxItems": max_triples_per_chunk, "items": triple_schema},
        },
        "required": ["triples"],
        "additionalProperties": False,
    }


MINI_SYSTEM_PROMPT = """You extract relationships from text into a knowledge graph.

OUTPUT FORMAT: a single JSON object with one key, "triples", holding an
array of objects. Each object has exactly these five fields:
subject, subject_type, predicate, target, target_type.

TYPES: PERSON, ORGANIZATION, CONCEPT, PRODUCT, LOCATION, EVENT, LAW, METRIC, PROCESS, DOCUMENT, TECHNOLOGY, PROVISION

PREDICATES (use ONLY these exact active forms):
is_a, instance_of, part_of, contains, includes, develops, authors, issues,
regulates, manages, governs, performs, enables, causes, requires, produces,
applies_to, affiliated_with, located_in, related_to, influences, depends_on

EXAMPLE:
TEXT: "The RBI regulates export credit through commercial banks."
{"triples": [
  {"subject": "RBI", "subject_type": "ORGANIZATION", "predicate": "regulates", "target": "export credit", "target_type": "CONCEPT"},
  {"subject": "commercial bank", "subject_type": "ORGANIZATION", "predicate": "performs", "target": "export credit", "target_type": "CONCEPT"}
]}

Extract from:"""

TEST_CHUNK = "The RBI regulates export credit through commercial banks."


def check_health(base_url: str) -> bool:
    print(f"[0] Health check: GET {base_url}/health")
    try:
        r = requests.get(f"{base_url}/health", timeout=10)
        ok = r.status_code == 200
        print(f"    -> status {r.status_code}  {'OK' if ok else 'FAILED'}")
        return ok
    except requests.exceptions.ConnectionError as e:
        print(f"    -> CONNECTION REFUSED: {e}")
        print("    Server isn't listening on this port -- confirm launch_vllm_server()")
        print("    actually returned successfully, or check the vLLM log file directly")
        print("    (kg_pipeline.tail_log() from a notebook cell, or `tail -f /content/vllm_server.log`).")
        return False
    except requests.exceptions.Timeout:
        print("    -> TIMED OUT. Server process may exist but isn't responding --")
        print("    check /content/vllm_server.log for what it's actually doing.")
        return False


def check_model(base_url: str):
    print(f"\n[0.5] Model registered: GET {base_url}/v1/models")
    try:
        r = requests.get(f"{base_url}/v1/models", timeout=10)
        r.raise_for_status()
        model_id = r.json()["data"][0]["id"]
        print(f"    -> serving: {model_id}")
        return model_id
    except Exception as e:
        print(f"    -> FAILED: {e}")
        return None


def step1_plain_request(base_url: str, model_id: str):
    print("\n[1] Plain completion request (no structured output)")
    prompt = f"{MINI_SYSTEM_PROMPT}\n\n{TEST_CHUNK}\n\nJSON:"
    payload = {"model": model_id, "prompt": prompt, "max_tokens": 150, "temperature": 0.1, "top_p": 0.9}
    t0 = time.time()
    try:
        r = requests.post(f"{base_url}/v1/completions", json=payload, timeout=120)
    except requests.exceptions.Timeout:
        print("    -> TIMED OUT after 120s. Model isn't generating -- check the")
        print("    server log for a crash during the forward pass, not just startup.")
        return None
    elapsed = time.time() - t0

    if r.status_code != 200:
        print(f"    -> HTTP {r.status_code}: {r.text[:500]}")
        return None

    text = r.json()["choices"][0]["text"]
    print(f"    -> {elapsed:.1f}s, {len(text)} chars returned")
    print(f"    -> raw output:\n{'-' * 60}\n{text.strip()}\n{'-' * 60}")

    if not text.strip():
        print("    -> EMPTY output. Model responded but generated nothing -- worth")
        print("    checking eos_token handling / whether it stopped immediately.")
        return text

    print("    PASS: server is generating text end-to-end (model load, AWQ")
    print("    quantization, and speculative decoding are all functioning)")
    return text


def step2_structured_request(base_url: str, model_id: str):
    print("\n[2] SAME prompt, WITH structured_outputs.json (JSON schema)")
    schema = build_triple_json_schema()
    prompt = f"{MINI_SYSTEM_PROMPT}\n\n{TEST_CHUNK}\n\nJSON:"
    payload = {
        "model": model_id, "prompt": prompt, "max_tokens": 200,
        "temperature": 0.1, "top_p": 0.9,
        "structured_outputs": {"json": schema},
    }
    t0 = time.time()
    try:
        r = requests.post(f"{base_url}/v1/completions", json=payload, timeout=30)
    except requests.exceptions.Timeout:
        print("    -> TIMED OUT after 30s (client-side). If the server itself is stuck")
        print("    compiling the schema, check the log -- xgrammar has its own internal")
        print("    ~5s compile timeout, so a healthy server should fail fast, not hang.")
        return

    elapsed = time.time() - t0

    if r.status_code != 200:
        print(f"    -> HTTP {r.status_code}: {r.text[:500]}")
        print("    If this is a 500 with 'Regex compilation timed out' in the SERVER log")
        print("    (not this client-side text), that's the old regex-based failure mode --")
        print("    make sure you're running the JSON-schema version of the pipeline, not")
        print("    an older copy still sending structured_outputs.regex.")
        print("    If this is a 422/'unexpected field' instead, structured_outputs.json")
        print("    isn't the field name your installed vLLM version expects -- check")
        print("    `vllm serve --help` / the structured-outputs docs for your version.")
        return

    text = r.json()["choices"][0]["text"]
    print(f"    -> {elapsed:.1f}s, {len(text)} chars returned")
    print(f"    -> raw output:\n{'-' * 60}\n{text.strip()}\n{'-' * 60}")

    try:
        parsed = json.loads(text)
        triples = parsed.get("triples", [])
        shape_ok = isinstance(triples, list) and all(
            isinstance(t, dict) and set(t.keys()) == {"subject", "subject_type", "predicate", "target", "target_type"}
            and t["subject_type"] in ALLOWED_ENTITY_TYPES and t["target_type"] in ALLOWED_ENTITY_TYPES
            and t["predicate"] in ALLOWED_PREDICATES
            for t in triples
        )
    except (json.JSONDecodeError, AttributeError):
        shape_ok = False

    if shape_ok:
        print(f"    PASS: valid JSON matching the schema exactly ({len(triples)} triple(s)) --")
        print("    structured output is REALLY constraining generation, not just present")
        print("    in the request and silently ignored.")
    else:
        print("    FAIL (silent-ignore signature): got a 200 response and text, but it's")
        print("    not valid JSON matching the schema. This is the specific failure mode")
        print("    this test exists to catch: the server likely accepted the request but")
        print("    never applied structured_outputs.json, so it generated freely. Compare")
        print("    this output to step 1's -- if they look similar, that confirms the")
        print("    constraint never engaged.")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--base-url", default="http://localhost:8000")
    args = parser.parse_args()

    print("=" * 70)
    print(f"vLLM server isolation test -- {args.base_url}")
    print("=" * 70)

    if not check_health(args.base_url):
        print("\nStopping here -- fix the connection/health issue before testing generation.")
        sys.exit(1)

    model_id = check_model(args.base_url)
    if model_id is None:
        print("\nStopping here -- server is up but /v1/models failed, which is odd.")
        sys.exit(1)

    plain_output = step1_plain_request(args.base_url, model_id)
    if plain_output is None:
        print("\nStopping here -- base generation isn't working, no point testing")
        print("structured output on top of a broken base case.")
        sys.exit(1)

    step2_structured_request(args.base_url, model_id)

    print("\n" + "=" * 70)
    print("Done. Read the PASS/FAIL lines above -- that tells you which layer,")
    print("if any, needs attention before trusting VLLMTripleExtractor on a real document.")
    print("=" * 70)


if __name__ == "__main__":
    main()

Writing test_vllm_server.py


In [ ]:
import kg_pipeline as kgp

# Tweak anything here before launching -- see PipelineConfig in
# kg_pipeline.py for the full list and the reasoning behind each default.
# Examples:
# kgp.config.batch_size = 4
# kgp.config.vllm_gpu_memory_utilization = 0.5
# kgp.config.ann_backend = "faiss"
# kgp.config.vllm_speculative_method = "draft_model"

# v2 harness -- which system runs? (the UI dropdown and the eval cells below build their own config
# from kgp.config via PipelineConfig.preset(name, kgp.config), so anything you set here carries over)
#   presets: v1_baseline | v2_evidence_only | v2_retrieval | v2_verification | v2_full
# Individual switches, e.g.:
# kgp.config.enable_verifier = False
# kgp.config.ontology_violation_action = "reject"      # default "review"
# kgp.config.attributed_action = "review"              # "According to X, ..." claims
# kgp.config.route_easy_conf = 0.85                    # UNCALIBRATED -- see the calibration cell below

kgp.config

## Pre-download the model

Decouples download time from vLLM's own startup time — without this,
"still waiting" during launch could mean either "still downloading" or
"still compiling", which was the exact ambiguity that caused problems
earlier in this project. Downloads into `~/.cache/huggingface`, which the
venv subprocess below reads from directly (no re-download).


In [15]:
import os
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

from huggingface_hub import snapshot_download

target_model = f"{kgp.config.model_name}-AWQ"
print(f"Pre-downloading {target_model} ...")
snapshot_download(target_model)

if kgp.config.vllm_speculative_method == "draft_model":
    print(f"Pre-downloading draft model {kgp.config.draft_model_name} ...")
    snapshot_download(kgp.config.draft_model_name)

print("Done.")

Pre-downloading Qwen/Qwen2.5-3B-Instruct-AWQ ...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Done.


## Launch + smoke-test the vLLM server

First launch uses `enforce_eager=True` — skips `torch.compile`/CUDA graph
capture for a startup that finishes in well under a minute, so this first
run tests *correctness* (model loads, AWQ quantization works, structured
output actually constrains generation) without waiting several minutes for
a compile. Progress prints the tail of the actual server log every ~30s,
not just a bare "still waiting".


In [19]:
proc, base_url = kgp.ensure_vllm_server(kgp.config, enforce_eager=True)
print("base_url:", base_url)

[PROCESS B] Launching: /content/vllm_env/bin/vllm serve Qwen/Qwen2.5-3B-Instruct-AWQ --port 8000 --dtype float16 --gpu-memory-utilization 0.55 --max-model-len 4096 --max-num-seqs 8 --quantization awq --enforce-eager --speculative-config {"method": "ngram", "num_speculative_tokens": 5, "prompt_lookup_max": 4}
...still starting (30s elapsed, timeout at 900s) | log: (APIServer pid=8956) INFO 08-26 06:23:43 [api_utils.py:273] non-default args: {'model_tag': 'Qwen/Qwen2.5-3B-Instruct-AWQ', 'model': 'Qwen/Qwen2.5-3B-Instruct-A
...still starting (60s elapsed, timeout at 900s) | log: Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
[PROCESS B] vLLM server up at http://localhost:8000 (target=Qwen/Qwen2.5-3B-Instruct-AWQ, spec=ngram) after ~90s
base_url: http://localhost:8000


In [ ]:
# If the cell above times out or you interrupt it, run this to see the
# last lines of the actual vLLM startup log (weight loading, compile
# progress, or the real error if it crashed).
kgp.tail_log(n_lines=60)

In [20]:
!python3 test_vllm_server.py --base-url {base_url}

vLLM server isolation test -- http://localhost:8000
[0] Health check: GET http://localhost:8000/health
    -> status 200  OK

[0.5] Model registered: GET http://localhost:8000/v1/models
    -> serving: Qwen/Qwen2.5-3B-Instruct-AWQ

[1] Plain completion request (no structured output)
    -> 7.9s, 386 chars returned
    -> raw output:
------------------------------------------------------------
```json
{
  "triples": [
    {
      "subject": "RBI",
      "subject_type": "ORGANIZATION",
      "predicate": "regulates",
      "target": "export credit",
      "target_type": "CONCEPT"
    },
    {
      "subject": "commercial bank",
      "subject_type": "ORGANIZATION",
      "predicate": "part_of",
      "target": "export credit",
      "target_type": "CONCEPT"
    }
  ]
}
```
------------------------------------------------------------
    PASS: server is generating text end-to-end (model load, AWQ
    quantization, and speculative decoding are all functioning)

[2] SAME prompt, WITH struct

## Optional: switch to throughput mode

`enforce_eager=True` above is for a fast correctness check. For a real
run, relaunch without it — `torch.compile` + CUDA graph capture, which can
take several minutes on a T4 (`tail_log()` will show progress). Skip this
cell if eager-mode throughput is fine for now; you can always come back
and run it later.


In [ ]:
kgp.shutdown_vllm_server()
proc, base_url = kgp.ensure_vllm_server(kgp.config, enforce_eager=False)
print("base_url:", base_url)

## Launch the UI

The **Pipeline preset** dropdown picks `v1_baseline` … `v2_full`. A v2 run also writes the audit trail (claim ledger, review queue, contradictions, entity memory, document outline, run manifest) to `./kg_output/`.


In [22]:
kgp.launch_ui()

## Where to look afterward

- **W&B**: the run link prints when `wandb.init()` fires inside the button
  handler — per-batch triple/confidence/GPU metrics, the vLLM `/metrics`
  series (prefixed `vllm/`), and the final graph summary
  (nodes/edges/communities) all land in the same run.
- **Langfuse**: open your project's **Traces** view — one trace per
  extraction batch, with a nested generation per chunk (prompt, raw
  completion, confidence, domain tag, validation-drop stats).
- **vLLM native metrics**: `curl http://localhost:8000/metrics` any time
  the server is up, or reuse `kg_pipeline._parse_vllm_metrics()` directly.
- **Nsight Systems** (optional, deep path): CUDA-kernel-level GPU/CPU
  timeline, beyond what W&B/vLLM's own metrics give you — see the
  "Nsight Systems profiling" section below. Not part of the always-on
  stack above; run it as a one-off investigation.

- **v2 audit trail** (`./kg_output/`): `claims_ledger.parquet|csv` (every candidate claim, accepted / review / rejected, with the reason), `review_queue.csv`, `contradictions.json`, `entity_memory.json`, `document_outline.json`, `run_manifest.json` (model / prompt / ontology / dataset / parser / embedding versions, threshold, hardware), `run_report.json`. `edges.parquet` / `nodes.parquet` carry provenance columns (evidence, page, chunk, verifier, supporting claims, contradiction flag).
- **v2 evaluation**: the cells below — scorecard, slices, failure taxonomy, threshold sweep, calibration, red-team set, ablation ladder, regression gate, drift.

## v2 harness — run, inspect, evaluate

These cells reuse the vLLM server from "Launch + smoke-test" above (via `base_url`); they do not need the UI. `run_document` stops at the resolved edge table (no cuGraph), so it is cheap enough to loop over a gold set.

In [ ]:
import os, json
import polars as pl
from IPython.display import display
import kg_pipeline as kgp
import kg_eval as ke

try:
    base_url
except NameError:
    base_url = f"http://localhost:{kgp.config.vllm_port}"

PRESET = "v2_full"                        # v1_baseline | v2_evidence_only | v2_retrieval | v2_verification | v2_full
PDF_PATH = "/content/last_uploaded.pdf"   # any PDF; a built-in demo text is used if it is missing

cfg = kgp.PipelineConfig.preset(PRESET, kgp.config)
pipe = kgp.StreamableKGPipeline(cfg, base_url=base_url, dataset_version="adhoc")

if os.path.exists(PDF_PATH):
    out = pipe.run_document(pdf_bytes=open(PDF_PATH, "rb").read(), doc_id="doc1")
else:
    print(f"{PDF_PATH} not found -- using the built-in demo text")
    out = pipe.run_document(text=ke.SEED_GOLD.docs[0].text, doc_id="doc1")

edges = out["edges"]
print(f"preset={PRESET}  edges={0 if edges is None else len(edges)}")
S = out["summary"]
for k in ["claims/candidates", "claims/accepted", "claims/review", "claims/rejected",
          "quality/unsupported_claim_rate", "quality/candidate_ungrounded_rate", "compute/verifier_calls",
          "compute/route_easy_frac", "compute/route_medium_frac", "compute/route_hard_frac", "latency/p95_s_per_chunk"]:
    print(f"  {k:36} {S.get(k, 0):.3f}")
if edges is not None and len(edges):
    cols = [c for c in ["source_str", "predicate_str", "target_str", "confidence", "page", "evidence", "verifier"] if c in edges.columns]
    display(edges.select(cols).head(20))

In [ ]:
# What was proposed, what was kept, and WHY -- the ledger holds every candidate claim.
paths = kgp.export_audit_artifacts(pipe, edges, "./kg_output") if edges is not None else {}
print({k: str(v) for k, v in paths.items()})

led = out["ledger"].to_polars()
if len(led):
    print(led.group_by("verdict").len().sort("verdict"))
    display(led.filter(pl.col("verdict") != "ACCEPT")
               .select(["verdict", "reject_reason", "subject", "predicate", "target", "evidence"]).head(15))

print(f"contradictions: {len(out['contradictions'])}")
for c in out["contradictions"][:5]:
    print(f"  {c.kind}: {c.note}")

In [ ]:
# Ontology-guided graph QA: answers carry the evidence and page of the edge they came from.
if edges is None or not len(edges):
    print("no edges to query")
else:
    qe = kgp.KGQueryEngine(edges)
    anchor = edges["target_str"].mode()[0]             # most frequent target; set your own entity here
    r = qe.query(anchor, role="target")                # "who/what points at <anchor>?"  (role="source" = "what does <anchor> point at?")
    print(f"anchor: {anchor}  ({r.get('anchor_type')})  edges examined: {r['edges_examined']}")
    for a in r["answers"][:5]:
        print(f"  {a['from']} --{a['predicate']}--> {a['to']}  conf={a['confidence']:.2f}  p.{a['page']}  | {a['evidence']}")
    if r["answers"]:
        a = r["answers"][0]
        print("why?", qe.explain(a["from"], a["predicate"], a["to"]))

    src = edges["source_str"].mode()[0]
    for constrained in (True, False):                  # schema-constrained search vs blind BFS: compare the cost
        tp = qe.typed_paths(src, "ORGANIZATION", max_hops=2, constrained=constrained)
        print(f"typed_paths({src!r} -> ORGANIZATION, constrained={constrained}): {len(tp['paths'])} paths, {tp['edges_examined']} edges examined")

### Evaluate against a gold set

`SEED_GOLD` is **synthetic** — three tiny documents written by the module author. It proves the harness runs; the numbers say nothing about the model. For real numbers build `gold_v1` with human annotators (cells below) and set `GOLD = ke.load_gold(...)`. With fewer than ~30 gold relations the scorecard says so and F1 differences are noise.

In [ ]:
GOLD = ke.SEED_GOLD            # -> ke.load_gold("/content/gold_v1.json") once you have human annotations
for line in ke.validate_gold(GOLD):
    print(line)

# Add permanent red-team cases BEFORE building the harness, e.g.:
# ke.REDTEAM_CASES.append(ke.RedTeamCase("my_case", "negation", "X does not require Y.", must_not_have=[("X", "requires", "Y")]))

harness = ke.EvalHarness(pipe, GOLD)        # measures whichever PRESET `pipe` was built with
report = harness.evaluate(label=PRESET)
ke.EvalHarness.scorecard(report)
eval_paths = ke.EvalHarness.save(report, "./kg_output/eval")
print(eval_paths)

In [ ]:
# Every miss / bad edge gets one failure label -- first-match heuristic, so skim before acting on the percentages.
fr = report["failure_rows"]
if len(fr):
    display(fr.select(["doc_id", "type", "mode", "subject", "predicate", "object", "detail"]).head(40))
else:
    print("no failures recorded")

### Ablation ladder (design doc section 26)

One independent pipeline per preset, same gold set, same red-team set, same server — "did retrieval / verification / ontology constraints actually help?". Each preset loads its own embedder, so this takes a few minutes on a T4.

In [ ]:
table, ablation = ke.compare_presets(GOLD, base_cfg=kgp.config, base_url=base_url)
display(table)

### Calibrate the confidence threshold

Per-claim confidence is `exp(scale · mean token logprob of the value tokens)` — **uncalibrated**. The sweep shows what each `confidence_filter` value would have given; `fit_calibrator` maps raw confidence to P(correct) (isotonic); `cv_calibration` reports held-out ECE before and after. Needs a few hundred labelled relations to mean anything.

In [ ]:
labelled = [x for d in report["doc_scores"] for x in d.labelled]     # (confidence, is_gold, key) per relation that passed the gate
scores, correct = [x[0] for x in labelled], [x[1] for x in labelled]
print(f"{len(labelled)} labelled relations" + ("  -- too few for calibration to be trustworthy" if len(labelled) < 100 else ""))
if labelled:
    print({k: v for k, v in ke.calibration_report(scores, correct).items() if k != "bins"})
    print(ke.cv_calibration(scores, correct))
    calibrator = ke.fit_calibrator(scores, correct)
display(report["sweep"])
choice = ke.select_threshold(report["sweep"], min_precision=0.90)
print(choice)
# kgp.config.confidence_filter = choice["threshold"]     # apply it

### Build `gold_v1` (human annotation)

1. Write one blank template per document for each annotator — deliberately **not** pre-filled from model output (anchoring on the model's guesses would inflate agreement).
2. Annotators fill entities (+ type, aliases), relations (+ evidence sentence, `coref`), known-wrong triples (`negatives`), contradictions, graph-QA questions, and the slice tags (`slices`, `language`, `script`, `code_mixed`).
3. Load, `validate_gold`, freeze as `gold_v1.json`, and don't edit it casually — the gate refuses to compare runs across different gold versions.
4. Have two people annotate a subset independently and run `annotator_agreement` (Cohen's kappa on entity types and predicates, F1 agreement on entities and relations) to see human disagreement next to model disagreement.

In [ ]:
DOCS = {d.doc_id: d.text for d in GOLD.docs}    # replace with {doc_id: text} of the documents you want annotated
os.makedirs("/content/gold_templates", exist_ok=True)
for doc_id, text in DOCS.items():
    tpl = ke.blank_annotation_template(doc_id, text, annotator="annotator_a")
    json.dump(tpl, open(f"/content/gold_templates/{doc_id}.json", "w"), indent=2, ensure_ascii=False)
print("templates in /content/gold_templates ->", sorted(os.listdir("/content/gold_templates")))

A_PATH, B_PATH = "/content/gold_annotator_a.json", "/content/gold_annotator_b.json"   # two independently annotated GoldSets
if os.path.exists(A_PATH) and os.path.exists(B_PATH):
    print(ke.annotator_agreement(ke.load_gold(A_PATH), ke.load_gold(B_PATH)))
else:
    print(f"annotator agreement: put two independently annotated gold files at {A_PATH} and {B_PATH}")

### Regression gate (design doc section 27)

PASS/FAIL on three axes — **quality** (F1, evidence accuracy, type accuracy, QA), **safety** (unsupported-edge rate, known-bad edges, ontology violations, red-team failures) and **latency** (p95, tokens/chunk, GPU/CPU memory). First run freezes a baseline; after you change a model / prompt / ontology / chunker / threshold, re-run the evaluate cell and this one. Tolerances live in `ke.DEFAULT_GATE_RULES` — they are starting points, edit them to your deployment objective.

In [ ]:
BASELINE = "./kg_output/eval/baseline.json"
if not os.path.exists(BASELINE):
    ke.save_baseline(report, BASELINE)
    print("baseline saved ->", BASELINE, "\n(change something, re-run the evaluate cell, then run this cell again)")
else:
    gate = ke.RegressionGate().check(report, ke.load_baseline(BASELINE))
    print(gate)

### Drift monitoring (design doc section 28)

No labels needed: compare the distribution of a production-like sample (confidence, document length, entities per document, relation / type mix, unsupported-claim, ontology-violation and review rates) against a frozen reference. PSI / KS / Jensen-Shannon flags are hints, and `low_n` rows (< 30 documents) are very noisy.

In [ ]:
SNAP_REF = "./kg_output/eval/reference_snapshot.json"
if not os.path.exists(SNAP_REF):
    ke.save_snapshot(report["snapshot"], SNAP_REF)
    print("reference snapshot saved ->", SNAP_REF)

SAMPLE_TEXTS = [d.text for d in GOLD.docs]       # replace with a sample of production-like documents
cur = ke.merge_snapshots([ke.snapshot_from_result(pipe.run_document(text=t, doc_id=f"prod{i}"), len(t))
                          for i, t in enumerate(SAMPLE_TEXTS)])
display(ke.drift_report(ke.load_snapshot(SNAP_REF), cur))

In [ ]:
import wandb
if wandb.run is None:
    wandb.init(project=kgp.config.wandb_project, name=f"eval-{PRESET}", reinit=True)
print("logged:", ke.log_to_wandb(report))       # scorecard metrics + tables; run manifest (model/prompt/ontology/dataset/parser/embedding/threshold/hardware) goes to config

In [16]:
!apt-get update -qq
!apt-get install -y --no-install-recommends gnupg2 wget ca-certificates

!wget -O- https://developer.download.nvidia.com/compute/cuda/repos/ubuntu1804/x86_64/7fa2af80.pub \
  | gpg --dearmor \
  | tee /usr/share/keyrings/nvidia-devtools-keyring.gpg > /dev/null

!echo "deb [signed-by=/usr/share/keyrings/nvidia-devtools-keyring.gpg] \
https://developer.download.nvidia.com/devtools/repos/ubuntu$(source /etc/lsb-release; echo "$DISTRIB_RELEASE" | tr -d .)/$(dpkg --print-architecture)/ /" \
  | tee /etc/apt/sources.list.d/nvidia-devtools.list

!apt-get update -qq
!apt-get install -y nsight-systems-cli

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ca-certificates is already the newest version (20260601~22.04.1).
wget is already the newest version (1.21.2-2ubuntu1.5).
gnupg2 is already the newest version (2.2.27-3ubuntu2.5).
0 upgraded, 0 newly installed, 0 to remove and 76 not upgraded.
--2026-08-26 06:18:21--  https://developer.download.nvidia.com/compute/cuda/repos/ubuntu1804/x86_64/7fa2af80.pub
Resolving developer.download.nvidia.com (developer.download.nvidia.com)... 23.32.152.136, 23.32.152.143
Connecting to developer.download.nvidia.com (developer.download.nvidia.com)|23.32.152.136|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1633 (1.6K) [application/octet-stream]
Saving to: ‘STDOUT’

-                   100%[====

In [17]:
!nsys --version

NVIDIA Nsight Systems version 2026.4.1.191-264138605071v0


In [18]:
!nsys status -e

NVIDIA Nsight Systems version 2026.4.1.191-264138605071v0

General Check
- Platform: Linux
- Timestamp counter supported: Yes

CPU Profiling Environment Check
- Root privilege: enabled
- Linux Kernel Paranoid Level: 2 (some features maybe not available)
- Linux Distribution: Ubuntu
- Linux Kernel Version (6.6.122+): OK
- Linux perf_event_open syscall available: OK
- Sampling trigger event available: OK
- Intel(c) Last Branch Record support: Not Available
- CPU Profiling Environment (process-tree): OK
- CPU Profiling Environment (system-wide): OK

See the product documentation at https://docs.nvidia.com/nsight-systems for more information,
including information on how to set the Linux Kernel Paranoid Level.


## Nsight Systems profiling

`nsys` can only trace a process it launches itself, from the start — it
can't attach to a process that's already running. That's fine for the
vLLM server (just launch a second, throwaway instance under `nsys
profile`), but it does mean there's no way to wrap *this already-running
Colab kernel* in `nsys` after the fact. So this profiles the pipeline in
two separate captures, matching the two-process split from the top of
the notebook:

- **A. The vLLM inference server** — a throwaway second instance on port
  8001 (never port 8000, which the UI above is using), launched through
  the same `launch_vllm_server()` the real one uses so the profiled
  process can't drift out of sync with the actual
  model/quantization/speculative-decoding config. `enforce_eager=True`
  here too: CUDA-graph-captured regions replay in a profile as one
  opaque launch, so eager mode is what gives individual, attributable
  kernels in the timeline.
- **B. `kg_pipeline`'s own code** — PDF parsing, chunking, entity
  resolution, and the RAPIDS (cuDF/cuGraph) graph analytics, all of
  which run in *this* kernel process rather than the vLLM one. Profiled
  by launching a small standalone driver script under `nsys` instead —
  it imports `kg_pipeline` and runs one real pass, picking up the NVTX
  ranges already added throughout the module (`pdf_parsing`, `chunking`,
  `vllm_http_call`, `triple_store_aggregate`, `leiden`, etc.), so the
  resulting timeline attributes time to named pipeline stages instead of
  raw, unlabeled CUDA/OS activity.

Both captures use `--sample=none --cpuctxsw=none`, matching the `nsys
status -e` result above — Colab's container restricts `perf_event_open`,
so CPU instruction sampling isn't available here regardless of what's
passed. `--trace=cuda,nvtx,osrt` doesn't need that permission and still
gives full GPU-kernel + NVTX + OS-runtime visibility. Each capture also
gets summarized with `nsys stats` right in the notebook output, so you
get a CLI answer to "where did GPU/CPU time go" with no extra tooling;
downloading the `.nsys-rep` for the full visual timeline in the desktop
Nsight Systems GUI is optional, for drilling into a specific kernel or
gap further.

### A. Profile the vLLM inference server


In [26]:
nsys_vllm_proc, nsys_vllm_base_url = kgp.launch_vllm_server(
    kgp.config,
    port=8001,
    enforce_eager=True,
    nsys_output="/content/vllm_profile",
    nsys_duration_s=300,
    log_path="/content/nsys_vllm.log",
)


[PROCESS B] Launching: nsys profile --force-overwrite=true -o /content/vllm_profile --trace=cuda,nvtx,osrt --sample=none --cpuctxsw=none --duration 300 /content/vllm_env/bin/vllm serve Qwen/Qwen2.5-3B-Instruct-AWQ --port 8001 --dtype float16 --gpu-memory-utilization 0.55 --max-model-len 4096 --max-num-seqs 8 --quantization awq --enforce-eager --speculative-config {"method": "ngram", "num_speculative_tokens": 5, "prompt_lookup_max": 4}
...still starting (30s elapsed, timeout at 900s) | log: 	/content/vllm_profile.nsys-rep
...still starting (60s elapsed, timeout at 900s) | log: 	/content/vllm_profile.nsys-rep
[PROCESS B] vLLM server up at http://localhost:8001 (target=Qwen/Qwen2.5-3B-Instruct-AWQ, spec=ngram) after ~65s


In [27]:
from pathlib import Path

pdf_path = Path("/content/last_uploaded.pdf")
if pdf_path.exists():
    sections = kgp.extract_sections_from_pdf(pdf_path.read_bytes())
    sample_text = " ".join(t for _, t in sections)[:6000]
else:
    sample_text = (
        "Marie Curie was a physicist and chemist who conducted pioneering "
        "research on radioactivity. She was the first woman to win a Nobel "
        "Prize, the first person to win a Nobel Prize twice, and the only "
        "person to win a Nobel Prize in two different scientific fields. "
    ) * 4

# Not real chunking -- this cell only needs to generate genuine inference
# GPU load during the capture window, not a usable knowledge graph.
words = sample_text.split()
chunk_size = max(40, len(words) // 4)
sample_chunks = [" ".join(words[i:i + chunk_size])
                  for i in range(0, min(len(words), chunk_size * 4), chunk_size)] or [sample_text]

nsys_extractor = kgp.VLLMTripleExtractor(kgp.config, base_url=nsys_vllm_base_url)
try:
    for _ in range(3):
        nsys_extractor.extract_batch(sample_chunks)
    print(f"Sent {3 * len(sample_chunks)} chunk-completions to the profiled server on port 8001.")
except Exception as e:
    print(f"Traffic generation failed ({e}) -- the capture will still contain server startup activity.")


Sent 12 chunk-completions to the profiled server on port 8001.


In [28]:
import subprocess
import time
from pathlib import Path

nsys_vllm_proc.terminate()
try:
    nsys_vllm_proc.wait(timeout=30)
except subprocess.TimeoutExpired:
    nsys_vllm_proc.kill()

report_path = Path("/content/vllm_profile.nsys-rep")
for _ in range(15):
    if report_path.exists() and report_path.stat().st_size > 0:
        break
    time.sleep(1)

if report_path.exists():
    print(f"Report ready: {report_path}  ({report_path.stat().st_size / 1024:.0f} KB)\n")
    !nsys stats --report cuda_gpu_kern_sum,cuda_api_sum,osrt_sum,nvtx_sum /content/vllm_profile.nsys-rep
else:
    print(f"No report at {report_path} yet -- check /content/nsys_vllm.log.")


Report ready: /content/vllm_profile.nsys-rep  (8698 KB)

Generating SQLite file /content/vllm_profile.sqlite from /content/vllm_profile.nsys-rep
Processing [/content/vllm_profile.sqlite] with [/opt/nvidia/nsight-systems-cli/2026.4.1/target-linux-x64/reports/cuda_gpu_kern_sum.py]... 
SKIPPED: /content/vllm_profile.sqlite does not contain CUDA kernel data.

Processing [/content/vllm_profile.sqlite] with [/opt/nvidia/nsight-systems-cli/2026.4.1/target-linux-x64/reports/cuda_api_sum.py]... 

 ** CUDA API Summary (cuda_api_sum):

 Time (%)  Total Time (ns)  Num Calls  Avg (ns)  Med (ns)  Min (ns)  Max (ns)  StdDev (ns)           Name          
 --------  ---------------  ---------  --------  --------  --------  --------  -----------  -----------------------
     61.3            7,842          1   7,842.0   7,842.0     7,842     7,842          0.0  cuModuleGetLoadingMode 
     38.7            4,947          2   2,473.5   2,473.5     2,469     2,478          6.4  cudaGetDeviceProperties

Proc

In [29]:
# Optional -- pulls the report to your machine for the full visual
# timeline in the desktop Nsight Systems GUI (File > Open). The CLI
# summary above already answers "where is GPU/CPU time going" without
# needing this; use the GUI to drill into a specific kernel or gap.
# Must be the same Nsight Systems version locally as `nsys --version`
# printed earlier, or the GUI may fail to open it.
from google.colab import files
files.download("/content/vllm_profile.nsys-rep")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### B. Profile `kg_pipeline`'s own code

Requires the *main* server (port 8000, from "Launch + smoke-test the
vLLM server" above) already up, and at least one PDF already processed
through the UI once (so `/content/last_uploaded.pdf` exists). The driver
script below reuses that running server via an explicit `base_url` — it
deliberately never calls `ensure_vllm_server()` / `launch_vllm_server()`
itself, since those clean up (kill) anything already bound to the port
they're given, which would take the UI's server down out from under it.

Builds its own RAPIDS context and embedder in a fresh process, alongside
the ones already resident in this kernel — fine sequentially, but avoid
running this cell while the UI is actively mid-pipeline on a T4 with
16GB of VRAM to share.


In [21]:
%%writefile profile_kg_pipeline.py
"""Standalone driver for Nsight Systems -- run as
`nsys profile ... python3 profile_kg_pipeline.py <pdf_path>`, never
imported directly. A fresh process is required because nsys can only
trace a process it launches itself, not the already-running notebook
kernel.

Talks to the vLLM server already running in the main kernel process
(port from PipelineConfig.vllm_port) via an explicit base_url -- NEVER
via ensure_vllm_server()/launch_vllm_server(), which would kill
whatever's already bound to that port (see the markdown cell above).
"""
import sys
import requests

import kg_pipeline as kgp

if __name__ == "__main__":
    pdf_path = sys.argv[1] if len(sys.argv) > 1 else "/content/last_uploaded.pdf"

    base_url = f"http://localhost:{kgp.config.vllm_port}"
    try:
        healthy = requests.get(f"{base_url}/health", timeout=5).status_code == 200
    except requests.exceptions.ConnectionError:
        healthy = False
    if not healthy:
        raise SystemExit(
            f"No healthy vLLM server at {base_url}. Run the 'Launch + "
            f"smoke-test the vLLM server' cell in the notebook first."
        )

    pdf_bytes = open(pdf_path, "rb").read()

    pipeline = kgp.StreamableKGPipeline(kgp.config, base_url=base_url)
    for _ in pipeline.process_pdf(pdf_bytes):
        pass
    edges_pl, metrics = pipeline.finalize_graph()
    kgp.export_to_parquet(edges_pl, metrics, output_dir="/content/kg_output_nsys_profile")
    print("Done.")


Writing profile_kg_pipeline.py


In [23]:
from pathlib import Path

for p in Path("/content").rglob("*.pdf"):
    print(p)

In [24]:
!find /content -maxdepth 3 -type f \( -iname "*.pdf" -o -iname "*uploaded*" \) -print

In [30]:
from pathlib import Path

assert Path("/content/last_uploaded.pdf").exists(), (
    "No PDF at /content/last_uploaded.pdf yet -- process one via the UI "
    "above at least once first."
)

!nsys profile \
  --force-overwrite=true \
  -o /content/kg_pipeline_profile \
  --trace=cuda,nvtx,osrt \
  --sample=none \
  --cpuctxsw=none \
  python3 profile_kg_pipeline.py /content/last_uploaded.pdf


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:310: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
[OBSERVE] Langfuse tracing disabled: LANGFUSE_PUBLIC_KEY/SECRET_KEY not set.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 776.07it/s]
[OCR/LAYOUT] Extracting sections...
[OCR/LAYOUT] Sections: 2
[CHUNK] Adaptive chunking...
[CHUNK] 16 chunks  |  Est. triples: 80-160
[PROCESS B] Extracting: 100%|██████████| 16/16 [00:50<00:00,  3.13s/chunk]
[PROCESS B] Extraction: 50.1s  |  Entities: 71
[VALIDATE] After confidence filter (0.4): 122 edges
[RESOLVE] Global entity resolution...
[RESOLVE] 3 label aliases merged, type conflicts reconciled by majority vote ->

In [31]:
!nsys stats --report cuda_gpu_kern_sum,cuda_api_sum,osrt_sum,nvtx_sum /content/kg_pipeline_profile.nsys-rep


Generating SQLite file /content/kg_pipeline_profile.sqlite from /content/kg_pipeline_profile.nsys-rep
Processing [/content/kg_pipeline_profile.sqlite] with [/opt/nvidia/nsight-systems-cli/2026.4.1/target-linux-x64/reports/cuda_gpu_kern_sum.py]... 

 ** CUDA GPU Kernel Summary (cuda_gpu_kern_sum):

 Time (%)  Total Time (ns)  Instances  Avg (ns)   Med (ns)   Min (ns)  Max (ns)  StdDev (ns)                                                  Name                                                
 --------  ---------------  ---------  ---------  ---------  --------  --------  -----------  ----------------------------------------------------------------------------------------------------
     40.3       14,996,622        102  147,025.7  112,749.0    40,095   556,562    130,992.6  volta_sgemm_128x64_tn                                                                               
     16.7        6,204,451         24  258,518.8  238,329.5    22,719   536,531    203,881.6  fmha_cutlassF_f32_alig

In [32]:
# Optional, same caveat as the download cell for capture A above. The
# nvtx_sum table above is the most direct answer to "where is CPU/GPU
# time going in kg_pipeline itself" -- one row per named pipeline stage.
from google.colab import files
files.download("/content/kg_pipeline_profile.nsys-rep")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Appendix: quick graph look

Reads back the parquet export and draws the top-60-by-PageRank subgraph.
Purely a sanity-check visualization — for real exploration, open
`nodes.csv`/`edges.csv` in Gephi or `edges.csv` in Cosmograph (both already
shaped for that in `export_to_parquet`).


In [38]:
!uv pip install --system pyvis dspy

Using Python 3.13.15 environment at: /usr
Checked 2 packages in 136ms


In [34]:
import polars as pl
from pyvis.network import Network
from IPython.display import IFrame

nodes = pl.read_parquet("./kg_output/nodes.parquet")
edges = pl.read_parquet("./kg_output/edges.parquet")

top_ids = set(nodes.sort("pagerank", descending=True).head(60)["id"].to_list())
sub_edges = edges.filter(pl.col("source").is_in(top_ids) & pl.col("target").is_in(top_ids))

id_to_label = dict(zip(nodes["id"].to_list(), nodes["label"].to_list()))
id_to_type = dict(zip(nodes["id"].to_list(), nodes["type"].to_list()))

net = Network(height="650px", width="100%", directed=True, cdn_resources="in_line")
for nid in top_ids:
    net.add_node(int(nid), label=id_to_label.get(nid, str(nid)), title=id_to_type.get(nid, ""))
for row in sub_edges.iter_rows(named=True):
    net.add_edge(int(row["source"]), int(row["target"]), title=row["label"], value=float(row["weight"]))

net.write_html("kg_preview.html")
IFrame("kg_preview.html", width="100%", height=670)

## Appendix: active learning (optional)

Not required for the main pipeline above. `VLLMDPODatasetBuilder` uses the
pipeline's own validation gates as the reward signal (no labeled data
needed) to bootstrap DPO preference pairs; `optimize_extraction_prompt`
runs DSPy's MIPROv2 against the same validators to search over prompt
phrasing. Both call the vLLM HTTP endpoint directly.


In [35]:
!uv pip install --system --no-cache-dir dspy trl peft datasets

Using Python 3.13.15 environment at: /usr
Resolved 109 packages in 1.41s
Prepared 15 packages in 5.25s
Uninstalled 3 packages in 66ms
Installed 15 packages in 131ms
 + boto3==1.43.80
 + botocore==1.43.80
 - datasets==4.0.0
 + datasets==5.0.1
 + diskcache==5.6.3
 + dspy==3.3.1
 + fastuuid==0.14.0
 + gepa==0.1.4
 - importlib-metadata==9.0.0
 + importlib-metadata==8.9.0
 + jmespath==1.1.0
 + json-repair==0.63.4
 + litellm==1.98.0
 - pyarrow==18.1.0
 + pyarrow==25.0.1
 + pydantic-settings==2.15.0
 + s3transfer==0.19.2
 + trl==1.10.0


In [36]:
pipeline = kgp._state["pipeline"]
assert pipeline is not None, "Process a PDF via the UI above at least once first."

# TODO: replace with real chunks -- e.g. pull a few sentences straight
# from a document you've already processed.
sample_chunks = [
    "The RBI regulates export credit through commercial banks.",
    "Section 7 of the Companies Act requires annual audits.",
]

builder = kgp.VLLMDPODatasetBuilder(pipeline.extractor)
builder.build_dataset(sample_chunks, out_path="./dpo_dataset.jsonl", n=4)

Building DPO pairs:   0%|          | 0/2 [00:00<?, ?it/s]

DPO dataset: 0/2 chunks yielded a usable pair -> ./dpo_dataset.jsonl


0

In [40]:
!uv pip install --system dspy

Using Python 3.13.15 environment at: /usr
Checked 1 package in 171ms


In [41]:
optimized_module = kgp.optimize_extraction_prompt(pipeline.extractor, sample_chunks, auto="light")

RuntimeError: dspy is required.
  pip install dspy

## Cleanup

In [ ]:
kgp.shutdown_vllm_server()
if kgp.wandb.run is not None:
    kgp.wandb.finish()
print("vLLM server stopped, W&B run closed.")

In [42]:
!uv pip install --system -q jupyterlab-nvidia-nsight

In [43]:
!nsys --version

NVIDIA Nsight Systems version 2026.4.1.191-264138605071v0


In [44]:
!jupyter lab --version

usage: jupyter [-h] [--version] [--config-dir] [--data-dir] [--runtime-dir]
               [--paths] [--json] [--debug]
               [subcommand]

Jupyter: Interactive Computing

positional arguments:
  subcommand     the subcommand to launch

options:
  -h, --help     show this help message and exit
  --version      show the versions of core jupyter packages and exit
  --config-dir   show Jupyter config dir
  --data-dir     show Jupyter data dir
  --runtime-dir  show Jupyter runtime dir
  --paths        show all Jupyter paths. Add --json for machine-readable
                 format.
  --json         output paths as machine-readable json
  --debug        output debug information about paths

Available subcommands: bundlerextension console dejavu events execute kernel
kernelgateway kernelspec migrate nbclassic nbconvert nbextension notebook run
server serverextension troubleshoot trust

Jupyter command `jupyter-lab` not found.


In [50]:
!pip list | grep -Ei "jupyter|lab|notebook"

alabaster                                1.0.0
google-colab                             1.0.0
jupyter_builder                          1.2.2
jupyter_client                           8.9.1
jupyter-console                          6.6.3
jupyter_core                             5.9.1
jupyter-events                           0.12.1
jupyter_kernel_gateway                   2.5.2
jupyter-leaflet                          0.20.0
jupyter-lsp                              2.3.1
jupyter_server                           2.20.0
jupyter_server_terminals                 0.5.4
jupyterlab                               4.6.3
jupyterlab_nvidia_nsight                 1.0.1
jupyterlab_pygments                      0.3.0
jupyterlab_server                        2.28.0
jupyterlab_widgets                       3.0.16
notebook                                 6.5.7
notebook_shim                            0.2.4
spanner-graph-notebook                   1.1.10


In [46]:
!uv pip install --system -U jupyterlab

Using Python 3.13.15 environment at: /usr
Resolved 90 packages in 1.61s
Prepared 29 packages in 1.39s
Uninstalled 16 packages in 192ms
Installed 29 packages in 142ms
 - argon2-cffi-bindings==25.1.0
 + argon2-cffi-bindings==26.1.0
 + asttokens==3.0.2
 + async-lru==2.3.0
 - beautifulsoup4==4.13.5
 + beautifulsoup4==4.15.0
 - charset-normalizer==3.4.9
 + charset-normalizer==3.5.1
 + comm==0.2.3
 - debugpy==1.8.15
 + debugpy==1.8.21
 + executing==2.2.1
 - idna==3.18
 + idna==3.19
 - ipykernel==6.17.1
 + ipykernel==7.3.0
 - ipython==7.34.0
 + ipython==9.16.1
 + ipython-pygments-lexers==1.1.1
 + json5==0.15.0
 + jupyter-builder==1.2.2
 - jupyter-client==7.4.9
 + jupyter-client==8.9.1
 + jupyter-lsp==2.3.1
 + jupyterlab==4.6.3
 + jupyterlab-server==2.28.0
 - nbclient==0.10.4
 + nbclient==0.11.0
 + nest-asyncio2==1.7.2
 - platformdirs==4.11.3
 + platformdirs==4.11.4
 - psutil==5.9.5
 + psutil==7.2.2
 + pure-eval==0.2.3
 - pyzmq==26.2.1
 + pyzmq==27.2.0
 - requests==2.32.4
 + requests==2.34.2
 

In [47]:
!jupyter lab --version

4.6.3


In [48]:
!uv pip install --system -U jupyterlab-nvidia-nsight

Using Python 3.13.15 environment at: /usr
Resolved 64 packages in 413ms
Checked 64 packages in 1ms


In [49]:
!jupyter labextension list

JupyterLab v4.6.3
/usr/local/share/jupyter/labextensions
        ipyevents v2.0.4 enabled OK
        jupyterlab-jupytext v1.4.6 enabled OK (python, jupytext)
        jupyterlab_pygments v0.3.0 enabled OK (python, jupyterlab_pygments)
        ipyparallel-labextension v8.8.0 enabled OK
        bqplot v0.5.48 enabled  X (python, bqplot)
        anywidget v0.9.21 enabled OK
        jupyterlab-plotly v5.24.1 enabled  X
        jupyter-leaflet v0.20.0 enabled OK
        jupyterlab-nvidia-nsight v1.0.1 enabled OK (python, jupyterlab_nvidia_nsight)
        @pyviz/jupyterlab_pyviz v3.0.6 enabled OK
        @jupyter-widgets/jupyterlab-manager v5.0.15 enabled OK (python, jupyterlab_widgets)


   The following extensions may be outdated or specify dependencies that are incompatible with the current version of jupyterlab:
        bqplot
        jupyterlab-plotly
        
   If you are a user, check if an update is available for these packages.
   If you are a developer, re-run with `--verbose` flag